In [ ]:
# ==========================================================
# CELL A1 — Environment Installation
# NumPy 1.x compatibility for escnn / lie_learn
# ==========================================================

!pip -q uninstall -y escnn lie-learn lie-learn-escience py3nj

!pip -q install "numpy<2" --force-reinstall
!pip -q install pandas==2.2.2 scikit-learn==1.5.2 scipy cython autograd joblib pillow tqdm
!pip -q install py3nj
!pip -q install lie-learn-escience==0.0.5 --no-deps
!pip -q install escnn==1.0.11 --no-deps

print("Installation completed.")

In [ ]:
# ==========================================================
# CELL A2 — Restart Runtime
# ==========================================================

import os
os.kill(os.getpid(), 9)

In [ ]:
# ==========================================================
# CELL A3 — Imports and Configuration
# ==========================================================

import os, re, json, time, glob, zipfile, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as torch_nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T

from sklearn.metrics import accuracy_score, f1_score, classification_report

from tqdm.auto import tqdm

from escnn import gspaces
from escnn import nn as enn

warnings.filterwarnings("ignore")

assert np.__version__.startswith("1."), f"NumPy harus 1.x, bukan {np.__version__}"

# -----------------------------
# Project paths
# -----------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")
DATA_RAW_DIR = PROJECT_ROOT / "data_raw"
BASELINE_OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PAPER_TABLES_DIR = BASELINE_OUTPUTS_DIR / "paper_tables"

EQUIV_OUTPUT_DIR = PROJECT_ROOT / "outputs_equiv_vgg16_phase1"
CHECKPOINT_DIR = EQUIV_OUTPUT_DIR / "checkpoints"
PRED_DIR = EQUIV_OUTPUT_DIR / "predictions"

EQUIV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Experiment constants
# -----------------------------
PROTOCOL = "A"
SEED = 42
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS_PHASE1 = 1

CLASS_NAMES_REFERENCE = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== CELL A3 READY ===")
print(f"Device  : {DEVICE}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"Torch   : {torch.__version__}")
print("escnn   : OK")

In [ ]:
# ==========================================================
# CELL B — Model Definition
# Equivariant VGG-style architecture using escnn p4 rotations
# ==========================================================

class EquivariantVGG16(torch_nn.Module):
    """
    VGG-style p4 rotation-equivariant CNN for TEM virus image classification.

    The model uses:
    - p4 rotational symmetry with rotations by 0, 90, 180, and 270 degrees
    - escnn R2Conv layers
    - IIDBatchNorm2d for equivariant feature fields
    - GroupPooling before the final classifier
    """

    def __init__(self, num_classes=14, N=4, dropout=0.30):
        super().__init__()

        self.r2_act = gspaces.rot2dOnR2(N=N)

        self.input_type = enn.FieldType(
            self.r2_act,
            [self.r2_act.trivial_repr] * 3
        )

        def regular_type(num_fields):
            return enn.FieldType(
                self.r2_act,
                [self.r2_act.regular_repr] * num_fields
            )

        def conv_bn_relu(in_type, out_type, kernel_size=3, padding=1):
            return enn.SequentialModule(
                enn.R2Conv(
                    in_type,
                    out_type,
                    kernel_size=kernel_size,
                    padding=padding,
                    bias=False
                ),
                enn.IIDBatchNorm2d(out_type),
                enn.ReLU(out_type, inplace=True)
            )

        t0 = self.input_type
        t1 = regular_type(16)
        t2 = regular_type(32)
        t3 = regular_type(64)
        t4 = regular_type(96)
        t5 = regular_type(128)

        self.block1 = enn.SequentialModule(
            conv_bn_relu(t0, t1),
            conv_bn_relu(t1, t1),
            enn.PointwiseMaxPool(t1, kernel_size=2, stride=2)
        )

        self.block2 = enn.SequentialModule(
            conv_bn_relu(t1, t2),
            conv_bn_relu(t2, t2),
            enn.PointwiseMaxPool(t2, kernel_size=2, stride=2)
        )

        self.block3 = enn.SequentialModule(
            conv_bn_relu(t2, t3),
            conv_bn_relu(t3, t3),
            conv_bn_relu(t3, t3),
            enn.PointwiseMaxPool(t3, kernel_size=2, stride=2)
        )

        self.block4 = enn.SequentialModule(
            conv_bn_relu(t3, t4),
            conv_bn_relu(t4, t4),
            conv_bn_relu(t4, t4),
            enn.PointwiseMaxPool(t4, kernel_size=2, stride=2)
        )

        self.block5 = enn.SequentialModule(
            conv_bn_relu(t4, t5),
            conv_bn_relu(t5, t5),
            conv_bn_relu(t5, t5),
            enn.PointwiseMaxPool(t5, kernel_size=2, stride=2)
        )

        self.group_pool = enn.GroupPooling(t5)

        final_channels = t5.size // N

        self.classifier = torch_nn.Sequential(
            torch_nn.AdaptiveAvgPool2d((1, 1)),
            torch_nn.Flatten(),
            torch_nn.Dropout(dropout),
            torch_nn.Linear(final_channels, 256),
            torch_nn.ReLU(inplace=True),
            torch_nn.Dropout(dropout),
            torch_nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = enn.GeometricTensor(x, self.input_type)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)

        x = self.group_pool(x)
        x = x.tensor

        logits = self.classifier(x)
        return logits


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model_test = EquivariantVGG16(num_classes=NUM_CLASSES).to(DEVICE)

print("=== CELL B READY ===")
print(f"Model                : {model_test.__class__.__name__}")
print(f"Trainable parameters : {count_trainable_parameters(model_test):,}")

del model_test
torch.cuda.empty_cache()

In [ ]:
# ==========================================================
# CELL C-REFRESH — Remount Google Drive
# ==========================================================

from google.colab import drive
from pathlib import Path
import shutil
import os
import time

try:
    drive.flush_and_unmount()
    print("Drive unmounted.")
except Exception as e:
    print("Unmount note:", e)

time.sleep(2)

if Path("/content/drive").exists():
    shutil.rmtree("/content/drive", ignore_errors=True)
    print("Local mount folder cleared.")

drive.mount("/content/drive", force_remount=True)

print("\nDrive remounted.")
print("/content/drive/MyDrive exists:", Path("/content/drive/MyDrive").exists())

In [ ]:
# ==========================================================
# CELL C — Dataset and DataLoader
# Protocol A official split
# ==========================================================

from pathlib import Path
import zipfile
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# ----------------------------------------------------------
# Project paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")
DATA_RAW_DIR = PROJECT_ROOT / "data_raw"

BASELINE_OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PAPER_TABLES_DIR = BASELINE_OUTPUTS_DIR / "paper_tables"

EQUIV_OUTPUT_DIR = PROJECT_ROOT / "outputs_equiv_vgg16_phase1"
CHECKPOINT_DIR = EQUIV_OUTPUT_DIR / "checkpoints"
PRED_DIR = EQUIV_OUTPUT_DIR / "predictions"

EQUIV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Experiment constants
# ----------------------------------------------------------
PROTOCOL = "A"
SEED = 42
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS_PHASE1 = 1

CLASS_NAMES_REFERENCE = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== Project Path Check ===")
print(f"PROJECT_ROOT exists : {PROJECT_ROOT.exists()} -> {PROJECT_ROOT}")
print(f"DATA_RAW_DIR exists : {DATA_RAW_DIR.exists()} -> {DATA_RAW_DIR}")

if not DATA_RAW_DIR.exists():
    raise FileNotFoundError(f"DATA_RAW_DIR was not found: {DATA_RAW_DIR}")

print("\n=== data_raw Contents ===")
for p in sorted(DATA_RAW_DIR.iterdir()):
    size_gb = p.stat().st_size / (1024**3) if p.is_file() else 0
    print(("DIR " if p.is_dir() else "FILE"), f"{size_gb:.2f} GB", p)


# ----------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def standardize_split_name(x):
    x = str(x).lower().strip()

    if x in ["train", "training"]:
        return "train"
    if x in ["val", "valid", "validation"]:
        return "val"
    if x in ["test", "testing"]:
        return "test"

    return x


# ----------------------------------------------------------
# Dataset extraction and location
# ----------------------------------------------------------
def find_dataset_folder():
    """
    Locate context_virus_1nm_256x256.
    If it is not extracted yet, extract the ZIP from data_raw to /content.
    """

    local_extract_root = Path("/content/tem_virus_dataset_extracted")

    candidate_dirs = [
        local_extract_root / "TEM virus dataset/context_virus_1nm_256x256",
        local_extract_root / "context_virus_1nm_256x256",
        DATA_RAW_DIR / "TEM virus dataset/context_virus_1nm_256x256",
        DATA_RAW_DIR / "context_virus_1nm_256x256",
        PROJECT_ROOT / "TEM virus dataset/context_virus_1nm_256x256",
        PROJECT_ROOT / "context_virus_1nm_256x256",
    ]

    for dataset_dir in candidate_dirs:
        if dataset_dir.exists():
            print(f"\nDataset folder found: {dataset_dir}")
            return dataset_dir

    search_roots = [
        local_extract_root,
        DATA_RAW_DIR,
        PROJECT_ROOT,
    ]

    for root in search_roots:
        if root.exists():
            found_dirs = sorted(list(root.rglob("context_virus_1nm_256x256")))
            if len(found_dirs) > 0:
                print(f"\nDataset folder found: {found_dirs[0]}")
                return found_dirs[0]

    zip_candidates = sorted(
        list(DATA_RAW_DIR.rglob("*.zip")) +
        list(DATA_RAW_DIR.rglob("*.ZIP"))
    )

    if len(zip_candidates) == 0:
        raise FileNotFoundError(
            f"No ZIP file found under DATA_RAW_DIR: {DATA_RAW_DIR}"
        )

    zip_path = zip_candidates[0]
    local_extract_root.mkdir(parents=True, exist_ok=True)

    print(f"\nExtracting dataset archive:")
    print(f"Source : {zip_path}")
    print(f"Target : {local_extract_root}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(local_extract_root)

    extracted_candidates = sorted(
        list(local_extract_root.rglob("context_virus_1nm_256x256"))
    )

    if len(extracted_candidates) == 0:
        raise FileNotFoundError(
            "Folder context_virus_1nm_256x256 was not found after extraction."
        )

    print(f"Dataset folder found after extraction: {extracted_candidates[0]}")
    return extracted_candidates[0]


# ----------------------------------------------------------
# Manifest construction
# ----------------------------------------------------------
def build_manifest_from_folder(dataset_dir):
    """
    Build manifest from official split folders:
    train / validation / test.
    """

    split_dir_map = {}

    for child in dataset_dir.iterdir():
        if child.is_dir():
            split_name = standardize_split_name(child.name)
            if split_name in ["train", "val", "test"]:
                split_dir_map[split_name] = child

    required_splits = ["train", "val", "test"]
    missing_splits = [s for s in required_splits if s not in split_dir_map]

    if len(missing_splits) > 0:
        detected = [p.name for p in dataset_dir.iterdir() if p.is_dir()]
        raise FileNotFoundError(
            f"Official split folders are incomplete. Missing: {missing_splits}. "
            f"Detected folders: {detected}"
        )

    rows = []

    for split_name in required_splits:
        split_dir = split_dir_map[split_name]
        class_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()])

        for class_dir in class_dirs:
            image_paths = sorted(list(class_dir.glob("*.tif")))

            for image_path in image_paths:
                rows.append({
                    "filepath": str(image_path),
                    "split": split_name,
                    "class_name": class_dir.name
                })

    manifest_df = pd.DataFrame(rows)

    if len(manifest_df) == 0:
        raise ValueError("Manifest is empty. No .tif images were found.")

    return manifest_df


def load_protocol_a_manifest():
    dataset_dir = find_dataset_folder()
    manifest_df = build_manifest_from_folder(dataset_dir)

    manifest_df["split"] = manifest_df["split"].apply(standardize_split_name)
    manifest_df = manifest_df[manifest_df["split"].isin(["train", "val", "test"])].copy()

    manifest_df["file_exists"] = manifest_df["filepath"].apply(lambda x: Path(x).exists())
    missing_files = int((~manifest_df["file_exists"]).sum())

    if missing_files > 0:
        raise FileNotFoundError(
            f"{missing_files} image files listed in the manifest were not found."
        )

    manifest_df = manifest_df.drop(columns=["file_exists"])

    return manifest_df


# ----------------------------------------------------------
# Dataset class
# ----------------------------------------------------------
class TEMVirusDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


# ----------------------------------------------------------
# DataLoader construction
# ----------------------------------------------------------
def make_dataloaders_phase1(seed=42):
    set_seed(seed)

    manifest_df = load_protocol_a_manifest()

    class_names = sorted(manifest_df["class_name"].unique().tolist())
    class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
    idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

    train_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomRotation(degrees=10),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    eval_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_df = manifest_df[manifest_df["split"] == "train"].copy()
    val_df = manifest_df[manifest_df["split"] == "val"].copy()
    test_df = manifest_df[manifest_df["split"] == "test"].copy()

    train_dataset = TEMVirusDataset(train_df, class_to_idx, transform=train_transform)
    val_dataset = TEMVirusDataset(val_df, class_to_idx, transform=eval_transform)
    test_dataset = TEMVirusDataset(test_df, class_to_idx, transform=eval_transform)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    split_summary = (
        manifest_df
        .groupby(["split", "class_name"])
        .size()
        .reset_index(name="n_images")
    )

    manifest_path = EQUIV_OUTPUT_DIR / "protocol_A_official_manifest_phase1.csv"
    split_summary_path = EQUIV_OUTPUT_DIR / "protocol_A_official_split_summary_phase1.csv"

    manifest_df.to_csv(manifest_path, index=False)
    split_summary.to_csv(split_summary_path, index=False)

    print("\n=== CELL C READY ===")
    print(f"Total images : {len(manifest_df)}")
    print(f"Train        : {len(train_df)}")
    print(f"Validation   : {len(val_df)}")
    print(f"Test         : {len(test_df)}")
    print(f"Classes      : {len(class_names)}")
    print(f"Class names  : {class_names}")

    print("\n=== Saved Files ===")
    print(f"Manifest      : {manifest_path}")
    print(f"Split summary : {split_summary_path}")

    print("\n=== Split Summary Preview ===")
    display(split_summary.head(20))

    return (
        train_loader,
        val_loader,
        test_loader,
        class_names,
        class_to_idx,
        idx_to_class,
        manifest_df,
        split_summary
    )


train_loader, val_loader, test_loader, class_names, class_to_idx, idx_to_class, manifest_df, split_summary = make_dataloaders_phase1(SEED)

In [ ]:
# ==========================================================
# CELL D — Training Function
# ==========================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch=1):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_targets = []

    progress_bar = tqdm(loader, desc=f"Train Epoch {epoch}", leave=True)

    for images, labels, _paths in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(labels.cpu().numpy().tolist())

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    epoch_loss = running_loss / len(loader.dataset)
    epoch_accuracy = accuracy_score(all_targets, all_preds)
    epoch_macro_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )
    epoch_weighted_f1 = f1_score(
        all_targets,
        all_preds,
        average="weighted",
        zero_division=0
    )

    metrics = {
        "loss": float(epoch_loss),
        "accuracy": float(epoch_accuracy),
        "macro_f1": float(epoch_macro_f1),
        "weighted_f1": float(epoch_weighted_f1)
    }

    return metrics

In [ ]:
# ==========================================================
# CELL E — Evaluation Function
# Accuracy, macro F1, weighted F1, and per-class F1
# ==========================================================

@torch.no_grad()
def evaluate_model(model, loader, criterion, device, class_names, split_name="test"):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_targets = []
    all_paths = []

    progress_bar = tqdm(loader, desc=f"Evaluate {split_name}", leave=True)

    for images, labels, paths in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        preds = torch.argmax(logits, dim=1)

        running_loss += loss.item() * images.size(0)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(labels.cpu().numpy().tolist())
        all_paths.extend(list(paths))

    avg_loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )
    weighted_f1 = f1_score(
        all_targets,
        all_preds,
        average="weighted",
        zero_division=0
    )
    per_class_f1_values = f1_score(
        all_targets,
        all_preds,
        average=None,
        zero_division=0
    )

    per_class_f1 = {
        class_names[i]: float(per_class_f1_values[i])
        for i in range(len(class_names))
    }

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    metrics = {
        "loss": float(avg_loss),
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "per_class_f1": per_class_f1,
        "classification_report": report_dict,
        "targets": all_targets,
        "preds": all_preds,
        "paths": all_paths
    }

    return metrics

In [ ]:
# ==========================================================
# CELL F — Run Protocol A, Seed 42, 1 Epoch (With Auto-Skip)
# ==========================================================

import gc
import random
import numpy as np
import torch
import time
import json
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

# ----------------------------------------------------------
# Model, loss, optimizer, scheduler
# ----------------------------------------------------------
model = EquivariantVGG16(
    num_classes=len(class_names),
    N=4,
    dropout=0.30
).to(DEVICE)

criterion = torch_nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=5
)

try:
    scaler = torch.amp.GradScaler(
        device="cuda",
        enabled=(DEVICE.type == "cuda")
    )
except TypeError:
    scaler = torch.cuda.amp.GradScaler(
        enabled=(DEVICE.type == "cuda")
    )

best_val_macro_f1 = -1.0

best_ckpt_path = CHECKPOINT_DIR / f"equiv_vgg16_p4_protocol_{PROTOCOL}_seed_{SEED}_best.pt"
history_path = EQUIV_OUTPUT_DIR / f"history_equiv_vgg16_p4_protocol_{PROTOCOL}_seed_{SEED}_phase1.csv"
pred_path = PRED_DIR / f"predictions_equiv_vgg16_p4_protocol_{PROTOCOL}_seed_{SEED}_epoch1_test.csv"

history = []

print("=== Experiment Configuration ===")
print(f"Protocol             : {PROTOCOL}")
print(f"Seed                 : {SEED}")
print(f"Epochs               : {MAX_EPOCHS_PHASE1}")
print(f"Batch size           : {BATCH_SIZE}")
print(f"Image size           : {IMG_SIZE}x{IMG_SIZE}")
print(f"Device               : {DEVICE}")
print(f"AMP                  : {DEVICE.type == 'cuda'}")
print(f"Optimizer            : AdamW")
print(f"Learning rate        : 1e-4")
print(f"Weight decay         : 1e-4")
print(f"Trainable parameters : {count_trainable_parameters(model):,}")

# ----------------------------------------------------------
# Training loop or Skip if Checkpoint Exists
# ----------------------------------------------------------
if best_ckpt_path.exists():
    print(f"\n[INFO] Checkpoint found at {best_ckpt_path}")
    print("[INFO] Skipping training phase and proceeding to evaluation...")
else:
    print("\n[INFO] No checkpoint found. Starting training phase...")
    start_time = time.time()

    for epoch in range(1, MAX_EPOCHS_PHASE1 + 1):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
            epoch=epoch
        )

        val_metrics = evaluate_model(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=DEVICE,
            class_names=class_names,
            split_name="validation"
        )

        scheduler.step(val_metrics["macro_f1"])

        epoch_result = {
            "protocol": PROTOCOL,
            "seed": SEED,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "learning_rate": optimizer.param_groups[0]["lr"]
        }

        history.append(epoch_result)

        print("\n=== Epoch Result ===")
        print(json.dumps(epoch_result, indent=2))

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]

            torch.save(
                {
                    "protocol": PROTOCOL,
                    "seed": SEED,
                    "epoch": epoch,
                    "model_name": "EquivariantVGG16_p4",
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_macro_f1": best_val_macro_f1,
                    "class_names": class_names,
                    "class_to_idx": class_to_idx,
                    "config": {
                        "num_classes": len(class_names),
                        "img_size": IMG_SIZE,
                        "batch_size": BATCH_SIZE,
                        "optimizer": "AdamW",
                        "learning_rate": 1e-4,
                        "weight_decay": 1e-4,
                        "scheduler": "ReduceLROnPlateau",
                        "scheduler_monitor": "val_macro_f1",
                        "augmentation": "rotation10_hflip_vflip_brightness_contrast_0.1",
                        "normalization": "ImageNet mean/std",
                        "amp": DEVICE.type == "cuda"
                    }
                },
                best_ckpt_path
            )
            print(f"Best checkpoint saved: {best_ckpt_path}")

    elapsed_time = time.time() - start_time

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

    print("\n=== Training Completed ===")
    print(f"Elapsed time : {elapsed_time / 60:.2f} minutes")
    print(f"History CSV  : {history_path}")
    print(f"Checkpoint   : {best_ckpt_path}")

# ----------------------------------------------------------
# Test evaluation using best checkpoint
# ----------------------------------------------------------
print("\n=== Running Test Evaluation ===")
checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])

test_metrics = evaluate_model(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    class_names=class_names,
    split_name="test"
)

prediction_df = pd.DataFrame({
    "filepath": test_metrics["paths"],
    "target": test_metrics["targets"],
    "prediction": test_metrics["preds"],
    "target_class": [class_names[i] for i in test_metrics["targets"]],
    "prediction_class": [class_names[i] for i in test_metrics["preds"]]
})

prediction_df.to_csv(pred_path, index=False)

print("\n=== Test Evaluation Completed ===")
print(f"Test accuracy    : {test_metrics['accuracy']:.4f}")
print(f"Test macro F1    : {test_metrics['macro_f1']:.4f}")
print(f"Test weighted F1 : {test_metrics['weighted_f1']:.4f}")
print(f"Prediction CSV   : {pred_path}")

gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ==========================================================
# CELL G — Final Experiment Result
# ==========================================================

final_result = {
    "protocol": PROTOCOL,
    "seed": SEED,
    "epoch": MAX_EPOCHS_PHASE1,
    "test_accuracy": round(float(test_metrics["accuracy"]), 4),
    "macro_f1": round(float(test_metrics["macro_f1"]), 4),
    "weighted_f1": round(float(test_metrics["weighted_f1"]), 4),
    "per_class_f1": {
        class_name: round(float(score), 4)
        for class_name, score in test_metrics["per_class_f1"].items()
    }
}

summary_path = EQUIV_OUTPUT_DIR / f"summary_equiv_vgg16_p4_protocol_{PROTOCOL}_seed_{SEED}_epoch1.json"

with open(summary_path, "w") as f:
    json.dump(final_result, f, indent=2)

print("=== Final Result ===")
print(json.dumps(final_result, indent=2))

print("\n=== Saved Artifacts ===")
print(f"Summary JSON   : {summary_path}")
print(f"History CSV    : {history_path}")
print(f"Prediction CSV : {pred_path}")
print(f"Checkpoint     : {best_ckpt_path}")

In [ ]:
# ==========================================================
# CELL H — CP2a Configuration and Baseline Split Reconstruction
# Protocol A official split reconstructed from DenseNet201 prediction files
# ==========================================================

import os
import json
import time
import math
import random
import zipfile
import warnings
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as torch_nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Main configuration
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")
DATA_RAW_DIR = PROJECT_ROOT / "data_raw"
BASELINE_OUTPUTS_DIR = PROJECT_ROOT / "outputs"

PHASE2_OUTPUT_DIR = BASELINE_OUTPUTS_DIR / "equivariant_phase2"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_equivariant"

PHASE2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL = "A_official"
SEED = 42
MAX_EPOCHS = 120
BATCH_SIZE = 24
NUM_WORKERS = 4
IMG_SIZE = 256

BASE_LR = 1e-4
WARMUP_START_LR = 1e-6
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

EARLY_STOP_PATIENCE = 25
SMOKE_TEST_EPOCH = 5
SMOKE_TEST_MIN_VAL_ACC = 0.35

RESUME_TRAINING = True

CLASS_NAMES_REFERENCE = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------
# Determinism
# ----------------------------------------------------------
def set_global_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(SEED)


# ----------------------------------------------------------
# Locate dataset folder
# ----------------------------------------------------------
def find_dataset_folder():
    local_extract_root = Path("/content/tem_virus_dataset_extracted")

    candidate_dirs = [
        local_extract_root / "TEM virus dataset/context_virus_1nm_256x256",
        local_extract_root / "context_virus_1nm_256x256",
        DATA_RAW_DIR / "TEM virus dataset/context_virus_1nm_256x256",
        DATA_RAW_DIR / "context_virus_1nm_256x256",
        PROJECT_ROOT / "TEM virus dataset/context_virus_1nm_256x256",
        PROJECT_ROOT / "context_virus_1nm_256x256",
    ]

    for dataset_dir in candidate_dirs:
        if dataset_dir.exists():
            return dataset_dir

    for root in [local_extract_root, DATA_RAW_DIR, PROJECT_ROOT]:
        if root.exists():
            found_dirs = sorted(list(root.rglob("context_virus_1nm_256x256")))
            if len(found_dirs) > 0:
                return found_dirs[0]

    zip_candidates = sorted(
        list(DATA_RAW_DIR.rglob("*.zip")) +
        list(DATA_RAW_DIR.rglob("*.ZIP"))
    )

    if len(zip_candidates) == 0:
        raise FileNotFoundError(f"No dataset ZIP found under: {DATA_RAW_DIR}")

    zip_path = zip_candidates[0]
    local_extract_root.mkdir(parents=True, exist_ok=True)

    print("Extracting dataset archive")
    print(f"Source: {zip_path}")
    print(f"Target: {local_extract_root}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(local_extract_root)

    extracted_candidates = sorted(
        list(local_extract_root.rglob("context_virus_1nm_256x256"))
    )

    if len(extracted_candidates) == 0:
        raise FileNotFoundError("context_virus_1nm_256x256 was not found after extraction.")

    return extracted_candidates[0]


def standardize_split_name(x):
    x = str(x).lower().strip()

    if x in ["train", "training"]:
        return "train"
    if x in ["val", "valid", "validation"]:
        return "val"
    if x in ["test", "testing"]:
        return "test"

    return x


def build_all_images_manifest(dataset_dir):
    rows = []

    for split_dir in sorted([p for p in dataset_dir.iterdir() if p.is_dir()]):
        split_name = standardize_split_name(split_dir.name)

        if split_name not in ["train", "val", "test"]:
            continue

        for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            class_name = class_dir.name

            for image_path in sorted(class_dir.glob("*.tif")):
                rows.append({
                    "filepath": str(image_path),
                    "original_split": split_name,
                    "class_name": class_name,
                    "filename": image_path.name,
                    "rel_key": f"{split_name}/{class_name}/{image_path.name}",
                    "class_file_key": f"{class_name}/{image_path.name}",
                    "basename": image_path.name
                })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise ValueError("No .tif images were found in the dataset folder.")

    return df


# ----------------------------------------------------------
# Baseline prediction files
# ----------------------------------------------------------
def find_baseline_prediction_file(expected_name):
    exact_path = BASELINE_OUTPUTS_DIR / expected_name

    if exact_path.exists():
        return exact_path

    candidates = sorted(list(BASELINE_OUTPUTS_DIR.rglob(expected_name)))

    if len(candidates) > 0:
        return candidates[0]

    print("Available CSV files containing DenseNet201 or predictions:")
    for p in sorted(BASELINE_OUTPUTS_DIR.rglob("*.csv"))[:100]:
        name = p.name.lower()
        if "densenet201" in name or "prediction" in name:
            print(p)

    raise FileNotFoundError(f"Baseline prediction file not found: {expected_name}")


BASELINE_VAL_PRED_PATH = find_baseline_prediction_file(
    "DenseNet201_A_official_seed42_validation_predictions.csv"
)
BASELINE_TEST_PRED_PATH = find_baseline_prediction_file(
    "DenseNet201_A_official_seed42_test_predictions.csv"
)

print("=== Baseline Prediction Files ===")
print(f"Validation predictions: {BASELINE_VAL_PRED_PATH}")
print(f"Test predictions      : {BASELINE_TEST_PRED_PATH}")


# ----------------------------------------------------------
# Robust key extraction from baseline prediction files
# ----------------------------------------------------------
def normalize_path_string(x):
    return str(x).replace("\\", "/").strip()


def standardize_rel_key_from_path(path_value):
    s = normalize_path_string(path_value)

    marker = "context_virus_1nm_256x256/"
    if marker in s:
        tail = s.split(marker, 1)[1]
    else:
        parts = [p for p in PurePosixPath(s).parts if p not in ["/", ""]]
        tail = "/".join(parts[-3:]) if len(parts) >= 3 else s

    parts = tail.split("/")
    if len(parts) >= 3:
        parts[0] = standardize_split_name(parts[0])
        return "/".join(parts[-3:])

    return None


def class_file_key_from_path(path_value):
    s = normalize_path_string(path_value)
    parts = [p for p in PurePosixPath(s).parts if p not in ["/", ""]]

    if len(parts) >= 2:
        parent = parts[-2]
        filename = parts[-1]

        if standardize_split_name(parent) not in ["train", "val", "test"]:
            return f"{parent}/{filename}"

    return None


def basename_from_path(path_value):
    s = normalize_path_string(path_value)
    return PurePosixPath(s).name


def detect_path_column(df):
    preferred = [
        "image_path", "filepath", "file_path", "path", "filename",
        "image", "img_path", "fname"
    ]

    lower_map = {c.lower(): c for c in df.columns}

    for c in preferred:
        if c in lower_map:
            return lower_map[c]

    for c in df.columns:
        sample = df[c].astype(str).head(20).str.lower()
        if sample.str.contains(".tif", regex=False).any():
            return c

    raise ValueError(
        "Could not detect image path or filename column in baseline prediction file. "
        f"Available columns: {list(df.columns)}"
    )


def detect_label_column(df):
    preferred = [
        "true_label", "target_class", "label_name", "class_name",
        "true_class", "target", "label"
    ]

    lower_map = {c.lower(): c for c in df.columns}

    for c in preferred:
        if c in lower_map:
            return lower_map[c]

    return None


def normalize_label_value(x, class_names):
    if pd.isna(x):
        return None

    value = str(x).strip()

    if value in class_names:
        return value

    try:
        idx = int(float(value))
        if 0 <= idx < len(class_names):
            return class_names[idx]
    except:
        pass

    value_lower = value.lower()
    for c in class_names:
        if value_lower == c.lower():
            return c

    return None


def resolve_prediction_filepaths(pred_df, all_df, split_label, class_names):
    path_col = detect_path_column(pred_df)
    label_col = detect_label_column(pred_df)

    temp = pred_df.copy()
    temp["_path_value"] = temp[path_col].astype(str)

    temp["rel_key_candidate"] = temp["_path_value"].apply(standardize_rel_key_from_path)
    temp["class_file_key_from_path"] = temp["_path_value"].apply(class_file_key_from_path)
    temp["basename_candidate"] = temp["_path_value"].apply(basename_from_path)

    if label_col is not None:
        temp["_label_name"] = temp[label_col].apply(lambda x: normalize_label_value(x, class_names))
        temp["class_file_key_from_label"] = temp.apply(
            lambda r: f"{r['_label_name']}/{r['basename_candidate']}"
            if r["_label_name"] is not None else None,
            axis=1
        )
    else:
        temp["class_file_key_from_label"] = None

    strategies = [
        ("rel_key_candidate", "rel_key"),
        ("class_file_key_from_label", "class_file_key"),
        ("class_file_key_from_path", "class_file_key"),
        ("basename_candidate", "basename")
    ]

    best = None

    for pred_key_col, all_key_col in strategies:
        pred_keys = temp[pred_key_col].dropna().astype(str).tolist()
        pred_key_set = set(pred_keys)

        if len(pred_key_set) == 0:
            continue

        all_key_counts = all_df[all_key_col].value_counts()
        duplicate_keys = set(all_key_counts[all_key_counts > 1].index)

        if all_key_col == "basename" and len(duplicate_keys) > 0:
            continue

        all_key_set = set(all_df[all_key_col].astype(str))
        matched_keys = pred_key_set.intersection(all_key_set)

        coverage = len(matched_keys) / max(1, len(pred_key_set))

        candidate = {
            "pred_key_col": pred_key_col,
            "all_key_col": all_key_col,
            "coverage": coverage,
            "matched_keys": matched_keys
        }

        if best is None or coverage > best["coverage"]:
            best = candidate

    if best is None or best["coverage"] < 0.95:
        print(f"\nBaseline prediction columns for {split_label}:")
        print(list(pred_df.columns))
        print("\nPreview:")
        display(pred_df.head())

        raise ValueError(
            f"Could not reliably match {split_label} prediction file to dataset images. "
            f"Best coverage: {0 if best is None else best['coverage']:.4f}"
        )

    matched_df = all_df[all_df[best["all_key_col"]].isin(best["matched_keys"])].copy()
    matched_filepaths = set(matched_df["filepath"].tolist())

    print(f"\n=== Split Match: {split_label} ===")
    print(f"Path column used     : {path_col}")
    print(f"Label column used    : {label_col}")
    print(f"Matching strategy    : {best['pred_key_col']} -> {best['all_key_col']}")
    print(f"Coverage             : {best['coverage']:.4f}")
    print(f"Expected rows        : {len(pred_df)}")
    print(f"Matched image files  : {len(matched_filepaths)}")

    return matched_filepaths


# ----------------------------------------------------------
# Reconstruct Protocol A split
# ----------------------------------------------------------
dataset_dir = find_dataset_folder()
all_images_df = build_all_images_manifest(dataset_dir)

class_names = sorted(all_images_df["class_name"].unique().tolist())

if class_names != CLASS_NAMES_REFERENCE:
    print("Detected class names:")
    print(class_names)
    print("Reference class names:")
    print(CLASS_NAMES_REFERENCE)
    raise ValueError("Detected class names do not match the expected 14-class reference.")

val_pred_df = pd.read_csv(BASELINE_VAL_PRED_PATH)
test_pred_df = pd.read_csv(BASELINE_TEST_PRED_PATH)

val_filepaths = resolve_prediction_filepaths(
    val_pred_df,
    all_images_df,
    split_label="validation",
    class_names=class_names
)

test_filepaths = resolve_prediction_filepaths(
    test_pred_df,
    all_images_df,
    split_label="test",
    class_names=class_names
)

overlap = val_filepaths.intersection(test_filepaths)
if len(overlap) > 0:
    raise ValueError(f"Validation and test file overlap detected: {len(overlap)} files")

protocol_df = all_images_df.copy()
protocol_df["split"] = "train"
protocol_df.loc[protocol_df["filepath"].isin(val_filepaths), "split"] = "val"
protocol_df.loc[protocol_df["filepath"].isin(test_filepaths), "split"] = "test"

split_counts = protocol_df["split"].value_counts().to_dict()

expected_counts = {"train": 5740, "val": 2249, "test": 1900}
print("\n=== Reconstructed Protocol A Split ===")
print(split_counts)

for split_name, expected_n in expected_counts.items():
    actual_n = int(split_counts.get(split_name, 0))
    if actual_n != expected_n:
        raise ValueError(
            f"Unexpected {split_name} count: {actual_n}. Expected: {expected_n}. "
            "Split reconstruction must be identical to the baseline."
        )

split_class_summary = (
    protocol_df
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="n_images")
)

manifest_path = PHASE2_OUTPUT_DIR / "EquivVGG16p4_A_official_seed42_reconstructed_manifest.csv"
split_summary_path = PHASE2_OUTPUT_DIR / "EquivVGG16p4_A_official_seed42_split_summary.csv"

protocol_df.to_csv(manifest_path, index=False)
split_class_summary.to_csv(split_summary_path, index=False)

print("\n=== CELL H READY ===")
print(f"Dataset folder : {dataset_dir}")
print(f"Total images   : {len(protocol_df)}")
print(f"Train          : {split_counts['train']}")
print(f"Validation     : {split_counts['val']}")
print(f"Test           : {split_counts['test']}")
print(f"Manifest saved : {manifest_path}")
print(f"Summary saved  : {split_summary_path}")

display(split_class_summary.head(30))

In [ ]:
# ==========================================================
# CELL I — Compute Dataset Mean/Std from Training Set
# Statistics are computed only from Protocol A training images
# ==========================================================

class GrayscaleStatsDataset(Dataset):
    def __init__(self, df, resize_size=256):
        self.df = df.reset_index(drop=True).copy()
        self.transform = T.Compose([
            T.Resize((resize_size, resize_size)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.df.iloc[idx]["filepath"]
        image = Image.open(image_path).convert("L")
        image = self.transform(image)
        return image


def compute_train_mean_std(train_df, batch_size=64, num_workers=2):
    stats_dataset = GrayscaleStatsDataset(train_df, resize_size=IMG_SIZE)

    stats_loader = DataLoader(
        stats_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    pixel_sum = torch.zeros(1, dtype=torch.float64)
    pixel_sq_sum = torch.zeros(1, dtype=torch.float64)
    pixel_count = 0

    for images in tqdm(stats_loader, desc="Computing train mean/std", leave=True):
        images = images.double()

        pixel_sum += images.sum(dim=[0, 2, 3])
        pixel_sq_sum += (images ** 2).sum(dim=[0, 2, 3])
        pixel_count += images.size(0) * images.size(2) * images.size(3)

    mean = pixel_sum / pixel_count
    variance = (pixel_sq_sum / pixel_count) - (mean ** 2)
    std = torch.sqrt(torch.clamp(variance, min=1e-12))

    return float(mean.item()), float(std.item())


stats_cache_path = PHASE2_OUTPUT_DIR / "EquivVGG16p4_A_official_seed42_train_mean_std.json"

train_df_for_stats = protocol_df[protocol_df["split"] == "train"].copy()

if stats_cache_path.exists():
    with open(stats_cache_path, "r") as f:
        stats_cache = json.load(f)

    train_mean = float(stats_cache["train_mean"])
    train_std = float(stats_cache["train_std"])

    print("Loaded cached training statistics.")
else:
    train_mean, train_std = compute_train_mean_std(
        train_df_for_stats,
        batch_size=64,
        num_workers=2
    )

    stats_cache = {
        "protocol": PROTOCOL,
        "seed": SEED,
        "image_size": IMG_SIZE,
        "train_images": int(len(train_df_for_stats)),
        "train_mean": train_mean,
        "train_std": train_std
    }

    with open(stats_cache_path, "w") as f:
        json.dump(stats_cache, f, indent=2)

DATASET_MEAN = [train_mean] * 3
DATASET_STD = [train_std] * 3

print("=== CELL I READY ===")
print(f"Train mean : {train_mean:.6f}")
print(f"Train std  : {train_std:.6f}")
print(f"Cache path : {stats_cache_path}")

In [ ]:
# ==========================================================
# CELL J — CP2a Transforms and DataLoaders
# Input resolution: 256 x 256
# ==========================================================

class TEMVirusCP2Dataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


class_names = sorted(protocol_df["class_name"].unique().tolist())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

train_transform = T.Compose([
    T.Resize((280, 280)),
    T.RandomResizedCrop(256, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=DATASET_MEAN, std=DATASET_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.08))
])

eval_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=DATASET_MEAN, std=DATASET_STD)
])

train_df = protocol_df[protocol_df["split"] == "train"].copy()
val_df = protocol_df[protocol_df["split"] == "val"].copy()
test_df = protocol_df[protocol_df["split"] == "test"].copy()

train_dataset = TEMVirusCP2Dataset(train_df, class_to_idx, transform=train_transform)
val_dataset = TEMVirusCP2Dataset(val_df, class_to_idx, transform=eval_transform)
test_dataset = TEMVirusCP2Dataset(test_df, class_to_idx, transform=eval_transform)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=train_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("=== CELL J READY ===")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Num workers    : {NUM_WORKERS}")
print(f"Image size     : {IMG_SIZE}")
print(f"Train images   : {len(train_dataset)}")
print(f"Val images     : {len(val_dataset)}")
print(f"Test images    : {len(test_dataset)}")
print(f"Classes        : {len(class_names)}")
print(f"Mean           : {DATASET_MEAN}")
print(f"Std            : {DATASET_STD}")

In [ ]:
# ==========================================================
# CELL K — Metrics, Evaluation, and Warmup-Cosine Scheduler
# ==========================================================

def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


class WarmupCosineScheduler:
    def __init__(
        self,
        optimizer,
        total_epochs=120,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6
    ):
        self.optimizer = optimizer
        self.total_epochs = total_epochs
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.last_epoch = -1

    def get_lr(self, epoch):
        if epoch < self.warmup_epochs:
            if self.warmup_epochs == 1:
                return self.base_lr

            progress = epoch / (self.warmup_epochs - 1)
            return self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

        cosine_epochs = max(1, self.total_epochs - self.warmup_epochs)
        progress = (epoch - self.warmup_epochs) / max(1, cosine_epochs - 1)
        progress = min(max(progress, 0.0), 1.0)

        cosine_value = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.min_lr + cosine_value * (self.base_lr - self.min_lr)

    def step(self, epoch):
        lr = self.get_lr(epoch)

        for group in self.optimizer.param_groups:
            group["lr"] = lr

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "total_epochs": self.total_epochs,
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "last_epoch": self.last_epoch
        }

    def load_state_dict(self, state_dict):
        self.total_epochs = state_dict["total_epochs"]
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.last_epoch = state_dict["last_epoch"]


def compute_basic_metrics(y_true, y_pred):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }

    return metrics


def train_one_epoch_cp2(model, loader, criterion, optimizer, scaler, device, epoch_display):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_preds = []

    progress_bar = tqdm(loader, desc=f"Train epoch {epoch_display}", leave=True)

    for images, labels, _paths in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = running_loss / len(loader.dataset)
    metrics = compute_basic_metrics(all_targets, all_preds)
    metrics["loss"] = float(avg_loss)

    return metrics


@torch.no_grad()
def evaluate_cp2(model, loader, criterion, device, class_names, split_name="validation"):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_paths = []
    all_probs = []

    progress_bar = tqdm(loader, desc=f"Evaluate {split_name}", leave=True)

    for images, labels, paths in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        probs = torch.softmax(logits.float(), dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_paths.extend(list(paths))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()

    avg_loss = running_loss / len(loader.dataset)
    base_metrics = compute_basic_metrics(all_targets, all_preds)
    base_metrics["loss"] = float(avg_loss)

    per_class_f1_values = f1_score(
        all_targets,
        all_preds,
        average=None,
        zero_division=0
    )

    per_class_f1 = {
        class_names[i]: float(per_class_f1_values[i])
        for i in range(len(class_names))
    }

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_txt = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        zero_division=0
    )

    cm = confusion_matrix(
        all_targets,
        all_preds,
        labels=list(range(len(class_names)))
    )

    pred_df = pd.DataFrame({
        "filename": [Path(p).name for p in all_paths],
        "image_path": all_paths,
        "true_label": [class_names[i] for i in all_targets],
        "pred_label": [class_names[i] for i in all_preds],
        "confidence": all_probs.max(axis=1)
    })

    for i, class_name in enumerate(class_names):
        safe_class_name = class_name.replace(" ", "_")
        pred_df[f"prob_{safe_class_name}"] = all_probs[:, i]

    outputs = {
        "metrics": base_metrics,
        "per_class_f1": per_class_f1,
        "classification_report": report_dict,
        "classification_report_txt": report_txt,
        "confusion_matrix": cm,
        "prediction_df": pred_df,
        "targets": all_targets,
        "preds": all_preds,
        "paths": all_paths
    }

    return outputs

In [ ]:
# ==========================================================
# CELL L — Checkpoint Resume and Early Stopping Utilities
# ==========================================================

LAST_CKPT_PATH = CHECKPOINT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_last.pt"
BEST_CKPT_PATH = CHECKPOINT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_best.pt"

HISTORY_CSV_PATH = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_training_history.csv"
HISTORY_JSON_PATH = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_training_history.json"
TRAINING_SUMMARY_JSON_PATH = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_training_summary.json"


def save_history(history):
    history_df = pd.DataFrame(history)
    history_df.to_csv(HISTORY_CSV_PATH, index=False)

    with open(HISTORY_JSON_PATH, "w") as f:
        json.dump(history, f, indent=2)

    return history_df


def save_last_checkpoint(
    model,
    optimizer,
    scheduler,
    epoch,
    best_metric,
    best_epoch,
    early_stop_counter,
    history,
    stopped_by_smoke_test=False,
    early_stopped=False
):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "best_metric": best_metric,
            "best_epoch": best_epoch,
            "early_stop_counter": early_stop_counter,
            "history": history,
            "seed": SEED,
            "protocol": PROTOCOL,
            "stopped_by_smoke_test": stopped_by_smoke_test,
            "early_stopped": early_stopped,
            "config": {
                "model": "EquivariantVGG16_p4",
                "epochs": MAX_EPOCHS,
                "batch_size": BATCH_SIZE,
                "image_size": IMG_SIZE,
                "optimizer": "AdamW",
                "base_lr": BASE_LR,
                "warmup_start_lr": WARMUP_START_LR,
                "min_lr": MIN_LR,
                "weight_decay": WEIGHT_DECAY,
                "label_smoothing": LABEL_SMOOTHING,
                "early_stop_patience": EARLY_STOP_PATIENCE,
                "monitor": "val_macro_f1"
            }
        },
        LAST_CKPT_PATH
    )


def save_best_model(model, optimizer, scheduler, epoch, val_macro_f1, val_metrics):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "val_macro_f1": val_macro_f1,
            "val_metrics": val_metrics,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "seed": SEED,
            "protocol": PROTOCOL,
            "config": {
                "model": "EquivariantVGG16_p4",
                "epochs": MAX_EPOCHS,
                "batch_size": BATCH_SIZE,
                "image_size": IMG_SIZE,
                "optimizer": "AdamW",
                "base_lr": BASE_LR,
                "warmup_start_lr": WARMUP_START_LR,
                "min_lr": MIN_LR,
                "weight_decay": WEIGHT_DECAY,
                "label_smoothing": LABEL_SMOOTHING
            }
        },
        BEST_CKPT_PATH
    )


def load_last_checkpoint(model, optimizer, scheduler):
    if RESUME_TRAINING and LAST_CKPT_PATH.exists():
        ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_metric = float(ckpt.get("best_metric", 0.0))
        best_epoch = ckpt.get("best_epoch", None)
        early_stop_counter = int(ckpt.get("early_stop_counter", 0))
        history = ckpt.get("history", [])

        print("=== Resume Checkpoint Found ===")
        print(f"Resuming from epoch : {start_epoch + 1}")
        print(f"Best val macro F1   : {best_metric:.4f}")
        print(f"Best epoch          : {None if best_epoch is None else best_epoch + 1}")
        print(f"Early stop counter  : {early_stop_counter}")

        return start_epoch, best_metric, best_epoch, early_stop_counter, history

    print("=== Fresh Training Run ===")
    print("No last checkpoint was loaded.")

    return 0, -1.0, None, 0, []


print(f"Last checkpoint : {LAST_CKPT_PATH}")
print(f"Best checkpoint : {BEST_CKPT_PATH}")
print(f"History CSV     : {HISTORY_CSV_PATH}")

In [ ]:
# ==========================================================
# CELL M — CP2a Full Training Loop
# 120 epochs, seed 42, Protocol A official
# ==========================================================

import gc

set_global_seed(SEED)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model = EquivariantVGG16(
    num_classes=len(class_names),
    N=4,
    dropout=0.30
).to(DEVICE)

criterion = torch_nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler = WarmupCosineScheduler(
    optimizer=optimizer,
    total_epochs=MAX_EPOCHS,
    warmup_epochs=5,
    warmup_start_lr=WARMUP_START_LR,
    base_lr=BASE_LR,
    min_lr=MIN_LR
)

try:
    scaler = torch.amp.GradScaler(
        device="cuda",
        enabled=(DEVICE.type == "cuda")
    )
except TypeError:
    scaler = torch.cuda.amp.GradScaler(
        enabled=(DEVICE.type == "cuda")
    )

start_epoch, best_metric, best_epoch, early_stop_counter, history = load_last_checkpoint(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler
)

training_start_time = time.time()
stopped_by_smoke_test = False
early_stopped = False

print("\n=== CP2a Training Configuration ===")
print(f"Protocol             : {PROTOCOL}")
print(f"Seed                 : {SEED}")
print(f"Max epochs           : {MAX_EPOCHS}")
print(f"Start epoch index    : {start_epoch}")
print(f"Batch size           : {BATCH_SIZE}")
print(f"Image size           : {IMG_SIZE}")
print(f"Device               : {DEVICE}")
print(f"Optimizer            : AdamW")
print(f"Base LR              : {BASE_LR}")
print(f"Warmup start LR      : {WARMUP_START_LR}")
print(f"Min LR               : {MIN_LR}")
print(f"Weight decay         : {WEIGHT_DECAY}")
print(f"Label smoothing      : {LABEL_SMOOTHING}")
print(f"Early stop patience  : {EARLY_STOP_PATIENCE}")
print(f"Trainable parameters : {count_trainable_parameters(model):,}")
print(f"Model size           : {estimate_model_size_mb(model):.2f} MB")

for epoch in range(start_epoch, MAX_EPOCHS):
    epoch_display = epoch + 1
    epoch_start_time = time.time()

    current_lr = scheduler.step(epoch)

    train_metrics = train_one_epoch_cp2(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
        epoch_display=epoch_display
    )

    val_outputs = evaluate_cp2(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
        class_names=class_names,
        split_name="validation"
    )

    val_metrics = val_outputs["metrics"]
    val_macro_f1 = val_metrics["macro_f1"]

    epoch_time_sec = time.time() - epoch_start_time

    improved = val_macro_f1 > best_metric + 1e-8

    if improved:
        best_metric = val_macro_f1
        best_epoch = epoch
        early_stop_counter = 0

        save_best_model(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            val_macro_f1=val_macro_f1,
            val_metrics=val_metrics
        )
    else:
        early_stop_counter += 1

    row = {
        "epoch": epoch_display,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "train_weighted_f1": train_metrics["weighted_f1"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_weighted_f1": val_metrics["weighted_f1"],
        "val_macro_precision": val_metrics["macro_precision"],
        "val_macro_recall": val_metrics["macro_recall"],
        "lr": current_lr,
        "epoch_time_sec": epoch_time_sec,
        "best_val_macro_f1_so_far": best_metric,
        "best_epoch_so_far": None if best_epoch is None else best_epoch + 1,
        "early_stop_counter": early_stop_counter,
        "improved": improved
    }

    history.append(row)
    history_df = save_history(history)

    save_last_checkpoint(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        best_metric=best_metric,
        best_epoch=best_epoch,
        early_stop_counter=early_stop_counter,
        history=history,
        stopped_by_smoke_test=stopped_by_smoke_test,
        early_stopped=early_stopped
    )

    print("\n=== Epoch Summary ===")
    print(json.dumps(row, indent=2))

    if epoch_display == SMOKE_TEST_EPOCH:
        if val_metrics["accuracy"] < SMOKE_TEST_MIN_VAL_ACC:
            stopped_by_smoke_test = True

            save_last_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                best_metric=best_metric,
                best_epoch=best_epoch,
                early_stop_counter=early_stop_counter,
                history=history,
                stopped_by_smoke_test=stopped_by_smoke_test,
                early_stopped=early_stopped
            )

            print("\n=== Smoke Test Stop ===")
            print(f"Validation accuracy at epoch {SMOKE_TEST_EPOCH}: {val_metrics['accuracy']:.4f}")
            print(f"Minimum required validation accuracy: {SMOKE_TEST_MIN_VAL_ACC:.4f}")
            break

        elif val_metrics["accuracy"] < 0.50:
            print("\n=== Smoke Test Warning ===")
            print(f"Validation accuracy at epoch {SMOKE_TEST_EPOCH}: {val_metrics['accuracy']:.4f}")
            print("Training continues with caution.")
        else:
            print("\n=== Smoke Test Passed ===")
            print(f"Validation accuracy at epoch {SMOKE_TEST_EPOCH}: {val_metrics['accuracy']:.4f}")

    if early_stop_counter >= EARLY_STOP_PATIENCE:
        early_stopped = True

        save_last_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            best_metric=best_metric,
            best_epoch=best_epoch,
            early_stop_counter=early_stop_counter,
            history=history,
            stopped_by_smoke_test=stopped_by_smoke_test,
            early_stopped=early_stopped
        )

        print("\n=== Early Stopping Triggered ===")
        print(f"No val_macro_f1 improvement for {EARLY_STOP_PATIENCE} epochs.")
        break

training_time_sec = time.time() - training_start_time

peak_gpu_memory_mb = None
if DEVICE.type == "cuda":
    peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

training_summary = {
    "protocol": PROTOCOL,
    "seed": SEED,
    "max_epochs": MAX_EPOCHS,
    "total_epochs_run": len(history),
    "best_epoch": None if best_epoch is None else best_epoch + 1,
    "best_val_macro_f1": None if best_metric < 0 else best_metric,
    "total_training_time_sec": training_time_sec,
    "peak_gpu_memory_mb": peak_gpu_memory_mb,
    "final_lr": optimizer.param_groups[0]["lr"],
    "early_stopped": early_stopped,
    "stopped_by_smoke_test": stopped_by_smoke_test,
    "model_params_count": count_trainable_parameters(model),
    "model_size_mb": estimate_model_size_mb(model),
    "last_checkpoint": str(LAST_CKPT_PATH),
    "best_checkpoint": str(BEST_CKPT_PATH),
    "history_csv": str(HISTORY_CSV_PATH),
    "history_json": str(HISTORY_JSON_PATH)
}

with open(TRAINING_SUMMARY_JSON_PATH, "w") as f:
    json.dump(training_summary, f, indent=2)

gc.collect()
torch.cuda.empty_cache()

print(json.dumps(training_summary, indent=2))

In [ ]:
# ==========================================================
# CELL N — Best Model Evaluation
# Evaluate validation and test sets using best val_macro_f1 checkpoint
# ==========================================================

def save_split_evaluation_outputs(split_name, eval_outputs):
    prefix = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_{split_name}"

    report_df = pd.DataFrame(eval_outputs["classification_report"]).transpose()
    report_csv_path = Path(str(prefix) + "_classification_report.csv")
    report_json_path = Path(str(prefix) + "_classification_report.json")
    report_txt_path = Path(str(prefix) + "_classification_report.txt")

    report_df.to_csv(report_csv_path)

    with open(report_json_path, "w") as f:
        json.dump(eval_outputs["classification_report"], f, indent=2)

    with open(report_txt_path, "w") as f:
        f.write(eval_outputs["classification_report_txt"])

    cm_df = pd.DataFrame(
        eval_outputs["confusion_matrix"],
        index=class_names,
        columns=class_names
    )
    cm_csv_path = Path(str(prefix) + "_confusion_matrix.csv")
    cm_df.to_csv(cm_csv_path)

    pred_csv_path = Path(str(prefix) + "_predictions.csv")
    eval_outputs["prediction_df"].to_csv(pred_csv_path, index=False)

    return {
        "classification_report_csv": str(report_csv_path),
        "classification_report_json": str(report_json_path),
        "classification_report_txt": str(report_txt_path),
        "confusion_matrix_csv": str(cm_csv_path),
        "predictions_csv": str(pred_csv_path)
    }


if not BEST_CKPT_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_CKPT_PATH}")

best_ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)

best_model = EquivariantVGG16(
    num_classes=len(class_names),
    N=4,
    dropout=0.30
).to(DEVICE)

best_model.load_state_dict(best_ckpt["model_state_dict"])

criterion_eval = torch_nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

validation_outputs = evaluate_cp2(
    model=best_model,
    loader=val_loader,
    criterion=criterion_eval,
    device=DEVICE,
    class_names=class_names,
    split_name="validation"
)

test_outputs = evaluate_cp2(
    model=best_model,
    loader=test_loader,
    criterion=criterion_eval,
    device=DEVICE,
    class_names=class_names,
    split_name="test"
)

validation_files = save_split_evaluation_outputs("validation", validation_outputs)
test_files = save_split_evaluation_outputs("test", test_outputs)

metrics_summary = {
    "protocol": PROTOCOL,
    "seed": SEED,
    "best_epoch": int(best_ckpt["epoch"]) + 1,
    "best_val_macro_f1_checkpoint": float(best_ckpt["val_macro_f1"]),
    "validation": validation_outputs["metrics"],
    "test": test_outputs["metrics"],
    "test_per_class_f1": test_outputs["per_class_f1"],
    "validation_files": validation_files,
    "test_files": test_files
}

metrics_summary_json_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_evaluation_metrics_summary.json"
metrics_summary_csv_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_evaluation_metrics_summary.csv"

with open(metrics_summary_json_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)

metrics_rows = []

for split_name, split_metrics in [
    ("validation", validation_outputs["metrics"]),
    ("test", test_outputs["metrics"])
]:
    row = {"split": split_name}
    row.update(split_metrics)
    metrics_rows.append(row)

metrics_summary_df = pd.DataFrame(metrics_rows)
metrics_summary_df.to_csv(metrics_summary_csv_path, index=False)

per_class_f1_df = pd.DataFrame({
    "class_name": list(test_outputs["per_class_f1"].keys()),
    "test_f1": list(test_outputs["per_class_f1"].values())
})

per_class_f1_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_test_per_class_f1.csv"
per_class_f1_df.to_csv(per_class_f1_path, index=False)

print(f"Best epoch               : {int(best_ckpt['epoch']) + 1}")
print(f"Best checkpoint val F1   : {float(best_ckpt['val_macro_f1']):.4f}")
print(f"Validation accuracy      : {validation_outputs['metrics']['accuracy']:.4f}")
print(f"Validation macro F1      : {validation_outputs['metrics']['macro_f1']:.4f}")
print(f"Test accuracy            : {test_outputs['metrics']['accuracy']:.4f}")
print(f"Test macro F1            : {test_outputs['metrics']['macro_f1']:.4f}")
print(f"Test weighted F1         : {test_outputs['metrics']['weighted_f1']:.4f}")
print(f"Metrics summary JSON     : {metrics_summary_json_path}")
print(f"Metrics summary CSV      : {metrics_summary_csv_path}")
print(f"Per-class F1 CSV         : {per_class_f1_path}")

display(per_class_f1_df)

In [ ]:
# ==========================================================
# CELL O — Learning Curve and Confusion Matrix Plots
# ==========================================================

import matplotlib.pyplot as plt

history_df = pd.read_csv(HISTORY_CSV_PATH)

learning_curve_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_learning_curve.png"
confusion_matrix_test_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_confusion_matrix_test.png"

# ----------------------------------------------------------
# Learning curve
# ----------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
axes[0, 0].set_title("Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
axes[0, 1].plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
axes[0, 1].set_title("Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
axes[1, 0].set_title("Validation Macro F1")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Macro F1")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history_df["epoch"], history_df["lr"], label="learning_rate")
axes[1, 1].set_title("Learning Rate Schedule")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Learning Rate")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(learning_curve_path, dpi=300, bbox_inches="tight")
plt.show()

# ----------------------------------------------------------
# Test confusion matrix
# ----------------------------------------------------------
test_cm = test_outputs["confusion_matrix"]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(test_cm)

ax.set_title("Test Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")

ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

for i in range(test_cm.shape[0]):
    for j in range(test_cm.shape[1]):
        ax.text(j, i, str(test_cm[i, j]), ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(confusion_matrix_test_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Learning curve saved      : {learning_curve_path}")
print(f"Test confusion matrix saved: {confusion_matrix_test_path}")

In [ ]:
# ==========================================================
# CELL P — Final CP2a Summary
# ==========================================================

with open(TRAINING_SUMMARY_JSON_PATH, "r") as f:
    training_summary_loaded = json.load(f)

with open(metrics_summary_json_path, "r") as f:
    metrics_summary_loaded = json.load(f)

final_cp2a_summary = {
    "protocol": PROTOCOL,
    "seed": SEED,
    "model": "EquivariantVGG16_p4",
    "best_epoch": metrics_summary_loaded["best_epoch"],
    "best_val_macro_f1": metrics_summary_loaded["best_val_macro_f1_checkpoint"],
    "test_accuracy": metrics_summary_loaded["test"]["accuracy"],
    "test_macro_f1": metrics_summary_loaded["test"]["macro_f1"],
    "test_weighted_f1": metrics_summary_loaded["test"]["weighted_f1"],
    "test_macro_precision": metrics_summary_loaded["test"]["macro_precision"],
    "test_macro_recall": metrics_summary_loaded["test"]["macro_recall"],
    "total_epochs_run": training_summary_loaded["total_epochs_run"],
    "total_training_time_sec": training_summary_loaded["total_training_time_sec"],
    "peak_gpu_memory_mb": training_summary_loaded["peak_gpu_memory_mb"],
    "early_stopped": training_summary_loaded["early_stopped"],
    "stopped_by_smoke_test": training_summary_loaded["stopped_by_smoke_test"],
    "best_checkpoint": str(BEST_CKPT_PATH),
    "last_checkpoint": str(LAST_CKPT_PATH),
    "training_history_csv": str(HISTORY_CSV_PATH),
    "learning_curve_png": str(learning_curve_path),
    "confusion_matrix_test_png": str(confusion_matrix_test_path),
    "metrics_summary_json": str(metrics_summary_json_path),
    "per_class_f1_csv": str(per_class_f1_path)
}

final_summary_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_{PROTOCOL}_seed{SEED}_CP2a_final_summary.json"

with open(final_summary_path, "w") as f:
    json.dump(final_cp2a_summary, f, indent=2)

print("=== CP2a Final Summary ===")
print(json.dumps(final_cp2a_summary, indent=2))

print("\n=== Required Review Files ===")
print(f"Training history CSV : {HISTORY_CSV_PATH}")
print(f"Learning curve PNG   : {learning_curve_path}")
print(f"Confusion matrix PNG : {confusion_matrix_test_path}")
print(f"Per-class F1 CSV     : {per_class_f1_path}")
print(f"Final summary JSON   : {final_summary_path}")

In [ ]:
# ==========================================================
# CELL Q — CP2a-v2 Config, Transforms, and Weighted Sampler
# ==========================================================

import os
import gc
import json
import time
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as torch_nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.transforms as T

from PIL import Image

# ----------------------------------------------------------
# CP2a-v2 configuration
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")
PHASE2_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "equivariant_phase2"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_equivariant"

PHASE2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL = "A"
SEED = 42
RUN_SUFFIX = "v2"

MAX_EPOCHS = 120
BATCH_SIZE = 32
NUM_WORKERS = 4
IMG_SIZE = 256

BASE_LR = 1e-4
WARMUP_START_LR = 1e-6
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05

EARLY_STOP_PATIENCE = 25
EARLY_GATE_EPOCH = 15

GRAD_CLIP_MAX_NORM = 1.0
COLLAPSE_VAL_ACC_THRESHOLD = 0.30
COLLAPSE_START_EPOCH = 10
COLLAPSE_PATIENCE = 3

RESUME_TRAINING = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------
# Determinism
# ----------------------------------------------------------
def set_global_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(SEED)

# ----------------------------------------------------------
# Dataset class
# ----------------------------------------------------------
class TEMVirusCP2v2Dataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

        self.labels = [
            self.class_to_idx[class_name]
            for class_name in self.df["class_name"].tolist()
        ]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


# ----------------------------------------------------------
# Class mapping
# ----------------------------------------------------------
class_names = sorted(protocol_df["class_name"].unique().tolist())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

# ----------------------------------------------------------
# Reduced augmentation policy
# ----------------------------------------------------------
train_transform_v2 = T.Compose([
    T.Resize((256, 256)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=DATASET_MEAN, std=DATASET_STD)
])

eval_transform_v2 = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=DATASET_MEAN, std=DATASET_STD)
])

train_df = protocol_df[protocol_df["split"] == "train"].copy()
val_df = protocol_df[protocol_df["split"] == "val"].copy()
test_df = protocol_df[protocol_df["split"] == "test"].copy()

train_dataset = TEMVirusCP2v2Dataset(train_df, class_to_idx, transform=train_transform_v2)
val_dataset = TEMVirusCP2v2Dataset(val_df, class_to_idx, transform=eval_transform_v2)
test_dataset = TEMVirusCP2v2Dataset(test_df, class_to_idx, transform=eval_transform_v2)

# ----------------------------------------------------------
# WeightedRandomSampler
# ----------------------------------------------------------
train_label_array = np.array(train_dataset.labels)
class_counts = np.bincount(train_label_array, minlength=len(class_names))

class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_label_array]
sample_weights = torch.DoubleTensor(sample_weights)

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

class_distribution_df = pd.DataFrame({
    "class_name": class_names,
    "class_index": list(range(len(class_names))),
    "train_count": class_counts.tolist(),
    "class_weight": class_weights.tolist()
})

class_distribution_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_class_sampling_weights.csv"
class_distribution_df.to_csv(class_distribution_path, index=False)

print("=== CELL Q READY ===")
print(f"Device       : {DEVICE}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Train images : {len(train_dataset)}")
print(f"Val images   : {len(val_dataset)}")
print(f"Test images  : {len(test_dataset)}")
print(f"Mean         : {DATASET_MEAN}")
print(f"Std          : {DATASET_STD}")
print(f"Sampler      : WeightedRandomSampler")
print(f"Class weights saved: {class_distribution_path}")

display(class_distribution_df)

In [ ]:
# ==========================================================
# CELL R — CP2a-v2 Training, Checkpoint, and Resume Utilities
# ==========================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

LAST_CKPT_PATH = CHECKPOINT_DIR / f"EquivVGG16p4_A_seed42_v2_last.pt"
BEST_CKPT_PATH = CHECKPOINT_DIR / f"EquivVGG16p4_A_seed42_v2_best.pt"

HISTORY_CSV_PATH = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_training_history.csv"
HISTORY_JSON_PATH = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_training_history.json"
TRAINING_SUMMARY_JSON_PATH = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_training_summary.json"


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


def compute_basic_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }


def save_history_v2(history):
    history_df = pd.DataFrame(history)
    history_df.to_csv(HISTORY_CSV_PATH, index=False)

    with open(HISTORY_JSON_PATH, "w") as f:
        json.dump(history, f, indent=2)

    return history_df


def save_last_checkpoint_v2(
    model,
    optimizer,
    scheduler,
    epoch,
    best_metric,
    best_epoch,
    early_stop_counter,
    consecutive_low_val_acc,
    history,
    stopped_by_early_gate=False,
    stopped_by_collapse=False,
    early_stopped=False
):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "best_metric": best_metric,
            "best_epoch": best_epoch,
            "early_stop_counter": early_stop_counter,
            "consecutive_low_val_acc": consecutive_low_val_acc,
            "history": history,
            "seed": SEED,
            "protocol": PROTOCOL,
            "run_suffix": RUN_SUFFIX,
            "stopped_by_early_gate": stopped_by_early_gate,
            "stopped_by_collapse": stopped_by_collapse,
            "early_stopped": early_stopped,
            "config": {
                "model": "EquivariantVGG16_p4",
                "epochs": MAX_EPOCHS,
                "batch_size": BATCH_SIZE,
                "image_size": IMG_SIZE,
                "optimizer": "AdamW",
                "base_lr": BASE_LR,
                "warmup_start_lr": WARMUP_START_LR,
                "min_lr": MIN_LR,
                "weight_decay": WEIGHT_DECAY,
                "label_smoothing": LABEL_SMOOTHING,
                "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
                "sampler": "WeightedRandomSampler",
                "early_stop_patience": EARLY_STOP_PATIENCE,
                "monitor": "val_macro_f1"
            }
        },
        LAST_CKPT_PATH
    )


def save_best_model_v2(model, optimizer, scheduler, epoch, val_macro_f1, val_metrics):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "val_macro_f1": val_macro_f1,
            "val_metrics": val_metrics,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "seed": SEED,
            "protocol": PROTOCOL,
            "run_suffix": RUN_SUFFIX,
            "config": {
                "model": "EquivariantVGG16_p4",
                "epochs": MAX_EPOCHS,
                "batch_size": BATCH_SIZE,
                "image_size": IMG_SIZE,
                "optimizer": "AdamW",
                "base_lr": BASE_LR,
                "warmup_start_lr": WARMUP_START_LR,
                "min_lr": MIN_LR,
                "weight_decay": WEIGHT_DECAY,
                "label_smoothing": LABEL_SMOOTHING,
                "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
                "sampler": "WeightedRandomSampler"
            }
        },
        BEST_CKPT_PATH
    )


def load_last_checkpoint_v2(model, optimizer, scheduler):
    if RESUME_TRAINING and LAST_CKPT_PATH.exists():
        ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_metric = float(ckpt.get("best_metric", -1.0))
        best_epoch = ckpt.get("best_epoch", None)
        early_stop_counter = int(ckpt.get("early_stop_counter", 0))
        consecutive_low_val_acc = int(ckpt.get("consecutive_low_val_acc", 0))
        history = ckpt.get("history", [])

        print("=== Resume Checkpoint Found ===")
        print(f"Resuming from epoch : {start_epoch + 1}")
        print(f"Best val macro F1   : {best_metric:.4f}")
        print(f"Best epoch          : {None if best_epoch is None else best_epoch + 1}")
        print(f"Early stop counter  : {early_stop_counter}")
        print(f"Low val-acc counter : {consecutive_low_val_acc}")

        return start_epoch, best_metric, best_epoch, early_stop_counter, consecutive_low_val_acc, history

    print("=== Fresh CP2a-v2 Training Run ===")
    return 0, -1.0, None, 0, 0, []


def train_one_epoch_cp2_v2(model, loader, criterion, optimizer, scaler, device, epoch_display):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_preds = []

    progress_bar = tqdm(loader, desc=f"Train v2 epoch {epoch_display}", leave=True)

    for images, labels, _paths in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        if torch.isnan(loss):
            raise ValueError(f"NaN loss detected at epoch {epoch_display}")

        scaler.scale(loss).backward()

        # Critical fix: unscale before gradient clipping
        scaler.unscale_(optimizer)
        grad_norm = torch_nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_MAX_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "grad_norm": f"{float(grad_norm):.3f}"
        })

    avg_loss = running_loss / len(loader.dataset)
    metrics = compute_basic_metrics(all_targets, all_preds)
    metrics["loss"] = float(avg_loss)
    metrics["grad_norm_last"] = float(grad_norm)

    return metrics


print(f"Last checkpoint : {LAST_CKPT_PATH}")
print(f"Best checkpoint : {BEST_CKPT_PATH}")
print(f"History CSV     : {HISTORY_CSV_PATH}")

In [ ]:
# ==========================================================
# CELL S — CP2a-v2 Full Training Loop
# 120 epochs, seed 42, Protocol A
# ==========================================================

set_global_seed(SEED)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model = EquivariantVGG16(
    num_classes=len(class_names),
    N=4,
    dropout=0.30
).to(DEVICE)

criterion = torch_nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler = WarmupCosineScheduler(
    optimizer=optimizer,
    total_epochs=MAX_EPOCHS,
    warmup_epochs=5,
    warmup_start_lr=WARMUP_START_LR,
    base_lr=BASE_LR,
    min_lr=MIN_LR
)

try:
    scaler = torch.amp.GradScaler(
        device="cuda",
        enabled=(DEVICE.type == "cuda")
    )
except TypeError:
    scaler = torch.cuda.amp.GradScaler(
        enabled=(DEVICE.type == "cuda")
    )

(
    start_epoch,
    best_metric,
    best_epoch,
    early_stop_counter,
    consecutive_low_val_acc,
    history
) = load_last_checkpoint_v2(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler
)

training_start_time = time.time()

stopped_by_early_gate = False
stopped_by_collapse = False
early_stopped = False
anomaly_log = []

print("\n=== CP2a-v2 Training Configuration ===")
print(f"Protocol             : {PROTOCOL}")
print(f"Seed                 : {SEED}")
print(f"Run suffix           : {RUN_SUFFIX}")
print(f"Max epochs           : {MAX_EPOCHS}")
print(f"Start epoch index    : {start_epoch}")
print(f"Batch size           : {BATCH_SIZE}")
print(f"Image size           : {IMG_SIZE}")
print(f"Device               : {DEVICE}")
print(f"Optimizer            : AdamW")
print(f"Base LR              : {BASE_LR}")
print(f"Warmup start LR      : {WARMUP_START_LR}")
print(f"Min LR               : {MIN_LR}")
print(f"Weight decay         : {WEIGHT_DECAY}")
print(f"Label smoothing      : {LABEL_SMOOTHING}")
print(f"Gradient clipping    : max_norm={GRAD_CLIP_MAX_NORM}")
print(f"Sampler              : WeightedRandomSampler")
print(f"Early stop patience  : {EARLY_STOP_PATIENCE}")
print(f"Trainable parameters : {count_trainable_parameters(model):,}")
print(f"Model size           : {estimate_model_size_mb(model):.2f} MB")

for epoch in range(start_epoch, MAX_EPOCHS):
    epoch_display = epoch + 1
    epoch_start_time = time.time()

    current_lr = scheduler.step(epoch)

    train_metrics = train_one_epoch_cp2_v2(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
        epoch_display=epoch_display
    )

    val_outputs = evaluate_cp2(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
        class_names=class_names,
        split_name="validation"
    )

    val_metrics = val_outputs["metrics"]
    val_macro_f1 = val_metrics["macro_f1"]
    val_acc = val_metrics["accuracy"]

    epoch_time_sec = time.time() - epoch_start_time

    improved = val_macro_f1 > best_metric + 1e-8

    if improved:
        best_metric = val_macro_f1
        best_epoch = epoch
        early_stop_counter = 0

        save_best_model_v2(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            val_macro_f1=val_macro_f1,
            val_metrics=val_metrics
        )
    else:
        early_stop_counter += 1

    # Collapse detection
    collapse_warning = False

    if epoch_display > COLLAPSE_START_EPOCH and val_acc < COLLAPSE_VAL_ACC_THRESHOLD:
        consecutive_low_val_acc += 1
        collapse_warning = True
        warning_msg = (
            f"Possible collapse: val_acc={val_acc:.4f} at epoch {epoch_display}, "
            f"consecutive_low_val_acc={consecutive_low_val_acc}"
        )
        anomaly_log.append(warning_msg)
        print("\nWARNING:", warning_msg)
    else:
        consecutive_low_val_acc = 0

    row = {
        "epoch": epoch_display,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "train_weighted_f1": train_metrics["weighted_f1"],
        "train_grad_norm_last": train_metrics["grad_norm_last"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_weighted_f1": val_metrics["weighted_f1"],
        "val_macro_precision": val_metrics["macro_precision"],
        "val_macro_recall": val_metrics["macro_recall"],
        "lr": current_lr,
        "epoch_time_sec": epoch_time_sec,
        "best_val_macro_f1_so_far": best_metric,
        "best_epoch_so_far": None if best_epoch is None else best_epoch + 1,
        "early_stop_counter": early_stop_counter,
        "consecutive_low_val_acc": consecutive_low_val_acc,
        "collapse_warning": collapse_warning,
        "improved": improved
    }

    history.append(row)
    history_df = save_history_v2(history)

    save_last_checkpoint_v2(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        best_metric=best_metric,
        best_epoch=best_epoch,
        early_stop_counter=early_stop_counter,
        consecutive_low_val_acc=consecutive_low_val_acc,
        history=history,
        stopped_by_early_gate=stopped_by_early_gate,
        stopped_by_collapse=stopped_by_collapse,
        early_stopped=early_stopped
    )

    print("\n=== Epoch Summary ===")
    print(json.dumps(row, indent=2))

    # Early gate at epoch 15
    if epoch_display == EARLY_GATE_EPOCH:
        if val_macro_f1 < 0.30:
            stopped_by_early_gate = True
            anomaly_log.append(
                f"Early gate stop: val_macro_f1={val_macro_f1:.4f} at epoch {EARLY_GATE_EPOCH}"
            )

            save_last_checkpoint_v2(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                best_metric=best_metric,
                best_epoch=best_epoch,
                early_stop_counter=early_stop_counter,
                consecutive_low_val_acc=consecutive_low_val_acc,
                history=history,
                stopped_by_early_gate=stopped_by_early_gate,
                stopped_by_collapse=stopped_by_collapse,
                early_stopped=early_stopped
            )

            print("\n=== Early Gate Stop ===")
            print(f"Validation macro F1 at epoch {EARLY_GATE_EPOCH}: {val_macro_f1:.4f}")
            break

        elif val_macro_f1 < 0.45:
            warning_msg = (
                f"Early gate warning: val_macro_f1={val_macro_f1:.4f} "
                f"at epoch {EARLY_GATE_EPOCH}"
            )
            anomaly_log.append(warning_msg)
            print("\n=== Early Gate Warning ===")
            print(warning_msg)
        else:
            print("\n=== Early Gate Passed ===")
            print(f"Validation macro F1 at epoch {EARLY_GATE_EPOCH}: {val_macro_f1:.4f}")

    # Collapse stop
    if consecutive_low_val_acc >= COLLAPSE_PATIENCE:
        stopped_by_collapse = True
        anomaly_log.append(
            f"Collapse stop: val_acc < {COLLAPSE_VAL_ACC_THRESHOLD} for "
            f"{COLLAPSE_PATIENCE} consecutive epochs"
        )

        save_last_checkpoint_v2(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            best_metric=best_metric,
            best_epoch=best_epoch,
            early_stop_counter=early_stop_counter,
            consecutive_low_val_acc=consecutive_low_val_acc,
            history=history,
            stopped_by_early_gate=stopped_by_early_gate,
            stopped_by_collapse=stopped_by_collapse,
            early_stopped=early_stopped
        )

        print("\n=== Collapse Stop Triggered ===")
        break

    # Early stopping
    if early_stop_counter >= EARLY_STOP_PATIENCE:
        early_stopped = True

        save_last_checkpoint_v2(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            best_metric=best_metric,
            best_epoch=best_epoch,
            early_stop_counter=early_stop_counter,
            consecutive_low_val_acc=consecutive_low_val_acc,
            history=history,
            stopped_by_early_gate=stopped_by_early_gate,
            stopped_by_collapse=stopped_by_collapse,
            early_stopped=early_stopped
        )

        print("\n=== Early Stopping Triggered ===")
        print(f"No val_macro_f1 improvement for {EARLY_STOP_PATIENCE} epochs.")
        break

training_time_sec = time.time() - training_start_time

peak_gpu_memory_mb = None
if DEVICE.type == "cuda":
    peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

training_summary_v2 = {
    "protocol": PROTOCOL,
    "seed": SEED,
    "run_suffix": RUN_SUFFIX,
    "max_epochs": MAX_EPOCHS,
    "total_epochs_run": len(history),
    "best_epoch": None if best_epoch is None else best_epoch + 1,
    "best_val_macro_f1": None if best_metric < 0 else best_metric,
    "total_training_time_sec": training_time_sec,
    "peak_gpu_memory_mb": peak_gpu_memory_mb,
    "final_lr": optimizer.param_groups[0]["lr"],
    "early_stopped": early_stopped,
    "stopped_by_early_gate": stopped_by_early_gate,
    "stopped_by_collapse": stopped_by_collapse,
    "anomaly_log": anomaly_log,
    "model_params_count": count_trainable_parameters(model),
    "model_size_mb": estimate_model_size_mb(model),
    "batch_size": BATCH_SIZE,
    "label_smoothing": LABEL_SMOOTHING,
    "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
    "sampler": "WeightedRandomSampler",
    "last_checkpoint": str(LAST_CKPT_PATH),
    "best_checkpoint": str(BEST_CKPT_PATH),
    "history_csv": str(HISTORY_CSV_PATH),
    "history_json": str(HISTORY_JSON_PATH)
}

with open(TRAINING_SUMMARY_JSON_PATH, "w") as f:
    json.dump(training_summary_v2, f, indent=2)

gc.collect()
torch.cuda.empty_cache()

print(json.dumps(training_summary_v2, indent=2))

In [ ]:
# ==========================================================
# CELL T — CP2a-v2 Best Model Evaluation and Save Outputs
# ==========================================================

from sklearn.metrics import classification_report, confusion_matrix

def save_split_evaluation_outputs_v2(split_name, eval_outputs):
    prefix = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_{split_name}"

    report_df = pd.DataFrame(eval_outputs["classification_report"]).transpose()
    report_csv_path = Path(str(prefix) + "_classification_report.csv")
    report_json_path = Path(str(prefix) + "_classification_report.json")
    report_txt_path = Path(str(prefix) + "_classification_report.txt")

    report_df.to_csv(report_csv_path)

    with open(report_json_path, "w") as f:
        json.dump(eval_outputs["classification_report"], f, indent=2)

    with open(report_txt_path, "w") as f:
        f.write(eval_outputs["classification_report_txt"])

    cm_df = pd.DataFrame(
        eval_outputs["confusion_matrix"],
        index=class_names,
        columns=class_names
    )

    cm_csv_path = Path(str(prefix) + "_confusion_matrix.csv")
    cm_df.to_csv(cm_csv_path)

    pred_csv_path = Path(str(prefix) + "_predictions.csv")
    eval_outputs["prediction_df"].to_csv(pred_csv_path, index=False)

    return {
        "classification_report_csv": str(report_csv_path),
        "classification_report_json": str(report_json_path),
        "classification_report_txt": str(report_txt_path),
        "confusion_matrix_csv": str(cm_csv_path),
        "predictions_csv": str(pred_csv_path)
    }


if not BEST_CKPT_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_CKPT_PATH}")

best_ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)

best_model = EquivariantVGG16(
    num_classes=len(class_names),
    N=4,
    dropout=0.30
).to(DEVICE)

best_model.load_state_dict(best_ckpt["model_state_dict"])

criterion_eval = torch_nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

validation_outputs_v2 = evaluate_cp2(
    model=best_model,
    loader=val_loader,
    criterion=criterion_eval,
    device=DEVICE,
    class_names=class_names,
    split_name="validation"
)

test_outputs_v2 = evaluate_cp2(
    model=best_model,
    loader=test_loader,
    criterion=criterion_eval,
    device=DEVICE,
    class_names=class_names,
    split_name="test"
)

validation_files_v2 = save_split_evaluation_outputs_v2("validation", validation_outputs_v2)
test_files_v2 = save_split_evaluation_outputs_v2("test", test_outputs_v2)

metrics_summary_v2 = {
    "protocol": PROTOCOL,
    "seed": SEED,
    "run_suffix": RUN_SUFFIX,
    "best_epoch": int(best_ckpt["epoch"]) + 1,
    "best_val_macro_f1_checkpoint": float(best_ckpt["val_macro_f1"]),
    "validation": validation_outputs_v2["metrics"],
    "test": test_outputs_v2["metrics"],
    "test_per_class_f1": test_outputs_v2["per_class_f1"],
    "validation_files": validation_files_v2,
    "test_files": test_files_v2
}

metrics_summary_json_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_evaluation_metrics_summary.json"
metrics_summary_csv_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_evaluation_metrics_summary.csv"

with open(metrics_summary_json_path, "w") as f:
    json.dump(metrics_summary_v2, f, indent=2)

metrics_rows = []

for split_name, split_metrics in [
    ("validation", validation_outputs_v2["metrics"]),
    ("test", test_outputs_v2["metrics"])
]:
    row = {"split": split_name}
    row.update(split_metrics)
    metrics_rows.append(row)

metrics_summary_df = pd.DataFrame(metrics_rows)
metrics_summary_df.to_csv(metrics_summary_csv_path, index=False)

per_class_f1_df_v2 = pd.DataFrame({
    "class_name": list(test_outputs_v2["per_class_f1"].keys()),
    "test_f1": list(test_outputs_v2["per_class_f1"].values())
})

per_class_f1_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_test_per_class_f1.csv"
per_class_f1_df_v2.to_csv(per_class_f1_path, index=False)

print(f"Best epoch               : {int(best_ckpt['epoch']) + 1}")
print(f"Best checkpoint val F1   : {float(best_ckpt['val_macro_f1']):.4f}")
print(f"Validation accuracy      : {validation_outputs_v2['metrics']['accuracy']:.4f}")
print(f"Validation macro F1      : {validation_outputs_v2['metrics']['macro_f1']:.4f}")
print(f"Test accuracy            : {test_outputs_v2['metrics']['accuracy']:.4f}")
print(f"Test macro F1            : {test_outputs_v2['metrics']['macro_f1']:.4f}")
print(f"Test weighted F1         : {test_outputs_v2['metrics']['weighted_f1']:.4f}")
print(f"Metrics summary JSON     : {metrics_summary_json_path}")
print(f"Metrics summary CSV      : {metrics_summary_csv_path}")
print(f"Per-class F1 CSV         : {per_class_f1_path}")

display(per_class_f1_df_v2)

In [ ]:
# ==========================================================
# CELL U — CP2a-v2 Learning Curve and Confusion Matrix Plots
# ==========================================================

import matplotlib.pyplot as plt

history_df_v2 = pd.read_csv(HISTORY_CSV_PATH)

learning_curve_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_learning_curve.png"
confusion_matrix_test_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_confusion_matrix_test.png"

# ----------------------------------------------------------
# Learning curve
# ----------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history_df_v2["epoch"], history_df_v2["train_loss"], label="train_loss")
axes[0, 0].plot(history_df_v2["epoch"], history_df_v2["val_loss"], label="val_loss")
axes[0, 0].set_title("Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history_df_v2["epoch"], history_df_v2["train_acc"], label="train_acc")
axes[0, 1].plot(history_df_v2["epoch"], history_df_v2["val_acc"], label="val_acc")
axes[0, 1].set_title("Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history_df_v2["epoch"], history_df_v2["val_macro_f1"], label="val_macro_f1")
axes[1, 0].set_title("Validation Macro F1")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Macro F1")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history_df_v2["epoch"], history_df_v2["lr"], label="learning_rate")
axes[1, 1].set_title("Learning Rate Schedule")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Learning Rate")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(learning_curve_path, dpi=300, bbox_inches="tight")
plt.show()

# ----------------------------------------------------------
# Test confusion matrix
# ----------------------------------------------------------
test_cm_v2 = test_outputs_v2["confusion_matrix"]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(test_cm_v2)

ax.set_title("Test Confusion Matrix — CP2a-v2")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")

ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

for i in range(test_cm_v2.shape[0]):
    for j in range(test_cm_v2.shape[1]):
        ax.text(j, i, str(test_cm_v2[i, j]), ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(confusion_matrix_test_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Learning curve saved       : {learning_curve_path}")
print(f"Test confusion matrix saved: {confusion_matrix_test_path}")

In [ ]:
# ==========================================================
# CELL V — CP2a-v2 Final Summary
# ==========================================================

with open(TRAINING_SUMMARY_JSON_PATH, "r") as f:
    training_summary_loaded_v2 = json.load(f)

with open(metrics_summary_json_path, "r") as f:
    metrics_summary_loaded_v2 = json.load(f)

astro_f1 = metrics_summary_loaded_v2["test_per_class_f1"].get("Astrovirus", None)
cowpox_f1 = metrics_summary_loaded_v2["test_per_class_f1"].get("Cowpox", None)
nipah_f1 = metrics_summary_loaded_v2["test_per_class_f1"].get("Nipah virus", None)

zero_f1_classes = [
    class_name
    for class_name, score in metrics_summary_loaded_v2["test_per_class_f1"].items()
    if float(score) == 0.0
]

final_cp2a_v2_summary = {
    "protocol": PROTOCOL,
    "seed": SEED,
    "run_suffix": RUN_SUFFIX,
    "model": "EquivariantVGG16_p4",
    "best_epoch": metrics_summary_loaded_v2["best_epoch"],
    "best_val_macro_f1": metrics_summary_loaded_v2["best_val_macro_f1_checkpoint"],
    "test_accuracy": metrics_summary_loaded_v2["test"]["accuracy"],
    "test_macro_f1": metrics_summary_loaded_v2["test"]["macro_f1"],
    "test_weighted_f1": metrics_summary_loaded_v2["test"]["weighted_f1"],
    "test_macro_precision": metrics_summary_loaded_v2["test"]["macro_precision"],
    "test_macro_recall": metrics_summary_loaded_v2["test"]["macro_recall"],
    "astrovirus_f1": astro_f1,
    "cowpox_f1": cowpox_f1,
    "nipah_virus_f1": nipah_f1,
    "zero_f1_classes": zero_f1_classes,
    "total_epochs_run": training_summary_loaded_v2["total_epochs_run"],
    "total_training_time_sec": training_summary_loaded_v2["total_training_time_sec"],
    "peak_gpu_memory_mb": training_summary_loaded_v2["peak_gpu_memory_mb"],
    "early_stopped": training_summary_loaded_v2["early_stopped"],
    "stopped_by_early_gate": training_summary_loaded_v2["stopped_by_early_gate"],
    "stopped_by_collapse": training_summary_loaded_v2["stopped_by_collapse"],
    "anomaly_log": training_summary_loaded_v2["anomaly_log"],
    "batch_size": training_summary_loaded_v2["batch_size"],
    "label_smoothing": training_summary_loaded_v2["label_smoothing"],
    "grad_clip_max_norm": training_summary_loaded_v2["grad_clip_max_norm"],
    "sampler": training_summary_loaded_v2["sampler"],
    "best_checkpoint": str(BEST_CKPT_PATH),
    "last_checkpoint": str(LAST_CKPT_PATH),
    "training_history_csv": str(HISTORY_CSV_PATH),
    "learning_curve_png": str(learning_curve_path),
    "confusion_matrix_test_png": str(confusion_matrix_test_path),
    "metrics_summary_json": str(metrics_summary_json_path),
    "per_class_f1_csv": str(per_class_f1_path)
}

final_summary_path = PHASE2_OUTPUT_DIR / f"EquivVGG16p4_A_seed42_v2_CP2a_v2_final_summary.json"

with open(final_summary_path, "w") as f:
    json.dump(final_cp2a_v2_summary, f, indent=2)

print("=== CP2a-v2 Final Summary ===")
print(json.dumps(final_cp2a_v2_summary, indent=2))

print(f"Training history CSV : {HISTORY_CSV_PATH}")
print(f"Learning curve PNG   : {learning_curve_path}")
print(f"Confusion matrix PNG : {confusion_matrix_test_path}")
print(f"Per-class F1 CSV     : {per_class_f1_path}")
print(f"Final summary JSON   : {final_summary_path}")


In [ ]:
# ==========================================================
# CP3-A — Install, Imports, and Configuration
# Architecture Sweep: EfficientNetV2-S and ConvNeXt-Tiny
# ==========================================================

!pip -q install timm

import os
import gc
import json
import time
import math
import random
import zipfile
import warnings
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as torch_nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T

import timm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Project paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")
DATA_RAW_DIR = PROJECT_ROOT / "data_raw"
BASELINE_OUTPUTS_DIR = PROJECT_ROOT / "outputs"

CP3_OUTPUT_DIR = BASELINE_OUTPUTS_DIR / "architecture_sweep_phase3"
CP3_CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_architecture_sweep"

CP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CP3_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Experiment config
# ----------------------------------------------------------
PROTOCOL = "A"
SEED = 42

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
MAX_EPOCHS = 30

BASE_LR = 1e-4
WARMUP_START_LR = 1e-6
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP_MAX_NORM = 1.0

EARLY_STOP_PATIENCE = 8
RESUME_TRAINING = True

EARLY_GATE_EPOCH_5 = 5
EARLY_GATE_EPOCH_15 = 15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES_REFERENCE = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

ARCHITECTURES = {
    "EffNetV2S": {
        "timm_name": "tf_efficientnetv2_s.in21k_ft_in1k",
        "display_name": "EfficientNetV2-S"
    },
    "ConvNeXtT": {
        "timm_name": "convnext_tiny.fb_in1k",
        "display_name": "ConvNeXt-Tiny"
    }
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ----------------------------------------------------------
# Determinism
# ----------------------------------------------------------
def set_global_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(SEED)

print(f"Device        : {DEVICE}")
print(f"Output dir    : {CP3_OUTPUT_DIR}")
print(f"Checkpoint dir: {CP3_CHECKPOINT_DIR}")
print(f"timm version  : {timm.__version__}")
print(f"Batch size    : {BATCH_SIZE}")

In [ ]:
# ==========================================================
# CP3-B — Protocol A Split Reconstruction
# Split is reconstructed from DenseNet201 baseline prediction files
# ==========================================================

def standardize_split_name(x):
    x = str(x).lower().strip()

    if x in ["train", "training"]:
        return "train"
    if x in ["val", "valid", "validation"]:
        return "val"
    if x in ["test", "testing"]:
        return "test"

    return x


def find_dataset_folder():
    local_extract_root = Path("/content/tem_virus_dataset_extracted")

    candidate_dirs = [
        local_extract_root / "TEM virus dataset/context_virus_1nm_256x256",
        local_extract_root / "context_virus_1nm_256x256",
        DATA_RAW_DIR / "TEM virus dataset/context_virus_1nm_256x256",
        DATA_RAW_DIR / "context_virus_1nm_256x256",
        PROJECT_ROOT / "TEM virus dataset/context_virus_1nm_256x256",
        PROJECT_ROOT / "context_virus_1nm_256x256",
    ]

    for dataset_dir in candidate_dirs:
        if dataset_dir.exists():
            return dataset_dir

    for root in [local_extract_root, DATA_RAW_DIR, PROJECT_ROOT]:
        if root.exists():
            found_dirs = sorted(list(root.rglob("context_virus_1nm_256x256")))
            if len(found_dirs) > 0:
                return found_dirs[0]

    zip_candidates = sorted(
        list(DATA_RAW_DIR.rglob("*.zip")) +
        list(DATA_RAW_DIR.rglob("*.ZIP"))
    )

    if len(zip_candidates) == 0:
        raise FileNotFoundError(f"No dataset ZIP found under: {DATA_RAW_DIR}")

    zip_path = zip_candidates[0]
    local_extract_root.mkdir(parents=True, exist_ok=True)

    print("Extracting dataset archive")
    print(f"Source: {zip_path}")
    print(f"Target: {local_extract_root}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(local_extract_root)

    extracted_candidates = sorted(
        list(local_extract_root.rglob("context_virus_1nm_256x256"))
    )

    if len(extracted_candidates) == 0:
        raise FileNotFoundError("context_virus_1nm_256x256 was not found after extraction.")

    return extracted_candidates[0]


def build_all_images_manifest(dataset_dir):
    rows = []

    for split_dir in sorted([p for p in dataset_dir.iterdir() if p.is_dir()]):
        split_name = standardize_split_name(split_dir.name)

        if split_name not in ["train", "val", "test"]:
            continue

        for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            class_name = class_dir.name

            for image_path in sorted(class_dir.glob("*.tif")):
                rows.append({
                    "filepath": str(image_path),
                    "original_split": split_name,
                    "class_name": class_name,
                    "filename": image_path.name,
                    "rel_key": f"{split_name}/{class_name}/{image_path.name}",
                    "class_file_key": f"{class_name}/{image_path.name}",
                    "basename": image_path.name
                })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise ValueError("No .tif images were found in the dataset folder.")

    return df


def find_baseline_prediction_file(expected_name):
    exact_path = BASELINE_OUTPUTS_DIR / expected_name

    if exact_path.exists():
        return exact_path

    candidates = sorted(list(BASELINE_OUTPUTS_DIR.rglob(expected_name)))

    if len(candidates) > 0:
        return candidates[0]

    raise FileNotFoundError(f"Baseline prediction file not found: {expected_name}")


def normalize_path_string(x):
    return str(x).replace("\\", "/").strip()


def standardize_rel_key_from_path(path_value):
    s = normalize_path_string(path_value)

    marker = "context_virus_1nm_256x256/"
    if marker in s:
        tail = s.split(marker, 1)[1]
    else:
        parts = [p for p in PurePosixPath(s).parts if p not in ["/", ""]]
        tail = "/".join(parts[-3:]) if len(parts) >= 3 else s

    parts = tail.split("/")
    if len(parts) >= 3:
        parts[0] = standardize_split_name(parts[0])
        return "/".join(parts[-3:])

    return None


def class_file_key_from_path(path_value):
    s = normalize_path_string(path_value)
    parts = [p for p in PurePosixPath(s).parts if p not in ["/", ""]]

    if len(parts) >= 2:
        parent = parts[-2]
        filename = parts[-1]

        if standardize_split_name(parent) not in ["train", "val", "test"]:
            return f"{parent}/{filename}"

    return None


def basename_from_path(path_value):
    s = normalize_path_string(path_value)
    return PurePosixPath(s).name


def detect_path_column(df):
    preferred = [
        "image_path", "filepath", "file_path", "path", "filename",
        "image", "img_path", "fname"
    ]

    lower_map = {c.lower(): c for c in df.columns}

    for c in preferred:
        if c in lower_map:
            return lower_map[c]

    for c in df.columns:
        sample = df[c].astype(str).head(20).str.lower()
        if sample.str.contains(".tif", regex=False).any():
            return c

    raise ValueError(
        "Could not detect image path or filename column. "
        f"Available columns: {list(df.columns)}"
    )


def detect_label_column(df):
    preferred = [
        "true_label", "target_class", "label_name", "class_name",
        "true_class", "target", "label"
    ]

    lower_map = {c.lower(): c for c in df.columns}

    for c in preferred:
        if c in lower_map:
            return lower_map[c]

    return None


def normalize_label_value(x, class_names):
    if pd.isna(x):
        return None

    value = str(x).strip()

    if value in class_names:
        return value

    try:
        idx = int(float(value))
        if 0 <= idx < len(class_names):
            return class_names[idx]
    except:
        pass

    value_lower = value.lower()
    for c in class_names:
        if value_lower == c.lower():
            return c

    return None


def resolve_prediction_filepaths(pred_df, all_df, split_label, class_names):
    path_col = detect_path_column(pred_df)
    label_col = detect_label_column(pred_df)

    temp = pred_df.copy()
    temp["_path_value"] = temp[path_col].astype(str)

    temp["rel_key_candidate"] = temp["_path_value"].apply(standardize_rel_key_from_path)
    temp["class_file_key_from_path"] = temp["_path_value"].apply(class_file_key_from_path)
    temp["basename_candidate"] = temp["_path_value"].apply(basename_from_path)

    if label_col is not None:
        temp["_label_name"] = temp[label_col].apply(
            lambda x: normalize_label_value(x, class_names)
        )
        temp["class_file_key_from_label"] = temp.apply(
            lambda r: f"{r['_label_name']}/{r['basename_candidate']}"
            if r["_label_name"] is not None else None,
            axis=1
        )
    else:
        temp["class_file_key_from_label"] = None

    strategies = [
        ("rel_key_candidate", "rel_key"),
        ("class_file_key_from_label", "class_file_key"),
        ("class_file_key_from_path", "class_file_key"),
        ("basename_candidate", "basename")
    ]

    best = None

    for pred_key_col, all_key_col in strategies:
        pred_keys = temp[pred_key_col].dropna().astype(str).tolist()
        pred_key_set = set(pred_keys)

        if len(pred_key_set) == 0:
            continue

        all_key_counts = all_df[all_key_col].value_counts()
        duplicate_keys = set(all_key_counts[all_key_counts > 1].index)

        if all_key_col == "basename" and len(duplicate_keys) > 0:
            continue

        all_key_set = set(all_df[all_key_col].astype(str))
        matched_keys = pred_key_set.intersection(all_key_set)
        coverage = len(matched_keys) / max(1, len(pred_key_set))

        candidate = {
            "pred_key_col": pred_key_col,
            "all_key_col": all_key_col,
            "coverage": coverage,
            "matched_keys": matched_keys
        }

        if best is None or coverage > best["coverage"]:
            best = candidate

    if best is None or best["coverage"] < 0.95:
        print(f"\nBaseline prediction columns for {split_label}:")
        print(list(pred_df.columns))
        display(pred_df.head())
        raise ValueError(
            f"Could not reliably match {split_label} prediction file. "
            f"Best coverage: {0 if best is None else best['coverage']:.4f}"
        )

    matched_df = all_df[all_df[best["all_key_col"]].isin(best["matched_keys"])].copy()
    matched_filepaths = set(matched_df["filepath"].tolist())

    print(f"\n=== Split Match: {split_label} ===")
    print(f"Path column used     : {path_col}")
    print(f"Label column used    : {label_col}")
    print(f"Matching strategy    : {best['pred_key_col']} -> {best['all_key_col']}")
    print(f"Coverage             : {best['coverage']:.4f}")
    print(f"Expected rows        : {len(pred_df)}")
    print(f"Matched image files  : {len(matched_filepaths)}")

    return matched_filepaths


# ----------------------------------------------------------
# Reconstruct Protocol A
# ----------------------------------------------------------
BASELINE_VAL_PRED_PATH = find_baseline_prediction_file(
    "DenseNet201_A_official_seed42_validation_predictions.csv"
)
BASELINE_TEST_PRED_PATH = find_baseline_prediction_file(
    "DenseNet201_A_official_seed42_test_predictions.csv"
)

dataset_dir = find_dataset_folder()
all_images_df = build_all_images_manifest(dataset_dir)

class_names = sorted(all_images_df["class_name"].unique().tolist())

if class_names != CLASS_NAMES_REFERENCE:
    raise ValueError(
        f"Class names mismatch.\nDetected: {class_names}\nExpected: {CLASS_NAMES_REFERENCE}"
    )

val_pred_df = pd.read_csv(BASELINE_VAL_PRED_PATH)
test_pred_df = pd.read_csv(BASELINE_TEST_PRED_PATH)

val_filepaths = resolve_prediction_filepaths(
    val_pred_df,
    all_images_df,
    split_label="validation",
    class_names=class_names
)

test_filepaths = resolve_prediction_filepaths(
    test_pred_df,
    all_images_df,
    split_label="test",
    class_names=class_names
)

overlap = val_filepaths.intersection(test_filepaths)
if len(overlap) > 0:
    raise ValueError(f"Validation and test file overlap detected: {len(overlap)} files")

protocol_df = all_images_df.copy()
protocol_df["split"] = "train"
protocol_df.loc[protocol_df["filepath"].isin(val_filepaths), "split"] = "val"
protocol_df.loc[protocol_df["filepath"].isin(test_filepaths), "split"] = "test"

split_counts = protocol_df["split"].value_counts().to_dict()
expected_counts = {"train": 5740, "val": 2249, "test": 1900}

print("\n=== Reconstructed Protocol A Split ===")
print(split_counts)

for split_name, expected_n in expected_counts.items():
    actual_n = int(split_counts.get(split_name, 0))
    if actual_n != expected_n:
        raise ValueError(
            f"Unexpected {split_name} count: {actual_n}. Expected: {expected_n}."
        )

class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

manifest_path = CP3_OUTPUT_DIR / "CP3_A_seed42_reconstructed_manifest.csv"
protocol_df.to_csv(manifest_path, index=False)

print(f"Dataset folder : {dataset_dir}")
print(f"Train          : {split_counts['train']}")
print(f"Validation     : {split_counts['val']}")
print(f"Test           : {split_counts['test']}")
print(f"Manifest saved : {manifest_path}")

In [ ]:
# ==========================================================
# CP3-C — Dataset, Dataloaders, Metrics, Scheduler, Utilities
# ==========================================================

class TEMVirusArchitectureDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_df = protocol_df[protocol_df["split"] == "train"].copy()
val_df = protocol_df[protocol_df["split"] == "val"].copy()
test_df = protocol_df[protocol_df["split"] == "test"].copy()

train_dataset = TEMVirusArchitectureDataset(train_df, class_to_idx, transform=train_transform)
val_dataset = TEMVirusArchitectureDataset(val_df, class_to_idx, transform=eval_transform)
test_dataset = TEMVirusArchitectureDataset(test_df, class_to_idx, transform=eval_transform)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=train_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


class WarmupCosineScheduler:
    def __init__(
        self,
        optimizer,
        total_epochs=30,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6
    ):
        self.optimizer = optimizer
        self.total_epochs = total_epochs
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.last_epoch = -1

    def get_lr(self, epoch):
        if epoch < self.warmup_epochs:
            progress = epoch / max(1, self.warmup_epochs - 1)
            return self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

        cosine_epochs = max(1, self.total_epochs - self.warmup_epochs)
        progress = (epoch - self.warmup_epochs) / max(1, cosine_epochs - 1)
        progress = min(max(progress, 0.0), 1.0)

        cosine_value = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.min_lr + cosine_value * (self.base_lr - self.min_lr)

    def step(self, epoch):
        lr = self.get_lr(epoch)

        for group in self.optimizer.param_groups:
            group["lr"] = lr

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "total_epochs": self.total_epochs,
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "last_epoch": self.last_epoch
        }

    def load_state_dict(self, state_dict):
        self.total_epochs = state_dict["total_epochs"]
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.last_epoch = state_dict["last_epoch"]


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


def compute_basic_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }


def train_one_epoch_arch(model, loader, criterion, optimizer, scaler, device, epoch_display, arch_name):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    last_grad_norm = 0.0

    progress_bar = tqdm(loader, desc=f"{arch_name} train epoch {epoch_display}", leave=True)

    for images, labels, _paths in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        if torch.isnan(loss):
            raise ValueError(f"NaN loss detected at epoch {epoch_display}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = torch_nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_MAX_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        last_grad_norm = float(grad_norm)

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "grad_norm": f"{last_grad_norm:.3f}"
        })

    avg_loss = running_loss / len(loader.dataset)
    metrics = compute_basic_metrics(all_targets, all_preds)
    metrics["loss"] = float(avg_loss)
    metrics["grad_norm_last"] = float(last_grad_norm)

    return metrics


@torch.no_grad()
def evaluate_arch(model, loader, criterion, device, class_names, split_name="validation", arch_name="model"):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_paths = []
    all_probs = []

    progress_bar = tqdm(loader, desc=f"{arch_name} evaluate {split_name}", leave=True)

    for images, labels, paths in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        probs = torch.softmax(logits.float(), dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_paths.extend(list(paths))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()

    avg_loss = running_loss / len(loader.dataset)
    base_metrics = compute_basic_metrics(all_targets, all_preds)
    base_metrics["loss"] = float(avg_loss)

    per_class_f1_values = f1_score(
        all_targets,
        all_preds,
        average=None,
        zero_division=0
    )

    per_class_f1 = {
        class_names[i]: float(per_class_f1_values[i])
        for i in range(len(class_names))
    }

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_txt = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        zero_division=0
    )

    cm = confusion_matrix(
        all_targets,
        all_preds,
        labels=list(range(len(class_names)))
    )

    pred_df = pd.DataFrame({
        "filename": [Path(p).name for p in all_paths],
        "image_path": all_paths,
        "true_label": [class_names[i] for i in all_targets],
        "pred_label": [class_names[i] for i in all_preds],
        "confidence": all_probs.max(axis=1)
    })

    for i, class_name in enumerate(class_names):
        safe_class_name = class_name.replace(" ", "_")
        pred_df[f"prob_{safe_class_name}"] = all_probs[:, i]

    return {
        "metrics": base_metrics,
        "per_class_f1": per_class_f1,
        "classification_report": report_dict,
        "classification_report_txt": report_txt,
        "confusion_matrix": cm,
        "prediction_df": pred_df,
        "targets": all_targets,
        "preds": all_preds,
        "paths": all_paths
    }


print(f"Train images : {len(train_dataset)}")
print(f"Val images   : {len(val_dataset)}")
print(f"Test images  : {len(test_dataset)}")
print(f"Class count  : {len(class_names)}")

In [ ]:
# ==========================================================
# CP3-D — Run Architecture Sweep
# EfficientNetV2-S and ConvNeXt-Tiny
# ==========================================================

def make_model(arch_key):
    timm_name = ARCHITECTURES[arch_key]["timm_name"]

    model = timm.create_model(
        timm_name,
        pretrained=True,
        num_classes=len(class_names)
    )

    return model


def get_arch_paths(arch_key):
    prefix = f"{arch_key}_{PROTOCOL}_seed{SEED}"

    return {
        "prefix": prefix,
        "best_ckpt": CP3_CHECKPOINT_DIR / f"{prefix}_best.pt",
        "last_ckpt": CP3_CHECKPOINT_DIR / f"{prefix}_last.pt",
        "history_csv": CP3_OUTPUT_DIR / f"{prefix}_training_history.csv",
        "history_json": CP3_OUTPUT_DIR / f"{prefix}_training_history.json",
        "training_summary_json": CP3_OUTPUT_DIR / f"{prefix}_training_summary.json",
        "final_summary_json": CP3_OUTPUT_DIR / f"{prefix}_CP3_final_summary.json",
        "learning_curve_png": CP3_OUTPUT_DIR / f"{prefix}_learning_curve.png",
        "confusion_matrix_test_png": CP3_OUTPUT_DIR / f"{prefix}_confusion_matrix_test.png",
        "metrics_summary_json": CP3_OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json",
        "metrics_summary_csv": CP3_OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.csv",
        "per_class_f1_csv": CP3_OUTPUT_DIR / f"{prefix}_test_per_class_f1.csv"
    }


def save_arch_history(history, paths):
    history_df = pd.DataFrame(history)
    history_df.to_csv(paths["history_csv"], index=False)

    with open(paths["history_json"], "w") as f:
        json.dump(history, f, indent=2)

    return history_df


def save_arch_last_checkpoint(
    model,
    optimizer,
    scheduler,
    epoch,
    best_metric,
    best_epoch,
    early_stop_counter,
    history,
    paths,
    arch_key,
    stopped_by_epoch5_gate=False,
    early_stopped=False
):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "best_metric": best_metric,
            "best_epoch": best_epoch,
            "early_stop_counter": early_stop_counter,
            "history": history,
            "architecture": arch_key,
            "timm_name": ARCHITECTURES[arch_key]["timm_name"],
            "seed": SEED,
            "protocol": PROTOCOL,
            "stopped_by_epoch5_gate": stopped_by_epoch5_gate,
            "early_stopped": early_stopped,
            "config": {
                "epochs": MAX_EPOCHS,
                "batch_size": BATCH_SIZE,
                "image_size": IMG_SIZE,
                "optimizer": "AdamW",
                "base_lr": BASE_LR,
                "warmup_start_lr": WARMUP_START_LR,
                "min_lr": MIN_LR,
                "weight_decay": WEIGHT_DECAY,
                "label_smoothing": LABEL_SMOOTHING,
                "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
                "early_stop_patience": EARLY_STOP_PATIENCE,
                "monitor": "val_macro_f1",
                "pretrained": True,
                "normalization": "ImageNet"
            }
        },
        paths["last_ckpt"]
    )


def save_arch_best_model(model, optimizer, scheduler, epoch, val_macro_f1, val_metrics, paths, arch_key):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "val_macro_f1": val_macro_f1,
            "val_metrics": val_metrics,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "architecture": arch_key,
            "timm_name": ARCHITECTURES[arch_key]["timm_name"],
            "seed": SEED,
            "protocol": PROTOCOL,
            "config": {
                "epochs": MAX_EPOCHS,
                "batch_size": BATCH_SIZE,
                "image_size": IMG_SIZE,
                "optimizer": "AdamW",
                "base_lr": BASE_LR,
                "warmup_start_lr": WARMUP_START_LR,
                "min_lr": MIN_LR,
                "weight_decay": WEIGHT_DECAY,
                "label_smoothing": LABEL_SMOOTHING,
                "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
                "pretrained": True,
                "normalization": "ImageNet"
            }
        },
        paths["best_ckpt"]
    )


def load_arch_last_checkpoint(model, optimizer, scheduler, paths):
    if RESUME_TRAINING and paths["last_ckpt"].exists():
        ckpt = torch.load(paths["last_ckpt"], map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_metric = float(ckpt.get("best_metric", -1.0))
        best_epoch = ckpt.get("best_epoch", None)
        early_stop_counter = int(ckpt.get("early_stop_counter", 0))
        history = ckpt.get("history", [])

        print("=== Resume Checkpoint Found ===")
        print(f"Resuming from epoch : {start_epoch + 1}")
        print(f"Best val macro F1   : {best_metric:.4f}")
        print(f"Best epoch          : {None if best_epoch is None else best_epoch + 1}")

        return start_epoch, best_metric, best_epoch, early_stop_counter, history

    print("=== Fresh Architecture Run ===")
    return 0, -1.0, None, 0, []


def save_split_evaluation_outputs_arch(arch_key, split_name, eval_outputs):
    prefix = CP3_OUTPUT_DIR / f"{arch_key}_{PROTOCOL}_seed{SEED}_{split_name}"

    report_df = pd.DataFrame(eval_outputs["classification_report"]).transpose()
    report_csv_path = Path(str(prefix) + "_classification_report.csv")
    report_json_path = Path(str(prefix) + "_classification_report.json")
    report_txt_path = Path(str(prefix) + "_classification_report.txt")

    report_df.to_csv(report_csv_path)

    with open(report_json_path, "w") as f:
        json.dump(eval_outputs["classification_report"], f, indent=2)

    with open(report_txt_path, "w") as f:
        f.write(eval_outputs["classification_report_txt"])

    cm_df = pd.DataFrame(
        eval_outputs["confusion_matrix"],
        index=class_names,
        columns=class_names
    )

    cm_csv_path = Path(str(prefix) + "_confusion_matrix.csv")
    cm_df.to_csv(cm_csv_path)

    pred_csv_path = Path(str(prefix) + "_predictions.csv")
    eval_outputs["prediction_df"].to_csv(pred_csv_path, index=False)

    return {
        "classification_report_csv": str(report_csv_path),
        "classification_report_json": str(report_json_path),
        "classification_report_txt": str(report_txt_path),
        "confusion_matrix_csv": str(cm_csv_path),
        "predictions_csv": str(pred_csv_path)
    }


def plot_arch_outputs(arch_key, paths, test_outputs):
    history_df = pd.read_csv(paths["history_csv"])

    # Learning curve
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    axes[0, 1].plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
    axes[1, 0].set_title("Validation Macro F1")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Macro F1")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(history_df["epoch"], history_df["lr"], label="learning_rate")
    axes[1, 1].set_title("Learning Rate Schedule")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Learning Rate")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f"{arch_key} — Training Curves", y=1.02)
    plt.tight_layout()
    plt.savefig(paths["learning_curve_png"], dpi=300, bbox_inches="tight")
    plt.show()

    # Confusion matrix
    test_cm = test_outputs["confusion_matrix"]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(test_cm)

    ax.set_title(f"{arch_key} — Test Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(test_cm.shape[0]):
        for j in range(test_cm.shape[1]):
            ax.text(j, i, str(test_cm[i, j]), ha="center", va="center", fontsize=8)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(paths["confusion_matrix_test_png"], dpi=300, bbox_inches="tight")
    plt.show()


def run_single_architecture(arch_key):
    set_global_seed(SEED)

    paths = get_arch_paths(arch_key)
    display_name = ARCHITECTURES[arch_key]["display_name"]
    timm_name = ARCHITECTURES[arch_key]["timm_name"]

    print("\n" + "=" * 80)
    print(f"Running architecture: {display_name}")
    print(f"timm model          : {timm_name}")
    print("=" * 80)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = make_model(arch_key).to(DEVICE)

    criterion = torch_nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = WarmupCosineScheduler(
        optimizer=optimizer,
        total_epochs=MAX_EPOCHS,
        warmup_epochs=5,
        warmup_start_lr=WARMUP_START_LR,
        base_lr=BASE_LR,
        min_lr=MIN_LR
    )

    try:
        scaler = torch.amp.GradScaler(
            device="cuda",
            enabled=(DEVICE.type == "cuda")
        )
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(
            enabled=(DEVICE.type == "cuda")
        )

    start_epoch, best_metric, best_epoch, early_stop_counter, history = load_arch_last_checkpoint(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        paths=paths
    )

    training_start_time = time.time()
    stopped_by_epoch5_gate = False
    early_stopped = False
    anomaly_log = []

    params_count = count_trainable_parameters(model)
    model_size_mb = estimate_model_size_mb(model)

    print(f"Trainable parameters : {params_count:,}")
    print(f"Model size           : {model_size_mb:.2f} MB")

    for epoch in range(start_epoch, MAX_EPOCHS):
        epoch_display = epoch + 1
        epoch_start_time = time.time()

        current_lr = scheduler.step(epoch)

        train_metrics = train_one_epoch_arch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
            epoch_display=epoch_display,
            arch_name=arch_key
        )

        val_outputs = evaluate_arch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=DEVICE,
            class_names=class_names,
            split_name="validation",
            arch_name=arch_key
        )

        val_metrics = val_outputs["metrics"]
        val_macro_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]

        epoch_time_sec = time.time() - epoch_start_time

        improved = val_macro_f1 > best_metric + 1e-8

        if improved:
            best_metric = val_macro_f1
            best_epoch = epoch
            early_stop_counter = 0

            save_arch_best_model(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                val_macro_f1=val_macro_f1,
                val_metrics=val_metrics,
                paths=paths,
                arch_key=arch_key
            )
        else:
            early_stop_counter += 1

        row = {
            "architecture": arch_key,
            "epoch": epoch_display,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_grad_norm_last": train_metrics["grad_norm_last"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "lr": current_lr,
            "epoch_time_sec": epoch_time_sec,
            "best_val_macro_f1_so_far": best_metric,
            "best_epoch_so_far": None if best_epoch is None else best_epoch + 1,
            "early_stop_counter": early_stop_counter,
            "improved": improved
        }

        history.append(row)
        save_arch_history(history, paths)

        save_arch_last_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            best_metric=best_metric,
            best_epoch=best_epoch,
            early_stop_counter=early_stop_counter,
            history=history,
            paths=paths,
            arch_key=arch_key,
            stopped_by_epoch5_gate=stopped_by_epoch5_gate,
            early_stopped=early_stopped
        )

        print("\n=== Epoch Summary ===")
        print(json.dumps(row, indent=2))

        # Epoch 5 gate
        if epoch_display == EARLY_GATE_EPOCH_5:
            if val_acc < 0.60:
                stopped_by_epoch5_gate = True
                anomaly_log.append(
                    f"Epoch 5 gate failed: val_acc={val_acc:.4f}"
                )

                save_arch_last_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch,
                    best_metric=best_metric,
                    best_epoch=best_epoch,
                    early_stop_counter=early_stop_counter,
                    history=history,
                    paths=paths,
                    arch_key=arch_key,
                    stopped_by_epoch5_gate=stopped_by_epoch5_gate,
                    early_stopped=early_stopped
                )

                print("\n=== Epoch 5 Gate Stop ===")
                print(f"Validation accuracy at epoch 5: {val_acc:.4f}")
                break

            elif val_acc < 0.80:
                msg = f"Epoch 5 gate warning: val_acc={val_acc:.4f}"
                anomaly_log.append(msg)
                print("\n=== Epoch 5 Gate Warning ===")
                print(msg)
            else:
                print("\n=== Epoch 5 Gate Passed ===")
                print(f"Validation accuracy at epoch 5: {val_acc:.4f}")

        # Epoch 15 gate
        if epoch_display == EARLY_GATE_EPOCH_15:
            if val_acc >= 0.90:
                print("\n=== Epoch 15 Gate: On Track ===")
                print(f"Validation accuracy at epoch 15: {val_acc:.4f}")
            elif val_acc >= 0.85:
                msg = f"Epoch 15 marginal: val_acc={val_acc:.4f}"
                anomaly_log.append(msg)
                print("\n=== Epoch 15 Gate: Marginal ===")
                print(msg)
            else:
                msg = f"Epoch 15 below target: val_acc={val_acc:.4f}"
                anomaly_log.append(msg)
                print("\n=== Epoch 15 Gate: Below Target ===")
                print(msg)

        # Early stopping
        if early_stop_counter >= EARLY_STOP_PATIENCE:
            early_stopped = True

            save_arch_last_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                best_metric=best_metric,
                best_epoch=best_epoch,
                early_stop_counter=early_stop_counter,
                history=history,
                paths=paths,
                arch_key=arch_key,
                stopped_by_epoch5_gate=stopped_by_epoch5_gate,
                early_stopped=early_stopped
            )

            print("\n=== Early Stopping Triggered ===")
            print(f"No val_macro_f1 improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

    total_training_time_sec = time.time() - training_start_time

    peak_gpu_memory_mb = None
    if DEVICE.type == "cuda":
        peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    training_summary = {
        "architecture": arch_key,
        "display_name": display_name,
        "timm_name": timm_name,
        "protocol": PROTOCOL,
        "seed": SEED,
        "max_epochs": MAX_EPOCHS,
        "total_epochs_run": len(history),
        "best_epoch": None if best_epoch is None else best_epoch + 1,
        "best_val_macro_f1": None if best_metric < 0 else best_metric,
        "total_training_time_sec": total_training_time_sec,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "final_lr": optimizer.param_groups[0]["lr"],
        "early_stopped": early_stopped,
        "stopped_by_epoch5_gate": stopped_by_epoch5_gate,
        "anomaly_log": anomaly_log,
        "params_count": params_count,
        "params_count_M": params_count / 1e6,
        "model_size_mb": model_size_mb,
        "best_checkpoint": str(paths["best_ckpt"]),
        "last_checkpoint": str(paths["last_ckpt"]),
        "history_csv": str(paths["history_csv"]),
        "history_json": str(paths["history_json"])
    }

    with open(paths["training_summary_json"], "w") as f:
        json.dump(training_summary, f, indent=2)

    # Evaluation using best checkpoint
    if not paths["best_ckpt"].exists():
        print(f"No best checkpoint found for {arch_key}; skipping final evaluation.")
        return training_summary

    best_ckpt = torch.load(paths["best_ckpt"], map_location=DEVICE)
    best_model = make_model(arch_key).to(DEVICE)
    best_model.load_state_dict(best_ckpt["model_state_dict"])

    criterion_eval = torch_nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    validation_outputs = evaluate_arch(
        model=best_model,
        loader=val_loader,
        criterion=criterion_eval,
        device=DEVICE,
        class_names=class_names,
        split_name="validation",
        arch_name=arch_key
    )

    test_outputs = evaluate_arch(
        model=best_model,
        loader=test_loader,
        criterion=criterion_eval,
        device=DEVICE,
        class_names=class_names,
        split_name="test",
        arch_name=arch_key
    )

    validation_files = save_split_evaluation_outputs_arch(arch_key, "validation", validation_outputs)
    test_files = save_split_evaluation_outputs_arch(arch_key, "test", test_outputs)

    metrics_summary = {
        "architecture": arch_key,
        "display_name": display_name,
        "timm_name": timm_name,
        "protocol": PROTOCOL,
        "seed": SEED,
        "best_epoch": int(best_ckpt["epoch"]) + 1,
        "best_val_macro_f1_checkpoint": float(best_ckpt["val_macro_f1"]),
        "validation": validation_outputs["metrics"],
        "test": test_outputs["metrics"],
        "test_per_class_f1": test_outputs["per_class_f1"],
        "validation_files": validation_files,
        "test_files": test_files
    }

    with open(paths["metrics_summary_json"], "w") as f:
        json.dump(metrics_summary, f, indent=2)

    metrics_rows = []
    for split_name, split_metrics in [
        ("validation", validation_outputs["metrics"]),
        ("test", test_outputs["metrics"])
    ]:
        row = {"split": split_name}
        row.update(split_metrics)
        metrics_rows.append(row)

    metrics_summary_df = pd.DataFrame(metrics_rows)
    metrics_summary_df.to_csv(paths["metrics_summary_csv"], index=False)

    per_class_f1_df = pd.DataFrame({
        "class_name": list(test_outputs["per_class_f1"].keys()),
        "test_f1": list(test_outputs["per_class_f1"].values())
    })
    per_class_f1_df.to_csv(paths["per_class_f1_csv"], index=False)

    plot_arch_outputs(arch_key, paths, test_outputs)

    final_summary = {
        "architecture": arch_key,
        "display_name": display_name,
        "timm_name": timm_name,
        "protocol": PROTOCOL,
        "seed": SEED,
        "best_epoch": metrics_summary["best_epoch"],
        "best_val_macro_f1": metrics_summary["best_val_macro_f1_checkpoint"],
        "test_accuracy": metrics_summary["test"]["accuracy"],
        "test_macro_f1": metrics_summary["test"]["macro_f1"],
        "test_weighted_f1": metrics_summary["test"]["weighted_f1"],
        "test_macro_precision": metrics_summary["test"]["macro_precision"],
        "test_macro_recall": metrics_summary["test"]["macro_recall"],
        "total_epochs_run": training_summary["total_epochs_run"],
        "total_training_time_sec": training_summary["total_training_time_sec"],
        "peak_gpu_memory_mb": training_summary["peak_gpu_memory_mb"],
        "params_count_M": training_summary["params_count_M"],
        "model_size_mb": training_summary["model_size_mb"],
        "early_stopped": training_summary["early_stopped"],
        "stopped_by_epoch5_gate": training_summary["stopped_by_epoch5_gate"],
        "anomaly_log": training_summary["anomaly_log"],
        "best_checkpoint": str(paths["best_ckpt"]),
        "last_checkpoint": str(paths["last_ckpt"]),
        "training_history_csv": str(paths["history_csv"]),
        "learning_curve_png": str(paths["learning_curve_png"]),
        "confusion_matrix_test_png": str(paths["confusion_matrix_test_png"]),
        "metrics_summary_json": str(paths["metrics_summary_json"]),
        "per_class_f1_csv": str(paths["per_class_f1_csv"])
    }

    with open(paths["final_summary_json"], "w") as f:
        json.dump(final_summary, f, indent=2)

    print("\n=== Architecture Final Summary ===")
    print(json.dumps(final_summary, indent=2))

    del model
    del best_model
    gc.collect()
    torch.cuda.empty_cache()

    return final_summary


cp3_results = []

for arch_key in ["EffNetV2S", "ConvNeXtT"]:
    result = run_single_architecture(arch_key)
    cp3_results.append(result)

In [ ]:
# ==========================================================
# CP3-E — Architecture Comparison Summary and Review Package
# ==========================================================

comparison_rows = []

for arch_key in ["EffNetV2S", "ConvNeXtT"]:
    paths = get_arch_paths(arch_key)

    if paths["final_summary_json"].exists():
        with open(paths["final_summary_json"], "r") as f:
            summary = json.load(f)

        row = {
            "architecture": arch_key,
            "display_name": summary.get("display_name"),
            "timm_name": summary.get("timm_name"),
            "best_epoch": summary.get("best_epoch"),
            "best_val_macro_f1": summary.get("best_val_macro_f1"),
            "test_accuracy": summary.get("test_accuracy"),
            "test_macro_f1": summary.get("test_macro_f1"),
            "test_weighted_f1": summary.get("test_weighted_f1"),
            "total_training_time_sec": summary.get("total_training_time_sec"),
            "peak_gpu_memory_mb": summary.get("peak_gpu_memory_mb"),
            "params_count_M": summary.get("params_count_M"),
            "model_size_mb": summary.get("model_size_mb"),
            "early_stopped": summary.get("early_stopped"),
            "stopped_by_epoch5_gate": summary.get("stopped_by_epoch5_gate"),
            "final_summary_json": str(paths["final_summary_json"])
        }
    else:
        row = {
            "architecture": arch_key,
            "display_name": ARCHITECTURES[arch_key]["display_name"],
            "timm_name": ARCHITECTURES[arch_key]["timm_name"],
            "best_epoch": None,
            "best_val_macro_f1": None,
            "test_accuracy": None,
            "test_macro_f1": None,
            "test_weighted_f1": None,
            "total_training_time_sec": None,
            "peak_gpu_memory_mb": None,
            "params_count_M": None,
            "model_size_mb": None,
            "early_stopped": None,
            "stopped_by_epoch5_gate": None,
            "final_summary_json": None
        }

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

comparison_path = CP3_OUTPUT_DIR / "CP3_architecture_comparison_summary.csv"
comparison_json_path = CP3_OUTPUT_DIR / "CP3_architecture_comparison_summary.json"

comparison_df.to_csv(comparison_path, index=False)

with open(comparison_json_path, "w") as f:
    json.dump(comparison_rows, f, indent=2)

print("=== CP3 Architecture Comparison ===")
display(comparison_df)

valid_df = comparison_df.dropna(subset=["test_accuracy"]).copy()

if len(valid_df) > 0:
    best_row = valid_df.sort_values("test_accuracy", ascending=False).iloc[0]
    best_arch = best_row["architecture"]

    print("\n=== Best Architecture by Test Accuracy ===")
    print(f"Architecture : {best_arch}")
    print(f"Test accuracy: {best_row['test_accuracy']:.4f}")
    print(f"Macro F1     : {best_row['test_macro_f1']:.4f}")
else:
    best_arch = None
    print("\nNo valid architecture result found.")

# ----------------------------------------------------------
# Package review files
# ----------------------------------------------------------
review_zip_path = CP3_OUTPUT_DIR / "CP3_architecture_sweep_review_files.zip"

review_files = [
    comparison_path,
    comparison_json_path
]

if best_arch is not None:
    best_paths = get_arch_paths(best_arch)

    review_files.extend([
        best_paths["history_csv"],
        best_paths["learning_curve_png"],
        best_paths["confusion_matrix_test_png"],
        best_paths["per_class_f1_csv"],
        best_paths["final_summary_json"]
    ])

with zipfile.ZipFile(review_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in review_files:
        file_path = Path(file_path)
        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
            print("Added:", file_path)
        else:
            print("Missing:", file_path)

print(f"Comparison CSV : {comparison_path}")
print(f"Comparison JSON: {comparison_json_path}")
print(f"Review ZIP     : {review_zip_path}")

In [ ]:
# ==========================================================
# CP3.5-0 — Configuration
# EfficientNetV2-S ceiling push experiments
# ==========================================================

import os
import gc
import json
import time
import math
import random
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import timm

from scipy.fft import dct, idct
from scipy.ndimage import generic_filter

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Required previous objects
# ----------------------------------------------------------
required_objects = ["protocol_df", "class_names", "class_to_idx"]

for obj_name in required_objects:
    if obj_name not in globals():
        raise RuntimeError(
            f"{obj_name} is not defined. Run CP3-A and CP3-B first."
        )

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cp3_5_ceiling_push"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_cp3_5_ceiling_push"
SIKDER_CACHE_DIR = PROJECT_ROOT / "preprocessed_sikder"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SIKDER_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# General experiment config
# ----------------------------------------------------------
SEED = 42
PROTOCOL = "A"

MODEL_NAME = "tf_efficientnetv2_s.in21k_ft_in1k"
MODEL_LABEL = "EffNetV2S"

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4

BASE_LR = 1e-4
WARMUP_START_LR = 1e-6
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
GRAD_CLIP_MAX_NORM = 1.0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

CP3_BASELINE_ACC = 0.9616
SIKDER_SOTA_ACC = 0.9744
SIKDER_SOTA_F1 = 0.9744

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------
# Experiment definitions
# ----------------------------------------------------------
EXPERIMENTS = {
    "sikderprep": {
        "suffix": "sikderprep",
        "title": "Sikder-style preprocessing",
        "max_epochs": 30,
        "early_stop_patience": 8,
        "label_smoothing": 0.1,
        "scheduler": "warmup_cosine",
        "use_sikder_cache": True,
        "use_mixup": False,
        "use_tta": False,
        "epoch5_min_val_acc": 0.70
    },
    "ttamixup": {
        "suffix": "ttamixup",
        "title": "TTA + Mixup + Warm Restarts",
        "max_epochs": 35,
        "early_stop_patience": 10,
        "label_smoothing": 0.1,
        "scheduler": "warmup_cosine_restarts",
        "use_sikder_cache": False,
        "use_mixup": True,
        "use_tta": True,
        "epoch5_min_val_acc": 0.80
    }
}

# ----------------------------------------------------------
# Determinism
# ----------------------------------------------------------
def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

# ----------------------------------------------------------
# Split dataframes
# ----------------------------------------------------------
train_df = protocol_df[protocol_df["split"] == "train"].copy()
val_df = protocol_df[protocol_df["split"] == "val"].copy()
test_df = protocol_df[protocol_df["split"] == "test"].copy()

print(f"Device      : {DEVICE}")
print(f"Train       : {len(train_df)}")
print(f"Validation  : {len(val_df)}")
print(f"Test        : {len(test_df)}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Checkpoint  : {CHECKPOINT_DIR}")
print(f"Cache dir   : {SIKDER_CACHE_DIR}")

In [ ]:
# ==========================================================
# CP3.5-1 — Dataset and Transform Builder
# ==========================================================

class TEMVirusDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None, use_sikder_cache=False):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.use_sikder_cache = use_sikder_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        original_path = Path(row["filepath"])
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        if self.use_sikder_cache:
            image_path = get_sikder_cache_path(original_path)
            if not image_path.exists():
                create_sikder_cache_image(original_path, image_path)
        else:
            image_path = original_path

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, str(image_path)


def build_transforms():
    train_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    eval_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    return train_transform, eval_transform


def build_dataloaders(use_sikder_cache=False):
    train_transform, eval_transform = build_transforms()

    train_dataset = TEMVirusDataset(
        train_df,
        class_to_idx,
        transform=train_transform,
        use_sikder_cache=use_sikder_cache
    )

    val_dataset = TEMVirusDataset(
        val_df,
        class_to_idx,
        transform=eval_transform,
        use_sikder_cache=use_sikder_cache
    )

    test_dataset = TEMVirusDataset(
        test_df,
        class_to_idx,
        transform=eval_transform,
        use_sikder_cache=use_sikder_cache
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader



In [ ]:
# ==========================================================
# CP3.5-2 — Sikder-style Preprocessing Cache
# ==========================================================

def local_std_dev_filter(image, window_size=5):
    def std_func(values):
        return np.std(values)

    return generic_filter(
        image.astype(np.float32),
        std_func,
        size=window_size
    )


def dct2d_denoise(image, threshold_ratio=0.05):
    dct_coef = dct(
        dct(image.astype(np.float32), axis=0, norm="ortho"),
        axis=1,
        norm="ortho"
    )

    threshold = np.max(np.abs(dct_coef)) * threshold_ratio
    dct_coef[np.abs(dct_coef) < threshold] = 0

    return idct(
        idct(dct_coef, axis=0, norm="ortho"),
        axis=1,
        norm="ortho"
    )


def sikder_preprocessing(image):
    filtered = local_std_dev_filter(image, window_size=5)
    denoised = dct2d_denoise(filtered, threshold_ratio=0.05)

    normalized = (
        (denoised - denoised.min()) /
        (denoised.max() - denoised.min() + 1e-8) * 255
    )

    return normalized.astype(np.uint8)


def get_sikder_cache_path(original_path):
    original_path = Path(original_path)

    split_name = original_path.parent.parent.name
    class_name = original_path.parent.name

    cache_dir = SIKDER_CACHE_DIR / split_name / class_name
    cache_dir.mkdir(parents=True, exist_ok=True)

    return cache_dir / f"{original_path.stem}.png"


def create_sikder_cache_image(original_path, cache_path):
    image = Image.open(original_path).convert("L")
    image_array = np.array(image)

    processed = sikder_preprocessing(image_array)
    Image.fromarray(processed).save(cache_path)


def build_sikder_cache(df, visual_check=True):
    created = 0
    skipped = 0
    samples = []

    for idx, row in tqdm(
        df.reset_index(drop=True).iterrows(),
        total=len(df),
        desc="Building Sikder cache"
    ):
        original_path = Path(row["filepath"])
        cache_path = get_sikder_cache_path(original_path)

        if cache_path.exists():
            skipped += 1
            continue

        image = Image.open(original_path).convert("L")
        original_array = np.array(image)

        processed_array = sikder_preprocessing(original_array)
        Image.fromarray(processed_array).save(cache_path)

        created += 1

        if visual_check and len(samples) < 3:
            samples.append((original_path.name, original_array, processed_array))

    print("=== Sikder Cache Summary ===")
    print(f"Created : {created}")
    print(f"Skipped : {skipped}")
    print(f"Total   : {len(df)}")

    if visual_check and len(samples) > 0:
        fig, axes = plt.subplots(len(samples), 2, figsize=(8, 4 * len(samples)))

        if len(samples) == 1:
            axes = np.array([axes])

        for i, (filename, original, processed) in enumerate(samples):
            axes[i, 0].imshow(original, cmap="gray")
            axes[i, 0].set_title(f"Original: {filename}")
            axes[i, 0].axis("off")

            axes[i, 1].imshow(processed, cmap="gray")
            axes[i, 1].set_title("Sikder-style preprocessing")
            axes[i, 1].axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
# ==========================================================
# CP3.5-3 — Training, Evaluation, and Output Utilities
# ==========================================================

def make_model():
    model = timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=len(class_names)
    )
    return model


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }


class WarmupCosineScheduler:
    def __init__(
        self,
        optimizer,
        total_epochs,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6
    ):
        self.optimizer = optimizer
        self.total_epochs = total_epochs
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.last_epoch = -1

    def get_lr(self, epoch):
        if epoch < self.warmup_epochs:
            progress = epoch / max(1, self.warmup_epochs - 1)
            return self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

        cosine_epochs = max(1, self.total_epochs - self.warmup_epochs)
        progress = (epoch - self.warmup_epochs) / max(1, cosine_epochs - 1)
        progress = min(max(progress, 0.0), 1.0)

        cosine_value = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.min_lr + cosine_value * (self.base_lr - self.min_lr)

    def step(self, epoch):
        lr = self.get_lr(epoch)

        for group in self.optimizer.param_groups:
            group["lr"] = lr

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "total_epochs": self.total_epochs,
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "last_epoch": self.last_epoch
        }

    def load_state_dict(self, state_dict):
        self.total_epochs = state_dict["total_epochs"]
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.last_epoch = state_dict["last_epoch"]


class WarmupCosineRestartScheduler:
    def __init__(
        self,
        optimizer,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6,
        t_0=10,
        t_mult=2
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.t_0 = t_0
        self.t_mult = t_mult
        self.last_epoch = -1

        self.restart_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=t_0,
            T_mult=t_mult,
            eta_min=min_lr
        )

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            progress = epoch / max(1, self.warmup_epochs - 1)
            lr = self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

            for group in self.optimizer.param_groups:
                group["lr"] = lr
        else:
            self.restart_scheduler.step(epoch - self.warmup_epochs)
            lr = self.optimizer.param_groups[0]["lr"]

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "t_0": self.t_0,
            "t_mult": self.t_mult,
            "last_epoch": self.last_epoch,
            "restart_scheduler": self.restart_scheduler.state_dict()
        }

    def load_state_dict(self, state_dict):
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.t_0 = state_dict["t_0"]
        self.t_mult = state_dict["t_mult"]
        self.last_epoch = state_dict["last_epoch"]
        self.restart_scheduler.load_state_dict(state_dict["restart_scheduler"])


def build_scheduler(optimizer, config):
    if config["scheduler"] == "warmup_cosine":
        return WarmupCosineScheduler(
            optimizer=optimizer,
            total_epochs=config["max_epochs"],
            warmup_epochs=5,
            warmup_start_lr=WARMUP_START_LR,
            base_lr=BASE_LR,
            min_lr=MIN_LR
        )

    if config["scheduler"] == "warmup_cosine_restarts":
        return WarmupCosineRestartScheduler(
            optimizer=optimizer,
            warmup_epochs=5,
            warmup_start_lr=WARMUP_START_LR,
            base_lr=BASE_LR,
            min_lr=MIN_LR,
            t_0=10,
            t_mult=2
        )

    raise ValueError(f"Unknown scheduler: {config['scheduler']}")

In [ ]:
# ==========================================================
# Mixup and TTA
# ==========================================================

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


def tta_predict(model, images):
    augmentations = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[2, 3])
    ]

    probs_list = []

    for aug in augmentations:
        aug_images = aug(images)
        logits = model(aug_images)
        probs = torch.softmax(logits.float(), dim=1)
        probs_list.append(probs)

    return torch.mean(torch.stack(probs_list, dim=0), dim=0)


def train_one_epoch(model, loader, criterion, optimizer, scaler, config, epoch):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    last_grad_norm = 0.0
    mixup_batches = 0

    progress = tqdm(loader, desc=f"{config['suffix']} train epoch {epoch}", leave=True)

    for images, labels, _paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        apply_mixup = config["use_mixup"] and (np.random.random() < 0.5)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            if apply_mixup:
                mixed_images, y_a, y_b, lam = mixup_data(images, labels, alpha=0.2)
                logits = model(mixed_images)
                loss = mixup_loss(criterion, logits, y_a, y_b, lam)
                mixup_batches += 1
            else:
                logits = model(images)
                loss = criterion(logits, labels)

        if torch.isnan(loss):
            raise ValueError(f"NaN loss detected at epoch {epoch}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_MAX_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        last_grad_norm = float(grad_norm)

        progress.set_postfix({
            "loss": f"{loss.item():.4f}",
            "grad": f"{last_grad_norm:.3f}",
            "mixup": mixup_batches
        })

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))
    metrics["grad_norm_last"] = float(last_grad_norm)
    metrics["mixup_batches"] = int(mixup_batches)

    return metrics


@torch.no_grad()
def evaluate_model(model, loader, criterion, config, split_name):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_paths = []
    all_probs = []

    progress = tqdm(loader, desc=f"{config['suffix']} evaluate {split_name}", leave=True)

    for images, labels, paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            if config["use_tta"]:
                probs = tta_predict(model, images)
                logits_for_loss = model(images)
                loss = criterion(logits_for_loss, labels)
            else:
                logits = model(images)
                loss = criterion(logits, labels)
                probs = torch.softmax(logits.float(), dim=1)

        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_paths.extend(list(paths))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))

    per_class_f1_values = f1_score(
        all_targets,
        all_preds,
        average=None,
        zero_division=0
    )

    per_class_f1 = {
        class_names[i]: float(per_class_f1_values[i])
        for i in range(len(class_names))
    }

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_txt = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        zero_division=0
    )

    cm = confusion_matrix(
        all_targets,
        all_preds,
        labels=list(range(len(class_names)))
    )

    pred_df = pd.DataFrame({
        "filename": [Path(p).name for p in all_paths],
        "image_path": all_paths,
        "true_label": [class_names[i] for i in all_targets],
        "pred_label": [class_names[i] for i in all_preds],
        "confidence": all_probs.max(axis=1)
    })

    for i, class_name in enumerate(class_names):
        safe_name = class_name.replace(" ", "_")
        pred_df[f"prob_{safe_name}"] = all_probs[:, i]

    return {
        "metrics": metrics,
        "per_class_f1": per_class_f1,
        "classification_report": report_dict,
        "classification_report_txt": report_txt,
        "confusion_matrix": cm,
        "prediction_df": pred_df
    }

In [ ]:
# ==========================================================
# CP3.5-4 — Checkpoint, Saving, Plotting, and Experiment Runner
# ==========================================================

def get_paths(suffix):
    prefix = f"{MODEL_LABEL}_{PROTOCOL}_seed{SEED}_{suffix}"

    return {
        "prefix": prefix,
        "best_ckpt": CHECKPOINT_DIR / f"{prefix}_best.pt",
        "last_ckpt": CHECKPOINT_DIR / f"{prefix}_last.pt",
        "history_csv": OUTPUT_DIR / f"{prefix}_training_history.csv",
        "history_json": OUTPUT_DIR / f"{prefix}_training_history.json",
        "training_summary_json": OUTPUT_DIR / f"{prefix}_training_summary.json",
        "final_summary_json": OUTPUT_DIR / f"{prefix}_CP3_5_final_summary.json",
        "metrics_json": OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json",
        "metrics_csv": OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.csv",
        "per_class_f1_csv": OUTPUT_DIR / f"{prefix}_test_per_class_f1.csv",
        "learning_curve_png": OUTPUT_DIR / f"{prefix}_learning_curve.png",
        "confusion_matrix_png": OUTPUT_DIR / f"{prefix}_confusion_matrix_test.png"
    }


def save_history(history, paths):
    history_df = pd.DataFrame(history)
    history_df.to_csv(paths["history_csv"], index=False)

    with open(paths["history_json"], "w") as f:
        json.dump(history, f, indent=2)

    return history_df


def save_last_checkpoint(model, optimizer, scheduler, epoch, best_metric, best_epoch, patience_counter, history, paths, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_metric": best_metric,
            "best_epoch": best_epoch,
            "patience_counter": patience_counter,
            "history": history,
            "config": config
        },
        paths["last_ckpt"]
    )


def save_best_checkpoint(model, optimizer, scheduler, epoch, val_macro_f1, val_metrics, paths, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_macro_f1": val_macro_f1,
            "val_metrics": val_metrics,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "config": config
        },
        paths["best_ckpt"]
    )


def load_last_checkpoint(model, optimizer, scheduler, paths):
    if paths["last_ckpt"].exists():
        ckpt = torch.load(paths["last_ckpt"], map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_metric = float(ckpt["best_metric"])
        best_epoch = ckpt["best_epoch"]
        patience_counter = int(ckpt["patience_counter"])
        history = ckpt["history"]

        print("=== Resume checkpoint found ===")
        print(f"Start epoch       : {start_epoch + 1}")
        print(f"Best val macro F1 : {best_metric:.4f}")
        print(f"Best epoch        : {None if best_epoch is None else best_epoch + 1}")

        return start_epoch, best_metric, best_epoch, patience_counter, history

    return 0, -1.0, None, 0, []


def save_split_outputs(suffix, split_name, outputs):
    prefix = OUTPUT_DIR / f"{MODEL_LABEL}_{PROTOCOL}_seed{SEED}_{suffix}_{split_name}"

    report_csv = Path(str(prefix) + "_classification_report.csv")
    report_json = Path(str(prefix) + "_classification_report.json")
    report_txt = Path(str(prefix) + "_classification_report.txt")
    cm_csv = Path(str(prefix) + "_confusion_matrix.csv")
    pred_csv = Path(str(prefix) + "_predictions.csv")

    pd.DataFrame(outputs["classification_report"]).transpose().to_csv(report_csv)

    with open(report_json, "w") as f:
        json.dump(outputs["classification_report"], f, indent=2)

    with open(report_txt, "w") as f:
        f.write(outputs["classification_report_txt"])

    cm_df = pd.DataFrame(
        outputs["confusion_matrix"],
        index=class_names,
        columns=class_names
    )
    cm_df.to_csv(cm_csv)

    outputs["prediction_df"].to_csv(pred_csv, index=False)

    return {
        "classification_report_csv": str(report_csv),
        "classification_report_json": str(report_json),
        "classification_report_txt": str(report_txt),
        "confusion_matrix_csv": str(cm_csv),
        "predictions_csv": str(pred_csv)
    }


def plot_outputs(paths, test_outputs, title):
    history_df = pd.read_csv(paths["history_csv"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    axes[0, 1].plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
    axes[1, 0].set_title("Validation Macro F1")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(history_df["epoch"], history_df["lr"], label="lr")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.savefig(paths["learning_curve_png"], dpi=300, bbox_inches="tight")
    plt.show()

    cm = test_outputs["confusion_matrix"]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm)

    ax.set_title(f"{title} — Test Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(paths["confusion_matrix_png"], dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ==========================================================
# Main experiment runner
# ==========================================================

def run_experiment(config):
    set_seed(SEED)

    suffix = config["suffix"]
    paths = get_paths(suffix)

    if config["use_sikder_cache"]:
        build_sikder_cache(protocol_df, visual_check=True)

    train_loader, val_loader, test_loader = build_dataloaders(
        use_sikder_cache=config["use_sikder_cache"]
    )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = make_model().to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])

    optimizer = optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = build_scheduler(optimizer, config)

    try:
        scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    start_epoch, best_metric, best_epoch, patience_counter, history = load_last_checkpoint(
        model,
        optimizer,
        scheduler,
        paths
    )

    params_count = count_trainable_parameters(model)
    model_size_mb = estimate_model_size_mb(model)

    stopped_by_gate = False
    early_stopped = False
    anomaly_log = []

    print("\n" + "=" * 80)
    print(f"Running experiment: {suffix}")
    print("=" * 80)
    print(f"Title          : {config['title']}")
    print(f"Epochs         : {config['max_epochs']}")
    print(f"Batch size     : {BATCH_SIZE}")
    print(f"Mixup          : {config['use_mixup']}")
    print(f"TTA eval       : {config['use_tta']}")
    print(f"Sikder cache   : {config['use_sikder_cache']}")
    print(f"Parameters     : {params_count:,}")
    print(f"Model size     : {model_size_mb:.2f} MB")

    start_time = time.time()

    for epoch in range(start_epoch, config["max_epochs"]):
        epoch_num = epoch + 1
        epoch_start = time.time()

        lr = scheduler.step(epoch)

        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            config,
            epoch_num
        )

        val_outputs = evaluate_model(
            model,
            val_loader,
            criterion,
            config,
            split_name="validation"
        )

        val_metrics = val_outputs["metrics"]
        val_macro_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]

        improved = val_macro_f1 > best_metric + 1e-8

        if improved:
            best_metric = val_macro_f1
            best_epoch = epoch
            patience_counter = 0

            save_best_checkpoint(
                model,
                optimizer,
                scheduler,
                epoch,
                val_macro_f1,
                val_metrics,
                paths,
                config
            )
        else:
            patience_counter += 1

        row = {
            "experiment": suffix,
            "epoch": epoch_num,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_grad_norm_last": train_metrics["grad_norm_last"],
            "train_mixup_batches": train_metrics.get("mixup_batches", 0),
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "lr": lr,
            "epoch_time_sec": time.time() - epoch_start,
            "best_val_macro_f1_so_far": best_metric,
            "best_epoch_so_far": None if best_epoch is None else best_epoch + 1,
            "patience_counter": patience_counter,
            "improved": improved
        }

        history.append(row)
        save_history(history, paths)

        save_last_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            best_metric,
            best_epoch,
            patience_counter,
            history,
            paths,
            config
        )

        print("\n=== Epoch Summary ===")
        print(json.dumps(row, indent=2))

        if epoch_num == 5 and val_acc < config["epoch5_min_val_acc"]:
            stopped_by_gate = True
            anomaly_log.append(f"Epoch 5 gate stopped: val_acc={val_acc:.4f}")
            print("\n=== Epoch 5 Gate Stop ===")
            break

        if patience_counter >= config["early_stop_patience"]:
            early_stopped = True
            print("\n=== Early Stopping Triggered ===")
            break

    training_time_sec = time.time() - start_time

    peak_gpu_memory_mb = None
    if DEVICE.type == "cuda":
        peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    training_summary = {
        "experiment": suffix,
        "model": MODEL_LABEL,
        "protocol": PROTOCOL,
        "seed": SEED,
        "total_epochs_run": len(history),
        "best_epoch": None if best_epoch is None else best_epoch + 1,
        "best_val_macro_f1": None if best_metric < 0 else best_metric,
        "total_training_time_sec": training_time_sec,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "early_stopped": early_stopped,
        "stopped_by_gate": stopped_by_gate,
        "anomaly_log": anomaly_log,
        "params_count_M": params_count / 1e6,
        "model_size_mb": model_size_mb,
        "config": config
    }

    with open(paths["training_summary_json"], "w") as f:
        json.dump(training_summary, f, indent=2)

    if not paths["best_ckpt"].exists():
        print(f"No best checkpoint found for {suffix}.")
        return training_summary

    best_ckpt = torch.load(paths["best_ckpt"], map_location=DEVICE)

    best_model = make_model().to(DEVICE)
    best_model.load_state_dict(best_ckpt["model_state_dict"])

    validation_outputs = evaluate_model(
        best_model,
        val_loader,
        criterion,
        config,
        split_name="validation"
    )

    test_outputs = evaluate_model(
        best_model,
        test_loader,
        criterion,
        config,
        split_name="test"
    )

    validation_files = save_split_outputs(suffix, "validation", validation_outputs)
    test_files = save_split_outputs(suffix, "test", test_outputs)

    metrics_summary = {
        "experiment": suffix,
        "best_epoch": int(best_ckpt["epoch"]) + 1,
        "best_val_macro_f1_checkpoint": float(best_ckpt["val_macro_f1"]),
        "validation": validation_outputs["metrics"],
        "test": test_outputs["metrics"],
        "test_per_class_f1": test_outputs["per_class_f1"],
        "validation_files": validation_files,
        "test_files": test_files
    }

    with open(paths["metrics_json"], "w") as f:
        json.dump(metrics_summary, f, indent=2)

    pd.DataFrame([
        {"split": "validation", **validation_outputs["metrics"]},
        {"split": "test", **test_outputs["metrics"]}
    ]).to_csv(paths["metrics_csv"], index=False)

    pd.DataFrame({
        "class_name": list(test_outputs["per_class_f1"].keys()),
        "test_f1": list(test_outputs["per_class_f1"].values())
    }).to_csv(paths["per_class_f1_csv"], index=False)

    plot_outputs(paths, test_outputs, config["title"])

    final_summary = {
        "experiment": suffix,
        "model": MODEL_LABEL,
        "protocol": PROTOCOL,
        "seed": SEED,
        "best_epoch": metrics_summary["best_epoch"],
        "best_val_macro_f1": metrics_summary["best_val_macro_f1_checkpoint"],
        "test_accuracy": metrics_summary["test"]["accuracy"],
        "test_macro_f1": metrics_summary["test"]["macro_f1"],
        "test_weighted_f1": metrics_summary["test"]["weighted_f1"],
        "test_macro_precision": metrics_summary["test"]["macro_precision"],
        "test_macro_recall": metrics_summary["test"]["macro_recall"],
        "improvement_vs_CP3": metrics_summary["test"]["accuracy"] - CP3_BASELINE_ACC,
        "beat_sikder_sota": (
            metrics_summary["test"]["accuracy"] >= SIKDER_SOTA_ACC and
            metrics_summary["test"]["macro_f1"] >= SIKDER_SOTA_F1
        ),
        "total_epochs_run": training_summary["total_epochs_run"],
        "total_training_time_sec": training_summary["total_training_time_sec"],
        "peak_gpu_memory_mb": training_summary["peak_gpu_memory_mb"],
        "early_stopped": training_summary["early_stopped"],
        "stopped_by_gate": training_summary["stopped_by_gate"],
        "anomaly_log": training_summary["anomaly_log"],
        "training_history_csv": str(paths["history_csv"]),
        "learning_curve_png": str(paths["learning_curve_png"]),
        "confusion_matrix_test_png": str(paths["confusion_matrix_png"]),
        "per_class_f1_csv": str(paths["per_class_f1_csv"]),
        "metrics_summary_json": str(paths["metrics_json"]),
        "final_summary_json": str(paths["final_summary_json"])
    }

    with open(paths["final_summary_json"], "w") as f:
        json.dump(final_summary, f, indent=2)

    print("\n=== Final Summary ===")
    print(json.dumps(final_summary, indent=2))

    del model
    del best_model
    gc.collect()
    torch.cuda.empty_cache()

    return final_summary

In [ ]:
# ==========================================================
# CP3.5-5 — Run Experiments and Generate Comparison Summary
# ==========================================================

cp3_5_results = []

for experiment_key in ["sikderprep", "ttamixup"]:
    result = run_experiment(EXPERIMENTS[experiment_key])
    cp3_5_results.append(result)

comparison_rows = []

for experiment_key in ["sikderprep", "ttamixup"]:
    paths = get_paths(experiment_key)

    if paths["final_summary_json"].exists():
        with open(paths["final_summary_json"], "r") as f:
            summary = json.load(f)

        comparison_rows.append({
            "experiment": experiment_key,
            "test_accuracy": summary.get("test_accuracy"),
            "test_macro_f1": summary.get("test_macro_f1"),
            "test_weighted_f1": summary.get("test_weighted_f1"),
            "best_val_macro_f1": summary.get("best_val_macro_f1"),
            "best_epoch": summary.get("best_epoch"),
            "total_epochs": summary.get("total_epochs_run"),
            "training_time_sec": summary.get("total_training_time_sec"),
            "peak_gpu_memory_mb": summary.get("peak_gpu_memory_mb"),
            "improvement_vs_CP3": summary.get("improvement_vs_CP3"),
            "beat_sikder_sota": summary.get("beat_sikder_sota"),
            "early_stopped": summary.get("early_stopped"),
            "stopped_by_gate": summary.get("stopped_by_gate"),
            "final_summary_json": str(paths["final_summary_json"])
        })
    else:
        comparison_rows.append({
            "experiment": experiment_key,
            "test_accuracy": None,
            "test_macro_f1": None,
            "test_weighted_f1": None,
            "best_val_macro_f1": None,
            "best_epoch": None,
            "total_epochs": None,
            "training_time_sec": None,
            "peak_gpu_memory_mb": None,
            "improvement_vs_CP3": None,
            "beat_sikder_sota": None,
            "early_stopped": None,
            "stopped_by_gate": None,
            "final_summary_json": None
        })

comparison_df = pd.DataFrame(comparison_rows)

comparison_csv = OUTPUT_DIR / "CP3_5_experiments_comparison.csv"
comparison_json = OUTPUT_DIR / "CP3_5_experiments_comparison.json"

comparison_df.to_csv(comparison_csv, index=False)

with open(comparison_json, "w") as f:
    json.dump(comparison_rows, f, indent=2)

print("=== CP3.5 Experiment Comparison ===")
display(comparison_df)

valid_df = comparison_df.dropna(subset=["test_accuracy"]).copy()

if len(valid_df) > 0:
    winning_row = valid_df.sort_values("test_accuracy", ascending=False).iloc[0]
    winning_experiment = winning_row["experiment"]

    print("\n=== Winning Experiment ===")
    print(f"Experiment  : {winning_experiment}")
    print(f"Accuracy    : {winning_row['test_accuracy']:.4f}")
    print(f"Macro F1    : {winning_row['test_macro_f1']:.4f}")
    print(f"Improvement : {winning_row['improvement_vs_CP3']:.4f}")
else:
    winning_experiment = None
    print("No valid experiment result found.")

review_zip = OUTPUT_DIR / "CP3_5_ceiling_push_review_files.zip"

review_files = [
    comparison_csv,
    comparison_json
]

if winning_experiment is not None:
    winner_paths = get_paths(winning_experiment)

    review_files.extend([
        winner_paths["history_csv"],
        winner_paths["learning_curve_png"],
        winner_paths["confusion_matrix_png"],
        winner_paths["per_class_f1_csv"],
        winner_paths["final_summary_json"]
    ])

    losing_experiment = "ttamixup" if winning_experiment == "sikderprep" else "sikderprep"
    losing_paths = get_paths(losing_experiment)

    if losing_paths["final_summary_json"].exists():
        review_files.append(losing_paths["final_summary_json"])

with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in review_files:
        file_path = Path(file_path)

        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
            print("Added:", file_path)
        else:
            print("Missing:", file_path)

print(f"Comparison CSV : {comparison_csv}")
print(f"Comparison JSON: {comparison_json}")
print(f"Review ZIP     : {review_zip}")

In [ ]:
# ==========================================================
# CP3.6-0 — Setup and Load Protocol A Split
# DenseNet201 + TTA + Mixup for ensemble compatibility
# ==========================================================

!pip -q install timm

import os
import gc
import json
import time
import math
import random
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import timm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")

CP3_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "architecture_sweep_phase3"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cp3_6_ensemble"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_cp3_6"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Config
# ----------------------------------------------------------
SEED = 42
PROTOCOL = "A"

MODEL_LABEL = "DenseNet201"
TIMM_MODEL_NAME = "densenet201"

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
MAX_EPOCHS = 35

BASE_LR = 1e-4
WARMUP_START_LR = 1e-6
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP_MAX_NORM = 1.0

MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.5

EARLY_STOP_PATIENCE = 10
EPOCH5_MIN_VAL_ACC = 0.80

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES_REFERENCE = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

# ----------------------------------------------------------
# Determinism
# ----------------------------------------------------------
def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

# ----------------------------------------------------------
# Load Protocol A manifest
# ----------------------------------------------------------
if "protocol_df" in globals():
    print("Using protocol_df from current notebook session.")
else:
    manifest_path = CP3_OUTPUT_DIR / "CP3_A_seed42_reconstructed_manifest.csv"

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Protocol A manifest not found: {manifest_path}\n"
            "Run CP3-A and CP3-B first to reconstruct the Protocol A split."
        )

    protocol_df = pd.read_csv(manifest_path)
    print(f"Loaded Protocol A manifest: {manifest_path}")

class_names = sorted(protocol_df["class_name"].unique().tolist())

if class_names != CLASS_NAMES_REFERENCE:
    raise ValueError(
        f"Class names mismatch.\nDetected: {class_names}\nExpected: {CLASS_NAMES_REFERENCE}"
    )

class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

train_df = protocol_df[protocol_df["split"] == "train"].copy()
val_df = protocol_df[protocol_df["split"] == "val"].copy()
test_df = protocol_df[protocol_df["split"] == "test"].copy()

print(f"Device      : {DEVICE}")
print(f"Train       : {len(train_df)}")
print(f"Validation  : {len(val_df)}")
print(f"Test        : {len(test_df)}")
print(f"Classes     : {len(class_names)}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Checkpoint  : {CHECKPOINT_DIR}")

In [ ]:
# ==========================================================
# CP3.6-1 — Dataset and Dataloaders
# ==========================================================

class TEMVirusDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_dataset = TEMVirusDataset(train_df, class_to_idx, transform=train_transform)
val_dataset = TEMVirusDataset(val_df, class_to_idx, transform=eval_transform)
test_dataset = TEMVirusDataset(test_df, class_to_idx, transform=eval_transform)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=train_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Batch size   : {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

In [ ]:
# ==========================================================
# CP3.6-2 — Training and Evaluation Utilities
# ==========================================================

def make_model():
    return timm.create_model(
        TIMM_MODEL_NAME,
        pretrained=True,
        num_classes=len(class_names)
    )


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }


class WarmupCosineRestartScheduler:
    def __init__(
        self,
        optimizer,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6,
        t_0=10,
        t_mult=2
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.t_0 = t_0
        self.t_mult = t_mult
        self.last_epoch = -1

        self.restart_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=t_0,
            T_mult=t_mult,
            eta_min=min_lr
        )

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            progress = epoch / max(1, self.warmup_epochs - 1)
            lr = self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

            for group in self.optimizer.param_groups:
                group["lr"] = lr
        else:
            self.restart_scheduler.step(epoch - self.warmup_epochs)
            lr = self.optimizer.param_groups[0]["lr"]

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "t_0": self.t_0,
            "t_mult": self.t_mult,
            "last_epoch": self.last_epoch,
            "restart_scheduler": self.restart_scheduler.state_dict()
        }

    def load_state_dict(self, state_dict):
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.t_0 = state_dict["t_0"]
        self.t_mult = state_dict["t_mult"]
        self.last_epoch = state_dict["last_epoch"]
        self.restart_scheduler.load_state_dict(state_dict["restart_scheduler"])


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


def tta_predict(model, images):
    augmentations = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[2, 3])
    ]

    probs_list = []

    for aug in augmentations:
        logits = model(aug(images))
        probs = torch.softmax(logits.float(), dim=1)
        probs_list.append(probs)

    return torch.mean(torch.stack(probs_list, dim=0), dim=0)


def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch_num):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    last_grad_norm = 0.0
    mixup_batches = 0

    progress = tqdm(loader, desc=f"DenseNet201 train epoch {epoch_num}", leave=True)

    for images, labels, _paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        apply_mixup = np.random.random() < MIXUP_PROB

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            if apply_mixup:
                mixed_images, y_a, y_b, lam = mixup_data(images, labels, alpha=MIXUP_ALPHA)
                logits = model(mixed_images)
                loss = mixup_loss(criterion, logits, y_a, y_b, lam)
                mixup_batches += 1
            else:
                logits = model(images)
                loss = criterion(logits, labels)

        if torch.isnan(loss):
            raise ValueError(f"NaN loss detected at epoch {epoch_num}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_MAX_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        last_grad_norm = float(grad_norm)

        progress.set_postfix({
            "loss": f"{loss.item():.4f}",
            "grad": f"{last_grad_norm:.3f}",
            "mixup": mixup_batches
        })

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))
    metrics["grad_norm_last"] = float(last_grad_norm)
    metrics["mixup_batches"] = int(mixup_batches)

    return metrics


@torch.no_grad()
def evaluate_with_tta(model, loader, criterion, split_name):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_paths = []
    all_probs = []

    progress = tqdm(loader, desc=f"DenseNet201 evaluate {split_name} with TTA", leave=True)

    for images, labels, paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            probs = tta_predict(model, images)
            logits_for_loss = model(images)
            loss = criterion(logits_for_loss, labels)

        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_paths.extend(list(paths))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))

    per_class_f1_values = f1_score(
        all_targets,
        all_preds,
        average=None,
        zero_division=0
    )

    per_class_f1 = {
        class_names[i]: float(per_class_f1_values[i])
        for i in range(len(class_names))
    }

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_txt = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        zero_division=0
    )

    cm = confusion_matrix(
        all_targets,
        all_preds,
        labels=list(range(len(class_names)))
    )

    pred_df = pd.DataFrame({
        "filename": [Path(p).name for p in all_paths],
        "image_path": all_paths,
        "true_label": [class_names[i] for i in all_targets],
        "pred_label": [class_names[i] for i in all_preds],
        "confidence": all_probs.max(axis=1)
    })

    for i, class_name in enumerate(class_names):
        safe_name = class_name.replace(" ", "_")
        pred_df[f"prob_{safe_name}"] = all_probs[:, i]

    return {
        "metrics": metrics,
        "per_class_f1": per_class_f1,
        "classification_report": report_dict,
        "classification_report_txt": report_txt,
        "confusion_matrix": cm,
        "prediction_df": pred_df,
        "softmax": all_probs,
        "targets": all_targets,
        "preds": all_preds,
        "paths": all_paths
    }

In [ ]:
# ==========================================================
# CP3.6-3 — Checkpoint, Saving, and Plotting
# ==========================================================

PREFIX = "DenseNet201_A_seed42_ttamixup"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{PREFIX}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{PREFIX}_last.pt"

HISTORY_CSV = OUTPUT_DIR / f"{PREFIX}_training_history.csv"
HISTORY_JSON = OUTPUT_DIR / f"{PREFIX}_training_history.json"
TRAINING_SUMMARY_JSON = OUTPUT_DIR / f"{PREFIX}_training_summary.json"
FINAL_SUMMARY_JSON = OUTPUT_DIR / f"{PREFIX}_CP3_6_final_summary.json"

METRICS_JSON = OUTPUT_DIR / f"{PREFIX}_evaluation_metrics_summary.json"
METRICS_CSV = OUTPUT_DIR / f"{PREFIX}_evaluation_metrics_summary.csv"
PER_CLASS_F1_CSV = OUTPUT_DIR / f"{PREFIX}_test_per_class_f1.csv"

VAL_SOFTMAX_NPY = OUTPUT_DIR / f"{PREFIX}_val_softmax.npy"
TEST_SOFTMAX_NPY = OUTPUT_DIR / f"{PREFIX}_test_softmax.npy"

LEARNING_CURVE_PNG = OUTPUT_DIR / f"{PREFIX}_learning_curve.png"
CONFUSION_MATRIX_PNG = OUTPUT_DIR / f"{PREFIX}_confusion_matrix_test.png"


def save_history(history):
    history_df = pd.DataFrame(history)
    history_df.to_csv(HISTORY_CSV, index=False)

    with open(HISTORY_JSON, "w") as f:
        json.dump(history, f, indent=2)

    return history_df


def save_last_checkpoint(model, optimizer, scheduler, epoch, best_metric, best_epoch, patience_counter, history, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_metric": best_metric,
            "best_epoch": best_epoch,
            "patience_counter": patience_counter,
            "history": history,
            "config": config
        },
        LAST_CKPT_PATH
    )


def save_best_checkpoint(model, optimizer, scheduler, epoch, val_macro_f1, val_metrics, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_macro_f1": val_macro_f1,
            "val_metrics": val_metrics,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "config": config
        },
        BEST_CKPT_PATH
    )


def load_last_checkpoint(model, optimizer, scheduler):
    if LAST_CKPT_PATH.exists():
        ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_metric = float(ckpt["best_metric"])
        best_epoch = ckpt["best_epoch"]
        patience_counter = int(ckpt["patience_counter"])
        history = ckpt["history"]

        print("=== Resume checkpoint found ===")
        print(f"Start epoch       : {start_epoch + 1}")
        print(f"Best val macro F1 : {best_metric:.4f}")
        print(f"Best epoch        : {None if best_epoch is None else best_epoch + 1}")

        return start_epoch, best_metric, best_epoch, patience_counter, history

    return 0, -1.0, None, 0, []


def save_split_outputs(split_name, outputs):
    prefix = OUTPUT_DIR / f"{PREFIX}_{split_name}"

    report_csv = Path(str(prefix) + "_classification_report.csv")
    report_json = Path(str(prefix) + "_classification_report.json")
    report_txt = Path(str(prefix) + "_classification_report.txt")
    cm_csv = Path(str(prefix) + "_confusion_matrix.csv")
    pred_csv = Path(str(prefix) + "_predictions.csv")

    pd.DataFrame(outputs["classification_report"]).transpose().to_csv(report_csv)

    with open(report_json, "w") as f:
        json.dump(outputs["classification_report"], f, indent=2)

    with open(report_txt, "w") as f:
        f.write(outputs["classification_report_txt"])

    cm_df = pd.DataFrame(
        outputs["confusion_matrix"],
        index=class_names,
        columns=class_names
    )
    cm_df.to_csv(cm_csv)

    outputs["prediction_df"].to_csv(pred_csv, index=False)

    return {
        "classification_report_csv": str(report_csv),
        "classification_report_json": str(report_json),
        "classification_report_txt": str(report_txt),
        "confusion_matrix_csv": str(cm_csv),
        "predictions_csv": str(pred_csv)
    }


def plot_outputs(test_outputs):
    history_df = pd.read_csv(HISTORY_CSV)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    axes[0, 1].plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
    axes[1, 0].set_title("Validation Macro F1")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(history_df["epoch"], history_df["lr"], label="lr")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle("DenseNet201 + TTA + Mixup", y=1.02)
    plt.tight_layout()
    plt.savefig(LEARNING_CURVE_PNG, dpi=300, bbox_inches="tight")
    plt.show()

    cm = test_outputs["confusion_matrix"]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm)

    ax.set_title("DenseNet201 + TTA + Mixup — Test Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(CONFUSION_MATRIX_PNG, dpi=300, bbox_inches="tight")
    plt.show()


print(f"Best checkpoint : {BEST_CKPT_PATH}")
print(f"Last checkpoint : {LAST_CKPT_PATH}")

In [ ]:
# ==========================================================
# CP3.6-4 — Train DenseNet201 + TTA + Mixup
# ==========================================================

set_seed(SEED)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model = make_model().to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler = WarmupCosineRestartScheduler(
    optimizer=optimizer,
    warmup_epochs=5,
    warmup_start_lr=WARMUP_START_LR,
    base_lr=BASE_LR,
    min_lr=MIN_LR,
    t_0=10,
    t_mult=2
)

try:
    scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

config = {
    "model": MODEL_LABEL,
    "timm_model_name": TIMM_MODEL_NAME,
    "protocol": PROTOCOL,
    "seed": SEED,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "image_size": IMG_SIZE,
    "optimizer": "AdamW",
    "base_lr": BASE_LR,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "mixup_alpha": MIXUP_ALPHA,
    "mixup_probability": MIXUP_PROB,
    "scheduler": "warmup_cosine_restarts",
    "tta": "original_hflip_vflip_both",
    "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
    "normalization": "ImageNet",
    "pretrained": True
}

start_epoch, best_metric, best_epoch, patience_counter, history = load_last_checkpoint(
    model,
    optimizer,
    scheduler
)

params_count = count_trainable_parameters(model)
model_size_mb = estimate_model_size_mb(model)

stopped_by_gate = False
early_stopped = False
anomaly_log = []

print("=== DenseNet201 TTA+Mixup Training ===")
print(f"Device       : {DEVICE}")
print(f"Epochs       : {MAX_EPOCHS}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Parameters   : {params_count:,}")
print(f"Model size   : {model_size_mb:.2f} MB")

training_start = time.time()

for epoch in range(start_epoch, MAX_EPOCHS):
    epoch_num = epoch + 1
    epoch_start = time.time()

    lr = scheduler.step(epoch)

    train_metrics = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
        epoch_num
    )

    val_outputs = evaluate_with_tta(
        model,
        val_loader,
        criterion,
        split_name="validation"
    )

    val_metrics = val_outputs["metrics"]
    val_macro_f1 = val_metrics["macro_f1"]
    val_acc = val_metrics["accuracy"]

    improved = val_macro_f1 > best_metric + 1e-8

    if improved:
        best_metric = val_macro_f1
        best_epoch = epoch
        patience_counter = 0

        save_best_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            val_macro_f1,
            val_metrics,
            config
        )
    else:
        patience_counter += 1

    row = {
        "epoch": epoch_num,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "train_weighted_f1": train_metrics["weighted_f1"],
        "train_grad_norm_last": train_metrics["grad_norm_last"],
        "train_mixup_batches": train_metrics["mixup_batches"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_weighted_f1": val_metrics["weighted_f1"],
        "val_macro_precision": val_metrics["macro_precision"],
        "val_macro_recall": val_metrics["macro_recall"],
        "lr": lr,
        "epoch_time_sec": time.time() - epoch_start,
        "best_val_macro_f1_so_far": best_metric,
        "best_epoch_so_far": None if best_epoch is None else best_epoch + 1,
        "patience_counter": patience_counter,
        "improved": improved
    }

    history.append(row)
    save_history(history)

    save_last_checkpoint(
        model,
        optimizer,
        scheduler,
        epoch,
        best_metric,
        best_epoch,
        patience_counter,
        history,
        config
    )

    print("\n=== Epoch Summary ===")
    print(json.dumps(row, indent=2))

    if epoch_num == 5 and val_acc < EPOCH5_MIN_VAL_ACC:
        stopped_by_gate = True
        anomaly_log.append(f"Epoch 5 gate stopped: val_acc={val_acc:.4f}")
        print("\n=== Epoch 5 Gate Stop ===")
        break

    if patience_counter >= EARLY_STOP_PATIENCE:
        early_stopped = True
        print("\n=== Early Stopping Triggered ===")
        break

training_time_sec = time.time() - training_start

peak_gpu_memory_mb = None
if DEVICE.type == "cuda":
    peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

training_summary = {
    "model": MODEL_LABEL,
    "protocol": PROTOCOL,
    "seed": SEED,
    "total_epochs_run": len(history),
    "best_epoch": None if best_epoch is None else best_epoch + 1,
    "best_val_macro_f1": None if best_metric < 0 else best_metric,
    "total_training_time_sec": training_time_sec,
    "peak_gpu_memory_mb": peak_gpu_memory_mb,
    "early_stopped": early_stopped,
    "stopped_by_gate": stopped_by_gate,
    "anomaly_log": anomaly_log,
    "params_count_M": params_count / 1e6,
    "model_size_mb": model_size_mb,
    "best_checkpoint": str(BEST_CKPT_PATH),
    "last_checkpoint": str(LAST_CKPT_PATH),
    "history_csv": str(HISTORY_CSV),
    "config": config
}

with open(TRAINING_SUMMARY_JSON, "w") as f:
    json.dump(training_summary, f, indent=2)

print(json.dumps(training_summary, indent=2))

In [ ]:
# ==========================================================
# CP3.6-5 — Evaluate Best Model and Save Softmax Predictions
# ==========================================================

if not BEST_CKPT_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_CKPT_PATH}")

best_ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)

best_model = make_model().to(DEVICE)
best_model.load_state_dict(best_ckpt["model_state_dict"])

criterion_eval = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

validation_outputs = evaluate_with_tta(
    best_model,
    val_loader,
    criterion_eval,
    split_name="validation"
)

test_outputs = evaluate_with_tta(
    best_model,
    test_loader,
    criterion_eval,
    split_name="test"
)

# Critical ensemble files
np.save(VAL_SOFTMAX_NPY, validation_outputs["softmax"])
np.save(TEST_SOFTMAX_NPY, test_outputs["softmax"])

validation_files = save_split_outputs("validation", validation_outputs)
test_files = save_split_outputs("test", test_outputs)

metrics_summary = {
    "model": MODEL_LABEL,
    "protocol": PROTOCOL,
    "seed": SEED,
    "best_epoch": int(best_ckpt["epoch"]) + 1,
    "best_val_macro_f1_checkpoint": float(best_ckpt["val_macro_f1"]),
    "validation": validation_outputs["metrics"],
    "test": test_outputs["metrics"],
    "test_per_class_f1": test_outputs["per_class_f1"],
    "validation_files": validation_files,
    "test_files": test_files,
    "val_softmax_npy": str(VAL_SOFTMAX_NPY),
    "test_softmax_npy": str(TEST_SOFTMAX_NPY)
}

with open(METRICS_JSON, "w") as f:
    json.dump(metrics_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **validation_outputs["metrics"]},
    {"split": "test", **test_outputs["metrics"]}
]).to_csv(METRICS_CSV, index=False)

per_class_f1_df = pd.DataFrame({
    "class_name": list(test_outputs["per_class_f1"].keys()),
    "test_f1": list(test_outputs["per_class_f1"].values())
})
per_class_f1_df.to_csv(PER_CLASS_F1_CSV, index=False)

plot_outputs(test_outputs)

# Validate softmax shape
val_softmax_check = np.load(VAL_SOFTMAX_NPY)
test_softmax_check = np.load(TEST_SOFTMAX_NPY)

final_summary = {
    "model": MODEL_LABEL,
    "protocol": PROTOCOL,
    "seed": SEED,
    "best_epoch": metrics_summary["best_epoch"],
    "best_val_macro_f1": metrics_summary["best_val_macro_f1_checkpoint"],
    "test_accuracy": metrics_summary["test"]["accuracy"],
    "test_macro_f1": metrics_summary["test"]["macro_f1"],
    "test_weighted_f1": metrics_summary["test"]["weighted_f1"],
    "test_macro_precision": metrics_summary["test"]["macro_precision"],
    "test_macro_recall": metrics_summary["test"]["macro_recall"],
    "total_epochs_run": training_summary["total_epochs_run"],
    "total_training_time_sec": training_summary["total_training_time_sec"],
    "peak_gpu_memory_mb": training_summary["peak_gpu_memory_mb"],
    "early_stopped": training_summary["early_stopped"],
    "stopped_by_gate": training_summary["stopped_by_gate"],
    "anomaly_log": training_summary["anomaly_log"],
    "val_softmax_shape": list(val_softmax_check.shape),
    "test_softmax_shape": list(test_softmax_check.shape),
    "val_softmax_saved": VAL_SOFTMAX_NPY.exists(),
    "test_softmax_saved": TEST_SOFTMAX_NPY.exists(),
    "val_softmax_npy": str(VAL_SOFTMAX_NPY),
    "test_softmax_npy": str(TEST_SOFTMAX_NPY),
    "training_history_csv": str(HISTORY_CSV),
    "learning_curve_png": str(LEARNING_CURVE_PNG),
    "confusion_matrix_test_png": str(CONFUSION_MATRIX_PNG),
    "per_class_f1_csv": str(PER_CLASS_F1_CSV),
    "metrics_summary_json": str(METRICS_JSON),
    "final_summary_json": str(FINAL_SUMMARY_JSON),
    "best_checkpoint": str(BEST_CKPT_PATH),
    "last_checkpoint": str(LAST_CKPT_PATH)
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(final_summary, f, indent=2)

print("=== CP3.6 Final Summary ===")
print(json.dumps(final_summary, indent=2))

print("\n=== Softmax Validation ===")
print(f"VAL softmax shape : {val_softmax_check.shape}")
print(f"TEST softmax shape: {test_softmax_check.shape}")
print(f"VAL softmax path  : {VAL_SOFTMAX_NPY}")
print(f"TEST softmax path : {TEST_SOFTMAX_NPY}")

In [ ]:
# ==========================================================
# CP3.6-6 — Package Review Files
# ==========================================================

review_files = [
    HISTORY_CSV,
    LEARNING_CURVE_PNG,
    CONFUSION_MATRIX_PNG,
    PER_CLASS_F1_CSV,
    FINAL_SUMMARY_JSON,
    METRICS_JSON,
    VAL_SOFTMAX_NPY,
    TEST_SOFTMAX_NPY
]

review_zip = OUTPUT_DIR / "DenseNet201_A_seed42_ttamixup_CP3_6_review_files.zip"

with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in review_files:
        file_path = Path(file_path)

        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
            print("Added:", file_path)
        else:
            print("Missing:", file_path)

print(f"Review ZIP: {review_zip}")

In [ ]:
# ==========================================================
# CP3.6-7 — Load / Generate Softmax Predictions
# Ensemble: EffNetV2S TTA+Mixup + DenseNet201 TTA+Mixup
# ==========================================================

!pip -q install timm

import os
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import timm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cp3_6_ensemble"
CP35_DIR = PROJECT_ROOT / "outputs" / "cp3_5_ceiling_push"

CP35_CKPT_DIR = PROJECT_ROOT / "checkpoints_cp3_5_ceiling_push"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Files
# ----------------------------------------------------------
EFFNET_VAL_SOFTMAX = CP35_DIR / "EffNetV2S_A_seed42_ttamixup_val_softmax.npy"
EFFNET_TEST_SOFTMAX = CP35_DIR / "EffNetV2S_A_seed42_ttamixup_test_softmax.npy"

DENSE_VAL_SOFTMAX = OUTPUT_DIR / "DenseNet201_A_seed42_ttamixup_val_softmax.npy"
DENSE_TEST_SOFTMAX = OUTPUT_DIR / "DenseNet201_A_seed42_ttamixup_test_softmax.npy"

EFFNET_VAL_PRED_CSV = CP35_DIR / "EffNetV2S_A_seed42_ttamixup_validation_predictions.csv"
EFFNET_TEST_PRED_CSV = CP35_DIR / "EffNetV2S_A_seed42_ttamixup_test_predictions.csv"

DENSE_VAL_PRED_CSV = OUTPUT_DIR / "DenseNet201_A_seed42_ttamixup_validation_predictions.csv"
DENSE_TEST_PRED_CSV = OUTPUT_DIR / "DenseNet201_A_seed42_ttamixup_test_predictions.csv"

EFFNET_BEST_CKPT = CP35_CKPT_DIR / "EffNetV2S_A_seed42_ttamixup_best.pt"

# ----------------------------------------------------------
# Constants
# ----------------------------------------------------------
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 4
IMG_SIZE = 224

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"Device     : {DEVICE}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"CP3.5 dir  : {CP35_DIR}")

In [ ]:
# ==========================================================
# Helper functions for labels and softmax regeneration
# ==========================================================

def label_to_index(value):
    if pd.isna(value):
        raise ValueError("Missing label value detected.")

    if isinstance(value, str):
        value = value.strip()
        if value in CLASS_TO_IDX:
            return CLASS_TO_IDX[value]

        try:
            idx = int(float(value))
            return idx
        except:
            raise ValueError(f"Unknown label: {value}")

    return int(value)


def get_existing_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    raise ValueError(
        f"None of these columns were found: {candidates}. "
        f"Available columns: {list(df.columns)}"
    )


class PredictionCSVDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform

        self.path_col = get_existing_column(
            self.df,
            ["image_path", "filepath", "file_path", "path"]
        )
        self.label_col = get_existing_column(
            self.df,
            ["true_label", "target_class", "label", "target"]
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row[self.path_col]
        label = label_to_index(row[self.label_col])

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


def make_effnet_model():
    model = timm.create_model(
        "tf_efficientnetv2_s.in21k_ft_in1k",
        pretrained=False,
        num_classes=len(CLASS_NAMES)
    )
    return model


def tta_predict(model, images):
    augmentations = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[2, 3])
    ]

    probs_list = []

    for aug in augmentations:
        logits = model(aug(images))
        probs = torch.softmax(logits.float(), dim=1)
        probs_list.append(probs)

    return torch.mean(torch.stack(probs_list, dim=0), dim=0)


@torch.no_grad()
def generate_softmax_from_checkpoint(csv_path, checkpoint_path, output_npy_path, split_name):
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"EffNetV2S checkpoint not found: {checkpoint_path}")

    transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    dataset = PredictionCSVDataset(csv_path, transform=transform)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

    model = make_effnet_model().to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    all_probs = []

    for images, labels, paths in tqdm(loader, desc=f"Generate EffNet softmax {split_name}"):
        images = images.to(DEVICE, non_blocking=True)

        probs = tta_predict(model, images)
        all_probs.append(probs.detach().cpu())

    softmax = torch.cat(all_probs, dim=0).numpy()
    np.save(output_npy_path, softmax)

    print(f"Saved {split_name} softmax: {output_npy_path}")
    print(f"Shape: {softmax.shape}")

    del model
    torch.cuda.empty_cache()

    return softmax


# ----------------------------------------------------------
# Generate EffNet softmax if missing
# ----------------------------------------------------------
if not EFFNET_VAL_SOFTMAX.exists():
    print("EffNet validation softmax not found. Generating from best checkpoint.")
    generate_softmax_from_checkpoint(
        EFFNET_VAL_PRED_CSV,
        EFFNET_BEST_CKPT,
        EFFNET_VAL_SOFTMAX,
        split_name="validation"
    )
else:
    print(f"EffNet validation softmax found: {EFFNET_VAL_SOFTMAX}")

if not EFFNET_TEST_SOFTMAX.exists():
    print("EffNet test softmax not found. Generating from best checkpoint.")
    generate_softmax_from_checkpoint(
        EFFNET_TEST_PRED_CSV,
        EFFNET_BEST_CKPT,
        EFFNET_TEST_SOFTMAX,
        split_name="test"
    )
else:
    print(f"EffNet test softmax found: {EFFNET_TEST_SOFTMAX}")

In [ ]:
# ==========================================================
# CP3.6-8 — Ensemble Strategies and Final Test Evaluation
# ==========================================================

# ----------------------------------------------------------
# Load softmax arrays
# ----------------------------------------------------------
effnet_val_softmax = np.load(EFFNET_VAL_SOFTMAX)
effnet_test_softmax = np.load(EFFNET_TEST_SOFTMAX)

dense_val_softmax = np.load(DENSE_VAL_SOFTMAX)
dense_test_softmax = np.load(DENSE_TEST_SOFTMAX)

assert effnet_val_softmax.shape == (2249, 14), f"EffNet val shape: {effnet_val_softmax.shape}"
assert effnet_test_softmax.shape == (1900, 14), f"EffNet test shape: {effnet_test_softmax.shape}"
assert dense_val_softmax.shape == (2249, 14), f"Dense val shape: {dense_val_softmax.shape}"
assert dense_test_softmax.shape == (1900, 14), f"Dense test shape: {dense_test_softmax.shape}"

# ----------------------------------------------------------
# Load labels and CSV predictions
# ----------------------------------------------------------
effnet_val_df = pd.read_csv(EFFNET_VAL_PRED_CSV)
effnet_test_df = pd.read_csv(EFFNET_TEST_PRED_CSV)

dense_val_df = pd.read_csv(DENSE_VAL_PRED_CSV)
dense_test_df = pd.read_csv(DENSE_TEST_PRED_CSV)

true_col_eff_val = get_existing_column(effnet_val_df, ["true_label", "target_class", "label", "target"])
true_col_eff_test = get_existing_column(effnet_test_df, ["true_label", "target_class", "label", "target"])

pred_col_eff_val = get_existing_column(effnet_val_df, ["pred_label", "prediction_class", "pred", "prediction"])
pred_col_eff_test = get_existing_column(effnet_test_df, ["pred_label", "prediction_class", "pred", "prediction"])

pred_col_dense_val = get_existing_column(dense_val_df, ["pred_label", "prediction_class", "pred", "prediction"])
pred_col_dense_test = get_existing_column(dense_test_df, ["pred_label", "prediction_class", "pred", "prediction"])

y_val = np.array([label_to_index(x) for x in effnet_val_df[true_col_eff_val].values])
y_test = np.array([label_to_index(x) for x in effnet_test_df[true_col_eff_test].values])

effnet_val_csv_pred = np.array([label_to_index(x) for x in effnet_val_df[pred_col_eff_val].values])
effnet_test_csv_pred = np.array([label_to_index(x) for x in effnet_test_df[pred_col_eff_test].values])

dense_val_csv_pred = np.array([label_to_index(x) for x in dense_val_df[pred_col_dense_val].values])
dense_test_csv_pred = np.array([label_to_index(x) for x in dense_test_df[pred_col_dense_test].values])

# ----------------------------------------------------------
# Sanity checks
# ----------------------------------------------------------
effnet_val_match = (effnet_val_softmax.argmax(axis=1) == effnet_val_csv_pred).mean()
effnet_test_match = (effnet_test_softmax.argmax(axis=1) == effnet_test_csv_pred).mean()

dense_val_match = (dense_val_softmax.argmax(axis=1) == dense_val_csv_pred).mean()
dense_test_match = (dense_test_softmax.argmax(axis=1) == dense_test_csv_pred).mean()

print("=== Softmax / CSV Prediction Alignment Check ===")
print(f"EffNet val match  : {effnet_val_match:.4f}")
print(f"EffNet test match : {effnet_test_match:.4f}")
print(f"Dense val match   : {dense_val_match:.4f}")
print(f"Dense test match  : {dense_test_match:.4f}")

if effnet_val_match < 0.95 or effnet_test_match < 0.95:
    raise ValueError("EffNet softmax order mismatch detected. Stop and debug before ensemble.")

if dense_val_match < 0.95 or dense_test_match < 0.95:
    raise ValueError("DenseNet softmax order mismatch detected. Stop and debug before ensemble.")

print("\nAlignment check passed.")

In [ ]:
# ==========================================================
# Ensemble functions
# ==========================================================

def evaluate_softmax(softmax, y_true, strategy_name):
    pred = softmax.argmax(axis=1)

    return {
        "strategy": strategy_name,
        "accuracy": float(accuracy_score(y_true, pred)),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, pred, average="macro", zero_division=0))
    }


def simple_average(a, b):
    return 0.5 * a + 0.5 * b


def weighted_by_val_f1(a, b, w_a, w_b):
    return w_a * a + w_b * b


def geometric_mean(a, b, eps=1e-10):
    out = np.exp((np.log(a + eps) + np.log(b + eps)) / 2.0)
    out = out / out.sum(axis=1, keepdims=True)
    return out


def max_confidence(a, b):
    a_conf = a.max(axis=1)
    b_conf = b.max(axis=1)
    use_a = a_conf > b_conf
    return np.where(use_a[:, None], a, b)


# ----------------------------------------------------------
# Weights from validation macro F1
# ----------------------------------------------------------
effnet_val_metrics = evaluate_softmax(effnet_val_softmax, y_val, "EffNetV2S_only")
dense_val_metrics = evaluate_softmax(dense_val_softmax, y_val, "DenseNet201_only")

w_effnet = effnet_val_metrics["macro_f1"] / (
    effnet_val_metrics["macro_f1"] + dense_val_metrics["macro_f1"]
)
w_dense = dense_val_metrics["macro_f1"] / (
    effnet_val_metrics["macro_f1"] + dense_val_metrics["macro_f1"]
)

print("=== Validation-derived weights ===")
print(f"EffNet weight : {w_effnet:.6f}")
print(f"Dense weight  : {w_dense:.6f}")

# ----------------------------------------------------------
# Validation strategies
# ----------------------------------------------------------
val_ensembles = {
    "EffNetV2S_only": effnet_val_softmax,
    "DenseNet201_only": dense_val_softmax,
    "simple_average": simple_average(effnet_val_softmax, dense_val_softmax),
    "weighted_by_valF1": weighted_by_val_f1(
        effnet_val_softmax,
        dense_val_softmax,
        w_effnet,
        w_dense
    ),
    "geometric_mean": geometric_mean(effnet_val_softmax, dense_val_softmax),
    "max_confidence": max_confidence(effnet_val_softmax, dense_val_softmax)
}

val_results = [
    evaluate_softmax(softmax, y_val, name)
    for name, softmax in val_ensembles.items()
]

val_df = pd.DataFrame(val_results)
val_strategy_path = OUTPUT_DIR / "CP3_6_ensemble_validation_strategies.csv"
val_df.to_csv(val_strategy_path, index=False)

print("=== Validation Strategy Results ===")
display(val_df)

best_strategy_name = val_df.loc[val_df["macro_f1"].idxmax(), "strategy"]

print(f"\nBest strategy by validation macro F1: {best_strategy_name}")

In [ ]:
# ==========================================================
# Apply best strategy to test set
# ==========================================================

test_ensembles = {
    "EffNetV2S_only": effnet_test_softmax,
    "DenseNet201_only": dense_test_softmax,
    "simple_average": simple_average(effnet_test_softmax, dense_test_softmax),
    "weighted_by_valF1": weighted_by_val_f1(
        effnet_test_softmax,
        dense_test_softmax,
        w_effnet,
        w_dense
    ),
    "geometric_mean": geometric_mean(effnet_test_softmax, dense_test_softmax),
    "max_confidence": max_confidence(effnet_test_softmax, dense_test_softmax)
}

ensemble_test = test_ensembles[best_strategy_name]
ensemble_val = val_ensembles[best_strategy_name]

ENSEMBLE_VAL_SOFTMAX = OUTPUT_DIR / "Ensemble2_A_seed42_val_softmax.npy"
ENSEMBLE_TEST_SOFTMAX = OUTPUT_DIR / "Ensemble2_A_seed42_test_softmax.npy"

np.save(ENSEMBLE_VAL_SOFTMAX, ensemble_val)
np.save(ENSEMBLE_TEST_SOFTMAX, ensemble_test)

test_pred = ensemble_test.argmax(axis=1)

test_acc = accuracy_score(y_test, test_pred)
test_macro_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test, test_pred, average="weighted", zero_division=0)
test_macro_precision = precision_score(y_test, test_pred, average="macro", zero_division=0)
test_macro_recall = recall_score(y_test, test_pred, average="macro", zero_division=0)

print("=" * 70)
print(f"FINAL ENSEMBLE TEST RESULTS — Strategy: {best_strategy_name}")
print("=" * 70)
print(f"Test Accuracy     : {test_acc:.6f}")
print(f"Test Macro F1     : {test_macro_f1:.6f}")
print(f"Test Weighted F1  : {test_weighted_f1:.6f}")
print(f"Test Macro Prec   : {test_macro_precision:.6f}")
print(f"Test Macro Recall : {test_macro_recall:.6f}")
print(f"Beat Sikder Acc   : {test_acc > 0.9744}")
print(f"Beat Sikder F1    : {test_macro_f1 > 0.9744}")
print(f"Hit 98% Accuracy  : {test_acc >= 0.98}")

In [ ]:
# ==========================================================
# Save final ensemble reports
# ==========================================================

effnet_test_pred = effnet_test_softmax.argmax(axis=1)
dense_test_pred = dense_test_softmax.argmax(axis=1)

per_class_f1_df = pd.DataFrame({
    "class_name": CLASS_NAMES,
    "ensemble_f1": f1_score(y_test, test_pred, average=None, zero_division=0),
    "effnet_only_f1": f1_score(y_test, effnet_test_pred, average=None, zero_division=0),
    "densenet_only_f1": f1_score(y_test, dense_test_pred, average=None, zero_division=0)
})

per_class_f1_df["ensemble_improvement_over_best_single"] = per_class_f1_df.apply(
    lambda r: r["ensemble_f1"] - max(r["effnet_only_f1"], r["densenet_only_f1"]),
    axis=1
)

per_class_path = OUTPUT_DIR / "Ensemble2_A_seed42_per_class_f1_comparison.csv"
per_class_f1_df.to_csv(per_class_path, index=False)

print("=== Per-class F1 Comparison ===")
display(per_class_f1_df)

cm = confusion_matrix(y_test, test_pred, labels=list(range(len(CLASS_NAMES))))
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)

cm_csv_path = OUTPUT_DIR / "Ensemble2_A_seed42_test_confusion_matrix.csv"
cm_df.to_csv(cm_csv_path)

report_dict = classification_report(
    y_test,
    test_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_txt = classification_report(
    y_test,
    test_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)

report_json_path = OUTPUT_DIR / "Ensemble2_A_seed42_test_classification_report.json"
report_txt_path = OUTPUT_DIR / "Ensemble2_A_seed42_test_classification_report.txt"

with open(report_json_path, "w") as f:
    json.dump(report_dict, f, indent=2)

with open(report_txt_path, "w") as f:
    f.write(report_txt)

# Predictions CSV
ensemble_pred_df = effnet_test_df.copy()
ensemble_pred_df["ensemble_pred_label"] = [CLASS_NAMES[i] for i in test_pred]
ensemble_pred_df["ensemble_confidence"] = ensemble_test.max(axis=1)

for i, class_name in enumerate(CLASS_NAMES):
    safe_name = class_name.replace(" ", "_")
    ensemble_pred_df[f"ensemble_prob_{safe_name}"] = ensemble_test[:, i]

ensemble_pred_csv = OUTPUT_DIR / "Ensemble2_A_seed42_test_predictions.csv"
ensemble_pred_df.to_csv(ensemble_pred_csv, index=False)

# Confusion matrix plot
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm)

ax.set_title(
    f"Ensemble2 Test Confusion Matrix\n"
    f"Acc={test_acc:.4f}, Macro F1={test_macro_f1:.4f}"
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

ax.set_xticks(np.arange(len(CLASS_NAMES)))
ax.set_yticks(np.arange(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax)
plt.tight_layout()

cm_png_path = OUTPUT_DIR / "Ensemble2_A_seed42_confusion_matrix_test.png"
plt.savefig(cm_png_path, dpi=300, bbox_inches="tight")
plt.show()

final_summary = {
    "ensemble_type": "2-model softmax ensemble: EfficientNetV2-S + DenseNet201",
    "protocol": "A",
    "seed": 42,
    "best_strategy_by_val_macro_f1": best_strategy_name,
    "weights": {
        "effnet_val_macro_f1": effnet_val_metrics["macro_f1"],
        "densenet_val_macro_f1": dense_val_metrics["macro_f1"],
        "w_effnet": float(w_effnet),
        "w_densenet": float(w_dense)
    },
    "test_accuracy": float(test_acc),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "test_macro_precision": float(test_macro_precision),
    "test_macro_recall": float(test_macro_recall),
    "beat_sikder_acc": bool(test_acc > 0.9744),
    "beat_sikder_f1": bool(test_macro_f1 > 0.9744),
    "hit_98_target_acc": bool(test_acc >= 0.98),
    "val_strategy_results": val_results,
    "softmax_files": {
        "effnet_val": str(EFFNET_VAL_SOFTMAX),
        "effnet_test": str(EFFNET_TEST_SOFTMAX),
        "dense_val": str(DENSE_VAL_SOFTMAX),
        "dense_test": str(DENSE_TEST_SOFTMAX),
        "ensemble_val": str(ENSEMBLE_VAL_SOFTMAX),
        "ensemble_test": str(ENSEMBLE_TEST_SOFTMAX)
    },
    "output_files": {
        "validation_strategies_csv": str(val_strategy_path),
        "per_class_f1_comparison_csv": str(per_class_path),
        "confusion_matrix_csv": str(cm_csv_path),
        "confusion_matrix_png": str(cm_png_path),
        "classification_report_json": str(report_json_path),
        "classification_report_txt": str(report_txt_path),
        "test_predictions_csv": str(ensemble_pred_csv)
    }
}

final_summary_path = OUTPUT_DIR / "Ensemble2_A_seed42_CP3_6_final_summary.json"

with open(final_summary_path, "w") as f:
    json.dump(final_summary, f, indent=2)

print("=== Step 2 Final Summary ===")
print(json.dumps(final_summary, indent=2))

In [ ]:
# ==========================================================
# CP3.6-9 — Package Ensemble Review Files
# ==========================================================

review_files = [
    OUTPUT_DIR / "Ensemble2_A_seed42_CP3_6_final_summary.json",
    OUTPUT_DIR / "CP3_6_ensemble_validation_strategies.csv",
    OUTPUT_DIR / "Ensemble2_A_seed42_per_class_f1_comparison.csv",
    OUTPUT_DIR / "Ensemble2_A_seed42_test_confusion_matrix.csv",
    OUTPUT_DIR / "Ensemble2_A_seed42_confusion_matrix_test.png",
    OUTPUT_DIR / "Ensemble2_A_seed42_test_classification_report.json",
    OUTPUT_DIR / "Ensemble2_A_seed42_test_classification_report.txt",
    OUTPUT_DIR / "Ensemble2_A_seed42_test_predictions.csv",
    OUTPUT_DIR / "Ensemble2_A_seed42_val_softmax.npy",
    OUTPUT_DIR / "Ensemble2_A_seed42_test_softmax.npy"
]

review_zip = OUTPUT_DIR / "Ensemble2_A_seed42_CP3_6_review_files.zip"

with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in review_files:
        file_path = Path(file_path)

        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
            print("Added:", file_path)
        else:
            print("Missing:", file_path)

print(f"Review ZIP: {review_zip}")

In [ ]:
# ==========================================================
# CP4-0 — Setup, Config, and Load Protocol A Manifest
# Multi-seed Protocol A: DenseNet201 and EfficientNetV2-S
# ==========================================================

!pip -q install timm

import os
import gc
import json
import time
import math
import random
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import timm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")

CP3_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "architecture_sweep_phase3"
CP35_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cp3_5_ceiling_push"
CP36_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cp3_6_ensemble"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cp4_multiseed"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_cp4"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# General config
# ----------------------------------------------------------
PROTOCOL = "A"
NEW_SEEDS = [123, 456, 789, 1024]
ALL_SEEDS = [42, 123, 456, 789, 1024]

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
MAX_EPOCHS = 35

BASE_LR = 1e-4
WARMUP_START_LR = 1e-6
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.5

GRAD_CLIP_MAX_NORM = 1.0
EARLY_STOP_PATIENCE = 10
EPOCH5_MIN_VAL_ACC = 0.80

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES_REFERENCE = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

ARCHITECTURES = {
    "DenseNet201": {
        "timm_name": "densenet201",
        "display_name": "DenseNet201"
    },
    "EffNetV2S": {
        "timm_name": "tf_efficientnetv2_s.in21k_ft_in1k",
        "display_name": "EfficientNetV2-S"
    }
}

# ----------------------------------------------------------
# Determinism
# ----------------------------------------------------------
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ----------------------------------------------------------
# Load Protocol A manifest
# ----------------------------------------------------------
manifest_path = CP3_OUTPUT_DIR / "CP3_A_seed42_reconstructed_manifest.csv"

if not manifest_path.exists():
    raise FileNotFoundError(
        f"Protocol A manifest not found: {manifest_path}\n"
        "Run CP3-A and CP3-B first to reconstruct Protocol A split."
    )

protocol_df = pd.read_csv(manifest_path)

class_names = sorted(protocol_df["class_name"].unique().tolist())

if class_names != CLASS_NAMES_REFERENCE:
    raise ValueError(
        f"Class names mismatch.\nDetected: {class_names}\nExpected: {CLASS_NAMES_REFERENCE}"
    )

class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

train_df = protocol_df[protocol_df["split"] == "train"].copy()
val_df = protocol_df[protocol_df["split"] == "val"].copy()
test_df = protocol_df[protocol_df["split"] == "test"].copy()

print(f"Device      : {DEVICE}")
print(f"Train       : {len(train_df)}")
print(f"Validation  : {len(val_df)}")
print(f"Test        : {len(test_df)}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Checkpoint  : {CHECKPOINT_DIR}")
print(f"Batch size  : {BATCH_SIZE}")

In [ ]:
# ==========================================================
# CP4-1 — Dataset, DataLoader, Scheduler, Mixup, and TTA
# ==========================================================

class TEMVirusDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


def build_transforms():
    train_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    eval_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    return train_transform, eval_transform


def build_dataloaders(seed):
    train_transform, eval_transform = build_transforms()

    train_dataset = TEMVirusDataset(train_df, class_to_idx, transform=train_transform)
    val_dataset = TEMVirusDataset(val_df, class_to_idx, transform=eval_transform)
    test_dataset = TEMVirusDataset(test_df, class_to_idx, transform=eval_transform)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


def make_model(arch_key):
    return timm.create_model(
        ARCHITECTURES[arch_key]["timm_name"],
        pretrained=True,
        num_classes=len(class_names)
    )


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }


class WarmupCosineRestartScheduler:
    def __init__(
        self,
        optimizer,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6,
        t_0=10,
        t_mult=2
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.t_0 = t_0
        self.t_mult = t_mult
        self.last_epoch = -1

        self.restart_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=t_0,
            T_mult=t_mult,
            eta_min=min_lr
        )

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            progress = epoch / max(1, self.warmup_epochs - 1)
            lr = self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

            for group in self.optimizer.param_groups:
                group["lr"] = lr
        else:
            self.restart_scheduler.step(epoch - self.warmup_epochs)
            lr = self.optimizer.param_groups[0]["lr"]

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "t_0": self.t_0,
            "t_mult": self.t_mult,
            "last_epoch": self.last_epoch,
            "restart_scheduler": self.restart_scheduler.state_dict()
        }

    def load_state_dict(self, state_dict):
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.t_0 = state_dict["t_0"]
        self.t_mult = state_dict["t_mult"]
        self.last_epoch = state_dict["last_epoch"]
        self.restart_scheduler.load_state_dict(state_dict["restart_scheduler"])


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


def tta_predict(model, images):
    augmentations = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[2, 3])
    ]

    probs_list = []

    for aug in augmentations:
        logits = model(aug(images))
        probs = torch.softmax(logits.float(), dim=1)
        probs_list.append(probs)

    return torch.mean(torch.stack(probs_list, dim=0), dim=0)

In [ ]:
# ==========================================================
# CP4-1 — Dataset, DataLoader, Scheduler, Mixup, and TTA
# ==========================================================

class TEMVirusDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


def build_transforms():
    train_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    eval_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    return train_transform, eval_transform


def build_dataloaders(seed):
    train_transform, eval_transform = build_transforms()

    train_dataset = TEMVirusDataset(train_df, class_to_idx, transform=train_transform)
    val_dataset = TEMVirusDataset(val_df, class_to_idx, transform=eval_transform)
    test_dataset = TEMVirusDataset(test_df, class_to_idx, transform=eval_transform)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


def make_model(arch_key):
    return timm.create_model(
        ARCHITECTURES[arch_key]["timm_name"],
        pretrained=True,
        num_classes=len(class_names)
    )


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }


class WarmupCosineRestartScheduler:
    def __init__(
        self,
        optimizer,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6,
        t_0=10,
        t_mult=2
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.t_0 = t_0
        self.t_mult = t_mult
        self.last_epoch = -1

        self.restart_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=t_0,
            T_mult=t_mult,
            eta_min=min_lr
        )

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            progress = epoch / max(1, self.warmup_epochs - 1)
            lr = self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

            for group in self.optimizer.param_groups:
                group["lr"] = lr
        else:
            self.restart_scheduler.step(epoch - self.warmup_epochs)
            lr = self.optimizer.param_groups[0]["lr"]

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "t_0": self.t_0,
            "t_mult": self.t_mult,
            "last_epoch": self.last_epoch,
            "restart_scheduler": self.restart_scheduler.state_dict()
        }

    def load_state_dict(self, state_dict):
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.t_0 = state_dict["t_0"]
        self.t_mult = state_dict["t_mult"]
        self.last_epoch = state_dict["last_epoch"]
        self.restart_scheduler.load_state_dict(state_dict["restart_scheduler"])


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


def tta_predict(model, images):
    augmentations = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[2, 3])
    ]

    probs_list = []

    for aug in augmentations:
        logits = model(aug(images))
        probs = torch.softmax(logits.float(), dim=1)
        probs_list.append(probs)

    return torch.mean(torch.stack(probs_list, dim=0), dim=0)

In [ ]:
# ==========================================================
# CP4-2 — Training and TTA Evaluation Functions
# ==========================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, arch_key, seed, epoch_num):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    last_grad_norm = 0.0
    mixup_batches = 0

    progress = tqdm(
        loader,
        desc=f"{arch_key} seed {seed} train epoch {epoch_num}",
        leave=True
    )

    for images, labels, _paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        apply_mixup = np.random.random() < MIXUP_PROB

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            if apply_mixup:
                mixed_images, y_a, y_b, lam = mixup_data(images, labels, alpha=MIXUP_ALPHA)
                logits = model(mixed_images)
                loss = mixup_loss(criterion, logits, y_a, y_b, lam)
                mixup_batches += 1
            else:
                logits = model(images)
                loss = criterion(logits, labels)

        if torch.isnan(loss):
            raise ValueError(f"NaN loss detected: {arch_key}, seed={seed}, epoch={epoch_num}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_MAX_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        last_grad_norm = float(grad_norm)

        progress.set_postfix({
            "loss": f"{loss.item():.4f}",
            "grad": f"{last_grad_norm:.3f}",
            "mixup": mixup_batches
        })

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))
    metrics["grad_norm_last"] = float(last_grad_norm)
    metrics["mixup_batches"] = int(mixup_batches)

    return metrics


@torch.no_grad()
def evaluate_with_tta(model, loader, criterion, arch_key, seed, split_name):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_paths = []
    all_probs = []

    progress = tqdm(
        loader,
        desc=f"{arch_key} seed {seed} evaluate {split_name} with TTA",
        leave=True
    )

    for images, labels, paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            probs = tta_predict(model, images)
            logits_for_loss = model(images)
            loss = criterion(logits_for_loss, labels)

        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_paths.extend(list(paths))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))

    per_class_f1_values = f1_score(
        all_targets,
        all_preds,
        average=None,
        zero_division=0
    )

    per_class_f1 = {
        class_names[i]: float(per_class_f1_values[i])
        for i in range(len(class_names))
    }

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_txt = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        zero_division=0
    )

    cm = confusion_matrix(
        all_targets,
        all_preds,
        labels=list(range(len(class_names)))
    )

    pred_df = pd.DataFrame({
        "filename": [Path(p).name for p in all_paths],
        "image_path": all_paths,
        "true_label": [class_names[i] for i in all_targets],
        "pred_label": [class_names[i] for i in all_preds],
        "confidence": all_probs.max(axis=1)
    })

    for i, class_name in enumerate(class_names):
        safe_name = class_name.replace(" ", "_")
        pred_df[f"prob_{safe_name}"] = all_probs[:, i]

    return {
        "metrics": metrics,
        "per_class_f1": per_class_f1,
        "classification_report": report_dict,
        "classification_report_txt": report_txt,
        "confusion_matrix": cm,
        "prediction_df": pred_df,
        "softmax": all_probs,
        "targets": all_targets,
        "preds": all_preds,
        "paths": all_paths
    }

In [ ]:
# ==========================================================
# CP4-3 — Saving, Checkpoint, Plotting, and Single-Run Function
# ==========================================================

def get_run_paths(arch_key, seed):
    prefix = f"{arch_key}_A_seed{seed}_ttamixup"

    return {
        "prefix": prefix,
        "best_ckpt": CHECKPOINT_DIR / f"{prefix}_best.pt",
        "last_ckpt": CHECKPOINT_DIR / f"{prefix}_last.pt",
        "history_csv": OUTPUT_DIR / f"{prefix}_training_history.csv",
        "history_json": OUTPUT_DIR / f"{prefix}_training_history.json",
        "training_summary_json": OUTPUT_DIR / f"{prefix}_training_summary.json",
        "final_summary_json": OUTPUT_DIR / f"{prefix}_CP4_final_summary.json",
        "metrics_json": OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json",
        "metrics_csv": OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.csv",
        "per_class_f1_csv": OUTPUT_DIR / f"{prefix}_test_per_class_f1.csv",
        "val_softmax": OUTPUT_DIR / f"{prefix}_val_softmax.npy",
        "test_softmax": OUTPUT_DIR / f"{prefix}_test_softmax.npy",
        "learning_curve_png": OUTPUT_DIR / f"{prefix}_learning_curve.png",
        "confusion_matrix_png": OUTPUT_DIR / f"{prefix}_confusion_matrix_test.png"
    }


def save_history(history, paths):
    history_df = pd.DataFrame(history)
    history_df.to_csv(paths["history_csv"], index=False)

    with open(paths["history_json"], "w") as f:
        json.dump(history, f, indent=2)

    return history_df


def save_last_checkpoint(model, optimizer, scheduler, epoch, best_metric, best_epoch, patience_counter, history, paths, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_metric": best_metric,
            "best_epoch": best_epoch,
            "patience_counter": patience_counter,
            "history": history,
            "config": config
        },
        paths["last_ckpt"]
    )


def save_best_checkpoint(model, optimizer, scheduler, epoch, val_macro_f1, val_metrics, paths, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_macro_f1": val_macro_f1,
            "val_metrics": val_metrics,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "config": config
        },
        paths["best_ckpt"]
    )


def load_last_checkpoint(model, optimizer, scheduler, paths):
    if paths["last_ckpt"].exists():
        ckpt = torch.load(paths["last_ckpt"], map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_metric = float(ckpt["best_metric"])
        best_epoch = ckpt["best_epoch"]
        patience_counter = int(ckpt["patience_counter"])
        history = ckpt["history"]

        print("=== Resume checkpoint found ===")
        print(f"Start epoch       : {start_epoch + 1}")
        print(f"Best val macro F1 : {best_metric:.4f}")
        print(f"Best epoch        : {None if best_epoch is None else best_epoch + 1}")

        return start_epoch, best_metric, best_epoch, patience_counter, history

    return 0, -1.0, None, 0, []


def save_split_outputs(arch_key, seed, split_name, outputs):
    prefix = OUTPUT_DIR / f"{arch_key}_A_seed{seed}_ttamixup_{split_name}"

    report_csv = Path(str(prefix) + "_classification_report.csv")
    report_json = Path(str(prefix) + "_classification_report.json")
    report_txt = Path(str(prefix) + "_classification_report.txt")
    cm_csv = Path(str(prefix) + "_confusion_matrix.csv")
    pred_csv = Path(str(prefix) + "_predictions.csv")

    pd.DataFrame(outputs["classification_report"]).transpose().to_csv(report_csv)

    with open(report_json, "w") as f:
        json.dump(outputs["classification_report"], f, indent=2)

    with open(report_txt, "w") as f:
        f.write(outputs["classification_report_txt"])

    cm_df = pd.DataFrame(
        outputs["confusion_matrix"],
        index=class_names,
        columns=class_names
    )
    cm_df.to_csv(cm_csv)

    outputs["prediction_df"].to_csv(pred_csv, index=False)

    return {
        "classification_report_csv": str(report_csv),
        "classification_report_json": str(report_json),
        "classification_report_txt": str(report_txt),
        "confusion_matrix_csv": str(cm_csv),
        "predictions_csv": str(pred_csv)
    }


def plot_outputs(paths, arch_key, seed, test_outputs):
    history_df = pd.read_csv(paths["history_csv"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    axes[0, 1].plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
    axes[1, 0].set_title("Validation Macro F1")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(history_df["epoch"], history_df["lr"], label="lr")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f"{arch_key} seed {seed} — TTA + Mixup", y=1.02)
    plt.tight_layout()
    plt.savefig(paths["learning_curve_png"], dpi=300, bbox_inches="tight")
    plt.show()

    cm = test_outputs["confusion_matrix"]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm)

    ax.set_title(f"{arch_key} seed {seed} — Test Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(paths["confusion_matrix_png"], dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ==========================================================
# CP4-4 — Single Model-Seed Runner
# ==========================================================

def run_model_seed(arch_key, seed, skip_completed=True):
    set_seed(seed)

    paths = get_run_paths(arch_key, seed)

    if skip_completed and paths["final_summary_json"].exists() and paths["test_softmax"].exists() and paths["val_softmax"].exists():
        print(f"Run already completed: {arch_key} seed {seed}")
        with open(paths["final_summary_json"], "r") as f:
            return json.load(f)

    train_loader, val_loader, test_loader = build_dataloaders(seed)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = make_model(arch_key).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = WarmupCosineRestartScheduler(
        optimizer=optimizer,
        warmup_epochs=5,
        warmup_start_lr=WARMUP_START_LR,
        base_lr=BASE_LR,
        min_lr=MIN_LR,
        t_0=10,
        t_mult=2
    )

    try:
        scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    config = {
        "model": arch_key,
        "timm_model_name": ARCHITECTURES[arch_key]["timm_name"],
        "protocol": PROTOCOL,
        "seed": seed,
        "max_epochs": MAX_EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE,
        "optimizer": "AdamW",
        "base_lr": BASE_LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "mixup_alpha": MIXUP_ALPHA,
        "mixup_probability": MIXUP_PROB,
        "scheduler": "warmup_cosine_restarts",
        "tta": "original_hflip_vflip_both",
        "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
        "normalization": "ImageNet",
        "pretrained": True
    }

    start_epoch, best_metric, best_epoch, patience_counter, history = load_last_checkpoint(
        model,
        optimizer,
        scheduler,
        paths
    )

    params_count = count_trainable_parameters(model)
    model_size_mb = estimate_model_size_mb(model)

    stopped_by_gate = False
    early_stopped = False
    anomaly_log = []

    print("\n" + "=" * 80)
    print(f"CP4 run: {arch_key}, seed {seed}")
    print("=" * 80)
    print(f"Device       : {DEVICE}")
    print(f"Epochs       : {MAX_EPOCHS}")
    print(f"Batch size   : {BATCH_SIZE}")
    print(f"Parameters   : {params_count:,}")
    print(f"Model size   : {model_size_mb:.2f} MB")

    training_start = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):
        epoch_num = epoch + 1
        epoch_start = time.time()

        lr = scheduler.step(epoch)

        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            arch_key,
            seed,
            epoch_num
        )

        val_outputs = evaluate_with_tta(
            model,
            val_loader,
            criterion,
            arch_key,
            seed,
            split_name="validation"
        )

        val_metrics = val_outputs["metrics"]
        val_macro_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]

        improved = val_macro_f1 > best_metric + 1e-8

        if improved:
            best_metric = val_macro_f1
            best_epoch = epoch
            patience_counter = 0

            save_best_checkpoint(
                model,
                optimizer,
                scheduler,
                epoch,
                val_macro_f1,
                val_metrics,
                paths,
                config
            )
        else:
            patience_counter += 1

        row = {
            "model": arch_key,
            "protocol": PROTOCOL,
            "seed": seed,
            "epoch": epoch_num,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_grad_norm_last": train_metrics["grad_norm_last"],
            "train_mixup_batches": train_metrics["mixup_batches"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "lr": lr,
            "epoch_time_sec": time.time() - epoch_start,
            "best_val_macro_f1_so_far": best_metric,
            "best_epoch_so_far": None if best_epoch is None else best_epoch + 1,
            "patience_counter": patience_counter,
            "improved": improved
        }

        history.append(row)
        save_history(history, paths)

        save_last_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            best_metric,
            best_epoch,
            patience_counter,
            history,
            paths,
            config
        )

        print("\n=== Epoch Summary ===")
        print(json.dumps(row, indent=2))

        if epoch_num == 5 and val_acc < EPOCH5_MIN_VAL_ACC:
            stopped_by_gate = True
            anomaly_log.append(f"Epoch 5 gate stopped: val_acc={val_acc:.4f}")
            print("\n=== Epoch 5 Gate Stop ===")
            break

        if patience_counter >= EARLY_STOP_PATIENCE:
            early_stopped = True
            print("\n=== Early Stopping Triggered ===")
            break

    training_time_sec = time.time() - training_start

    peak_gpu_memory_mb = None
    if DEVICE.type == "cuda":
        peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    training_summary = {
        "model": arch_key,
        "protocol": PROTOCOL,
        "seed": seed,
        "total_epochs_run": len(history),
        "best_epoch": None if best_epoch is None else best_epoch + 1,
        "best_val_macro_f1": None if best_metric < 0 else best_metric,
        "total_training_time_sec": training_time_sec,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "early_stopped": early_stopped,
        "stopped_by_gate": stopped_by_gate,
        "anomaly_log": anomaly_log,
        "params_count_M": params_count / 1e6,
        "model_size_mb": model_size_mb,
        "best_checkpoint": str(paths["best_ckpt"]),
        "last_checkpoint": str(paths["last_ckpt"]),
        "history_csv": str(paths["history_csv"]),
        "config": config
    }

    with open(paths["training_summary_json"], "w") as f:
        json.dump(training_summary, f, indent=2)

    if not paths["best_ckpt"].exists():
        raise FileNotFoundError(f"No best checkpoint was saved for {arch_key} seed {seed}")

    best_ckpt = torch.load(paths["best_ckpt"], map_location=DEVICE)

    best_model = make_model(arch_key).to(DEVICE)
    best_model.load_state_dict(best_ckpt["model_state_dict"])

    criterion_eval = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    validation_outputs = evaluate_with_tta(
        best_model,
        val_loader,
        criterion_eval,
        arch_key,
        seed,
        split_name="validation"
    )

    test_outputs = evaluate_with_tta(
        best_model,
        test_loader,
        criterion_eval,
        arch_key,
        seed,
        split_name="test"
    )

    np.save(paths["val_softmax"], validation_outputs["softmax"])
    np.save(paths["test_softmax"], test_outputs["softmax"])

    validation_files = save_split_outputs(arch_key, seed, "validation", validation_outputs)
    test_files = save_split_outputs(arch_key, seed, "test", test_outputs)

    metrics_summary = {
        "model": arch_key,
        "protocol": PROTOCOL,
        "seed": seed,
        "best_epoch": int(best_ckpt["epoch"]) + 1,
        "best_val_macro_f1_checkpoint": float(best_ckpt["val_macro_f1"]),
        "validation": validation_outputs["metrics"],
        "test": test_outputs["metrics"],
        "test_per_class_f1": test_outputs["per_class_f1"],
        "validation_files": validation_files,
        "test_files": test_files,
        "val_softmax_npy": str(paths["val_softmax"]),
        "test_softmax_npy": str(paths["test_softmax"])
    }

    with open(paths["metrics_json"], "w") as f:
        json.dump(metrics_summary, f, indent=2)

    pd.DataFrame([
        {"split": "validation", **validation_outputs["metrics"]},
        {"split": "test", **test_outputs["metrics"]}
    ]).to_csv(paths["metrics_csv"], index=False)

    pd.DataFrame({
        "class_name": list(test_outputs["per_class_f1"].keys()),
        "test_f1": list(test_outputs["per_class_f1"].values())
    }).to_csv(paths["per_class_f1_csv"], index=False)

    plot_outputs(paths, arch_key, seed, test_outputs)

    val_softmax_shape = list(np.load(paths["val_softmax"]).shape)
    test_softmax_shape = list(np.load(paths["test_softmax"]).shape)

    final_summary = {
        "model": arch_key,
        "protocol": PROTOCOL,
        "seed": seed,
        "best_epoch": metrics_summary["best_epoch"],
        "best_val_macro_f1": metrics_summary["best_val_macro_f1_checkpoint"],
        "test_accuracy": metrics_summary["test"]["accuracy"],
        "test_macro_f1": metrics_summary["test"]["macro_f1"],
        "test_weighted_f1": metrics_summary["test"]["weighted_f1"],
        "test_macro_precision": metrics_summary["test"]["macro_precision"],
        "test_macro_recall": metrics_summary["test"]["macro_recall"],
        "total_epochs_run": training_summary["total_epochs_run"],
        "total_training_time_sec": training_summary["total_training_time_sec"],
        "peak_gpu_memory_mb": training_summary["peak_gpu_memory_mb"],
        "early_stopped": training_summary["early_stopped"],
        "stopped_by_gate": training_summary["stopped_by_gate"],
        "anomaly_log": training_summary["anomaly_log"],
        "val_softmax_shape": val_softmax_shape,
        "test_softmax_shape": test_softmax_shape,
        "val_softmax_saved": paths["val_softmax"].exists(),
        "test_softmax_saved": paths["test_softmax"].exists(),
        "val_softmax_npy": str(paths["val_softmax"]),
        "test_softmax_npy": str(paths["test_softmax"]),
        "training_history_csv": str(paths["history_csv"]),
        "learning_curve_png": str(paths["learning_curve_png"]),
        "confusion_matrix_test_png": str(paths["confusion_matrix_png"]),
        "per_class_f1_csv": str(paths["per_class_f1_csv"]),
        "metrics_summary_json": str(paths["metrics_json"]),
        "final_summary_json": str(paths["final_summary_json"]),
        "best_checkpoint": str(paths["best_ckpt"]),
        "last_checkpoint": str(paths["last_ckpt"])
    }

    with open(paths["final_summary_json"], "w") as f:
        json.dump(final_summary, f, indent=2)

    print("\n=== CP4 Run Final Summary ===")
    print(json.dumps(final_summary, indent=2))

    del model
    del best_model
    gc.collect()
    torch.cuda.empty_cache()

    return final_summary

In [ ]:
# ==========================================================
# CP4-5 — Run 8 New Protocol A Runs
# DenseNet201 seeds 123/456/789/1024
# EffNetV2S seeds 123/456/789/1024
# ==========================================================

run_plan = []

for arch_key in ["DenseNet201", "EffNetV2S"]:
    for seed in NEW_SEEDS:
        run_plan.append((arch_key, seed))

cp4_run_results = []
cp4_error_log = []

for arch_key, seed in run_plan:
    try:
        result = run_model_seed(
            arch_key=arch_key,
            seed=seed,
            skip_completed=True
        )
        cp4_run_results.append(result)

    except RuntimeError as e:
        error_message = str(e)

        if "out of memory" in error_message.lower():
            torch.cuda.empty_cache()
            error_record = {
                "model": arch_key,
                "seed": seed,
                "error_type": "OOM",
                "error_message": error_message
            }
        else:
            error_record = {
                "model": arch_key,
                "seed": seed,
                "error_type": "RuntimeError",
                "error_message": error_message
            }

        cp4_error_log.append(error_record)

        print("\n=== RUN ERROR ===")
        print(json.dumps(error_record, indent=2))
        print("Continuing to next run.")

    except Exception as e:
        error_record = {
            "model": arch_key,
            "seed": seed,
            "error_type": type(e).__name__,
            "error_message": str(e)
        }

        cp4_error_log.append(error_record)

        print("\n=== RUN ERROR ===")
        print(json.dumps(error_record, indent=2))
        print("Continuing to next run.")

error_log_path = OUTPUT_DIR / "CP4_protocolA_error_log.json"

with open(error_log_path, "w") as f:
    json.dump(cp4_error_log, f, indent=2)

print(f"Completed runs : {len(cp4_run_results)}")
print(f"Failed runs    : {len(cp4_error_log)}")
print(f"Error log      : {error_log_path}")

In [ ]:
# ==========================================================
# CP4-6 — Aggregate Summary Including Existing Seed 42
# ==========================================================

def safe_load_json(path):
    path = Path(path)

    if not path.exists():
        return None

    with open(path, "r") as f:
        return json.load(f)


def build_summary_row_from_json(summary, model_name, seed, source):
    if summary is None:
        return {
            "model": model_name,
            "protocol": PROTOCOL,
            "seed": seed,
            "source": source,
            "test_accuracy": None,
            "test_macro_f1": None,
            "test_weighted_f1": None,
            "test_macro_precision": None,
            "test_macro_recall": None,
            "best_epoch": None,
            "best_val_macro_f1": None,
            "total_epochs": None,
            "training_time_sec": None,
            "peak_gpu_memory_mb": None,
            "val_softmax_saved": False,
            "test_softmax_saved": False,
            "val_softmax_shape": None,
            "test_softmax_shape": None,
            "final_summary_json": None,
            "status": "missing"
        }

    return {
        "model": model_name,
        "protocol": summary.get("protocol", PROTOCOL),
        "seed": seed,
        "source": source,
        "test_accuracy": summary.get("test_accuracy"),
        "test_macro_f1": summary.get("test_macro_f1"),
        "test_weighted_f1": summary.get("test_weighted_f1"),
        "test_macro_precision": summary.get("test_macro_precision"),
        "test_macro_recall": summary.get("test_macro_recall"),
        "best_epoch": summary.get("best_epoch"),
        "best_val_macro_f1": summary.get("best_val_macro_f1"),
        "total_epochs": summary.get("total_epochs_run"),
        "training_time_sec": summary.get("total_training_time_sec"),
        "peak_gpu_memory_mb": summary.get("peak_gpu_memory_mb"),
        "val_softmax_saved": summary.get("val_softmax_saved", Path(summary.get("val_softmax_npy", "")).exists() if summary.get("val_softmax_npy") else None),
        "test_softmax_saved": summary.get("test_softmax_saved", Path(summary.get("test_softmax_npy", "")).exists() if summary.get("test_softmax_npy") else None),
        "val_softmax_shape": summary.get("val_softmax_shape"),
        "test_softmax_shape": summary.get("test_softmax_shape"),
        "final_summary_json": summary.get("final_summary_json"),
        "status": "ok"
    }


summary_rows = []

# Existing seed 42 — DenseNet201
dense42_summary_path = CP36_OUTPUT_DIR / "DenseNet201_A_seed42_ttamixup_CP3_6_final_summary.json"
dense42_summary = safe_load_json(dense42_summary_path)
summary_rows.append(
    build_summary_row_from_json(
        dense42_summary,
        model_name="DenseNet201",
        seed=42,
        source="existing_cp3_6"
    )
)

# Existing seed 42 — EffNetV2S
eff42_summary_path = CP35_OUTPUT_DIR / "EffNetV2S_A_seed42_ttamixup_CP3_5_final_summary.json"
eff42_summary = safe_load_json(eff42_summary_path)
summary_rows.append(
    build_summary_row_from_json(
        eff42_summary,
        model_name="EffNetV2S",
        seed=42,
        source="existing_cp3_5"
    )
)

# New CP4 runs
for arch_key in ["DenseNet201", "EffNetV2S"]:
    for seed in NEW_SEEDS:
        paths = get_run_paths(arch_key, seed)
        summary = safe_load_json(paths["final_summary_json"])

        summary_rows.append(
            build_summary_row_from_json(
                summary,
                model_name=arch_key,
                seed=seed,
                source="cp4_new_run"
            )
        )

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "CP4_protocolA_multiseed_summary.csv"
summary_json = OUTPUT_DIR / "CP4_protocolA_multiseed_summary.json"

summary_df.to_csv(summary_csv, index=False)

with open(summary_json, "w") as f:
    json.dump(summary_rows, f, indent=2)

print("=== CP4 Protocol A Multi-seed Summary ===")
display(summary_df)

# ----------------------------------------------------------
# Softmax verification
# ----------------------------------------------------------
softmax_rows = []

# Existing seed 42
softmax_files = [
    {
        "model": "DenseNet201",
        "seed": 42,
        "val_softmax": CP36_OUTPUT_DIR / "DenseNet201_A_seed42_ttamixup_val_softmax.npy",
        "test_softmax": CP36_OUTPUT_DIR / "DenseNet201_A_seed42_ttamixup_test_softmax.npy"
    },
    {
        "model": "EffNetV2S",
        "seed": 42,
        "val_softmax": CP35_OUTPUT_DIR / "EffNetV2S_A_seed42_ttamixup_val_softmax.npy",
        "test_softmax": CP35_OUTPUT_DIR / "EffNetV2S_A_seed42_ttamixup_test_softmax.npy"
    }
]

# New runs
for arch_key in ["DenseNet201", "EffNetV2S"]:
    for seed in NEW_SEEDS:
        paths = get_run_paths(arch_key, seed)
        softmax_files.append({
            "model": arch_key,
            "seed": seed,
            "val_softmax": paths["val_softmax"],
            "test_softmax": paths["test_softmax"]
        })

for item in softmax_files:
    val_path = Path(item["val_softmax"])
    test_path = Path(item["test_softmax"])

    val_exists = val_path.exists()
    test_exists = test_path.exists()

    val_shape = list(np.load(val_path).shape) if val_exists else None
    test_shape = list(np.load(test_path).shape) if test_exists else None

    softmax_rows.append({
        "model": item["model"],
        "seed": item["seed"],
        "val_softmax_exists": val_exists,
        "test_softmax_exists": test_exists,
        "val_softmax_shape": val_shape,
        "test_softmax_shape": test_shape,
        "val_softmax_path": str(val_path),
        "test_softmax_path": str(test_path)
    })

softmax_df = pd.DataFrame(softmax_rows)

softmax_csv = OUTPUT_DIR / "CP4_protocolA_softmax_verification.csv"
softmax_df.to_csv(softmax_csv, index=False)

print("\n=== Softmax Verification ===")
display(softmax_df)

# ----------------------------------------------------------
# Aggregate by model
# ----------------------------------------------------------
valid_summary_df = summary_df.dropna(subset=["test_accuracy"]).copy()

aggregate_df = (
    valid_summary_df
    .groupby("model")
    .agg(
        n_runs=("test_accuracy", "count"),
        mean_test_accuracy=("test_accuracy", "mean"),
        std_test_accuracy=("test_accuracy", "std"),
        mean_test_macro_f1=("test_macro_f1", "mean"),
        std_test_macro_f1=("test_macro_f1", "std"),
        mean_test_weighted_f1=("test_weighted_f1", "mean"),
        std_test_weighted_f1=("test_weighted_f1", "std"),
        mean_training_time_sec=("training_time_sec", "mean")
    )
    .reset_index()
)

aggregate_csv = OUTPUT_DIR / "CP4_protocolA_multiseed_aggregate_by_model.csv"
aggregate_df.to_csv(aggregate_csv, index=False)

print("\n=== Aggregate by Model ===")
display(aggregate_df)

# ----------------------------------------------------------
# Package review files
# ----------------------------------------------------------
review_files = [
    summary_csv,
    summary_json,
    softmax_csv,
    aggregate_csv,
    OUTPUT_DIR / "CP4_protocolA_error_log.json"
]

review_zip = OUTPUT_DIR / "CP4_protocolA_multiseed_review_files.zip"

with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in review_files:
        file_path = Path(file_path)

        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
            print("Added:", file_path)
        else:
            print("Missing:", file_path)

print(f"Summary CSV     : {summary_csv}")
print(f"Summary JSON    : {summary_json}")
print(f"Softmax CSV     : {softmax_csv}")
print(f"Aggregate CSV   : {aggregate_csv}")
print(f"Review ZIP      : {review_zip}")

In [ ]:
# ==========================================================
# Protocol B-G14 Split Files Audit — READ ONLY
# No training. No deletion. No file modification.
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import json
import re

ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus/")

print("=" * 90)
print("PROTOCOL B-G14 SPLIT FILES AUDIT — READ ONLY")
print("=" * 90)
print(f"Root exists: {ROOT.exists()}")
print(f"Root path  : {ROOT}")

# ----------------------------------------------------------
# Search candidate files
# ----------------------------------------------------------
b_g14_candidates = []

b_g14_patterns = [
    "B_G14", "B-G14", "BG14", "b_g14", "b-g14",
    "B_conservative_G14", "conservative_G14",
    "G14_RAWSource", "RAWSource",
    "protocol_B", "protocolB",
    "B_source", "source_aware",
    "B_official"
]

print("\n[1] Scanning for B-G14 / G14 / RAWSource related CSV and JSON files...")

for suffix in ["*.csv", "*.json"]:
    for path in ROOT.rglob(suffix):
        name_lower = path.name.lower()
        path_lower = str(path).lower()

        if any(pattern.lower() in name_lower or pattern.lower() in path_lower for pattern in b_g14_patterns):
            b_g14_candidates.append(path)

b_g14_candidates = sorted(
    set(b_g14_candidates),
    key=lambda p: p.stat().st_size if p.exists() else 0,
    reverse=True
)

print(f"\nFound {len(b_g14_candidates)} candidate files:")
for path in b_g14_candidates[:80]:
    size_mb = path.stat().st_size / (1024 * 1024)
    rel = path.relative_to(ROOT)
    print(f"  {size_mb:>8.2f} MB  {rel}")

# ----------------------------------------------------------
# Find prediction files
# ----------------------------------------------------------
print("\n" + "=" * 90)
print("[2] CHECKING B-G14 PREDICTION FILES FOR SPLIT RECONSTRUCTION")
print("=" * 90)

prediction_files = []

for p in b_g14_candidates:
    name = p.name.lower()
    path_text = str(p).lower()

    is_prediction = "prediction" in name or "predictions" in name
    is_val_or_test = any(x in name for x in ["validation", "val", "test"])
    is_csv = p.suffix.lower() == ".csv"

    if is_csv and is_prediction and is_val_or_test:
        prediction_files.append(p)

prediction_files = sorted(set(prediction_files))

print(f"\nFound {len(prediction_files)} B-G14/G14 prediction CSV files:")

prediction_summary = []

for pf in prediction_files:
    rel = pf.relative_to(ROOT)
    print("\n" + "-" * 90)
    print(f"FILE: {rel}")

    try:
        df_head = pd.read_csv(pf, nrows=3)
        df_full = pd.read_csv(pf)

        print(f"Rows   : {len(df_full)}")
        print(f"Columns: {list(df_full.columns)}")

        print("\nSample row:")
        print(df_head.iloc[0].to_dict() if len(df_head) > 0 else "EMPTY FILE")

        columns_lower = [c.lower() for c in df_full.columns]

        has_path_col = any(c in columns_lower for c in [
            "image_path", "filepath", "file_path", "path", "filename"
        ])

        has_true_label_col = any(c in columns_lower for c in [
            "true_label", "target_class", "label", "target", "class_name"
        ])

        has_pred_label_col = any(c in columns_lower for c in [
            "pred_label", "prediction", "pred", "prediction_class"
        ])

        split_type = "unknown"
        lname = pf.name.lower()
        if "test" in lname:
            split_type = "test"
        elif "validation" in lname or "val" in lname:
            split_type = "validation"

        seed_match = re.search(r"seed[_]?(\d+)", pf.name.lower())
        seed_value = int(seed_match.group(1)) if seed_match else None

        prediction_summary.append({
            "file": str(rel),
            "rows": len(df_full),
            "split_type": split_type,
            "seed": seed_value,
            "has_path_col": has_path_col,
            "has_true_label_col": has_true_label_col,
            "has_pred_label_col": has_pred_label_col,
            "reconstruct_usable": has_path_col and has_true_label_col
        })

        print("\nUsability check:")
        print(f"  split_type          : {split_type}")
        print(f"  seed                : {seed_value}")
        print(f"  has_path_col        : {has_path_col}")
        print(f"  has_true_label_col  : {has_true_label_col}")
        print(f"  has_pred_label_col  : {has_pred_label_col}")
        print(f"  reconstruct usable  : {has_path_col and has_true_label_col}")

    except Exception as e:
        print(f"ERROR reading file: {e}")

prediction_summary_df = pd.DataFrame(prediction_summary)

print("\n" + "=" * 90)
print("[3] PREDICTION FILE SUMMARY")
print("=" * 90)

if len(prediction_summary_df) > 0:
    display(prediction_summary_df)
else:
    print("No B-G14 prediction files found.")

# ----------------------------------------------------------
# Detect seed availability
# ----------------------------------------------------------
print("\n" + "=" * 90)
print("[4] DETECTING B-G14 SEED AVAILABILITY")
print("=" * 90)

if len(prediction_summary_df) > 0:
    usable_df = prediction_summary_df[prediction_summary_df["reconstruct_usable"] == True].copy()

    print("\nUsable prediction files:")
    display(usable_df)

    seed_split_table = (
        usable_df
        .groupby(["seed", "split_type"])
        .size()
        .reset_index(name="file_count")
        .sort_values(["seed", "split_type"])
    )

    print("\nSeed/split availability:")
    display(seed_split_table)

    seeds_with_val = set(usable_df[usable_df["split_type"] == "validation"]["seed"].dropna().astype(int).tolist())
    seeds_with_test = set(usable_df[usable_df["split_type"] == "test"]["seed"].dropna().astype(int).tolist())
    seeds_with_both = sorted(seeds_with_val.intersection(seeds_with_test))

    print(f"\nSeeds with validation predictions: {sorted(seeds_with_val)}")
    print(f"Seeds with test predictions      : {sorted(seeds_with_test)}")
    print(f"Seeds with BOTH val and test     : {seeds_with_both}")
else:
    seeds_with_both = []
    print("No usable prediction files available for seed detection.")

# ----------------------------------------------------------
# Training history files
# ----------------------------------------------------------
print("\n" + "=" * 90)
print("[5] CHECKING B-G14 TRAINING HISTORY FILES")
print("=" * 90)

history_files = []

for path in ROOT.rglob("*training_history*.csv"):
    name_lower = path.name.lower()
    path_lower = str(path).lower()

    if any(pattern.lower() in name_lower or pattern.lower() in path_lower for pattern in b_g14_patterns):
        history_files.append(path)

history_files = sorted(set(history_files))

print(f"\nFound {len(history_files)} B-G14/G14 training history files:")

history_seeds = set()

for hf in history_files:
    rel = hf.relative_to(ROOT)
    print(f"  {rel}")

    match = re.search(r"seed[_]?(\d+)", hf.name.lower())
    if match:
        history_seeds.add(int(match.group(1)))

print(f"\nSeeds detected from history files: {sorted(history_seeds)}")

# ----------------------------------------------------------
# Split definition / paper table files
# ----------------------------------------------------------
print("\n" + "=" * 90)
print("[6] CHECKING PAPER TABLES AND EXPLICIT SPLIT FILES")
print("=" * 90)

paper_tables_dirs = list(ROOT.rglob("*paper_table*"))

if len(paper_tables_dirs) == 0:
    print("\nNo paper_table directories found.")
else:
    for d in paper_tables_dirs:
        if d.is_dir():
            print(f"\n{d.relative_to(ROOT)}/")
            for f in sorted(d.iterdir()):
                if f.is_file():
                    size_kb = f.stat().st_size / 1024
                    print(f"  - {f.name} ({size_kb:.1f} KB)")

print("\n[Searching explicit split/manifest files...]")

split_files = []

for path in ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in [".csv", ".json", ".txt"]:
        name_lower = path.name.lower()
        path_lower = str(path).lower()

        if (
            "split" in name_lower or
            "manifest" in name_lower or
            "protocol" in name_lower
        ) and any(pattern.lower() in name_lower or pattern.lower() in path_lower for pattern in b_g14_patterns):
            split_files.append(path)

split_files = sorted(set(split_files), key=lambda p: p.stat().st_size if p.exists() else 0, reverse=True)

print(f"\nFound {len(split_files)} explicit B-G14/G14 split/manifest/protocol files:")
for sf in split_files[:80]:
    rel = sf.relative_to(ROOT)
    size_kb = sf.stat().st_size / 1024
    print(f"  {size_kb:>8.1f} KB  {rel}")

# ----------------------------------------------------------
# Final decision summary
# ----------------------------------------------------------
print("\n" + "=" * 90)
print("[7] FINAL AUDIT DECISION")
print("=" * 90)

has_reconstructable_split = False

if len(prediction_summary_df) > 0:
    has_val = any(prediction_summary_df["split_type"] == "validation")
    has_test = any(prediction_summary_df["split_type"] == "test")
    has_usable = any(prediction_summary_df["reconstruct_usable"] == True)

    has_reconstructable_split = bool(has_val and has_test and has_usable)

print(f"Prediction CSV files found      : {len(prediction_files)}")
print(f"Usable prediction files found   : {len(prediction_summary_df[prediction_summary_df['reconstruct_usable'] == True]) if len(prediction_summary_df) > 0 else 0}")
print(f"Seeds with both val/test        : {seeds_with_both}")
print(f"Training history seeds detected : {sorted(history_seeds)}")
print(f"Explicit split files found      : {len(split_files)}")

if has_reconstructable_split:
    print("\nDECISION: B-G14 split likely CAN be reconstructed from available prediction files.")
else:
    print("\nDECISION: B-G14 split reconstruction is NOT confirmed from prediction files.")
    print("Need either B-G14 validation/test prediction CSVs or explicit B-G14 manifest/split file.")

print("\n" + "=" * 90)
print("AUDIT COMPLETE — READ ONLY")
print("=" * 90)

In [ ]:
# ==========================================================
# CP4B-0 — Setup, Config, and Load Protocol B-G14 Split
# Multi-seed Protocol B-G14: DenseNet201 and EfficientNetV2-S
# ==========================================================

!pip -q install timm

import os
import gc
import json
import time
import math
import random
import zipfile
import warnings
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import timm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")

BASE_OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cp4_multiseed"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_cp4"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

B_G14_MANIFEST = BASE_OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"
B_G14_VAL_PRED = BASE_OUTPUT_DIR / "DenseNet201_B_G14_seed42_validation_predictions.csv"
B_G14_TEST_PRED = BASE_OUTPUT_DIR / "DenseNet201_B_G14_seed42_test_predictions.csv"

# ----------------------------------------------------------
# General config
# ----------------------------------------------------------
PROTOCOL = "B_G14"
PROTOCOL_DISPLAY = "B-G14"
SEEDS = [42, 123, 456, 789, 1024]

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
MAX_EPOCHS = 35

BASE_LR = 1e-4
WARMUP_START_LR = 1e-6
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.5

GRAD_CLIP_MAX_NORM = 1.0
EARLY_STOP_PATIENCE = 10
EPOCH5_WARN_VAL_ACC = 0.75
EPOCH5_STOP_VAL_ACC = 0.60

VAL_EXPECTED = 2251
TEST_EXPECTED = 1903

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES_REFERENCE = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

ARCHITECTURES = {
    "DenseNet201": {
        "timm_name": "densenet201",
        "display_name": "DenseNet201"
    },
    "EffNetV2S": {
        "timm_name": "tf_efficientnetv2_s.in21k_ft_in1k",
        "display_name": "EfficientNetV2-S"
    }
}

# ----------------------------------------------------------
# Determinism
# ----------------------------------------------------------
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ----------------------------------------------------------
# Utility functions for split loading
# ----------------------------------------------------------
def find_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    return None


def normalize_class_name(value):
    value = str(value).strip()
    value = value.replace("_", " ")

    if value == "Nipah virus":
        return "Nipah virus"
    if value == "Rift Valley":
        return "Rift Valley"

    return value


def normalize_split_name(value):
    value = str(value).strip().lower()

    if value in ["val", "valid", "validation"]:
        return "validation"
    if value == "test":
        return "test"
    if value == "train":
        return "train"

    return value


def make_split_from_manifest_or_predictions():
    if not B_G14_MANIFEST.exists():
        raise FileNotFoundError(f"B-G14 manifest not found: {B_G14_MANIFEST}")

    manifest_df = pd.read_csv(B_G14_MANIFEST)

    path_col = find_col(manifest_df, ["filepath", "image_path", "file_path", "path"])
    split_col = find_col(manifest_df, ["split", "split_name", "set", "subset"])
    class_col = find_col(manifest_df, ["class_name", "true_class", "label_name", "class"])
    label_col = find_col(manifest_df, ["true_label", "label", "target", "class_idx", "class_index"])

    if path_col is None:
        raise ValueError(
            f"No filepath-like column found in manifest. Columns: {list(manifest_df.columns)}"
        )

    work_df = manifest_df.copy()
    work_df["filepath"] = work_df[path_col].astype(str)

    if class_col is not None:
        work_df["class_name"] = work_df[class_col].apply(normalize_class_name)
    else:
        work_df["class_name"] = work_df["filepath"].apply(lambda p: Path(p).parent.name)
        work_df["class_name"] = work_df["class_name"].apply(normalize_class_name)

    # Case 1: manifest already has split assignment
    if split_col is not None:
        work_df["split"] = work_df[split_col].apply(normalize_split_name)
        split_df = work_df[["filepath", "class_name", "split"]].copy()

    # Case 2: fallback to prediction CSV val/test and manifest all filepaths
    else:
        if not B_G14_VAL_PRED.exists() or not B_G14_TEST_PRED.exists():
            raise FileNotFoundError(
                "Manifest has no split column and prediction fallback files are missing."
            )

        val_pred = pd.read_csv(B_G14_VAL_PRED)
        test_pred = pd.read_csv(B_G14_TEST_PRED)

        val_path_col = find_col(val_pred, ["filepath", "image_path", "file_path", "path"])
        test_path_col = find_col(test_pred, ["filepath", "image_path", "file_path", "path"])

        val_paths = set(val_pred[val_path_col].astype(str).tolist())
        test_paths = set(test_pred[test_path_col].astype(str).tolist())

        def assign_split(path):
            if path in val_paths:
                return "validation"
            if path in test_paths:
                return "test"
            return "train"

        work_df["split"] = work_df["filepath"].apply(assign_split)
        split_df = work_df[["filepath", "class_name", "split"]].copy()

    split_df["class_name"] = split_df["class_name"].apply(normalize_class_name)

    class_names_detected = sorted(split_df["class_name"].unique().tolist())

    if class_names_detected != CLASS_NAMES_REFERENCE:
        print("Detected classes:", class_names_detected)
        print("Expected classes:", CLASS_NAMES_REFERENCE)
        raise ValueError("Class name mismatch in B-G14 split.")

    return split_df


protocol_df = make_split_from_manifest_or_predictions()

class_names = sorted(protocol_df["class_name"].unique().tolist())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

train_df = protocol_df[protocol_df["split"] == "train"].copy()
val_df = protocol_df[protocol_df["split"] == "validation"].copy()
test_df = protocol_df[protocol_df["split"] == "test"].copy()

print("=== CP4B-0 SPLIT VERIFICATION ===")
print(f"Device      : {DEVICE}")
print(f"Manifest    : {B_G14_MANIFEST}")
print(f"Train       : {len(train_df)}")
print(f"Validation  : {len(val_df)}")
print(f"Test        : {len(test_df)}")
print(f"Classes     : {len(class_names)}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Checkpoint  : {CHECKPOINT_DIR}")

if len(val_df) != VAL_EXPECTED or len(test_df) != TEST_EXPECTED:
    raise ValueError(
        f"B-G14 split size mismatch. Expected val={VAL_EXPECTED}, test={TEST_EXPECTED}; "
        f"got val={len(val_df)}, test={len(test_df)}. STOP before training."
    )

print("Split verified: B-G14 val/test sizes match Phase 1 baseline.")

In [ ]:
# ==========================================================
# CP4B-1 — Dataset, DataLoader, Scheduler, Mixup, and TTA
# ==========================================================

class TEMVirusDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


def build_transforms():
    train_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    eval_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    return train_transform, eval_transform


def build_dataloaders(seed):
    train_transform, eval_transform = build_transforms()

    train_dataset = TEMVirusDataset(train_df, class_to_idx, transform=train_transform)
    val_dataset = TEMVirusDataset(val_df, class_to_idx, transform=eval_transform)
    test_dataset = TEMVirusDataset(test_df, class_to_idx, transform=eval_transform)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


def make_model(arch_key):
    return timm.create_model(
        ARCHITECTURES[arch_key]["timm_name"],
        pretrained=True,
        num_classes=len(class_names)
    )


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }


class WarmupCosineRestartScheduler:
    def __init__(
        self,
        optimizer,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6,
        t_0=10,
        t_mult=2
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.t_0 = t_0
        self.t_mult = t_mult
        self.last_epoch = -1

        self.restart_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=t_0,
            T_mult=t_mult,
            eta_min=min_lr
        )

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            progress = epoch / max(1, self.warmup_epochs - 1)
            lr = self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

            for group in self.optimizer.param_groups:
                group["lr"] = lr
        else:
            self.restart_scheduler.step(epoch - self.warmup_epochs)
            lr = self.optimizer.param_groups[0]["lr"]

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "t_0": self.t_0,
            "t_mult": self.t_mult,
            "last_epoch": self.last_epoch,
            "restart_scheduler": self.restart_scheduler.state_dict()
        }

    def load_state_dict(self, state_dict):
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.t_0 = state_dict["t_0"]
        self.t_mult = state_dict["t_mult"]
        self.last_epoch = state_dict["last_epoch"]
        self.restart_scheduler.load_state_dict(state_dict["restart_scheduler"])


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


def tta_predict(model, images):
    augmentations = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[2, 3])
    ]

    probs_list = []

    for aug in augmentations:
        logits = model(aug(images))
        probs = torch.softmax(logits.float(), dim=1)
        probs_list.append(probs)

    return torch.mean(torch.stack(probs_list, dim=0), dim=0)

In [ ]:
# ==========================================================
# CP4B-2 — Training and TTA Evaluation Functions
# ==========================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, arch_key, seed, epoch_num):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    last_grad_norm = 0.0
    mixup_batches = 0

    progress = tqdm(
        loader,
        desc=f"{arch_key} B-G14 seed {seed} train epoch {epoch_num}",
        leave=True
    )

    for images, labels, _paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        apply_mixup = np.random.random() < MIXUP_PROB

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            if apply_mixup:
                mixed_images, y_a, y_b, lam = mixup_data(images, labels, alpha=MIXUP_ALPHA)
                logits = model(mixed_images)
                loss = mixup_loss(criterion, logits, y_a, y_b, lam)
                mixup_batches += 1
            else:
                logits = model(images)
                loss = criterion(logits, labels)

        if torch.isnan(loss):
            raise ValueError(f"NaN loss detected: {arch_key}, seed={seed}, epoch={epoch_num}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_MAX_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        last_grad_norm = float(grad_norm)

        progress.set_postfix({
            "loss": f"{loss.item():.4f}",
            "grad": f"{last_grad_norm:.3f}",
            "mixup": mixup_batches
        })

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))
    metrics["grad_norm_last"] = float(last_grad_norm)
    metrics["mixup_batches"] = int(mixup_batches)

    return metrics


@torch.no_grad()
def evaluate_with_tta(model, loader, criterion, arch_key, seed, split_name):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_paths = []
    all_probs = []

    progress = tqdm(
        loader,
        desc=f"{arch_key} B-G14 seed {seed} evaluate {split_name} with TTA",
        leave=True
    )

    for images, labels, paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            probs = tta_predict(model, images)
            logits_for_loss = model(images)
            loss = criterion(logits_for_loss, labels)

        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_paths.extend(list(paths))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))

    per_class_f1_values = f1_score(
        all_targets,
        all_preds,
        average=None,
        zero_division=0
    )

    per_class_f1 = {
        class_names[i]: float(per_class_f1_values[i])
        for i in range(len(class_names))
    }

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_txt = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        zero_division=0
    )

    cm = confusion_matrix(
        all_targets,
        all_preds,
        labels=list(range(len(class_names)))
    )

    pred_df = pd.DataFrame({
        "filename": [Path(p).name for p in all_paths],
        "image_path": all_paths,
        "true_label": [class_names[i] for i in all_targets],
        "pred_label": [class_names[i] for i in all_preds],
        "confidence": all_probs.max(axis=1)
    })

    for i, class_name in enumerate(class_names):
        safe_name = class_name.replace(" ", "_")
        pred_df[f"prob_{safe_name}"] = all_probs[:, i]

    return {
        "metrics": metrics,
        "per_class_f1": per_class_f1,
        "classification_report": report_dict,
        "classification_report_txt": report_txt,
        "confusion_matrix": cm,
        "prediction_df": pred_df,
        "softmax": all_probs,
        "targets": all_targets,
        "preds": all_preds,
        "paths": all_paths
    }

In [ ]:
# ==========================================================
# CP4B-3 — Saving, Checkpoint, Plotting, Storage Helpers
# ==========================================================

def get_run_paths(arch_key, seed):
    prefix = f"{arch_key}_B_G14_seed{seed}_ttamixup"

    return {
        "prefix": prefix,
        "best_ckpt": CHECKPOINT_DIR / f"{prefix}_best.pt",
        "last_ckpt": CHECKPOINT_DIR / f"{prefix}_last.pt",
        "history_csv": OUTPUT_DIR / f"{prefix}_training_history.csv",
        "history_json": OUTPUT_DIR / f"{prefix}_training_history.json",
        "training_summary_json": OUTPUT_DIR / f"{prefix}_training_summary.json",
        "final_summary_json": OUTPUT_DIR / f"{prefix}_CP4_final_summary.json",
        "metrics_json": OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json",
        "metrics_csv": OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.csv",
        "per_class_f1_csv": OUTPUT_DIR / f"{prefix}_test_per_class_f1.csv",
        "val_softmax": OUTPUT_DIR / f"{prefix}_val_softmax.npy",
        "test_softmax": OUTPUT_DIR / f"{prefix}_test_softmax.npy",
        "learning_curve_png": OUTPUT_DIR / f"{prefix}_learning_curve.png",
        "confusion_matrix_png": OUTPUT_DIR / f"{prefix}_confusion_matrix_test.png"
    }


def get_drive_free_mb():
    try:
        usage = shutil.disk_usage("/content/drive")
        return usage.free / (1024 ** 2)
    except Exception:
        return None


def storage_gate():
    free_mb = get_drive_free_mb()

    if free_mb is None:
        print("Storage gate: unable to estimate Drive free space from runtime.")
        return

    print(f"Estimated /content/drive free space: {free_mb:.1f} MB")

    if free_mb < 800:
        raise RuntimeError(
            f"Storage gate STOP: free space < 800 MB ({free_mb:.1f} MB)."
        )

    if free_mb < 1500:
        print(f"Storage gate WARNING: free space is low ({free_mb:.1f} MB).")


def cleanup_last_checkpoint_if_finalized(paths):
    if (
        paths["final_summary_json"].exists()
        and paths["val_softmax"].exists()
        and paths["test_softmax"].exists()
        and paths["last_ckpt"].exists()
    ):
        size_mb = paths["last_ckpt"].stat().st_size / (1024 ** 2)
        paths["last_ckpt"].unlink()
        print(f"Cleaned _last.pt: {paths['last_ckpt'].name} ({size_mb:.1f} MB)")


def save_history(history, paths):
    history_df = pd.DataFrame(history)
    history_df.to_csv(paths["history_csv"], index=False)

    with open(paths["history_json"], "w") as f:
        json.dump(history, f, indent=2)

    return history_df


def save_last_checkpoint(model, optimizer, scheduler, epoch, best_metric, best_epoch, patience_counter, history, paths, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_metric": best_metric,
            "best_epoch": best_epoch,
            "patience_counter": patience_counter,
            "history": history,
            "config": config
        },
        paths["last_ckpt"]
    )


def save_best_checkpoint(model, optimizer, scheduler, epoch, val_macro_f1, val_metrics, paths, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_macro_f1": val_macro_f1,
            "val_metrics": val_metrics,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "config": config
        },
        paths["best_ckpt"]
    )


def load_last_checkpoint(model, optimizer, scheduler, paths):
    if paths["last_ckpt"].exists():
        ckpt = torch.load(paths["last_ckpt"], map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_metric = float(ckpt["best_metric"])
        best_epoch = ckpt["best_epoch"]
        patience_counter = int(ckpt["patience_counter"])
        history = ckpt["history"]

        print("=== Resume checkpoint found ===")
        print(f"Start epoch       : {start_epoch + 1}")
        print(f"Best val macro F1 : {best_metric:.4f}")
        print(f"Best epoch        : {None if best_epoch is None else best_epoch + 1}")

        return start_epoch, best_metric, best_epoch, patience_counter, history

    return 0, -1.0, None, 0, []


def save_split_outputs(arch_key, seed, split_name, outputs):
    prefix = OUTPUT_DIR / f"{arch_key}_B_G14_seed{seed}_ttamixup_{split_name}"

    report_csv = Path(str(prefix) + "_classification_report.csv")
    report_json = Path(str(prefix) + "_classification_report.json")
    report_txt = Path(str(prefix) + "_classification_report.txt")
    cm_csv = Path(str(prefix) + "_confusion_matrix.csv")
    pred_csv = Path(str(prefix) + "_predictions.csv")

    pd.DataFrame(outputs["classification_report"]).transpose().to_csv(report_csv)

    with open(report_json, "w") as f:
        json.dump(outputs["classification_report"], f, indent=2)

    with open(report_txt, "w") as f:
        f.write(outputs["classification_report_txt"])

    cm_df = pd.DataFrame(
        outputs["confusion_matrix"],
        index=class_names,
        columns=class_names
    )
    cm_df.to_csv(cm_csv)

    outputs["prediction_df"].to_csv(pred_csv, index=False)

    return {
        "classification_report_csv": str(report_csv),
        "classification_report_json": str(report_json),
        "classification_report_txt": str(report_txt),
        "confusion_matrix_csv": str(cm_csv),
        "predictions_csv": str(pred_csv)
    }


def plot_outputs(paths, arch_key, seed, test_outputs):
    history_df = pd.read_csv(paths["history_csv"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    axes[0, 1].plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
    axes[1, 0].set_title("Validation Macro F1")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(history_df["epoch"], history_df["lr"], label="lr")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f"{arch_key} B-G14 seed {seed} — TTA + Mixup", y=1.02)
    plt.tight_layout()
    plt.savefig(paths["learning_curve_png"], dpi=300, bbox_inches="tight")
    plt.show()

    cm = test_outputs["confusion_matrix"]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm)

    ax.set_title(f"{arch_key} B-G14 seed {seed} — Test Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(paths["confusion_matrix_png"], dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ==========================================================
# CP4B-4 — Single Model-Seed Runner
# ==========================================================

def run_model_seed_bg14(arch_key, seed, skip_completed=True):
    set_seed(seed)

    paths = get_run_paths(arch_key, seed)

    if skip_completed and paths["final_summary_json"].exists() and paths["test_softmax"].exists() and paths["val_softmax"].exists():
        print(f"Run already completed: {arch_key} B-G14 seed {seed}")
        cleanup_last_checkpoint_if_finalized(paths)
        with open(paths["final_summary_json"], "r") as f:
            return json.load(f)

    storage_gate()

    train_loader, val_loader, test_loader = build_dataloaders(seed)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = make_model(arch_key).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = WarmupCosineRestartScheduler(
        optimizer=optimizer,
        warmup_epochs=5,
        warmup_start_lr=WARMUP_START_LR,
        base_lr=BASE_LR,
        min_lr=MIN_LR,
        t_0=10,
        t_mult=2
    )

    try:
        scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    config = {
        "model": arch_key,
        "timm_model_name": ARCHITECTURES[arch_key]["timm_name"],
        "protocol": PROTOCOL,
        "protocol_display": PROTOCOL_DISPLAY,
        "seed": seed,
        "max_epochs": MAX_EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE,
        "optimizer": "AdamW",
        "base_lr": BASE_LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "mixup_alpha": MIXUP_ALPHA,
        "mixup_probability": MIXUP_PROB,
        "scheduler": "warmup_cosine_restarts",
        "tta": "original_hflip_vflip_both",
        "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
        "normalization": "ImageNet",
        "pretrained": True,
        "split_source": str(B_G14_MANIFEST)
    }

    start_epoch, best_metric, best_epoch, patience_counter, history = load_last_checkpoint(
        model,
        optimizer,
        scheduler,
        paths
    )

    params_count = count_trainable_parameters(model)
    model_size_mb = estimate_model_size_mb(model)

    stopped_by_gate = False
    early_stopped = False
    anomaly_log = []

    print("\n" + "=" * 80)
    print(f"CP4B run: {arch_key}, B-G14, seed {seed}")
    print("=" * 80)
    print(f"Device       : {DEVICE}")
    print(f"Epochs       : {MAX_EPOCHS}")
    print(f"Batch size   : {BATCH_SIZE}")
    print(f"Parameters   : {params_count:,}")
    print(f"Model size   : {model_size_mb:.2f} MB")

    training_start = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):
        epoch_num = epoch + 1
        epoch_start = time.time()

        lr = scheduler.step(epoch)

        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            arch_key,
            seed,
            epoch_num
        )

        val_outputs = evaluate_with_tta(
            model,
            val_loader,
            criterion,
            arch_key,
            seed,
            split_name="validation"
        )

        val_metrics = val_outputs["metrics"]
        val_macro_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]

        improved = val_macro_f1 > best_metric + 1e-8

        if improved:
            best_metric = val_macro_f1
            best_epoch = epoch
            patience_counter = 0

            save_best_checkpoint(
                model,
                optimizer,
                scheduler,
                epoch,
                val_macro_f1,
                val_metrics,
                paths,
                config
            )
        else:
            patience_counter += 1

        row = {
            "model": arch_key,
            "protocol": PROTOCOL,
            "seed": seed,
            "epoch": epoch_num,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_grad_norm_last": train_metrics["grad_norm_last"],
            "train_mixup_batches": train_metrics["mixup_batches"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "lr": lr,
            "epoch_time_sec": time.time() - epoch_start,
            "best_val_macro_f1_so_far": best_metric,
            "best_epoch_so_far": None if best_epoch is None else best_epoch + 1,
            "patience_counter": patience_counter,
            "improved": improved
        }

        history.append(row)
        save_history(history, paths)

        save_last_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            best_metric,
            best_epoch,
            patience_counter,
            history,
            paths,
            config
        )

        print("\n=== Epoch Summary ===")
        print(json.dumps(row, indent=2))

        if epoch_num == 5:
            if val_acc < EPOCH5_WARN_VAL_ACC:
                anomaly_log.append(f"Epoch 5 warning: val_acc={val_acc:.4f}")
                print(f"\nEpoch 5 warning: val_acc={val_acc:.4f}")

            if val_acc < EPOCH5_STOP_VAL_ACC:
                stopped_by_gate = True
                anomaly_log.append(f"Epoch 5 gate stopped: val_acc={val_acc:.4f}")
                print("\n=== Epoch 5 Gate Stop ===")
                break

        if patience_counter >= EARLY_STOP_PATIENCE:
            early_stopped = True
            print("\n=== Early Stopping Triggered ===")
            break

    training_time_sec = time.time() - training_start

    peak_gpu_memory_mb = None
    if DEVICE.type == "cuda":
        peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    training_summary = {
        "model": arch_key,
        "protocol": PROTOCOL,
        "seed": seed,
        "total_epochs_run": len(history),
        "best_epoch": None if best_epoch is None else best_epoch + 1,
        "best_val_macro_f1": None if best_metric < 0 else best_metric,
        "total_training_time_sec": training_time_sec,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "early_stopped": early_stopped,
        "stopped_by_gate": stopped_by_gate,
        "anomaly_log": anomaly_log,
        "params_count_M": params_count / 1e6,
        "model_size_mb": model_size_mb,
        "best_checkpoint": str(paths["best_ckpt"]),
        "last_checkpoint": str(paths["last_ckpt"]),
        "history_csv": str(paths["history_csv"]),
        "config": config
    }

    with open(paths["training_summary_json"], "w") as f:
        json.dump(training_summary, f, indent=2)

    if not paths["best_ckpt"].exists():
        raise FileNotFoundError(f"No best checkpoint was saved for {arch_key} B-G14 seed {seed}")

    best_ckpt = torch.load(paths["best_ckpt"], map_location=DEVICE)

    best_model = make_model(arch_key).to(DEVICE)
    best_model.load_state_dict(best_ckpt["model_state_dict"])

    criterion_eval = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    validation_outputs = evaluate_with_tta(
        best_model,
        val_loader,
        criterion_eval,
        arch_key,
        seed,
        split_name="validation"
    )

    test_outputs = evaluate_with_tta(
        best_model,
        test_loader,
        criterion_eval,
        arch_key,
        seed,
        split_name="test"
    )

    np.save(paths["val_softmax"], validation_outputs["softmax"])
    np.save(paths["test_softmax"], test_outputs["softmax"])

    validation_files = save_split_outputs(arch_key, seed, "validation", validation_outputs)
    test_files = save_split_outputs(arch_key, seed, "test", test_outputs)

    metrics_summary = {
        "model": arch_key,
        "protocol": PROTOCOL,
        "seed": seed,
        "best_epoch": int(best_ckpt["epoch"]) + 1,
        "best_val_macro_f1_checkpoint": float(best_ckpt["val_macro_f1"]),
        "validation": validation_outputs["metrics"],
        "test": test_outputs["metrics"],
        "test_per_class_f1": test_outputs["per_class_f1"],
        "validation_files": validation_files,
        "test_files": test_files,
        "val_softmax_npy": str(paths["val_softmax"]),
        "test_softmax_npy": str(paths["test_softmax"])
    }

    with open(paths["metrics_json"], "w") as f:
        json.dump(metrics_summary, f, indent=2)

    pd.DataFrame([
        {"split": "validation", **validation_outputs["metrics"]},
        {"split": "test", **test_outputs["metrics"]}
    ]).to_csv(paths["metrics_csv"], index=False)

    pd.DataFrame({
        "class_name": list(test_outputs["per_class_f1"].keys()),
        "test_f1": list(test_outputs["per_class_f1"].values())
    }).to_csv(paths["per_class_f1_csv"], index=False)

    plot_outputs(paths, arch_key, seed, test_outputs)

    val_softmax_shape = list(np.load(paths["val_softmax"]).shape)
    test_softmax_shape = list(np.load(paths["test_softmax"]).shape)

    final_summary = {
        "model": arch_key,
        "protocol": PROTOCOL,
        "seed": seed,
        "best_epoch": metrics_summary["best_epoch"],
        "best_val_macro_f1": metrics_summary["best_val_macro_f1_checkpoint"],
        "test_accuracy": metrics_summary["test"]["accuracy"],
        "test_macro_f1": metrics_summary["test"]["macro_f1"],
        "test_weighted_f1": metrics_summary["test"]["weighted_f1"],
        "test_macro_precision": metrics_summary["test"]["macro_precision"],
        "test_macro_recall": metrics_summary["test"]["macro_recall"],
        "total_epochs_run": training_summary["total_epochs_run"],
        "total_training_time_sec": training_summary["total_training_time_sec"],
        "peak_gpu_memory_mb": training_summary["peak_gpu_memory_mb"],
        "early_stopped": training_summary["early_stopped"],
        "stopped_by_gate": training_summary["stopped_by_gate"],
        "anomaly_log": training_summary["anomaly_log"],
        "val_softmax_shape": val_softmax_shape,
        "test_softmax_shape": test_softmax_shape,
        "val_softmax_saved": paths["val_softmax"].exists(),
        "test_softmax_saved": paths["test_softmax"].exists(),
        "val_softmax_npy": str(paths["val_softmax"]),
        "test_softmax_npy": str(paths["test_softmax"]),
        "training_history_csv": str(paths["history_csv"]),
        "learning_curve_png": str(paths["learning_curve_png"]),
        "confusion_matrix_test_png": str(paths["confusion_matrix_png"]),
        "per_class_f1_csv": str(paths["per_class_f1_csv"]),
        "metrics_summary_json": str(paths["metrics_json"]),
        "final_summary_json": str(paths["final_summary_json"]),
        "best_checkpoint": str(paths["best_ckpt"]),
        "last_checkpoint": str(paths["last_ckpt"])
    }

    with open(paths["final_summary_json"], "w") as f:
        json.dump(final_summary, f, indent=2)

    cleanup_last_checkpoint_if_finalized(paths)

    print("\n=== CP4B Run Final Summary ===")
    print(json.dumps(final_summary, indent=2))

    del model
    del best_model
    gc.collect()
    torch.cuda.empty_cache()

    return final_summary

In [ ]:
# ==========================================================
# CP4B-5 — Run 10 Protocol B-G14 Runs
# DenseNet201 seeds 42/123/456/789/1024
# EffNetV2S seeds 42/123/456/789/1024
# ==========================================================

run_plan = []

for arch_key in ["DenseNet201", "EffNetV2S"]:
    for seed in SEEDS:
        run_plan.append((arch_key, seed))

cp4b_run_results = []
cp4b_error_log = []

for arch_key, seed in run_plan:
    try:
        result = run_model_seed_bg14(
            arch_key=arch_key,
            seed=seed,
            skip_completed=True
        )
        cp4b_run_results.append(result)

    except RuntimeError as e:
        error_message = str(e)

        if "storage gate stop" in error_message.lower():
            error_record = {
                "model": arch_key,
                "seed": seed,
                "error_type": "StorageGateStop",
                "error_message": error_message
            }
            cp4b_error_log.append(error_record)
            print("\n=== STORAGE GATE STOP ===")
            print(json.dumps(error_record, indent=2))
            print("Stopping all remaining B-G14 runs.")
            break

        elif "out of memory" in error_message.lower():
            torch.cuda.empty_cache()
            error_record = {
                "model": arch_key,
                "seed": seed,
                "error_type": "OOM",
                "error_message": error_message
            }
        else:
            error_record = {
                "model": arch_key,
                "seed": seed,
                "error_type": "RuntimeError",
                "error_message": error_message
            }

        cp4b_error_log.append(error_record)

        print("\n=== RUN ERROR ===")
        print(json.dumps(error_record, indent=2))
        print("Continuing to next run.")

    except Exception as e:
        error_record = {
            "model": arch_key,
            "seed": seed,
            "error_type": type(e).__name__,
            "error_message": str(e)
        }

        cp4b_error_log.append(error_record)

        print("\n=== RUN ERROR ===")
        print(json.dumps(error_record, indent=2))
        print("Continuing to next run.")

error_log_path = OUTPUT_DIR / "CP4_protocolBG14_error_log.json"

with open(error_log_path, "w") as f:
    json.dump(cp4b_error_log, f, indent=2)

print(f"Completed runs : {len(cp4b_run_results)}")
print(f"Failed runs    : {len(cp4b_error_log)}")
print(f"Error log      : {error_log_path}")

In [ ]:
# ==========================================================
# CP4B-6 — Aggregate Summary, Softmax Check, Review ZIP
# ==========================================================

def safe_load_json(path):
    path = Path(path)

    if not path.exists():
        return None

    with open(path, "r") as f:
        return json.load(f)


def build_summary_row_from_json(summary, model_name, seed):
    if summary is None:
        return {
            "model": model_name,
            "protocol": PROTOCOL,
            "seed": seed,
            "test_accuracy": None,
            "test_macro_f1": None,
            "test_weighted_f1": None,
            "test_macro_precision": None,
            "test_macro_recall": None,
            "best_epoch": None,
            "best_val_macro_f1": None,
            "total_epochs": None,
            "training_time_sec": None,
            "peak_gpu_memory_mb": None,
            "val_softmax_saved": False,
            "test_softmax_saved": False,
            "val_softmax_shape": None,
            "test_softmax_shape": None,
            "final_summary_json": None,
            "status": "missing"
        }

    return {
        "model": model_name,
        "protocol": summary.get("protocol", PROTOCOL),
        "seed": seed,
        "test_accuracy": summary.get("test_accuracy"),
        "test_macro_f1": summary.get("test_macro_f1"),
        "test_weighted_f1": summary.get("test_weighted_f1"),
        "test_macro_precision": summary.get("test_macro_precision"),
        "test_macro_recall": summary.get("test_macro_recall"),
        "best_epoch": summary.get("best_epoch"),
        "best_val_macro_f1": summary.get("best_val_macro_f1"),
        "total_epochs": summary.get("total_epochs_run"),
        "training_time_sec": summary.get("total_training_time_sec"),
        "peak_gpu_memory_mb": summary.get("peak_gpu_memory_mb"),
        "val_softmax_saved": summary.get("val_softmax_saved"),
        "test_softmax_saved": summary.get("test_softmax_saved"),
        "val_softmax_shape": summary.get("val_softmax_shape"),
        "test_softmax_shape": summary.get("test_softmax_shape"),
        "final_summary_json": summary.get("final_summary_json"),
        "status": "ok"
    }


summary_rows = []

for arch_key in ["DenseNet201", "EffNetV2S"]:
    for seed in SEEDS:
        paths = get_run_paths(arch_key, seed)
        summary = safe_load_json(paths["final_summary_json"])
        summary_rows.append(build_summary_row_from_json(summary, arch_key, seed))

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "CP4_protocolBG14_multiseed_summary.csv"
summary_json = OUTPUT_DIR / "CP4_protocolBG14_multiseed_summary.json"

summary_df.to_csv(summary_csv, index=False)

with open(summary_json, "w") as f:
    json.dump(summary_rows, f, indent=2)

print("=== CP4 Protocol B-G14 Multi-seed Summary ===")
display(summary_df)

# ----------------------------------------------------------
# Softmax verification
# ----------------------------------------------------------
softmax_rows = []

for arch_key in ["DenseNet201", "EffNetV2S"]:
    for seed in SEEDS:
        paths = get_run_paths(arch_key, seed)

        val_path = paths["val_softmax"]
        test_path = paths["test_softmax"]

        val_exists = val_path.exists()
        test_exists = test_path.exists()

        val_shape = list(np.load(val_path).shape) if val_exists else None
        test_shape = list(np.load(test_path).shape) if test_exists else None

        softmax_rows.append({
            "model": arch_key,
            "seed": seed,
            "val_softmax_exists": val_exists,
            "test_softmax_exists": test_exists,
            "val_softmax_shape": val_shape,
            "test_softmax_shape": test_shape,
            "val_softmax_path": str(val_path),
            "test_softmax_path": str(test_path)
        })

softmax_df = pd.DataFrame(softmax_rows)

softmax_csv = OUTPUT_DIR / "CP4_protocolBG14_softmax_verification.csv"
softmax_df.to_csv(softmax_csv, index=False)

print("\n=== B-G14 Softmax Verification ===")
display(softmax_df)

# ----------------------------------------------------------
# Aggregate by model
# ----------------------------------------------------------
valid_summary_df = summary_df.dropna(subset=["test_accuracy"]).copy()

aggregate_df = (
    valid_summary_df
    .groupby("model")
    .agg(
        n_runs=("test_accuracy", "count"),
        mean_test_accuracy=("test_accuracy", "mean"),
        std_test_accuracy=("test_accuracy", "std"),
        mean_test_macro_f1=("test_macro_f1", "mean"),
        std_test_macro_f1=("test_macro_f1", "std"),
        mean_test_weighted_f1=("test_weighted_f1", "mean"),
        std_test_weighted_f1=("test_weighted_f1", "std"),
        mean_training_time_sec=("training_time_sec", "mean")
    )
    .reset_index()
)

aggregate_csv = OUTPUT_DIR / "CP4_protocolBG14_multiseed_aggregate_by_model.csv"
aggregate_df.to_csv(aggregate_csv, index=False)

print("\n=== B-G14 Aggregate by Model ===")
display(aggregate_df)

# ----------------------------------------------------------
# Split verification summary
# ----------------------------------------------------------
split_verification = {
    "protocol": PROTOCOL,
    "split_source": str(B_G14_MANIFEST),
    "train_count": int(len(train_df)),
    "validation_count": int(len(val_df)),
    "test_count": int(len(test_df)),
    "expected_validation_count": VAL_EXPECTED,
    "expected_test_count": TEST_EXPECTED,
    "validation_match": bool(len(val_df) == VAL_EXPECTED),
    "test_match": bool(len(test_df) == TEST_EXPECTED),
    "class_names": class_names
}

split_verification_json = OUTPUT_DIR / "CP4_protocolBG14_split_verification.json"

with open(split_verification_json, "w") as f:
    json.dump(split_verification, f, indent=2)

# ----------------------------------------------------------
# Package review files
# ----------------------------------------------------------
review_files = [
    summary_csv,
    summary_json,
    softmax_csv,
    aggregate_csv,
    split_verification_json,
    OUTPUT_DIR / "CP4_protocolBG14_error_log.json"
]

review_zip = OUTPUT_DIR / "CP4_protocolBG14_multiseed_review_files.zip"

with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in review_files:
        file_path = Path(file_path)

        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
            print("Added:", file_path)
        else:
            print("Missing:", file_path)

print(f"Summary CSV      : {summary_csv}")
print(f"Summary JSON     : {summary_json}")
print(f"Softmax CSV      : {softmax_csv}")
print(f"Aggregate CSV    : {aggregate_csv}")
print(f"Split verify JSON: {split_verification_json}")
print(f"Review ZIP       : {review_zip}")

In [ ]:
# ==========================================================
# CP4 P#3 — A-clean-strict & C-G09 Split Files Audit
# READ ONLY — no training, no delete, no file modification
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import re

ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus/")

def audit_protocol(protocol_patterns, label, expected_seeds, expected_val_count=None, expected_test_count=None):
    print("=" * 90)
    print(f"AUDIT: {label}")
    print(f"Patterns: {protocol_patterns}")
    print("=" * 90)

    predictions = []
    for path in ROOT.rglob("*predictions*.csv"):
        name_lower = path.name.lower()
        path_lower = str(path).lower()

        if any(p.lower() in name_lower or p.lower() in path_lower for p in protocol_patterns):
            predictions.append(path)

    main_predictions = [
        p for p in predictions
        if "bundle_for_claude" not in str(p).lower()
    ]

    main_predictions = sorted(set(main_predictions))

    print(f"\n[1] Found {len(main_predictions)} prediction files excluding bundle duplicates:")
    for p in main_predictions:
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"    {size_mb:>7.2f} MB  {p.relative_to(ROOT)}")

    seeds_found_val = set()
    seeds_found_test = set()

    for p in main_predictions:
        match = re.search(r"seed[_]?(\d+)", p.name.lower())
        if match:
            seed = int(match.group(1))

            if "validation" in p.name.lower() or "val" in p.name.lower():
                seeds_found_val.add(seed)
            elif "test" in p.name.lower():
                seeds_found_test.add(seed)

    seeds_both = seeds_found_val & seeds_found_test
    expected_set = set(expected_seeds)
    missing = expected_set - seeds_both

    print("\n[2] Seeds detected:")
    print(f"    Validation predictions: {sorted(seeds_found_val)}")
    print(f"    Test predictions      : {sorted(seeds_found_test)}")
    print(f"    Both val+test         : {sorted(seeds_both)}")
    print(f"    Expected seeds        : {sorted(expected_set)}")
    print(f"    Missing seeds         : {sorted(missing) if missing else 'NONE ✓'}")

    print("\n[3] Schema and row-count check:")

    schema_rows = []

    for sample_file in main_predictions:
        try:
            df_head = pd.read_csv(sample_file, nrows=3)
            full_df = pd.read_csv(sample_file)

            cols = list(full_df.columns)

            has_filepath = any(c in cols for c in ["filepath", "image_path", "file_path", "path", "filename"])
            has_true_label = any(c in cols for c in ["true_label", "label", "target", "class_idx", "class_index"])
            has_true_class = any(c in cols for c in ["true_class", "class_name", "label_name", "class"])
            has_pred_label = any(c in cols for c in ["pred_label", "prediction", "pred", "prediction_class"])

            split_type = "unknown"
            lname = sample_file.name.lower()
            if "test" in lname:
                split_type = "test"
            elif "validation" in lname or "val" in lname:
                split_type = "validation"

            seed_match = re.search(r"seed[_]?(\d+)", sample_file.name.lower())
            seed = int(seed_match.group(1)) if seed_match else None

            schema_rows.append({
                "file": str(sample_file.relative_to(ROOT)),
                "seed": seed,
                "split_type": split_type,
                "rows": len(full_df),
                "has_filepath": has_filepath,
                "has_true_label": has_true_label,
                "has_true_class": has_true_class,
                "has_pred_label": has_pred_label,
                "usable": has_filepath and has_true_label
            })

        except Exception as e:
            schema_rows.append({
                "file": str(sample_file.relative_to(ROOT)),
                "seed": None,
                "split_type": "error",
                "rows": None,
                "has_filepath": False,
                "has_true_label": False,
                "has_true_class": False,
                "has_pred_label": False,
                "usable": False,
                "error": str(e)
            })

    schema_df = pd.DataFrame(schema_rows)

    if len(schema_df) > 0:
        display(schema_df)
    else:
        print("    No prediction files found.")

    print("\n[4] Searching for explicit manifest/split files:")

    manifest_files = []

    for path in ROOT.rglob("*"):
        if path.is_file() and path.suffix.lower() in [".csv", ".json", ".txt"]:
            name_lower = path.name.lower()
            path_lower = str(path).lower()

            looks_like_manifest = (
                "manifest" in name_lower or
                "split" in name_lower or
                "protocol" in name_lower
            )

            matches_protocol = any(
                p.lower() in name_lower or p.lower() in path_lower
                for p in protocol_patterns
            )

            if looks_like_manifest and matches_protocol:
                manifest_files.append(path)

    main_manifests = [
        m for m in sorted(set(manifest_files))
        if "bundle_for_claude" not in str(m).lower()
    ]

    if main_manifests:
        for m in main_manifests:
            size_kb = m.stat().st_size / 1024
            print(f"    {size_kb:>8.1f} KB  {m.relative_to(ROOT)}")
    else:
        print("    None found. Split may need reconstruction from prediction CSVs.")

    usable_count = int(schema_df["usable"].sum()) if len(schema_df) > 0 and "usable" in schema_df.columns else 0
    reconstruct_ok = (
        len(missing) == 0 and
        usable_count >= 2 * len(expected_seeds)
    )

    print(f"\n[5] FINAL ASSESSMENT for {label}:")
    print(f"    Prediction files usable       : {usable_count}")
    print(f"    Required usable files minimum : {2 * len(expected_seeds)}")
    print(f"    Split reconstruction possible: {'YES ✓' if reconstruct_ok else 'NO ✗'}")

    if not reconstruct_ok:
        print("    Reason: missing seed(s), missing val/test pair(s), or unusable prediction schema.")

    return {
        "protocol": label,
        "seeds_with_both_val_test": sorted(seeds_both),
        "missing_seeds": sorted(missing),
        "reconstruct_ok": reconstruct_ok,
        "main_manifests": [str(m.relative_to(ROOT)) for m in main_manifests],
        "main_predictions_count": len(main_predictions),
        "usable_prediction_count": usable_count
    }


print("\n" + "█" * 90)
print("CP4 P#3 — PROTOCOL A-CLEAN-STRICT & C-G09 SPLIT FILES AUDIT")
print("READ ONLY — NO TRAINING — NO MODIFICATION")
print("█" * 90 + "\n")

results = []

result_aclean = audit_protocol(
    protocol_patterns=[
        "A_clean_strict",
        "A-clean-strict",
        "A_clean",
        "Aclean",
        "clean_strict",
        "hamming0",
        "duplicate_only"
    ],
    label="Protocol A-clean-strict",
    expected_seeds=[42, 123, 456],
    expected_val_count=2249,
    expected_test_count=1900
)
results.append(result_aclean)

print("\n")

result_cg09 = audit_protocol(
    protocol_patterns=[
        "C_G09",
        "C-G09",
        "CG09",
        "G09_Date_Magnification",
        "Date_Magnification",
        "B_G09",
        "G09",
        "C_official",
        "C_RAWSource"
    ],
    label="Protocol C-G09",
    expected_seeds=[42, 123, 456],
    expected_val_count=None,
    expected_test_count=None
)
results.append(result_cg09)

print("\n" + "█" * 90)
print("OVERALL SUMMARY")
print("█" * 90)

for r in results:
    status = "✓ READY" if r["reconstruct_ok"] else "✗ MISSING"
    print(f"{status}  {r['protocol']}")
    print(f"       Seeds available : {r['seeds_with_both_val_test']}")
    print(f"       Missing seeds   : {r['missing_seeds'] if r['missing_seeds'] else 'NONE'}")
    print(f"       Prediction files: {r['main_predictions_count']}")
    print(f"       Usable files    : {r['usable_prediction_count']}")
    print(f"       Manifests       : {r['main_manifests'] if r['main_manifests'] else 'None'}")

print("\nDECISION:")
all_ok = all(r["reconstruct_ok"] for r in results)

if all_ok:
    print("Both protocols ready.")
else:
    print("Some protocols missing files.")

In [ ]:
# ==========================================================
# CP4C-0 — Setup, Config, and Split Verification
# Protocol A-clean-strict and Protocol C-G09
# ==========================================================

!pip -q install timm

import os
import gc
import sys
import json
import time
import random
import zipfile
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import timm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")
BASE_OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cp4_multiseed"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_cp4"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# General config
# ----------------------------------------------------------
SEEDS = [42, 123, 456]

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
MAX_EPOCHS = 35

BASE_LR = 1e-4
WARMUP_START_LR = 1e-6
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.5

GRAD_CLIP_MAX_NORM = 1.0
EARLY_STOP_PATIENCE = 10

EPOCH5_STOP_VAL_ACC = 0.70
EPOCH5_WARN_VAL_ACC = 0.80

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES_REFERENCE = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

ARCHITECTURES = {
    "DenseNet201": {
        "timm_name": "densenet201",
        "display_name": "DenseNet201"
    },
    "EffNetV2S": {
        "timm_name": "tf_efficientnetv2_s.in21k_ft_in1k",
        "display_name": "EfficientNetV2-S"
    }
}

PROTOCOL_CONFIGS = {
    "A_clean_strict": {
        "display_name": "A-clean-strict",
        "manifest": BASE_OUTPUT_DIR / "v31_split_manifest_A_clean_strict_hamming0_duplicate_only.csv",
        "val_pred_seed42": BASE_OUTPUT_DIR / "DenseNet201_A_clean_strict_seed42_validation_predictions.csv",
        "test_pred_seed42": BASE_OUTPUT_DIR / "DenseNet201_A_clean_strict_seed42_test_predictions.csv"
    },
    "C_G09": {
        "display_name": "C-G09",
        "manifest": BASE_OUTPUT_DIR / "v5_split_manifest_B_G09_Date_Magnification.csv",
        "val_pred_seed42": BASE_OUTPUT_DIR / "DenseNet201_C_G09_seed42_validation_predictions.csv",
        "test_pred_seed42": BASE_OUTPUT_DIR / "DenseNet201_C_G09_seed42_test_predictions.csv"
    }
}

# ----------------------------------------------------------
# Determinism
# ----------------------------------------------------------
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def find_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    return None


def normalize_class_name(value):
    value = str(value).strip().replace("_", " ")

    if value.lower() == "nipah virus":
        return "Nipah virus"
    if value.lower() == "rift valley":
        return "Rift Valley"

    return value


def normalize_split_name(value):
    value = str(value).strip().lower()

    if value in ["val", "valid", "validation"]:
        return "validation"
    if value == "test":
        return "test"
    if value == "train":
        return "train"

    return value


def get_drive_free_gb():
    usage = shutil.disk_usage("/content/drive/MyDrive")
    return usage.free / (1024 ** 3)


def storage_gate_startup():
    free_gb = get_drive_free_gb()
    print(f"Drive free space estimate: {free_gb:.2f} GB")

    if free_gb < 0.8:
        raise RuntimeError(f"Storage critical: {free_gb:.2f} GB free. Stop before training.")
    elif free_gb < 1.5:
        print(f"Storage warning: {free_gb:.2f} GB free. Continuing with caution.")


storage_gate_startup()


def load_protocol_split(protocol_tag, cfg):
    manifest_path = cfg["manifest"]
    val_pred_path = cfg["val_pred_seed42"]
    test_pred_path = cfg["test_pred_seed42"]

    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")

    if not val_pred_path.exists():
        raise FileNotFoundError(f"Validation prediction file not found: {val_pred_path}")

    if not test_pred_path.exists():
        raise FileNotFoundError(f"Test prediction file not found: {test_pred_path}")

    manifest = pd.read_csv(manifest_path)
    val_pred = pd.read_csv(val_pred_path)
    test_pred = pd.read_csv(test_pred_path)

    print("\n" + "=" * 90)
    print(f"SPLIT VERIFICATION: {cfg['display_name']} ({protocol_tag})")
    print("=" * 90)
    print(f"Manifest: {manifest_path}")
    print(f"Manifest rows: {len(manifest)}")
    print(f"Manifest columns: {list(manifest.columns)}")
    print(f"Seed42 validation predictions: {len(val_pred)}")
    print(f"Seed42 test predictions      : {len(test_pred)}")

    path_col = find_col(manifest, ["filepath", "image_path", "file_path", "path"])
    split_col = find_col(manifest, ["split", "set", "partition", "split_assignment", "subset"])
    class_col = find_col(manifest, ["class_name", "true_class", "label_name", "class"])
    label_col = find_col(manifest, ["true_label", "label", "target", "class_idx", "class_index"])

    if path_col is None:
        raise ValueError(f"No filepath-like column found in manifest. Columns: {list(manifest.columns)}")

    val_path_col = find_col(val_pred, ["filepath", "image_path", "file_path", "path"])
    test_path_col = find_col(test_pred, ["filepath", "image_path", "file_path", "path"])

    if val_path_col is None or test_path_col is None:
        raise ValueError("Baseline prediction CSV must contain filepath-like column.")

    work_df = manifest.copy()
    work_df["filepath"] = work_df[path_col].astype(str)

    if class_col is not None:
        work_df["class_name"] = work_df[class_col].apply(normalize_class_name)
    elif label_col is not None:
        label_to_class = {i: CLASS_NAMES_REFERENCE[i] for i in range(len(CLASS_NAMES_REFERENCE))}
        work_df["class_name"] = work_df[label_col].astype(int).map(label_to_class)
    else:
        work_df["class_name"] = work_df["filepath"].apply(lambda p: normalize_class_name(Path(p).parent.name))

    # Prefer explicit split column if available
    if split_col is not None:
        work_df["split"] = work_df[split_col].apply(normalize_split_name)
        print(f"Found split column: {split_col}")
        print(work_df["split"].value_counts())
    else:
        print("No explicit split column. Reconstructing split from baseline seed42 predictions.")

        val_paths = set(val_pred[val_path_col].astype(str).tolist())
        test_paths = set(test_pred[test_path_col].astype(str).tolist())

        def assign_split(path):
            if path in val_paths:
                return "validation"
            if path in test_paths:
                return "test"
            return "train"

        work_df["split"] = work_df["filepath"].apply(assign_split)

    split_df = work_df[["filepath", "class_name", "split"]].copy()

    class_names_detected = sorted(split_df["class_name"].dropna().unique().tolist())

    if class_names_detected != CLASS_NAMES_REFERENCE:
        print("Detected classes:", class_names_detected)
        print("Expected classes:", CLASS_NAMES_REFERENCE)
        raise ValueError(f"Class name mismatch for {protocol_tag}.")

    train_count = int((split_df["split"] == "train").sum())
    val_count = int((split_df["split"] == "validation").sum())
    test_count = int((split_df["split"] == "test").sum())

    expected_val_count = len(val_pred)
    expected_test_count = len(test_pred)

    print("\nSplit counts:")
    print(f"Train      : {train_count}")
    print(f"Validation : {val_count}")
    print(f"Test       : {test_count}")

    print("\nExpected from baseline predictions:")
    print(f"Validation : {expected_val_count}")
    print(f"Test       : {expected_test_count}")

    val_diff_rate = abs(val_count - expected_val_count) / max(1, expected_val_count)
    test_diff_rate = abs(test_count - expected_test_count) / max(1, expected_test_count)

    if val_diff_rate > 0.02 or test_diff_rate > 0.02:
        raise ValueError(
            f"{protocol_tag} split mismatch >2%. "
            f"val_count={val_count}, expected={expected_val_count}; "
            f"test_count={test_count}, expected={expected_test_count}"
        )

    print("Split verification passed.")

    return split_df, {
        "protocol": protocol_tag,
        "display_name": cfg["display_name"],
        "manifest": str(manifest_path),
        "val_prediction_seed42": str(val_pred_path),
        "test_prediction_seed42": str(test_pred_path),
        "train_count": train_count,
        "validation_count": val_count,
        "test_count": test_count,
        "expected_validation_count": expected_val_count,
        "expected_test_count": expected_test_count,
        "validation_match_within_2pct": bool(val_diff_rate <= 0.02),
        "test_match_within_2pct": bool(test_diff_rate <= 0.02),
        "class_names": CLASS_NAMES_REFERENCE
    }


PROTOCOL_SPLITS = {}
SPLIT_VERIFICATION = {}

for protocol_tag, cfg in PROTOCOL_CONFIGS.items():
    split_df, verification = load_protocol_split(protocol_tag, cfg)
    PROTOCOL_SPLITS[protocol_tag] = split_df
    SPLIT_VERIFICATION[protocol_tag] = verification

split_verification_path = OUTPUT_DIR / "CP4_protocolAcleanCG09_split_verification.json"

with open(split_verification_path, "w") as f:
    json.dump(SPLIT_VERIFICATION, f, indent=2)

print(f"Device: {DEVICE}")
print(f"Split verification JSON: {split_verification_path}")

In [ ]:
# ==========================================================
# CP4C-1 — Dataset, Dataloader, Scheduler, Mixup, TTA
# ==========================================================

class TEMVirusDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["filepath"]
        class_name = row["class_name"]
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("L")
        image = Image.merge("RGB", (image, image, image))

        if self.transform is not None:
            image = self.transform(image)

        return image, label, image_path


class_names = CLASS_NAMES_REFERENCE
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}


def build_transforms():
    train_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    eval_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    return train_transform, eval_transform


def build_dataloaders(protocol_tag, seed):
    protocol_df = PROTOCOL_SPLITS[protocol_tag]

    train_df = protocol_df[protocol_df["split"] == "train"].copy()
    val_df = protocol_df[protocol_df["split"] == "validation"].copy()
    test_df = protocol_df[protocol_df["split"] == "test"].copy()

    train_transform, eval_transform = build_transforms()

    train_dataset = TEMVirusDataset(train_df, class_to_idx, transform=train_transform)
    val_dataset = TEMVirusDataset(val_df, class_to_idx, transform=eval_transform)
    test_dataset = TEMVirusDataset(test_df, class_to_idx, transform=eval_transform)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


def make_model(arch_key):
    return timm.create_model(
        ARCHITECTURES[arch_key]["timm_name"],
        pretrained=True,
        num_classes=len(class_names)
    )


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model):
    total_bytes = 0

    for param in model.parameters():
        total_bytes += param.numel() * param.element_size()

    for buffer in model.buffers():
        total_bytes += buffer.numel() * buffer.element_size()

    return total_bytes / (1024 ** 2)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    }


class WarmupCosineRestartScheduler:
    def __init__(
        self,
        optimizer,
        warmup_epochs=5,
        warmup_start_lr=1e-6,
        base_lr=1e-4,
        min_lr=1e-6,
        t_0=10,
        t_mult=2
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.warmup_start_lr = warmup_start_lr
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.t_0 = t_0
        self.t_mult = t_mult
        self.last_epoch = -1

        self.restart_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=t_0,
            T_mult=t_mult,
            eta_min=min_lr
        )

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            progress = epoch / max(1, self.warmup_epochs - 1)
            lr = self.warmup_start_lr + progress * (self.base_lr - self.warmup_start_lr)

            for group in self.optimizer.param_groups:
                group["lr"] = lr
        else:
            self.restart_scheduler.step(epoch - self.warmup_epochs)
            lr = self.optimizer.param_groups[0]["lr"]

        self.last_epoch = epoch
        return lr

    def state_dict(self):
        return {
            "warmup_epochs": self.warmup_epochs,
            "warmup_start_lr": self.warmup_start_lr,
            "base_lr": self.base_lr,
            "min_lr": self.min_lr,
            "t_0": self.t_0,
            "t_mult": self.t_mult,
            "last_epoch": self.last_epoch,
            "restart_scheduler": self.restart_scheduler.state_dict()
        }

    def load_state_dict(self, state_dict):
        self.warmup_epochs = state_dict["warmup_epochs"]
        self.warmup_start_lr = state_dict["warmup_start_lr"]
        self.base_lr = state_dict["base_lr"]
        self.min_lr = state_dict["min_lr"]
        self.t_0 = state_dict["t_0"]
        self.t_mult = state_dict["t_mult"]
        self.last_epoch = state_dict["last_epoch"]
        self.restart_scheduler.load_state_dict(state_dict["restart_scheduler"])


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


def tta_predict(model, images):
    augmentations = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[2, 3])
    ]

    probs_list = []

    for aug in augmentations:
        logits = model(aug(images))
        probs = torch.softmax(logits.float(), dim=1)
        probs_list.append(probs)

    return torch.mean(torch.stack(probs_list, dim=0), dim=0)

In [ ]:
# ==========================================================
# CP4C-2 — Training and Evaluation Functions
# ==========================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, arch_key, protocol_tag, seed, epoch_num):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    last_grad_norm = 0.0
    mixup_batches = 0

    progress = tqdm(
        loader,
        desc=f"{arch_key} {protocol_tag} seed {seed} train epoch {epoch_num}",
        leave=True
    )

    for images, labels, _paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        apply_mixup = np.random.random() < MIXUP_PROB

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            if apply_mixup:
                mixed_images, y_a, y_b, lam = mixup_data(images, labels, alpha=MIXUP_ALPHA)
                logits = model(mixed_images)
                loss = mixup_loss(criterion, logits, y_a, y_b, lam)
                mixup_batches += 1
            else:
                logits = model(images)
                loss = criterion(logits, labels)

        if torch.isnan(loss):
            raise ValueError(f"NaN loss detected: {arch_key}, {protocol_tag}, seed={seed}, epoch={epoch_num}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_MAX_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        preds = torch.argmax(logits.detach(), dim=1)

        running_loss += loss.item() * images.size(0)
        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        last_grad_norm = float(grad_norm)

        progress.set_postfix({
            "loss": f"{loss.item():.4f}",
            "grad": f"{last_grad_norm:.3f}",
            "mixup": mixup_batches
        })

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))
    metrics["grad_norm_last"] = float(last_grad_norm)
    metrics["mixup_batches"] = int(mixup_batches)

    return metrics


@torch.no_grad()
def evaluate_with_tta(model, loader, criterion, arch_key, protocol_tag, seed, split_name):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_paths = []
    all_probs = []

    progress = tqdm(
        loader,
        desc=f"{arch_key} {protocol_tag} seed {seed} evaluate {split_name} with TTA",
        leave=True
    )

    for images, labels, paths in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            probs = tta_predict(model, images)
            logits_for_loss = model(images)
            loss = criterion(logits_for_loss, labels)

        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_targets.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_paths.extend(list(paths))
        all_probs.append(probs.detach().cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))

    per_class_f1_values = f1_score(
        all_targets,
        all_preds,
        average=None,
        zero_division=0
    )

    per_class_f1 = {
        class_names[i]: float(per_class_f1_values[i])
        for i in range(len(class_names))
    }

    report_dict = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_txt = classification_report(
        all_targets,
        all_preds,
        target_names=class_names,
        zero_division=0
    )

    cm = confusion_matrix(
        all_targets,
        all_preds,
        labels=list(range(len(class_names)))
    )

    pred_df = pd.DataFrame({
        "filename": [Path(p).name for p in all_paths],
        "image_path": all_paths,
        "true_label": [class_names[i] for i in all_targets],
        "pred_label": [class_names[i] for i in all_preds],
        "confidence": all_probs.max(axis=1)
    })

    for i, class_name in enumerate(class_names):
        safe_name = class_name.replace(" ", "_")
        pred_df[f"prob_{safe_name}"] = all_probs[:, i]

    return {
        "metrics": metrics,
        "per_class_f1": per_class_f1,
        "classification_report": report_dict,
        "classification_report_txt": report_txt,
        "confusion_matrix": cm,
        "prediction_df": pred_df,
        "softmax": all_probs,
        "targets": all_targets,
        "preds": all_preds,
        "paths": all_paths
    }

In [ ]:
# ==========================================================
# CP4C-3 — Save, Checkpoint, Plot, Storage Helpers
# ==========================================================

def get_run_paths(arch_key, protocol_tag, seed):
    prefix = f"{arch_key}_{protocol_tag}_seed{seed}_ttamixup"

    return {
        "prefix": prefix,
        "best_ckpt": CHECKPOINT_DIR / f"{prefix}_best.pt",
        "last_ckpt": CHECKPOINT_DIR / f"{prefix}_last.pt",
        "history_csv": OUTPUT_DIR / f"{prefix}_training_history.csv",
        "history_json": OUTPUT_DIR / f"{prefix}_training_history.json",
        "training_summary_json": OUTPUT_DIR / f"{prefix}_training_summary.json",
        "final_summary_json": OUTPUT_DIR / f"{prefix}_CP4_final_summary.json",
        "metrics_json": OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json",
        "metrics_csv": OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.csv",
        "per_class_f1_csv": OUTPUT_DIR / f"{prefix}_test_per_class_f1.csv",
        "val_softmax": OUTPUT_DIR / f"{prefix}_val_softmax.npy",
        "test_softmax": OUTPUT_DIR / f"{prefix}_test_softmax.npy",
        "learning_curve_png": OUTPUT_DIR / f"{prefix}_learning_curve.png",
        "confusion_matrix_png": OUTPUT_DIR / f"{prefix}_confusion_matrix_test.png"
    }


def storage_gate():
    free_gb = get_drive_free_gb()
    print(f"Drive free space estimate: {free_gb:.2f} GB")

    if free_gb < 0.8:
        raise RuntimeError(f"Storage gate STOP: free space <0.8 GB ({free_gb:.2f} GB).")

    if free_gb < 1.5:
        print(f"Storage gate WARNING: free space is low ({free_gb:.2f} GB).")


def cleanup_last_checkpoint_if_finalized(paths):
    if (
        paths["final_summary_json"].exists()
        and paths["val_softmax"].exists()
        and paths["test_softmax"].exists()
        and paths["last_ckpt"].exists()
    ):
        size_mb = paths["last_ckpt"].stat().st_size / (1024 ** 2)
        paths["last_ckpt"].unlink()
        print(f"Cleaned _last.pt: {paths['last_ckpt'].name} ({size_mb:.1f} MB)")


def save_history(history, paths):
    history_df = pd.DataFrame(history)
    history_df.to_csv(paths["history_csv"], index=False)

    with open(paths["history_json"], "w") as f:
        json.dump(history, f, indent=2)

    return history_df


def save_last_checkpoint(model, optimizer, scheduler, epoch, best_metric, best_epoch, patience_counter, history, paths, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_metric": best_metric,
            "best_epoch": best_epoch,
            "patience_counter": patience_counter,
            "history": history,
            "config": config
        },
        paths["last_ckpt"]
    )


def save_best_checkpoint(model, optimizer, scheduler, epoch, val_macro_f1, val_metrics, paths, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_macro_f1": val_macro_f1,
            "val_metrics": val_metrics,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "config": config
        },
        paths["best_ckpt"]
    )


def load_last_checkpoint(model, optimizer, scheduler, paths):
    if paths["last_ckpt"].exists():
        ckpt = torch.load(paths["last_ckpt"], map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_metric = float(ckpt["best_metric"])
        best_epoch = ckpt["best_epoch"]
        patience_counter = int(ckpt["patience_counter"])
        history = ckpt["history"]

        print("=== Resume checkpoint found ===")
        print(f"Start epoch       : {start_epoch + 1}")
        print(f"Best val macro F1 : {best_metric:.4f}")
        print(f"Best epoch        : {None if best_epoch is None else best_epoch + 1}")

        return start_epoch, best_metric, best_epoch, patience_counter, history

    return 0, -1.0, None, 0, []


def save_split_outputs(arch_key, protocol_tag, seed, split_name, outputs):
    prefix = OUTPUT_DIR / f"{arch_key}_{protocol_tag}_seed{seed}_ttamixup_{split_name}"

    report_csv = Path(str(prefix) + "_classification_report.csv")
    report_json = Path(str(prefix) + "_classification_report.json")
    report_txt = Path(str(prefix) + "_classification_report.txt")
    cm_csv = Path(str(prefix) + "_confusion_matrix.csv")
    pred_csv = Path(str(prefix) + "_predictions.csv")

    pd.DataFrame(outputs["classification_report"]).transpose().to_csv(report_csv)

    with open(report_json, "w") as f:
        json.dump(outputs["classification_report"], f, indent=2)

    with open(report_txt, "w") as f:
        f.write(outputs["classification_report_txt"])

    cm_df = pd.DataFrame(
        outputs["confusion_matrix"],
        index=class_names,
        columns=class_names
    )
    cm_df.to_csv(cm_csv)

    outputs["prediction_df"].to_csv(pred_csv, index=False)

    return {
        "classification_report_csv": str(report_csv),
        "classification_report_json": str(report_json),
        "classification_report_txt": str(report_txt),
        "confusion_matrix_csv": str(cm_csv),
        "predictions_csv": str(pred_csv)
    }


def plot_outputs(paths, arch_key, protocol_tag, seed, test_outputs):
    history_df = pd.read_csv(paths["history_csv"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    axes[0, 1].plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    axes[0, 1].set_title("Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
    axes[1, 0].set_title("Validation Macro F1")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(history_df["epoch"], history_df["lr"], label="lr")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f"{arch_key} {protocol_tag} seed {seed} — TTA + Mixup", y=1.02)
    plt.tight_layout()
    plt.savefig(paths["learning_curve_png"], dpi=300, bbox_inches="tight")
    plt.show()

    cm = test_outputs["confusion_matrix"]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm)

    ax.set_title(f"{arch_key} {protocol_tag} seed {seed} — Test Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(paths["confusion_matrix_png"], dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ==========================================================
# CP4C-4 — Single Protocol-Model-Seed Runner
# ==========================================================

def run_protocol_model_seed(protocol_tag, arch_key, seed, skip_completed=True):
    set_seed(seed)

    paths = get_run_paths(arch_key, protocol_tag, seed)

    if skip_completed and paths["final_summary_json"].exists() and paths["test_softmax"].exists() and paths["val_softmax"].exists():
        print(f"Run already completed: {arch_key} {protocol_tag} seed {seed}")
        cleanup_last_checkpoint_if_finalized(paths)
        with open(paths["final_summary_json"], "r") as f:
            return json.load(f)

    storage_gate()

    train_loader, val_loader, test_loader = build_dataloaders(protocol_tag, seed)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = make_model(arch_key).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = WarmupCosineRestartScheduler(
        optimizer=optimizer,
        warmup_epochs=5,
        warmup_start_lr=WARMUP_START_LR,
        base_lr=BASE_LR,
        min_lr=MIN_LR,
        t_0=10,
        t_mult=2
    )

    try:
        scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    config = {
        "model": arch_key,
        "timm_model_name": ARCHITECTURES[arch_key]["timm_name"],
        "protocol": protocol_tag,
        "protocol_display": PROTOCOL_CONFIGS[protocol_tag]["display_name"],
        "seed": seed,
        "max_epochs": MAX_EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE,
        "optimizer": "AdamW",
        "base_lr": BASE_LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "mixup_alpha": MIXUP_ALPHA,
        "mixup_probability": MIXUP_PROB,
        "scheduler": "warmup_cosine_restarts",
        "tta": "original_hflip_vflip_both",
        "grad_clip_max_norm": GRAD_CLIP_MAX_NORM,
        "normalization": "ImageNet",
        "pretrained": True,
        "split_source": str(PROTOCOL_CONFIGS[protocol_tag]["manifest"])
    }

    start_epoch, best_metric, best_epoch, patience_counter, history = load_last_checkpoint(
        model,
        optimizer,
        scheduler,
        paths
    )

    params_count = count_trainable_parameters(model)
    model_size_mb = estimate_model_size_mb(model)

    stopped_by_gate = False
    early_stopped = False
    anomaly_log = []

    print("\n" + "=" * 90)
    print(f"CP4C run: {arch_key}, {protocol_tag}, seed {seed}")
    print("=" * 90)
    print(f"Device       : {DEVICE}")
    print(f"Epochs       : {MAX_EPOCHS}")
    print(f"Batch size   : {BATCH_SIZE}")
    print(f"Parameters   : {params_count:,}")
    print(f"Model size   : {model_size_mb:.2f} MB")

    training_start = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):
        epoch_num = epoch + 1
        epoch_start = time.time()

        lr = scheduler.step(epoch)

        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            arch_key,
            protocol_tag,
            seed,
            epoch_num
        )

        val_outputs = evaluate_with_tta(
            model,
            val_loader,
            criterion,
            arch_key,
            protocol_tag,
            seed,
            split_name="validation"
        )

        val_metrics = val_outputs["metrics"]
        val_macro_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]

        improved = val_macro_f1 > best_metric + 1e-8

        if improved:
            best_metric = val_macro_f1
            best_epoch = epoch
            patience_counter = 0

            save_best_checkpoint(
                model,
                optimizer,
                scheduler,
                epoch,
                val_macro_f1,
                val_metrics,
                paths,
                config
            )
        else:
            patience_counter += 1

        row = {
            "model": arch_key,
            "protocol": protocol_tag,
            "seed": seed,
            "epoch": epoch_num,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_grad_norm_last": train_metrics["grad_norm_last"],
            "train_mixup_batches": train_metrics["mixup_batches"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "lr": lr,
            "epoch_time_sec": time.time() - epoch_start,
            "best_val_macro_f1_so_far": best_metric,
            "best_epoch_so_far": None if best_epoch is None else best_epoch + 1,
            "patience_counter": patience_counter,
            "improved": improved
        }

        history.append(row)
        save_history(history, paths)

        save_last_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            best_metric,
            best_epoch,
            patience_counter,
            history,
            paths,
            config
        )

        print("\n=== Epoch Summary ===")
        print(json.dumps(row, indent=2))

        if epoch_num == 5:
            if val_acc < EPOCH5_WARN_VAL_ACC:
                anomaly_log.append(f"Epoch 5 warning: val_acc={val_acc:.4f}")
                print(f"\nEpoch 5 warning: val_acc={val_acc:.4f}")

            if val_acc < EPOCH5_STOP_VAL_ACC:
                stopped_by_gate = True
                anomaly_log.append(f"Epoch 5 gate stopped: val_acc={val_acc:.4f}")
                print("\n=== Epoch 5 Gate Stop ===")
                break

        if patience_counter >= EARLY_STOP_PATIENCE:
            early_stopped = True
            print("\n=== Early Stopping Triggered ===")
            break

    training_time_sec = time.time() - training_start

    peak_gpu_memory_mb = None
    if DEVICE.type == "cuda":
        peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    training_summary = {
        "model": arch_key,
        "protocol": protocol_tag,
        "seed": seed,
        "total_epochs_run": len(history),
        "best_epoch": None if best_epoch is None else best_epoch + 1,
        "best_val_macro_f1": None if best_metric < 0 else best_metric,
        "total_training_time_sec": training_time_sec,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "early_stopped": early_stopped,
        "stopped_by_gate": stopped_by_gate,
        "anomaly_log": anomaly_log,
        "params_count_M": params_count / 1e6,
        "model_size_mb": model_size_mb,
        "best_checkpoint": str(paths["best_ckpt"]),
        "last_checkpoint": str(paths["last_ckpt"]),
        "history_csv": str(paths["history_csv"]),
        "config": config
    }

    with open(paths["training_summary_json"], "w") as f:
        json.dump(training_summary, f, indent=2)

    if not paths["best_ckpt"].exists():
        raise FileNotFoundError(f"No best checkpoint saved for {arch_key} {protocol_tag} seed {seed}")

    best_ckpt = torch.load(paths["best_ckpt"], map_location=DEVICE)

    best_model = make_model(arch_key).to(DEVICE)
    best_model.load_state_dict(best_ckpt["model_state_dict"])

    criterion_eval = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    validation_outputs = evaluate_with_tta(
        best_model,
        val_loader,
        criterion_eval,
        arch_key,
        protocol_tag,
        seed,
        split_name="validation"
    )

    test_outputs = evaluate_with_tta(
        best_model,
        test_loader,
        criterion_eval,
        arch_key,
        protocol_tag,
        seed,
        split_name="test"
    )

    np.save(paths["val_softmax"], validation_outputs["softmax"])
    np.save(paths["test_softmax"], test_outputs["softmax"])

    validation_files = save_split_outputs(arch_key, protocol_tag, seed, "validation", validation_outputs)
    test_files = save_split_outputs(arch_key, protocol_tag, seed, "test", test_outputs)

    metrics_summary = {
        "model": arch_key,
        "protocol": protocol_tag,
        "seed": seed,
        "best_epoch": int(best_ckpt["epoch"]) + 1,
        "best_val_macro_f1_checkpoint": float(best_ckpt["val_macro_f1"]),
        "validation": validation_outputs["metrics"],
        "test": test_outputs["metrics"],
        "test_per_class_f1": test_outputs["per_class_f1"],
        "validation_files": validation_files,
        "test_files": test_files,
        "val_softmax_npy": str(paths["val_softmax"]),
        "test_softmax_npy": str(paths["test_softmax"])
    }

    with open(paths["metrics_json"], "w") as f:
        json.dump(metrics_summary, f, indent=2)

    pd.DataFrame([
        {"split": "validation", **validation_outputs["metrics"]},
        {"split": "test", **test_outputs["metrics"]}
    ]).to_csv(paths["metrics_csv"], index=False)

    pd.DataFrame({
        "class_name": list(test_outputs["per_class_f1"].keys()),
        "test_f1": list(test_outputs["per_class_f1"].values())
    }).to_csv(paths["per_class_f1_csv"], index=False)

    plot_outputs(paths, arch_key, protocol_tag, seed, test_outputs)

    val_softmax_shape = list(np.load(paths["val_softmax"]).shape)
    test_softmax_shape = list(np.load(paths["test_softmax"]).shape)

    final_summary = {
        "model": arch_key,
        "protocol": protocol_tag,
        "seed": seed,
        "best_epoch": metrics_summary["best_epoch"],
        "best_val_macro_f1": metrics_summary["best_val_macro_f1_checkpoint"],
        "test_accuracy": metrics_summary["test"]["accuracy"],
        "test_macro_f1": metrics_summary["test"]["macro_f1"],
        "test_weighted_f1": metrics_summary["test"]["weighted_f1"],
        "test_macro_precision": metrics_summary["test"]["macro_precision"],
        "test_macro_recall": metrics_summary["test"]["macro_recall"],
        "total_epochs_run": training_summary["total_epochs_run"],
        "total_training_time_sec": training_summary["total_training_time_sec"],
        "peak_gpu_memory_mb": training_summary["peak_gpu_memory_mb"],
        "early_stopped": training_summary["early_stopped"],
        "stopped_by_gate": training_summary["stopped_by_gate"],
        "anomaly_log": training_summary["anomaly_log"],
        "val_softmax_shape": val_softmax_shape,
        "test_softmax_shape": test_softmax_shape,
        "val_softmax_saved": paths["val_softmax"].exists(),
        "test_softmax_saved": paths["test_softmax"].exists(),
        "val_softmax_npy": str(paths["val_softmax"]),
        "test_softmax_npy": str(paths["test_softmax"]),
        "training_history_csv": str(paths["history_csv"]),
        "learning_curve_png": str(paths["learning_curve_png"]),
        "confusion_matrix_test_png": str(paths["confusion_matrix_png"]),
        "per_class_f1_csv": str(paths["per_class_f1_csv"]),
        "metrics_summary_json": str(paths["metrics_json"]),
        "final_summary_json": str(paths["final_summary_json"]),
        "best_checkpoint": str(paths["best_ckpt"]),
        "last_checkpoint": str(paths["last_ckpt"])
    }

    with open(paths["final_summary_json"], "w") as f:
        json.dump(final_summary, f, indent=2)

    cleanup_last_checkpoint_if_finalized(paths)

    print("\n=== CP4C Run Final Summary ===")
    print(json.dumps(final_summary, indent=2))

    del model
    del best_model
    gc.collect()
    torch.cuda.empty_cache()

    return final_summary

In [ ]:
# ==========================================================
# CP4C-5 — Run 12 Runs
# A-clean-strict + C-G09
# DenseNet201 + EffNetV2S
# Seeds 42, 123, 456
# ==========================================================

run_plan = []

for protocol_tag in ["A_clean_strict", "C_G09"]:
    for arch_key in ["DenseNet201", "EffNetV2S"]:
        for seed in SEEDS:
            run_plan.append((protocol_tag, arch_key, seed))

cp4c_run_results = []
cp4c_error_log = []

for protocol_tag, arch_key, seed in run_plan:
    try:
        result = run_protocol_model_seed(
            protocol_tag=protocol_tag,
            arch_key=arch_key,
            seed=seed,
            skip_completed=True
        )
        cp4c_run_results.append(result)

    except RuntimeError as e:
        error_message = str(e)

        if "storage gate stop" in error_message.lower():
            error_record = {
                "protocol": protocol_tag,
                "model": arch_key,
                "seed": seed,
                "error_type": "StorageGateStop",
                "error_message": error_message
            }
            cp4c_error_log.append(error_record)
            print("\n=== STORAGE GATE STOP ===")
            print(json.dumps(error_record, indent=2))
            print("Stopping all remaining CP4C runs.")
            break

        elif "out of memory" in error_message.lower():
            torch.cuda.empty_cache()
            error_record = {
                "protocol": protocol_tag,
                "model": arch_key,
                "seed": seed,
                "error_type": "OOM",
                "error_message": error_message
            }
        else:
            error_record = {
                "protocol": protocol_tag,
                "model": arch_key,
                "seed": seed,
                "error_type": "RuntimeError",
                "error_message": error_message
            }

        cp4c_error_log.append(error_record)

        print("\n=== RUN ERROR ===")
        print(json.dumps(error_record, indent=2))
        print("Continuing to next run.")

    except Exception as e:
        error_record = {
            "protocol": protocol_tag,
            "model": arch_key,
            "seed": seed,
            "error_type": type(e).__name__,
            "error_message": str(e)
        }

        cp4c_error_log.append(error_record)

        print("\n=== RUN ERROR ===")
        print(json.dumps(error_record, indent=2))
        print("Continuing to next run.")

error_log_path = OUTPUT_DIR / "CP4_protocolAcleanCG09_error_log.json"

with open(error_log_path, "w") as f:
    json.dump(cp4c_error_log, f, indent=2)

print(f"Completed runs : {len(cp4c_run_results)}")
print(f"Failed runs    : {len(cp4c_error_log)}")
print(f"Error log      : {error_log_path}")

In [ ]:
# ==========================================================
# CP4C-6 — Aggregate Summary, Softmax Check, Review ZIP
# ==========================================================

def safe_load_json(path):
    path = Path(path)

    if not path.exists():
        return None

    with open(path, "r") as f:
        return json.load(f)


def build_summary_row_from_json(summary, protocol_tag, model_name, seed):
    if summary is None:
        return {
            "model": model_name,
            "protocol": protocol_tag,
            "seed": seed,
            "test_accuracy": None,
            "test_macro_f1": None,
            "test_weighted_f1": None,
            "test_macro_precision": None,
            "test_macro_recall": None,
            "best_epoch": None,
            "best_val_macro_f1": None,
            "total_epochs": None,
            "training_time_sec": None,
            "peak_gpu_memory_mb": None,
            "val_softmax_saved": False,
            "test_softmax_saved": False,
            "val_softmax_shape": None,
            "test_softmax_shape": None,
            "final_summary_json": None,
            "status": "missing"
        }

    return {
        "model": model_name,
        "protocol": summary.get("protocol", protocol_tag),
        "seed": seed,
        "test_accuracy": summary.get("test_accuracy"),
        "test_macro_f1": summary.get("test_macro_f1"),
        "test_weighted_f1": summary.get("test_weighted_f1"),
        "test_macro_precision": summary.get("test_macro_precision"),
        "test_macro_recall": summary.get("test_macro_recall"),
        "best_epoch": summary.get("best_epoch"),
        "best_val_macro_f1": summary.get("best_val_macro_f1"),
        "total_epochs": summary.get("total_epochs_run"),
        "training_time_sec": summary.get("total_training_time_sec"),
        "peak_gpu_memory_mb": summary.get("peak_gpu_memory_mb"),
        "val_softmax_saved": summary.get("val_softmax_saved"),
        "test_softmax_saved": summary.get("test_softmax_saved"),
        "val_softmax_shape": summary.get("val_softmax_shape"),
        "test_softmax_shape": summary.get("test_softmax_shape"),
        "final_summary_json": summary.get("final_summary_json"),
        "status": "ok"
    }


summary_rows = []

for protocol_tag in ["A_clean_strict", "C_G09"]:
    for arch_key in ["DenseNet201", "EffNetV2S"]:
        for seed in SEEDS:
            paths = get_run_paths(arch_key, protocol_tag, seed)
            summary = safe_load_json(paths["final_summary_json"])
            summary_rows.append(build_summary_row_from_json(summary, protocol_tag, arch_key, seed))

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "CP4_protocolAcleanCG09_multiseed_summary.csv"
summary_json = OUTPUT_DIR / "CP4_protocolAcleanCG09_multiseed_summary.json"

summary_df.to_csv(summary_csv, index=False)

with open(summary_json, "w") as f:
    json.dump(summary_rows, f, indent=2)

print("=== CP4 A-clean-strict and C-G09 Multi-seed Summary ===")
display(summary_df)

# ----------------------------------------------------------
# Softmax verification
# ----------------------------------------------------------
softmax_rows = []

for protocol_tag in ["A_clean_strict", "C_G09"]:
    for arch_key in ["DenseNet201", "EffNetV2S"]:
        for seed in SEEDS:
            paths = get_run_paths(arch_key, protocol_tag, seed)

            val_path = paths["val_softmax"]
            test_path = paths["test_softmax"]

            val_exists = val_path.exists()
            test_exists = test_path.exists()

            val_shape = list(np.load(val_path).shape) if val_exists else None
            test_shape = list(np.load(test_path).shape) if test_exists else None

            softmax_rows.append({
                "protocol": protocol_tag,
                "model": arch_key,
                "seed": seed,
                "val_softmax_exists": val_exists,
                "test_softmax_exists": test_exists,
                "val_softmax_shape": val_shape,
                "test_softmax_shape": test_shape,
                "val_softmax_path": str(val_path),
                "test_softmax_path": str(test_path)
            })

softmax_df = pd.DataFrame(softmax_rows)

softmax_csv = OUTPUT_DIR / "CP4_protocolAcleanCG09_softmax_verification.csv"
softmax_df.to_csv(softmax_csv, index=False)

print("\n=== CP4 A-clean-strict and C-G09 Softmax Verification ===")
display(softmax_df)

# ----------------------------------------------------------
# Aggregate by protocol and model
# ----------------------------------------------------------
valid_summary_df = summary_df.dropna(subset=["test_accuracy"]).copy()

aggregate_df = (
    valid_summary_df
    .groupby(["protocol", "model"])
    .agg(
        n_runs=("test_accuracy", "count"),
        mean_test_accuracy=("test_accuracy", "mean"),
        std_test_accuracy=("test_accuracy", "std"),
        mean_test_macro_f1=("test_macro_f1", "mean"),
        std_test_macro_f1=("test_macro_f1", "std"),
        mean_test_weighted_f1=("test_weighted_f1", "mean"),
        std_test_weighted_f1=("test_weighted_f1", "std"),
        mean_training_time_sec=("training_time_sec", "mean")
    )
    .reset_index()
)

aggregate_csv = OUTPUT_DIR / "CP4_protocolAcleanCG09_multiseed_aggregate_by_model.csv"
aggregate_df.to_csv(aggregate_csv, index=False)

print("\n=== CP4 A-clean-strict and C-G09 Aggregate by Protocol and Model ===")
display(aggregate_df)

# ----------------------------------------------------------
# Package review files
# ----------------------------------------------------------
review_files = [
    summary_csv,
    summary_json,
    softmax_csv,
    aggregate_csv,
    split_verification_path,
    OUTPUT_DIR / "CP4_protocolAcleanCG09_error_log.json"
]

review_zip = OUTPUT_DIR / "CP4_protocolAcleanCG09_multiseed_review_files.zip"

with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in review_files:
        file_path = Path(file_path)

        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
            print("Added:", file_path)
        else:
            print("Missing:", file_path)

print(f"Summary CSV      : {summary_csv}")
print(f"Summary JSON     : {summary_json}")
print(f"Softmax CSV      : {softmax_csv}")
print(f"Aggregate CSV    : {aggregate_csv}")
print(f"Split verify JSON: {split_verification_path}")
print(f"Review ZIP       : {review_zip}")

In [ ]:
# ==========================================================
# CP4D-0 — Final Ensemble Setup
# All protocols, all seeds, DenseNet201 + EffNetV2S
# READ ONLY for existing softmax/predictions
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Riset_TEM_Virus")

CP4_DIR = PROJECT_ROOT / "outputs" / "cp4_multiseed"
CP35_DIR = PROJECT_ROOT / "outputs" / "cp3_5_ceiling_push"
CP36_DIR = PROJECT_ROOT / "outputs" / "cp3_6_ensemble"

ENSEMBLE_OUT = PROJECT_ROOT / "outputs" / "cp4_ensemble"
ENSEMBLE_OUT.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Constants
# ----------------------------------------------------------
CLASS_NAMES = [
    "Adenovirus", "Astrovirus", "CCHF", "Cowpox", "Ebola", "Influenza",
    "Lassa", "Marburg", "Nipah virus", "Norovirus", "Orf", "Papilloma",
    "Rift Valley", "Rotavirus"
]

CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

PROTOCOLS = {
    "A": {
        "display_name": "Protocol A official",
        "seeds": [42, 123, 456, 789, 1024],
        "tag": "A",
        "expected_val": 2249,
        "expected_test": 1900,
        "baseline_acc_mean": 0.9585,
        "baseline_acc_std": 0.0091,
        "baseline_f1_mean": 0.9504,
        "baseline_f1_std": 0.0100
    },
    "B_G14": {
        "display_name": "Protocol B-G14",
        "seeds": [42, 123, 456, 789, 1024],
        "tag": "B_G14",
        "expected_val": 2251,
        "expected_test": 1903,
        "baseline_acc_mean": 0.9589,
        "baseline_acc_std": 0.0039,
        "baseline_f1_mean": 0.9601,
        "baseline_f1_std": 0.0043
    },
    "A_clean_strict": {
        "display_name": "Protocol A-clean strict",
        "seeds": [42, 123, 456],
        "tag": "A_clean_strict",
        "expected_val": 2249,
        "expected_test": 1900,
        "baseline_acc_mean": 0.9612,
        "baseline_acc_std": 0.0077,
        "baseline_f1_mean": 0.9536,
        "baseline_f1_std": 0.0116
    },
    "C_G09": {
        "display_name": "Protocol C-G09",
        "seeds": [42, 123, 456],
        "tag": "C_G09",
        "expected_val": 2312,
        "expected_test": 1937,
        "baseline_acc_mean": 0.9463,
        "baseline_acc_std": 0.0108,
        "baseline_f1_mean": 0.9229,
        "baseline_f1_std": 0.0161
    }
}

print(f"CP4 dir      : {CP4_DIR}")
print(f"Ensemble out : {ENSEMBLE_OUT}")

In [ ]:
# ==========================================================
# Helper functions
# ==========================================================

def label_to_index(value):
    if pd.isna(value):
        raise ValueError("Missing label value detected.")

    if isinstance(value, str):
        value = value.strip()

        if value in CLASS_TO_IDX:
            return CLASS_TO_IDX[value]

        value_norm = value.replace("_", " ")
        if value_norm in CLASS_TO_IDX:
            return CLASS_TO_IDX[value_norm]

        try:
            return int(float(value))
        except:
            raise ValueError(f"Unknown label value: {value}")

    return int(value)


def find_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    return None


def load_softmax(model, protocol_tag, seed, split):
    """
    split: 'val' or 'test'
    """
    filename = f"{model}_{protocol_tag}_seed{seed}_ttamixup_{split}_softmax.npy"
    primary = CP4_DIR / filename

    if primary.exists():
        return np.load(primary), primary

    # Special fallback for Protocol A seed 42 from CP3.5 / CP3.6
    if protocol_tag == "A" and seed == 42:
        if model == "EffNetV2S":
            fallback = CP35_DIR / f"EffNetV2S_A_seed42_ttamixup_{split}_softmax.npy"
        elif model == "DenseNet201":
            fallback = CP36_DIR / f"DenseNet201_A_seed42_ttamixup_{split}_softmax.npy"
        else:
            fallback = None

        if fallback is not None and fallback.exists():
            return np.load(fallback), fallback

    raise FileNotFoundError(f"Softmax not found: {filename}")


def load_prediction_df(model, protocol_tag, seed, split):
    """
    split: 'val' or 'test'
    """
    split_name = "validation" if split == "val" else "test"
    filename = f"{model}_{protocol_tag}_seed{seed}_ttamixup_{split_name}_predictions.csv"
    primary = CP4_DIR / filename

    if primary.exists():
        return pd.read_csv(primary), primary

    # Special fallback for Protocol A seed 42 from CP3.5 / CP3.6
    if protocol_tag == "A" and seed == 42:
        if model == "EffNetV2S":
            fallback = CP35_DIR / f"EffNetV2S_A_seed42_ttamixup_{split_name}_predictions.csv"
        elif model == "DenseNet201":
            fallback = CP36_DIR / f"DenseNet201_A_seed42_ttamixup_{split_name}_predictions.csv"
        else:
            fallback = None

        if fallback is not None and fallback.exists():
            return pd.read_csv(fallback), fallback

    raise FileNotFoundError(f"Predictions CSV not found: {filename}")


def load_true_labels(model, protocol_tag, seed, split):
    df, path = load_prediction_df(model, protocol_tag, seed, split)

    true_col = find_col(df, ["true_label", "target_class", "label", "target", "class_name"])
    if true_col is None:
        raise ValueError(f"No true-label column found in {path}. Columns: {list(df.columns)}")

    y = np.array([label_to_index(v) for v in df[true_col].values])
    return y, df, path


def load_pred_labels_if_available(model, protocol_tag, seed, split):
    df, path = load_prediction_df(model, protocol_tag, seed, split)

    pred_col = find_col(df, ["pred_label", "prediction_class", "prediction", "pred"])
    if pred_col is None:
        return None, path

    y_pred_csv = np.array([label_to_index(v) for v in df[pred_col].values])
    return y_pred_csv, path


def evaluate_predictions(y_true, softmax):
    pred = softmax.argmax(axis=1)

    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, pred, average="macro", zero_division=0))
    }


# Ensemble strategies
def ensemble_simple(s1, s2):
    return 0.5 * s1 + 0.5 * s2


def ensemble_weighted(s1, s2, w1, w2):
    total = w1 + w2
    if total <= 0:
        return ensemble_simple(s1, s2)
    return (w1 / total) * s1 + (w2 / total) * s2


def ensemble_geometric_mean(s1, s2):
    eps = 1e-10
    out = np.exp((np.log(s1 + eps) + np.log(s2 + eps)) / 2.0)
    out = out / out.sum(axis=1, keepdims=True)
    return out


def ensemble_max_confidence(s1, s2):
    max1 = s1.max(axis=1)
    max2 = s2.max(axis=1)
    use_1 = max1 > max2
    return np.where(use_1[:, None], s1, s2)


def build_strategy_softmax(strategy, dense_sm, eff_sm, dense_val_f1=None, eff_val_f1=None):
    if strategy == "simple_average":
        return ensemble_simple(dense_sm, eff_sm)

    if strategy == "weighted_by_val_f1":
        return ensemble_weighted(dense_sm, eff_sm, dense_val_f1, eff_val_f1)

    if strategy == "geometric_mean":
        return ensemble_geometric_mean(dense_sm, eff_sm)

    if strategy == "max_confidence":
        return ensemble_max_confidence(dense_sm, eff_sm)

    raise ValueError(f"Unknown strategy: {strategy}")

In [ ]:
# ==========================================================
# CP4D-1 — Compute Ensemble for All Protocol-Seed Combinations
# ==========================================================

STRATEGIES = [
    "simple_average",
    "weighted_by_val_f1",
    "geometric_mean",
    "max_confidence"
]

seed_level_rows = []
strategy_detail_rows = []
softmax_verification_rows = []
error_log = []

for protocol_key, protocol_info in PROTOCOLS.items():
    tag = protocol_info["tag"]

    print("\n" + "=" * 90)
    print(f"Processing {protocol_key} — {protocol_info['display_name']}")
    print("=" * 90)

    for seed in protocol_info["seeds"]:
        try:
            # --------------------------------------------------
            # Load softmax
            # --------------------------------------------------
            dense_val_sm, dense_val_path = load_softmax("DenseNet201", tag, seed, "val")
            dense_test_sm, dense_test_path = load_softmax("DenseNet201", tag, seed, "test")
            eff_val_sm, eff_val_path = load_softmax("EffNetV2S", tag, seed, "val")
            eff_test_sm, eff_test_path = load_softmax("EffNetV2S", tag, seed, "test")

            # --------------------------------------------------
            # Load labels
            # --------------------------------------------------
            y_val, val_df, val_pred_path = load_true_labels("DenseNet201", tag, seed, "val")
            y_test, test_df, test_pred_path = load_true_labels("DenseNet201", tag, seed, "test")

            # --------------------------------------------------
            # Shape checks
            # --------------------------------------------------
            expected_val = protocol_info["expected_val"]
            expected_test = protocol_info["expected_test"]

            assert dense_val_sm.shape == eff_val_sm.shape, f"Val model softmax mismatch: {dense_val_sm.shape} vs {eff_val_sm.shape}"
            assert dense_test_sm.shape == eff_test_sm.shape, f"Test model softmax mismatch: {dense_test_sm.shape} vs {eff_test_sm.shape}"

            assert dense_val_sm.shape == (expected_val, 14), f"Dense val shape mismatch: {dense_val_sm.shape}, expected {(expected_val, 14)}"
            assert eff_val_sm.shape == (expected_val, 14), f"Eff val shape mismatch: {eff_val_sm.shape}, expected {(expected_val, 14)}"
            assert dense_test_sm.shape == (expected_test, 14), f"Dense test shape mismatch: {dense_test_sm.shape}, expected {(expected_test, 14)}"
            assert eff_test_sm.shape == (expected_test, 14), f"Eff test shape mismatch: {eff_test_sm.shape}, expected {(expected_test, 14)}"

            assert len(y_val) == expected_val, f"y_val length mismatch: {len(y_val)}"
            assert len(y_test) == expected_test, f"y_test length mismatch: {len(y_test)}"

            # --------------------------------------------------
            # Optional alignment check against CSV predictions
            # --------------------------------------------------
            dense_test_csv_pred, dense_test_csv_path = load_pred_labels_if_available("DenseNet201", tag, seed, "test")
            eff_test_csv_pred, eff_test_csv_path = load_pred_labels_if_available("EffNetV2S", tag, seed, "test")

            dense_test_match = None
            eff_test_match = None

            if dense_test_csv_pred is not None:
                dense_test_match = float((dense_test_sm.argmax(axis=1) == dense_test_csv_pred).mean())

            if eff_test_csv_pred is not None:
                eff_test_match = float((eff_test_sm.argmax(axis=1) == eff_test_csv_pred).mean())

            if dense_test_match is not None and dense_test_match < 0.95:
                raise ValueError(f"DenseNet201 softmax/CSV alignment low: {dense_test_match:.4f}")

            if eff_test_match is not None and eff_test_match < 0.95:
                raise ValueError(f"EffNetV2S softmax/CSV alignment low: {eff_test_match:.4f}")

            softmax_verification_rows.append({
                "protocol": protocol_key,
                "protocol_tag": tag,
                "seed": seed,
                "dense_val_shape": list(dense_val_sm.shape),
                "dense_test_shape": list(dense_test_sm.shape),
                "eff_val_shape": list(eff_val_sm.shape),
                "eff_test_shape": list(eff_test_sm.shape),
                "expected_val": expected_val,
                "expected_test": expected_test,
                "dense_test_match_csv": dense_test_match,
                "eff_test_match_csv": eff_test_match,
                "dense_val_softmax_path": str(dense_val_path),
                "dense_test_softmax_path": str(dense_test_path),
                "eff_val_softmax_path": str(eff_val_path),
                "eff_test_softmax_path": str(eff_test_path)
            })

            # --------------------------------------------------
            # Single-model validation and test metrics
            # --------------------------------------------------
            dense_val_metrics = evaluate_predictions(y_val, dense_val_sm)
            eff_val_metrics = evaluate_predictions(y_val, eff_val_sm)

            dense_test_metrics = evaluate_predictions(y_test, dense_test_sm)
            eff_test_metrics = evaluate_predictions(y_test, eff_test_sm)

            dense_val_f1 = dense_val_metrics["macro_f1"]
            eff_val_f1 = eff_val_metrics["macro_f1"]

            # --------------------------------------------------
            # Evaluate strategies on validation
            # --------------------------------------------------
            strategy_val_results = {}

            for strategy in STRATEGIES:
                val_ensemble = build_strategy_softmax(
                    strategy,
                    dense_val_sm,
                    eff_val_sm,
                    dense_val_f1=dense_val_f1,
                    eff_val_f1=eff_val_f1
                )

                val_metrics = evaluate_predictions(y_val, val_ensemble)
                strategy_val_results[strategy] = val_metrics

                strategy_detail_rows.append({
                    "protocol": protocol_key,
                    "seed": seed,
                    "split": "validation",
                    "strategy": strategy,
                    **val_metrics
                })

            best_strategy = max(
                STRATEGIES,
                key=lambda s: strategy_val_results[s]["macro_f1"]
            )

            best_val_macro_f1 = strategy_val_results[best_strategy]["macro_f1"]

            # --------------------------------------------------
            # Apply best strategy to test
            # --------------------------------------------------
            test_ensemble = build_strategy_softmax(
                best_strategy,
                dense_test_sm,
                eff_test_sm,
                dense_val_f1=dense_val_f1,
                eff_val_f1=eff_val_f1
            )

            test_metrics = evaluate_predictions(y_test, test_ensemble)

            # Save ensemble val/test softmax for reproducibility
            val_ensemble = build_strategy_softmax(
                best_strategy,
                dense_val_sm,
                eff_val_sm,
                dense_val_f1=dense_val_f1,
                eff_val_f1=eff_val_f1
            )

            ensemble_val_path = ENSEMBLE_OUT / f"Ensemble2_{protocol_key}_seed{seed}_val_softmax.npy"
            ensemble_test_path = ENSEMBLE_OUT / f"Ensemble2_{protocol_key}_seed{seed}_test_softmax.npy"

            np.save(ensemble_val_path, val_ensemble)
            np.save(ensemble_test_path, test_ensemble)

            # Save ensemble predictions CSV
            test_pred = test_ensemble.argmax(axis=1)
            pred_df = test_df.copy()
            pred_df["ensemble_pred_label"] = [CLASS_NAMES[i] for i in test_pred]
            pred_df["ensemble_confidence"] = test_ensemble.max(axis=1)

            for i, class_name in enumerate(CLASS_NAMES):
                safe_name = class_name.replace(" ", "_")
                pred_df[f"ensemble_prob_{safe_name}"] = test_ensemble[:, i]

            ensemble_pred_path = ENSEMBLE_OUT / f"Ensemble2_{protocol_key}_seed{seed}_test_predictions.csv"
            pred_df.to_csv(ensemble_pred_path, index=False)

            # Per-class F1 comparison
            per_class_f1_df = pd.DataFrame({
                "class_name": CLASS_NAMES,
                "ensemble_f1": f1_score(y_test, test_ensemble.argmax(axis=1), average=None, zero_division=0),
                "densenet_f1": f1_score(y_test, dense_test_sm.argmax(axis=1), average=None, zero_division=0),
                "effnet_f1": f1_score(y_test, eff_test_sm.argmax(axis=1), average=None, zero_division=0)
            })

            per_class_f1_df["ensemble_improvement_over_best_single"] = per_class_f1_df.apply(
                lambda r: r["ensemble_f1"] - max(r["densenet_f1"], r["effnet_f1"]),
                axis=1
            )

            per_class_path = ENSEMBLE_OUT / f"Ensemble2_{protocol_key}_seed{seed}_per_class_f1.csv"
            per_class_f1_df.to_csv(per_class_path, index=False)

            # Classification report and confusion matrix
            report_dict = classification_report(
                y_test,
                test_pred,
                target_names=CLASS_NAMES,
                output_dict=True,
                zero_division=0
            )

            report_path = ENSEMBLE_OUT / f"Ensemble2_{protocol_key}_seed{seed}_classification_report.json"
            with open(report_path, "w") as f:
                json.dump(report_dict, f, indent=2)

            cm = confusion_matrix(y_test, test_pred, labels=list(range(len(CLASS_NAMES))))
            cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
            cm_path = ENSEMBLE_OUT / f"Ensemble2_{protocol_key}_seed{seed}_confusion_matrix.csv"
            cm_df.to_csv(cm_path)

            best_single_acc = max(dense_test_metrics["accuracy"], eff_test_metrics["accuracy"])
            best_single_f1 = max(dense_test_metrics["macro_f1"], eff_test_metrics["macro_f1"])

            row = {
                "protocol": protocol_key,
                "protocol_tag": tag,
                "seed": seed,
                "best_strategy_by_val_macro_f1": best_strategy,
                "best_val_macro_f1": best_val_macro_f1,

                "ensemble_test_accuracy": test_metrics["accuracy"],
                "ensemble_test_macro_f1": test_metrics["macro_f1"],
                "ensemble_test_weighted_f1": test_metrics["weighted_f1"],
                "ensemble_test_macro_precision": test_metrics["macro_precision"],
                "ensemble_test_macro_recall": test_metrics["macro_recall"],

                "densenet_test_accuracy": dense_test_metrics["accuracy"],
                "densenet_test_macro_f1": dense_test_metrics["macro_f1"],
                "densenet_test_weighted_f1": dense_test_metrics["weighted_f1"],

                "effnet_test_accuracy": eff_test_metrics["accuracy"],
                "effnet_test_macro_f1": eff_test_metrics["macro_f1"],
                "effnet_test_weighted_f1": eff_test_metrics["weighted_f1"],

                "ensemble_improvement_acc_vs_best_single": test_metrics["accuracy"] - best_single_acc,
                "ensemble_improvement_f1_vs_best_single": test_metrics["macro_f1"] - best_single_f1,

                "beat_sikder_accuracy_0_9744": bool(test_metrics["accuracy"] > 0.9744),
                "beat_sikder_macro_f1_0_9744": bool(test_metrics["macro_f1"] > 0.9744),
                "hit_98_accuracy": bool(test_metrics["accuracy"] >= 0.98),

                "ensemble_val_softmax_path": str(ensemble_val_path),
                "ensemble_test_softmax_path": str(ensemble_test_path),
                "ensemble_test_predictions_csv": str(ensemble_pred_path),
                "per_class_f1_csv": str(per_class_path),
                "classification_report_json": str(report_path),
                "confusion_matrix_csv": str(cm_path)
            }

            seed_level_rows.append(row)

            print(
                f"{protocol_key:15s} seed={seed:<4d} "
                f"strategy={best_strategy:18s} "
                f"acc={test_metrics['accuracy']:.4f} "
                f"macroF1={test_metrics['macro_f1']:.4f} "
                f"delta_acc={row['ensemble_improvement_acc_vs_best_single']:+.4f}"
            )

        except Exception as e:
            error_record = {
                "protocol": protocol_key,
                "protocol_tag": tag,
                "seed": seed,
                "error_type": type(e).__name__,
                "error_message": str(e)
            }
            error_log.append(error_record)

            print(f"ERROR — {protocol_key} seed {seed}: {type(e).__name__}: {e}")

# Save detailed outputs
seed_level_df = pd.DataFrame(seed_level_rows)
strategy_detail_df = pd.DataFrame(strategy_detail_rows)
softmax_verification_df = pd.DataFrame(softmax_verification_rows)

seed_level_csv = ENSEMBLE_OUT / "CP4_ensemble_seed_level.csv"
strategy_detail_csv = ENSEMBLE_OUT / "CP4_ensemble_strategy_validation_details.csv"
softmax_verification_csv = ENSEMBLE_OUT / "CP4_ensemble_softmax_verification.csv"
error_log_json = ENSEMBLE_OUT / "CP4_ensemble_error_log.json"

seed_level_df.to_csv(seed_level_csv, index=False)
strategy_detail_df.to_csv(strategy_detail_csv, index=False)
softmax_verification_df.to_csv(softmax_verification_csv, index=False)

with open(error_log_json, "w") as f:
    json.dump(error_log, f, indent=2)

print(f"Seed-level CSV      : {seed_level_csv}")
print(f"Strategy details CSV: {strategy_detail_csv}")
print(f"Softmax verify CSV  : {softmax_verification_csv}")
print(f"Error log JSON      : {error_log_json}")
print(f"Errors              : {len(error_log)}")

In [ ]:
# ==========================================================
# CP4D-2 — Aggregate Summary and Final Paper Tables
# ==========================================================

seed_level_df = pd.read_csv(ENSEMBLE_OUT / "CP4_ensemble_seed_level.csv")

aggregate_rows = []

for protocol_key, protocol_info in PROTOCOLS.items():
    sub = seed_level_df[seed_level_df["protocol"] == protocol_key].copy()

    if len(sub) == 0:
        continue

    strategy_counts = sub["best_strategy_by_val_macro_f1"].value_counts().to_dict()

    aggregate_rows.append({
        "protocol": protocol_key,
        "display_name": protocol_info["display_name"],
        "n_seeds": len(sub),

        "baseline_accuracy_mean": protocol_info["baseline_acc_mean"],
        "baseline_accuracy_std": protocol_info["baseline_acc_std"],
        "baseline_macro_f1_mean": protocol_info["baseline_f1_mean"],
        "baseline_macro_f1_std": protocol_info["baseline_f1_std"],

        "densenet_accuracy_mean": sub["densenet_test_accuracy"].mean(),
        "densenet_accuracy_std": sub["densenet_test_accuracy"].std(),
        "densenet_macro_f1_mean": sub["densenet_test_macro_f1"].mean(),
        "densenet_macro_f1_std": sub["densenet_test_macro_f1"].std(),

        "effnet_accuracy_mean": sub["effnet_test_accuracy"].mean(),
        "effnet_accuracy_std": sub["effnet_test_accuracy"].std(),
        "effnet_macro_f1_mean": sub["effnet_test_macro_f1"].mean(),
        "effnet_macro_f1_std": sub["effnet_test_macro_f1"].std(),

        "ensemble_accuracy_mean": sub["ensemble_test_accuracy"].mean(),
        "ensemble_accuracy_std": sub["ensemble_test_accuracy"].std(),
        "ensemble_macro_f1_mean": sub["ensemble_test_macro_f1"].mean(),
        "ensemble_macro_f1_std": sub["ensemble_test_macro_f1"].std(),
        "ensemble_weighted_f1_mean": sub["ensemble_test_weighted_f1"].mean(),
        "ensemble_weighted_f1_std": sub["ensemble_test_weighted_f1"].std(),

        "ensemble_best_seed_accuracy": sub["ensemble_test_accuracy"].max(),
        "ensemble_best_seed_macro_f1": sub["ensemble_test_macro_f1"].max(),

        "ensemble_mean_acc_improvement_vs_best_single": sub["ensemble_improvement_acc_vs_best_single"].mean(),
        "ensemble_mean_f1_improvement_vs_best_single": sub["ensemble_improvement_f1_vs_best_single"].mean(),

        "beat_sikder_acc_count": int(sub["beat_sikder_accuracy_0_9744"].sum()),
        "beat_sikder_f1_count": int(sub["beat_sikder_macro_f1_0_9744"].sum()),
        "hit_98_acc_count": int(sub["hit_98_accuracy"].sum()),

        "strategies_used": json.dumps(strategy_counts)
    })

aggregate_df = pd.DataFrame(aggregate_rows)

aggregate_csv = ENSEMBLE_OUT / "CP4_ensemble_aggregate_by_protocol.csv"
aggregate_json = ENSEMBLE_OUT / "CP4_ensemble_aggregate_by_protocol.json"

aggregate_df.to_csv(aggregate_csv, index=False)

with open(aggregate_json, "w") as f:
    json.dump(aggregate_rows, f, indent=2)

print("=== Aggregated Ensemble Results by Protocol ===")
display(aggregate_df)

# ----------------------------------------------------------
# Final compact paper table
# ----------------------------------------------------------
paper_rows = []

for _, row in aggregate_df.iterrows():
    protocol = row["protocol"]

    paper_rows.append({
        "protocol": protocol,
        "baseline_denseNet201_accuracy": f"{row['baseline_accuracy_mean']:.4f} ± {row['baseline_accuracy_std']:.4f}",
        "densenet201_tta_mixup_accuracy": f"{row['densenet_accuracy_mean']:.4f} ± {row['densenet_accuracy_std']:.4f}",
        "effnetv2s_tta_mixup_accuracy": f"{row['effnet_accuracy_mean']:.4f} ± {row['effnet_accuracy_std']:.4f}",
        "ensemble_accuracy": f"{row['ensemble_accuracy_mean']:.4f} ± {row['ensemble_accuracy_std']:.4f}",
        "baseline_denseNet201_macro_f1": f"{row['baseline_macro_f1_mean']:.4f} ± {row['baseline_macro_f1_std']:.4f}",
        "densenet201_tta_mixup_macro_f1": f"{row['densenet_macro_f1_mean']:.4f} ± {row['densenet_macro_f1_std']:.4f}",
        "effnetv2s_tta_mixup_macro_f1": f"{row['effnet_macro_f1_mean']:.4f} ± {row['effnet_macro_f1_std']:.4f}",
        "ensemble_macro_f1": f"{row['ensemble_macro_f1_mean']:.4f} ± {row['ensemble_macro_f1_std']:.4f}",
        "ensemble_best_seed_accuracy": f"{row['ensemble_best_seed_accuracy']:.4f}",
        "strategies_used": row["strategies_used"]
    })

paper_table_df = pd.DataFrame(paper_rows)

paper_table_csv = ENSEMBLE_OUT / "CP4_final_paper_table_all_protocols.csv"
paper_table_df.to_csv(paper_table_csv, index=False)

print("\n=== Final Paper Table — All Protocols ===")
display(paper_table_df)

# ----------------------------------------------------------
# Method ranking table
# ----------------------------------------------------------
ranking_rows = []

for _, row in aggregate_df.iterrows():
    protocol = row["protocol"]

    methods = [
        ("Baseline DenseNet201", row["baseline_accuracy_mean"], row["baseline_macro_f1_mean"]),
        ("DenseNet201 TTA+Mixup", row["densenet_accuracy_mean"], row["densenet_macro_f1_mean"]),
        ("EffNetV2S TTA+Mixup", row["effnet_accuracy_mean"], row["effnet_macro_f1_mean"]),
        ("Ensemble2", row["ensemble_accuracy_mean"], row["ensemble_macro_f1_mean"])
    ]

    methods_sorted = sorted(methods, key=lambda x: x[1], reverse=True)

    for rank, (method, acc, macro_f1) in enumerate(methods_sorted, start=1):
        ranking_rows.append({
            "protocol": protocol,
            "rank_by_accuracy": rank,
            "method": method,
            "accuracy_mean": acc,
            "macro_f1_mean": macro_f1
        })

ranking_df = pd.DataFrame(ranking_rows)

ranking_csv = ENSEMBLE_OUT / "CP4_method_ranking_by_protocol.csv"
ranking_df.to_csv(ranking_csv, index=False)

print("\n=== Method Ranking by Protocol ===")
display(ranking_df)

print(f"Aggregate CSV : {aggregate_csv}")
print(f"Paper table   : {paper_table_csv}")
print(f"Ranking CSV   : {ranking_csv}")

In [ ]:
# ==========================================================
# CP4D-3 — Package Final Ensemble Review Files
# ==========================================================

review_files = [
    ENSEMBLE_OUT / "CP4_ensemble_seed_level.csv",
    ENSEMBLE_OUT / "CP4_ensemble_strategy_validation_details.csv",
    ENSEMBLE_OUT / "CP4_ensemble_softmax_verification.csv",
    ENSEMBLE_OUT / "CP4_ensemble_error_log.json",
    ENSEMBLE_OUT / "CP4_ensemble_aggregate_by_protocol.csv",
    ENSEMBLE_OUT / "CP4_ensemble_aggregate_by_protocol.json",
    ENSEMBLE_OUT / "CP4_final_paper_table_all_protocols.csv",
    ENSEMBLE_OUT / "CP4_method_ranking_by_protocol.csv"
]

review_zip = ENSEMBLE_OUT / "CP4_ensemble_review_files.zip"

with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in review_files:
        file_path = Path(file_path)

        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
            print("Added:", file_path)
        else:
            print("Missing:", file_path)

print(f"Review ZIP: {review_zip}")

In [ ]:
import timm

# Nama model EfficientNetV2-S di library timm sesuai dengan dictionary ARCHITECTURES yang ada
timm_name = "tf_efficientnetv2_s.in21k_ft_in1k"
num_classes = 14  # Sesuai dengan dataset (14 kelas virus)

print(f"Membuat model {timm_name}...")
# Kita set pretrained=False karena kita hanya ingin menghitung jumlah parameternya
model = timm.create_model(timm_name, pretrained=False, num_classes=num_classes)

# Menghitung total parameter dan parameter yang dapat dilatih (trainable)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameter             : {total_params:,}")
print(f"Parameter yang dapat dilatih: {trainable_params:,}")

In [ ]:
# Per-Class F1 Aggregate
# Output: outputs/cp4_paper_prep/per_class_f1_aggregate.csv

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import traceback

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'Adenovirus', 'Astrovirus', 'CCHF', 'Cowpox', 'Ebola', 'Influenza', 'Lassa',
    'Marburg', 'Nipah virus', 'Norovirus', 'Orf', 'Papilloma', 'Rift Valley', 'Rotavirus'
]

PROTOCOL_SEEDS = {
    'A': [42, 123, 456, 789, 1024],
    'B_G14': [42, 123, 456, 789, 1024],
    'A_clean_strict': [42, 123, 456],
    'C_G09': [42, 123, 456],
}

MODELS = {
    'DenseNet201': 'DenseNet201_TTA_Mixup',
    'EffNetV2S': 'EffNetV2S_TTA_Mixup',
}


def find_column(df, candidates):
    """Find a column using case-insensitive matching."""
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def normalize_per_class_df(df, file_path):
    """Return dataframe with class_name and f1 columns."""
    class_col = find_column(df, [
        'class_name', 'class', 'label', 'target', 'virus_class', 'category'
    ])
    f1_col = find_column(df, [
        'f1', 'f1_score', 'f1-score', 'test_f1', 'F1', 'F1-score'
    ])

    if class_col is None:
        # Common case: sklearn classification_report exported with unnamed index.
        first_col = df.columns[0]
        if df[first_col].astype(str).isin(CLASS_NAMES).any():
            class_col = first_col

    if f1_col is None:
        # Try fuzzy matching.
        for col in df.columns:
            normalized = col.lower().replace('_', '').replace('-', '').replace(' ', '')
            if normalized in ['f1', 'f1score', 'scoref1']:
                f1_col = col
                break

    if class_col is None or f1_col is None:
        raise ValueError(
            f"Cannot identify class/F1 columns in {file_path}. "
            f"Columns found: {list(df.columns)}"
        )

    out = df[[class_col, f1_col]].copy()
    out.columns = ['class_name', 'f1']

    # If class labels are numeric indices, map them to class names.
    if pd.api.types.is_numeric_dtype(out['class_name']):
        out['class_name'] = out['class_name'].astype(int).map(
            {i: name for i, name in enumerate(CLASS_NAMES)}
        )

    out['class_name'] = out['class_name'].astype(str)
    out = out[out['class_name'].isin(CLASS_NAMES)].copy()
    out['f1'] = pd.to_numeric(out['f1'], errors='coerce')
    out = out.dropna(subset=['f1'])

    return out


def candidate_paths(model, protocol, seed):
    """Build prioritized candidate paths for a model/protocol/seed per-class F1 file."""
    paths = []

    # Special verified fallback/source locations for Protocol A seed 42.
    if protocol == 'A' and seed == 42 and model == 'DenseNet201':
        paths.append(ROOT / 'outputs/cp3_6_ensemble/DenseNet201_A_seed42_ttamixup_test_per_class_f1.csv')
    if protocol == 'A' and seed == 42 and model == 'EffNetV2S':
        paths.append(ROOT / 'outputs/cp3_5_ceiling_push/EffNetV2S_A_seed42_ttamixup_test_per_class_f1.csv')

    # Standard CP4 multiseed location.
    paths.append(ROOT / f'outputs/cp4_multiseed/{model}_{protocol}_seed{seed}_ttamixup_test_per_class_f1.csv')

    return paths


def load_first_existing(paths):
    """Load first existing CSV from candidate paths."""
    for p in paths:
        if p.exists():
            return p, pd.read_csv(p)
    return None, None


def main():
    print("=" * 80)
    print("SCRIPT 1: PER-CLASS F1 AGGREGATE")
    print("=" * 80)

    records = []
    missing_files = []
    loaded_count = 0

    for protocol, seeds in PROTOCOL_SEEDS.items():
        for model, method_name in MODELS.items():
            for seed in seeds:
                paths = candidate_paths(model, protocol, seed)
                found_path, df = load_first_existing(paths)

                if df is None:
                    missing_files.append({
                        'protocol': protocol,
                        'model': model,
                        'seed': seed,
                        'checked_paths': [str(p) for p in paths],
                    })
                    continue

                try:
                    per_class = normalize_per_class_df(df, found_path)
                    loaded_count += 1

                    for _, row in per_class.iterrows():
                        records.append({
                            'protocol': protocol,
                            'method': method_name,
                            'seed': seed,
                            'class_name': row['class_name'],
                            'f1': float(row['f1']),
                            'source_file': str(found_path),
                        })

                    print(f"Loaded: {found_path}")

                except Exception as e:
                    print(f"[WARNING] Failed to parse {found_path}: {e}")
                    missing_files.append({
                        'protocol': protocol,
                        'model': model,
                        'seed': seed,
                        'checked_paths': [str(found_path)],
                        'error': str(e),
                    })

    if not records:
        print("\n[ERROR] No per-class F1 records were loaded. No output generated.")
        return

    raw_df = pd.DataFrame(records)

    agg = (
        raw_df
        .groupby(['protocol', 'method', 'class_name'], as_index=False)
        .agg(
            mean_f1=('f1', 'mean'),
            std_f1=('f1', lambda x: float(np.std(x, ddof=1)) if len(x) > 1 else 0.0),
            n_seeds=('seed', 'nunique')
        )
    )

    # Keep deterministic ordering.
    protocol_order = {p: i for i, p in enumerate(PROTOCOL_SEEDS.keys())}
    method_order = {m: i for i, m in enumerate(MODELS.values())}
    class_order = {c: i for i, c in enumerate(CLASS_NAMES)}

    agg['_protocol_order'] = agg['protocol'].map(protocol_order)
    agg['_method_order'] = agg['method'].map(method_order)
    agg['_class_order'] = agg['class_name'].map(class_order)
    agg = (
        agg.sort_values(['_protocol_order', '_method_order', '_class_order'])
           .drop(columns=['_protocol_order', '_method_order', '_class_order'])
           .reset_index(drop=True)
    )

    output_path = OUT_DIR / 'per_class_f1_aggregate.csv'
    agg.to_csv(output_path, index=False)

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Loaded per-seed CSV files : {loaded_count}")
    print(f"Raw records loaded        : {len(raw_df)}")
    print(f"Rows generated            : {len(agg)}")
    print(f"Expected rows             : 112")
    print(f"Output saved to           : {output_path}")

    if missing_files:
        print("\nMissing or unreadable files:")
        for item in missing_files:
            print(f"- protocol={item['protocol']} model={item['model']} seed={item['seed']}")
            for p in item['checked_paths']:
                print(f"  checked: {p}")
            if 'error' in item:
                print(f"  error: {item['error']}")
    else:
        print("\nAll expected per-seed per-class files were found.")

    print("\nPreview:")
    print(agg.head(20).to_string(index=False))


if __name__ == "__main__":
    try:
        main()
    except Exception:
        print("[FATAL ERROR] Script failed unexpectedly.")
        traceback.print_exc()

In [ ]:
# Per-Class F1 Ensemble Protocol A Seed 42
# Output: outputs/cp4_paper_prep/Ensemble2_A_seed42_per_class_f1_comparison.csv

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import traceback
from sklearn.metrics import f1_score

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'Adenovirus', 'Astrovirus', 'CCHF', 'Cowpox', 'Ebola', 'Influenza', 'Lassa',
    'Marburg', 'Nipah virus', 'Norovirus', 'Orf', 'Papilloma', 'Rift Valley', 'Rotavirus'
]

LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}


def find_column(df, candidates):
    """Find a column using case-insensitive matching."""
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def extract_true_labels(pred_csv_path):
    """Extract y_true from a prediction CSV."""
    df = pd.read_csv(pred_csv_path)

    y_col = find_column(df, [
        'y_true', 'true_label', 'label', 'target', 'true_class',
        'ground_truth', 'actual', 'class_id', 'true_idx'
    ])

    if y_col is None:
        raise ValueError(
            f"Cannot identify true-label column in {pred_csv_path}. "
            f"Columns found: {list(df.columns)}"
        )

    y = df[y_col]

    if pd.api.types.is_numeric_dtype(y):
        y_true = y.astype(int).to_numpy()
    else:
        y_true = y.astype(str).map(LABEL_TO_IDX)
        if y_true.isna().any():
            bad_values = sorted(set(y.astype(str)[y_true.isna()]))
            raise ValueError(
                f"Found unmapped label values in {pred_csv_path}: {bad_values}"
            )
        y_true = y_true.astype(int).to_numpy()

    return y_true


def load_required_npy(path):
    """Load NPY file with informative error."""
    if not path.exists():
        raise FileNotFoundError(f"Missing required softmax file: {path}")
    arr = np.load(path)
    if arr.ndim != 2 or arr.shape[1] != len(CLASS_NAMES):
        raise ValueError(
            f"Unexpected softmax shape for {path}: {arr.shape}. "
            f"Expected (n_samples, {len(CLASS_NAMES)})."
        )
    return arr


def main():
    print("=" * 80)
    print("SCRIPT 2: PER-CLASS F1 COMPARISON FOR ENSEMBLE PROTOCOL A SEED 42")
    print("=" * 80)

    dense_softmax_path = ROOT / 'outputs/cp3_6_ensemble/DenseNet201_A_seed42_ttamixup_test_softmax.npy'
    eff_softmax_path = ROOT / 'outputs/cp3_5_ceiling_push/EffNetV2S_A_seed42_ttamixup_test_softmax.npy'
    ens_softmax_path = ROOT / 'outputs/cp4_ensemble/Ensemble2_A_seed42_test_softmax.npy'

    prediction_candidates = [
        ROOT / 'outputs/cp3_6_ensemble/DenseNet201_A_seed42_ttamixup_test_predictions.csv',
        ROOT / 'outputs/cp3_5_ceiling_push/EffNetV2S_A_seed42_ttamixup_test_predictions.csv',
        ROOT / 'outputs/cp4_multiseed/DenseNet201_A_seed42_ttamixup_test_predictions.csv',
        ROOT / 'outputs/cp4_multiseed/EffNetV2S_A_seed42_ttamixup_test_predictions.csv',
    ]

    pred_csv_path = next((p for p in prediction_candidates if p.exists()), None)

    if pred_csv_path is None:
        print("[ERROR] No prediction CSV found for Protocol A seed 42.")
        print("Checked:")
        for p in prediction_candidates:
            print(f"  - {p}")
        return

    print(f"Loading true labels from: {pred_csv_path}")

    dense_softmax = load_required_npy(dense_softmax_path)
    eff_softmax = load_required_npy(eff_softmax_path)
    ens_softmax = load_required_npy(ens_softmax_path)
    y_true = extract_true_labels(pred_csv_path)

    n = len(y_true)
    for name, arr in [
        ('DenseNet201', dense_softmax),
        ('EffNetV2S', eff_softmax),
        ('Ensemble', ens_softmax),
    ]:
        if arr.shape[0] != n:
            raise ValueError(
                f"Length mismatch for {name}: softmax has {arr.shape[0]} rows, "
                f"but y_true has {n} labels."
            )

    y_pred_dense = np.argmax(dense_softmax, axis=1)
    y_pred_eff = np.argmax(eff_softmax, axis=1)
    y_pred_ens = np.argmax(ens_softmax, axis=1)

    labels = list(range(len(CLASS_NAMES)))

    dense_f1 = f1_score(y_true, y_pred_dense, labels=labels, average=None, zero_division=0)
    eff_f1 = f1_score(y_true, y_pred_eff, labels=labels, average=None, zero_division=0)
    ens_f1 = f1_score(y_true, y_pred_ens, labels=labels, average=None, zero_division=0)

    rows = []
    for i, class_name in enumerate(CLASS_NAMES):
        best_single = max(dense_f1[i], eff_f1[i])
        rows.append({
            'class_name': class_name,
            'DenseNet201_only_F1': dense_f1[i],
            'EffNetV2S_only_F1': eff_f1[i],
            'Ensemble_F1': ens_f1[i],
            'Ensemble_vs_Best_Single': ens_f1[i] - best_single,
        })

    out_df = pd.DataFrame(rows)

    output_path = OUT_DIR / 'Ensemble2_A_seed42_per_class_f1_comparison.csv'
    out_df.to_csv(output_path, index=False)

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Samples evaluated       : {n}")
    print(f"Classes                 : {len(CLASS_NAMES)}")
    print(f"Rows generated          : {len(out_df)}")
    print(f"DenseNet201 macro F1    : {dense_f1.mean():.6f}")
    print(f"EffNetV2S macro F1      : {eff_f1.mean():.6f}")
    print(f"Ensemble macro F1       : {ens_f1.mean():.6f}")
    print(f"Output saved to         : {output_path}")

    print("\nPreview:")
    print(out_df.to_string(index=False))


if __name__ == "__main__":
    try:
        main()
    except Exception:
        print("[FATAL ERROR] Script failed unexpectedly.")
        traceback.print_exc()

In [ ]:
# Statistical Tests Results
# Output: outputs/cp4_paper_prep/statistical_tests_results.csv

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import json
import traceback
from scipy.stats import wilcoxon

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL_SEEDS = {
    'A': [42, 123, 456, 789, 1024],
    'B_G14': [42, 123, 456, 789, 1024],
    'A_clean_strict': [42, 123, 456],
    'C_G09': [42, 123, 456],
}

BASELINE_TAGS = {
    'A': 'A_official',
    'B_G14': 'B_G14',
    'A_clean_strict': 'A_clean_strict',
    'C_G09': 'C_G09',
}

SUMMARY_FILES = [
    ROOT / 'outputs/cp4_multiseed/CP4_protocolA_multiseed_summary.csv',
    ROOT / 'outputs/cp4_multiseed/CP4_protocolBG14_multiseed_summary.csv',
    ROOT / 'outputs/cp4_multiseed/CP4_protocolAcleanCG09_multiseed_summary.csv',
]

ENSEMBLE_SEED_FILE = ROOT / 'outputs/cp4_ensemble/CP4_ensemble_seed_level.csv'


def find_column(df, candidates):
    """Find a column using case-insensitive matching."""
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def normalize_protocol_value(x):
    """Normalize protocol labels from different CSV conventions."""
    s = str(x).strip()
    mapping = {
        'Protocol A': 'A',
        'A official': 'A',
        'A_official': 'A',
        'official': 'A',
        'B-G14': 'B_G14',
        'B_G14': 'B_G14',
        'Protocol B-G14': 'B_G14',
        'A-clean strict': 'A_clean_strict',
        'A_clean_strict': 'A_clean_strict',
        'A-clean-strict': 'A_clean_strict',
        'Protocol A-clean-strict': 'A_clean_strict',
        'C-G09': 'C_G09',
        'C_G09': 'C_G09',
        'Protocol C-G09': 'C_G09',
    }
    return mapping.get(s, s)


def infer_protocol_from_filename(path):
    """Infer protocol if a summary file does not contain protocol column."""
    name = path.name.lower()
    if 'protocola_' in name or 'protocola_' in name or 'protocola.' in name:
        return 'A'
    if 'protocolbg14' in name or 'b_g14' in name or 'bg14' in name:
        return 'B_G14'
    if 'acleancg09' in name:
        return None
    return None


def flatten_dict(d, parent_key=''):
    """Flatten nested JSON dictionaries."""
    items = {}
    if isinstance(d, dict):
        for k, v in d.items():
            new_key = f"{parent_key}.{k}" if parent_key else str(k)
            if isinstance(v, dict):
                items.update(flatten_dict(v, new_key))
            else:
                items[new_key] = v
    return items


def extract_accuracy_from_json(path):
    """Extract test accuracy from a JSON metrics file."""
    with open(path, 'r') as f:
        data = json.load(f)

    flat = flatten_dict(data)
    candidates = [
        'test_accuracy', 'test_acc', 'accuracy', 'test.metrics.accuracy',
        'metrics.test_accuracy', 'test.accuracy', 'test_acc_mean'
    ]

    lower_map = {k.lower(): k for k in flat.keys()}

    for cand in candidates:
        if cand.lower() in lower_map:
            val = flat[lower_map[cand.lower()]]
            try:
                return float(val)
            except Exception:
                pass

    # Fuzzy fallback: prefer keys containing both "test" and "acc".
    for k, v in flat.items():
        kl = k.lower()
        if 'test' in kl and ('accuracy' in kl or kl.endswith('acc') or 'acc' in kl):
            try:
                return float(v)
            except Exception:
                continue

    # Last fallback: any accuracy key.
    for k, v in flat.items():
        kl = k.lower()
        if 'accuracy' in kl or kl.endswith('acc') or 'acc' in kl:
            try:
                return float(v)
            except Exception:
                continue

    raise ValueError(f"Could not extract test accuracy from {path}. Keys: {list(flat.keys())}")


def load_baseline_accuracies():
    """Load Phase 1 baseline DenseNet201 per-seed test accuracies."""
    result = {p: {} for p in PROTOCOL_SEEDS}
    missing = []

    for protocol, seeds in PROTOCOL_SEEDS.items():
        tag = BASELINE_TAGS[protocol]

        for seed in seeds:
            patterns = [
                f'DenseNet201_{tag}_seed{seed}_evaluation_metrics_summary.json',
                f'DenseNet201*{tag}*seed{seed}*evaluation_metrics_summary.json',
            ]

            matches = []
            for pattern in patterns:
                matches.extend(list((ROOT / 'outputs/checkpoints').rglob(pattern)))
                matches.extend(list((ROOT / 'outputs').rglob(pattern)))

            # Deduplicate and sort for deterministic selection.
            matches = sorted(set(matches))

            if not matches:
                missing.append((protocol, seed, tag))
                continue

            chosen = matches[0]
            try:
                acc = extract_accuracy_from_json(chosen)
                result[protocol][seed] = acc
                print(f"Baseline loaded: protocol={protocol} seed={seed} acc={acc:.6f} from {chosen}")
            except Exception as e:
                print(f"[WARNING] Failed baseline JSON parse: {chosen} | {e}")
                missing.append((protocol, seed, tag))

    return result, missing


def load_dn_ttamixup_accuracies():
    """Load DenseNet201 TTA+mixup per-seed test accuracies from CP4 multiseed summaries."""
    result = {p: {} for p in PROTOCOL_SEEDS}
    loaded_any = False

    for path in SUMMARY_FILES:
        if not path.exists():
            print(f"[WARNING] Missing summary file: {path}")
            continue

        df = pd.read_csv(path)
        loaded_any = True
        print(f"Loaded multiseed summary: {path} shape={df.shape}")

        seed_col = find_column(df, ['seed', 'random_seed'])
        protocol_col = find_column(df, ['protocol', 'protocol_key', 'split', 'protocol_name'])
        model_col = find_column(df, ['model', 'architecture', 'method', 'model_name'])
        acc_col = find_column(df, [
            'test_accuracy', 'accuracy', 'test_acc', 'acc', 'mean_test_accuracy'
        ])

        if seed_col is None or acc_col is None:
            print(f"[WARNING] Cannot parse seed/accuracy columns in {path}. Columns: {list(df.columns)}")
            continue

        work = df.copy()

        if protocol_col is not None:
            work['_protocol'] = work[protocol_col].apply(normalize_protocol_value)
        else:
            inferred = infer_protocol_from_filename(path)
            if inferred is None and 'acleancg09' in path.name.lower():
                # This file may contain both A_clean_strict and C_G09 rows.
                possible_col = find_column(work, ['protocol', 'protocol_key', 'split'])
                if possible_col is not None:
                    work['_protocol'] = work[possible_col].apply(normalize_protocol_value)
                else:
                    print(f"[WARNING] Cannot infer protocol from {path}. Columns: {list(df.columns)}")
                    continue
            else:
                work['_protocol'] = inferred

        if model_col is not None:
            # Keep DenseNet201 rows when model/method column exists.
            mask = work[model_col].astype(str).str.contains('DenseNet201', case=False, na=False)
            if mask.any():
                work = work[mask].copy()

        for _, row in work.iterrows():
            protocol = normalize_protocol_value(row['_protocol'])
            if protocol not in result:
                continue
            try:
                seed = int(row[seed_col])
                acc = float(row[acc_col])
            except Exception:
                continue
            result[protocol][seed] = acc

    if not loaded_any:
        print("[WARNING] No CP4 multiseed summary files were loaded.")

    return result


def load_ensemble_accuracies():
    """Load ensemble per-seed test accuracies."""
    result = {p: {} for p in PROTOCOL_SEEDS}

    if not ENSEMBLE_SEED_FILE.exists():
        print(f"[WARNING] Missing ensemble seed-level file: {ENSEMBLE_SEED_FILE}")
        return result

    df = pd.read_csv(ENSEMBLE_SEED_FILE)
    print(f"Loaded ensemble seed-level file: {ENSEMBLE_SEED_FILE} shape={df.shape}")

    seed_col = find_column(df, ['seed', 'random_seed'])
    protocol_col = find_column(df, ['protocol', 'protocol_key', 'split', 'protocol_name'])
    acc_col = find_column(df, [
        'ensemble_test_accuracy', 'test_accuracy', 'accuracy',
        'ensemble_accuracy', 'test_acc'
    ])

    if seed_col is None or protocol_col is None or acc_col is None:
        print(f"[WARNING] Cannot parse ensemble file. Columns: {list(df.columns)}")
        return result

    for _, row in df.iterrows():
        protocol = normalize_protocol_value(row[protocol_col])
        if protocol not in result:
            continue
        try:
            seed = int(row[seed_col])
            acc = float(row[acc_col])
        except Exception:
            continue
        result[protocol][seed] = acc

    return result


def paired_cohens_d(diff):
    """Compute paired Cohen's d."""
    diff = np.asarray(diff, dtype=float)
    if len(diff) < 2:
        return np.nan
    sd = np.std(diff, ddof=1)
    if sd == 0:
        return np.inf if np.mean(diff) != 0 else 0.0
    return float(np.mean(diff) / sd)


def bootstrap_ci_mean(diff, n_iter=10000, seed=42):
    """Bootstrap 95% CI for mean paired difference."""
    diff = np.asarray(diff, dtype=float)
    if len(diff) == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot_means = []
    n = len(diff)

    for _ in range(n_iter):
        sample = rng.choice(diff, size=n, replace=True)
        boot_means.append(np.mean(sample))

    return float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5))


def safe_wilcoxon(diff):
    """Run Wilcoxon signed-rank test with graceful handling."""
    diff = np.asarray(diff, dtype=float)

    if len(diff) < 2:
        return np.nan

    if np.allclose(diff, 0):
        return 1.0

    try:
        stat, p = wilcoxon(diff, zero_method='wilcox', alternative='two-sided', mode='auto')
        return float(p)
    except Exception as e:
        print(f"[WARNING] Wilcoxon failed for diff={diff}: {e}")
        return np.nan


def build_comparison_row(protocol, comparison, left_values, right_values):
    """Build one statistical comparison row from paired seed dictionaries."""
    matched_seeds = sorted(set(left_values.keys()) & set(right_values.keys()))
    diffs = []

    for seed in matched_seeds:
        diffs.append(float(left_values[seed]) - float(right_values[seed]))

    diffs = np.asarray(diffs, dtype=float)

    if len(diffs) == 0:
        return {
            'protocol': protocol,
            'comparison': comparison,
            'n_seeds': 0,
            'matched_seeds': '',
            'mean_diff': np.nan,
            'std_diff': np.nan,
            'wilcoxon_p': np.nan,
            'cohens_d': np.nan,
            'ci_low': np.nan,
            'ci_high': np.nan,
            'bonferroni_threshold': 0.05 / 12,
            'significant_after_bonferroni': False,
        }

    ci_low, ci_high = bootstrap_ci_mean(diffs, n_iter=10000, seed=42)

    return {
        'protocol': protocol,
        'comparison': comparison,
        'n_seeds': len(diffs),
        'matched_seeds': ','.join(map(str, matched_seeds)),
        'mean_diff': float(np.mean(diffs)),
        'std_diff': float(np.std(diffs, ddof=1)) if len(diffs) > 1 else 0.0,
        'wilcoxon_p': safe_wilcoxon(diffs),
        'cohens_d': paired_cohens_d(diffs),
        'ci_low': ci_low,
        'ci_high': ci_high,
        'bonferroni_threshold': 0.05 / 12,
        'significant_after_bonferroni': False,
    }


def main():
    print("=" * 80)
    print("SCRIPT 3: STATISTICAL TESTS RESULTS")
    print("=" * 80)

    baseline, baseline_missing = load_baseline_accuracies()
    dn_ttamixup = load_dn_ttamixup_accuracies()
    ensemble = load_ensemble_accuracies()

    rows = []

    for protocol in PROTOCOL_SEEDS:
        rows.append(build_comparison_row(
            protocol=protocol,
            comparison='TTA+Mixup_DN_vs_Baseline',
            left_values=dn_ttamixup[protocol],
            right_values=baseline[protocol],
        ))

        rows.append(build_comparison_row(
            protocol=protocol,
            comparison='Ensemble_vs_Baseline',
            left_values=ensemble[protocol],
            right_values=baseline[protocol],
        ))

        rows.append(build_comparison_row(
            protocol=protocol,
            comparison='Ensemble_vs_TTAMixup_DN',
            left_values=ensemble[protocol],
            right_values=dn_ttamixup[protocol],
        ))

    out_df = pd.DataFrame(rows)

    # Apply Bonferroni significance flag.
    threshold = 0.05 / 12
    out_df['significant_after_bonferroni'] = (
        pd.to_numeric(out_df['wilcoxon_p'], errors='coerce') < threshold
    )

    output_path = OUT_DIR / 'statistical_tests_results.csv'
    out_df.to_csv(output_path, index=False)

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Rows generated              : {len(out_df)}")
    print(f"Expected rows               : 12")
    print(f"Bonferroni threshold        : {threshold:.6f}")
    print(f"Output saved to             : {output_path}")

    print("\nLoaded counts by protocol:")
    for protocol in PROTOCOL_SEEDS:
        print(
            f"- {protocol:<15} "
            f"baseline={len(baseline[protocol])} | "
            f"DN_TTA_mixup={len(dn_ttamixup[protocol])} | "
            f"ensemble={len(ensemble[protocol])}"
        )

    if baseline_missing:
        print("\nMissing baseline JSON files:")
        for protocol, seed, tag in baseline_missing:
            print(f"- protocol={protocol} seed={seed} expected_tag={tag}")

    print("\nResult preview:")
    print(out_df.to_string(index=False))


if __name__ == "__main__":
    try:
        main()
    except Exception:
        print("[FATAL ERROR] Script failed unexpectedly.")
        traceback.print_exc()

In [ ]:
# Confusion Matrix Protocol A Best Seed (Ensemble)
# Output: outputs/cp4_paper_prep/Ensemble2_A_best_seed_confusion_matrix.csv
# Additional Output: outputs/cp4_paper_prep/Ensemble2_A_best_seed_confusion_matrix_normalized.csv

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import traceback
from sklearn.metrics import confusion_matrix

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'Adenovirus', 'Astrovirus', 'CCHF', 'Cowpox', 'Ebola', 'Influenza', 'Lassa',
    'Marburg', 'Nipah virus', 'Norovirus', 'Orf', 'Papilloma', 'Rift Valley', 'Rotavirus'
]

LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

ENSEMBLE_SEED_FILE = ROOT / 'outputs/cp4_ensemble/CP4_ensemble_seed_level.csv'


def find_column(df, candidates):
    """Find a column using case-insensitive matching."""
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def normalize_protocol_value(x):
    """Normalize protocol labels from different CSV conventions."""
    s = str(x).strip()
    mapping = {
        'Protocol A': 'A',
        'A official': 'A',
        'A_official': 'A',
        'official': 'A',
        'B-G14': 'B_G14',
        'B_G14': 'B_G14',
        'Protocol B-G14': 'B_G14',
        'A-clean strict': 'A_clean_strict',
        'A_clean_strict': 'A_clean_strict',
        'A-clean-strict': 'A_clean_strict',
        'C-G09': 'C_G09',
        'C_G09': 'C_G09',
    }
    return mapping.get(s, s)


def extract_true_labels(pred_csv_path):
    """Extract y_true from a prediction CSV."""
    df = pd.read_csv(pred_csv_path)

    y_col = find_column(df, [
        'y_true', 'true_label', 'label', 'target', 'true_class',
        'ground_truth', 'actual', 'class_id', 'true_idx'
    ])

    if y_col is None:
        raise ValueError(
            f"Cannot identify true-label column in {pred_csv_path}. "
            f"Columns found: {list(df.columns)}"
        )

    y = df[y_col]

    if pd.api.types.is_numeric_dtype(y):
        y_true = y.astype(int).to_numpy()
    else:
        y_true = y.astype(str).map(LABEL_TO_IDX)
        if y_true.isna().any():
            bad_values = sorted(set(y.astype(str)[y_true.isna()]))
            raise ValueError(
                f"Found unmapped label values in {pred_csv_path}: {bad_values}"
            )
        y_true = y_true.astype(int).to_numpy()

    return y_true


def load_best_protocol_a_seed():
    """Identify best Protocol A ensemble seed from CP4_ensemble_seed_level.csv."""
    if not ENSEMBLE_SEED_FILE.exists():
        raise FileNotFoundError(f"Missing ensemble seed-level file: {ENSEMBLE_SEED_FILE}")

    df = pd.read_csv(ENSEMBLE_SEED_FILE)

    protocol_col = find_column(df, ['protocol', 'protocol_key', 'split', 'protocol_name'])
    seed_col = find_column(df, ['seed', 'random_seed'])
    acc_col = find_column(df, [
        'ensemble_test_accuracy', 'test_accuracy', 'accuracy',
        'ensemble_accuracy', 'test_acc'
    ])

    if protocol_col is None or seed_col is None or acc_col is None:
        raise ValueError(
            f"Cannot parse {ENSEMBLE_SEED_FILE}. "
            f"Columns found: {list(df.columns)}"
        )

    work = df.copy()
    work['_protocol'] = work[protocol_col].apply(normalize_protocol_value)
    work = work[work['_protocol'] == 'A'].copy()

    if work.empty:
        raise ValueError("No Protocol A rows found in ensemble seed-level CSV.")

    work[acc_col] = pd.to_numeric(work[acc_col], errors='coerce')
    work = work.dropna(subset=[acc_col])

    if work.empty:
        raise ValueError("Protocol A rows exist, but no valid ensemble accuracy values were found.")

    best_row = work.sort_values(acc_col, ascending=False).iloc[0]
    best_seed = int(best_row[seed_col])
    best_acc = float(best_row[acc_col])

    return best_seed, best_acc, work


def main():
    print("=" * 80)
    print("SCRIPT 4: PROTOCOL A BEST-SEED ENSEMBLE CONFUSION MATRIX")
    print("=" * 80)

    best_seed, best_acc, protocol_a_df = load_best_protocol_a_seed()

    print(f"Best Protocol A ensemble seed identified: {best_seed}")
    print(f"Best Protocol A ensemble accuracy       : {best_acc:.6f}")
    print("\nProtocol A seed-level candidates:")
    print(protocol_a_df.to_string(index=False))

    softmax_path = ROOT / f'outputs/cp4_ensemble/Ensemble2_A_seed{best_seed}_test_softmax.npy'

    prediction_candidates = [
        ROOT / f'outputs/cp3_6_ensemble/DenseNet201_A_seed{best_seed}_ttamixup_test_predictions.csv',
        ROOT / f'outputs/cp3_5_ceiling_push/EffNetV2S_A_seed{best_seed}_ttamixup_test_predictions.csv',
        ROOT / f'outputs/cp4_multiseed/DenseNet201_A_seed{best_seed}_ttamixup_test_predictions.csv',
        ROOT / f'outputs/cp4_multiseed/EffNetV2S_A_seed{best_seed}_ttamixup_test_predictions.csv',
    ]

    pred_csv_path = next((p for p in prediction_candidates if p.exists()), None)

    if not softmax_path.exists():
        raise FileNotFoundError(f"Missing ensemble softmax file: {softmax_path}")

    if pred_csv_path is None:
        print("[ERROR] No prediction CSV found for best seed.")
        print("Checked:")
        for p in prediction_candidates:
            print(f"  - {p}")
        return

    print(f"\nLoading ensemble softmax from: {softmax_path}")
    print(f"Loading true labels from     : {pred_csv_path}")

    softmax = np.load(softmax_path)

    if softmax.ndim != 2 or softmax.shape[1] != len(CLASS_NAMES):
        raise ValueError(
            f"Unexpected softmax shape: {softmax.shape}. "
            f"Expected (n_samples, {len(CLASS_NAMES)})."
        )

    y_true = extract_true_labels(pred_csv_path)

    if len(y_true) != softmax.shape[0]:
        raise ValueError(
            f"Length mismatch: y_true has {len(y_true)} labels, "
            f"softmax has {softmax.shape[0]} rows."
        )

    y_pred = np.argmax(softmax, axis=1)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES)))
    )

    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_df.index.name = 'true_class'

    row_sums = cm_df.sum(axis=1).replace(0, np.nan)
    cm_norm_df = cm_df.div(row_sums, axis=0).fillna(0.0)
    cm_norm_df.index.name = 'true_class'

    output_raw = OUT_DIR / 'Ensemble2_A_best_seed_confusion_matrix.csv'
    output_norm = OUT_DIR / 'Ensemble2_A_best_seed_confusion_matrix_normalized.csv'

    cm_df.to_csv(output_raw)
    cm_norm_df.to_csv(output_norm)

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Best seed                 : {best_seed}")
    print(f"Best ensemble accuracy    : {best_acc:.6f}")
    print(f"Samples evaluated         : {len(y_true)}")
    print(f"Raw confusion matrix      : {output_raw}")
    print(f"Normalized confusion matrix: {output_norm}")

    print("\nRaw confusion matrix preview:")
    print(cm_df.to_string())

    print("\nRow-normalized confusion matrix preview:")
    print(cm_norm_df.round(4).to_string())


if __name__ == "__main__":
    try:
        main()
    except Exception:
        print("[FATAL ERROR] Script failed unexpectedly.")
        traceback.print_exc()

In [ ]:
# Table 1 - Overall Accuracy
# Output:
#   outputs/cp4_paper_prep/tables/table_1_overall_accuracy.tex
#   outputs/cp4_paper_prep/tables/table_1_overall_accuracy.md

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAPTION = (
    "Test accuracy across protocols and methods. Values are mean $\\pm$ standard "
    "deviation across seeds. Best-seed value reported separately for ensemble."
)

def fmt(mean, sd):
    return f"{mean:.4f} ± {sd:.4f}"

def bold_if_best(value, best_value):
    return f"**{value}**" if value == best_value else value

def latex_bold_if_best(value, is_best):
    return f"\\textbf{{{value}}}" if is_best else value

def main():
    """Generate Table 1 in Markdown and LaTeX formats."""
    rows = [
        {
            "Protocol": "A",
            "Baseline DN201": fmt(0.9585, 0.0091),
            "DN201+TTA+mixup": fmt(0.9697, 0.0053),
            "EffNetV2-S+TTA+mixup": fmt(0.9651, 0.0087),
            "Ensemble": fmt(0.9709, 0.0052),
            "Best Seed Ens": "0.9768",
            "_means": [0.9585, 0.9697, 0.9651, 0.9709],
        },
        {
            "Protocol": "B-G14",
            "Baseline DN201": fmt(0.9589, 0.0039),
            "DN201+TTA+mixup": fmt(0.9636, 0.0055),
            "EffNetV2-S+TTA+mixup": fmt(0.9605, 0.0056),
            "Ensemble": fmt(0.9663, 0.0042),
            "Best Seed Ens": "0.9721",
            "_means": [0.9589, 0.9636, 0.9605, 0.9663],
        },
        {
            "Protocol": "A-clean-strict",
            "Baseline DN201": fmt(0.9612, 0.0077),
            "DN201+TTA+mixup": fmt(0.9728, 0.0034),
            "EffNetV2-S+TTA+mixup": fmt(0.9647, 0.0071),
            "Ensemble": fmt(0.9716, 0.0005),
            "Best Seed Ens": "0.9721",
            "_means": [0.9612, 0.9728, 0.9647, 0.9716],
        },
        {
            "Protocol": "C-G09",
            "Baseline DN201": fmt(0.9463, 0.0108),
            "DN201+TTA+mixup": fmt(0.9596, 0.0016),
            "EffNetV2-S+TTA+mixup": fmt(0.9601, 0.0021),
            "Ensemble": fmt(0.9633, 0.0015),
            "Best Seed Ens": "0.9649",
            "_means": [0.9463, 0.9596, 0.9601, 0.9633],
        },
    ]

    method_cols = ["Baseline DN201", "DN201+TTA+mixup", "EffNetV2-S+TTA+mixup", "Ensemble"]

    md_rows = []
    tex_rows = []
    for row in rows:
        best_idx = max(range(len(row["_means"])), key=lambda i: row["_means"][i])
        md_row = {k: row[k] for k in ["Protocol"] + method_cols + ["Best Seed Ens"]}
        tex_row = md_row.copy()

        md_row[method_cols[best_idx]] = f"**{md_row[method_cols[best_idx]]}**"
        tex_row[method_cols[best_idx]] = f"\\textbf{{{tex_row[method_cols[best_idx]]}}}"

        md_rows.append(md_row)
        tex_rows.append(tex_row)

    md_df = pd.DataFrame(md_rows)
    tex_df = pd.DataFrame(tex_rows)

    md_path = OUT_DIR / "table_1_overall_accuracy.md"
    tex_path = OUT_DIR / "table_1_overall_accuracy.tex"

    md_text = f"**Table 1.** {CAPTION}\n\n" + md_df.to_markdown(index=False)
    md_path.write_text(md_text, encoding="utf-8")

    latex = tex_df.to_latex(index=False, escape=False, caption=CAPTION, label="tab:overall_accuracy")
    tex_path.write_text(latex, encoding="utf-8")

    print(md_text)
    print(f"\nSaved Markdown: {md_path}")
    print(f"Saved LaTeX   : {tex_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Table 2 - Macro F1
# Output:
#   outputs/cp4_paper_prep/tables/table_2_macro_f1.tex
#   outputs/cp4_paper_prep/tables/table_2_macro_f1.md

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAPTION = "Macro F1 scores across protocols and methods (mean $\\pm$ SD)."

def fmt(mean, sd):
    return f"{mean:.4f} ± {sd:.4f}"

def main():
    """Generate Table 2 in Markdown and LaTeX formats."""
    rows = [
        {
            "Protocol": "A",
            "Baseline DN201": fmt(0.9504, 0.0100),
            "DN201+TTA+mixup": fmt(0.9598, 0.0083),
            "EffNetV2-S+TTA+mixup": fmt(0.9602, 0.0104),
            "Ensemble": fmt(0.9640, 0.0074),
            "_means": [0.9504, 0.9598, 0.9602, 0.9640],
        },
        {
            "Protocol": "B-G14",
            "Baseline DN201": fmt(0.9601, 0.0043),
            "DN201+TTA+mixup": fmt(0.9635, 0.0050),
            "EffNetV2-S+TTA+mixup": fmt(0.9615, 0.0051),
            "Ensemble": fmt(0.9678, 0.0041),
            "_means": [0.9601, 0.9635, 0.9615, 0.9678],
        },
        {
            "Protocol": "A-clean-strict",
            "Baseline DN201": fmt(0.9536, 0.0116),
            "DN201+TTA+mixup": fmt(0.9630, 0.0052),
            "EffNetV2-S+TTA+mixup": fmt(0.9586, 0.0084),
            "Ensemble": fmt(0.9654, 0.0019),
            "_means": [0.9536, 0.9630, 0.9586, 0.9654],
        },
        {
            "Protocol": "C-G09",
            "Baseline DN201": fmt(0.9229, 0.0161),
            "DN201+TTA+mixup": fmt(0.9345, 0.0032),
            "EffNetV2-S+TTA+mixup": fmt(0.9336, 0.0005),
            "Ensemble": fmt(0.9398, 0.0027),
            "_means": [0.9229, 0.9345, 0.9336, 0.9398],
        },
    ]

    method_cols = ["Baseline DN201", "DN201+TTA+mixup", "EffNetV2-S+TTA+mixup", "Ensemble"]

    md_rows, tex_rows = [], []
    for row in rows:
        best_idx = max(range(len(row["_means"])), key=lambda i: row["_means"][i])
        md_row = {k: row[k] for k in ["Protocol"] + method_cols}
        tex_row = md_row.copy()

        md_row[method_cols[best_idx]] = f"**{md_row[method_cols[best_idx]]}**"
        tex_row[method_cols[best_idx]] = f"\\textbf{{{tex_row[method_cols[best_idx]]}}}"

        md_rows.append(md_row)
        tex_rows.append(tex_row)

    md_df = pd.DataFrame(md_rows)
    tex_df = pd.DataFrame(tex_rows)

    md_path = OUT_DIR / "table_2_macro_f1.md"
    tex_path = OUT_DIR / "table_2_macro_f1.tex"

    md_text = f"**Table 2.** {CAPTION}\n\n" + md_df.to_markdown(index=False)
    md_path.write_text(md_text, encoding="utf-8")

    latex = tex_df.to_latex(index=False, escape=False, caption=CAPTION, label="tab:macro_f1")
    tex_path.write_text(latex, encoding="utf-8")

    print(md_text)
    print(f"\nSaved Markdown: {md_path}")
    print(f"Saved LaTeX   : {tex_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Script 3: Table 3 - Per-Class F1 by Protocol
# Output:
#   outputs/cp4_paper_prep/tables/table_3_per_class_f1_by_protocol.tex
#   outputs/cp4_paper_prep/tables/table_3_per_class_f1_by_protocol.md

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
SOURCE_CSV = ROOT / 'outputs/cp4_paper_prep/per_class_f1_aggregate.csv'
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'Adenovirus', 'Astrovirus', 'CCHF', 'Cowpox', 'Ebola', 'Influenza', 'Lassa',
    'Marburg', 'Nipah virus', 'Norovirus', 'Orf', 'Papilloma', 'Rift Valley', 'Rotavirus'
]

METHOD = 'DenseNet201_TTA_Mixup'

PROTOCOLS = {
    'A': 'A',
    'B_G14': 'B-G14',
    'A_clean_strict': 'A-clean-strict',
    'C_G09': 'C-G09',
}

CAPTION = (
    "Per-class F1 scores for DenseNet201+TTA+mixup across protocols "
    "(mean across seeds). This single-method view enables direct "
    "cross-protocol comparison; ensemble-based per-class comparison is shown "
    "separately for Protocol A seed 42 in the per-class ensemble analysis."
)

def main():
    """Generate Table 3 from per_class_f1_aggregate.csv using DenseNet201+TTA+mixup only."""
    if not SOURCE_CSV.exists():
        raise FileNotFoundError(f"Missing source CSV: {SOURCE_CSV}")

    df = pd.read_csv(SOURCE_CSV)

    required_cols = {'protocol', 'method', 'class_name', 'mean_f1'}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(
            f"Missing required columns: {missing_cols}. "
            f"Found columns: {list(df.columns)}"
        )

    output = pd.DataFrame({'Class': CLASS_NAMES})

    for protocol_key, protocol_label in PROTOCOLS.items():
        sub = df[
            (df['protocol'] == protocol_key) &
            (df['method'] == METHOD)
        ].copy()

        if sub.empty:
            print(f"[WARNING] No rows found for protocol={protocol_key}, method={METHOD}")
            output[protocol_label] = "NA"
            continue

        sub['mean_f1'] = pd.to_numeric(sub['mean_f1'], errors='coerce')
        f1_map = dict(zip(sub['class_name'], sub['mean_f1']))

        output[protocol_label] = output['Class'].map(f1_map).map(
            lambda x: f"{x:.3f}" if pd.notna(x) else "NA"
        )

    md_path = OUT_DIR / "table_3_per_class_f1_by_protocol.md"
    tex_path = OUT_DIR / "table_3_per_class_f1_by_protocol.tex"

    md_text = f"**Table 3.** {CAPTION}\n\n" + output.to_markdown(index=False)
    md_path.write_text(md_text, encoding="utf-8")

    latex = output.to_latex(
        index=False,
        escape=False,
        caption=CAPTION,
        label="tab:per_class_f1_protocol"
    )
    tex_path.write_text(latex, encoding="utf-8")

    print(md_text)
    print(f"\nSaved Markdown: {md_path}")
    print(f"Saved LaTeX   : {tex_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Table 4 - Statistical Tests Summary
# Output:
#   outputs/cp4_paper_prep/tables/table_4_statistical_tests_summary.tex
#   outputs/cp4_paper_prep/tables/table_4_statistical_tests_summary.md

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAPTION = (
    "Statistical comparison of methods. Wilcoxon signed-rank test, Cohen's d effect size, "
    "and bootstrap 95% CI shown. No comparisons reach Bonferroni-corrected threshold "
    "($\\alpha/12 = 0.00417$); however, multiple comparisons show large effect sizes "
    "(Cohen's d > 0.8) and CIs largely excluding zero."
)

def main():
    """Generate Table 4 from hard-coded verified statistical test results."""
    rows = [
        ["A", "TTA+mixup DN vs Baseline", 5, 0.0112, 0.98, 0.1250, "[0.0018, 0.0196]"],
        ["A", "Ensemble vs Baseline", 5, 0.0124, 1.07, 0.1250, "[0.0031, 0.0215]"],
        ["A", "Ensemble vs TTA+mixup DN", 5, 0.0013, 0.71, 0.2500, "[-0.0001, 0.0026]"],
        ["B-G14", "TTA+mixup DN vs Baseline", 5, 0.0047, 0.51, 0.4375, "[-0.0021, 0.0127]"],
        ["B-G14", "Ensemble vs Baseline", 5, 0.0074, 0.95, 0.1250, "[0.0019, 0.0132]"],
        ["B-G14", "Ensemble vs TTA+mixup DN", 5, 0.0026, 0.88, 0.2500, "[0.0004, 0.0048]"],
        ["A-clean-strict", "TTA+mixup DN vs Baseline", 3, 0.0116, 2.44, 0.2500, "[0.0068, 0.0163]"],
        ["A-clean-strict", "Ensemble vs Baseline", 3, 0.0104, 1.28, 0.2500, "[0.0042, 0.0195]"],
        ["A-clean-strict", "Ensemble vs TTA+mixup DN", 3, -0.0012, -0.32, 0.7500, "[-0.0042, 0.0032]"],
        ["C-G09", "TTA+mixup DN vs Baseline", 3, 0.0133, 1.39, 0.2500, "[0.0077, 0.0243]"],
        ["C-G09", "Ensemble vs Baseline", 3, 0.0170, 1.59, 0.2500, "[0.0103, 0.0294]"],
        ["C-G09", "Ensemble vs TTA+mixup DN", 3, 0.0038, 2.91, 0.2500, "[0.0026, 0.0052]"],
    ]

    df = pd.DataFrame(rows, columns=[
        "Protocol", "Comparison", "n", "Mean Diff", "Cohen's d", "Wilcoxon p", "95% CI"
    ])

    df["Mean Diff"] = df["Mean Diff"].map(lambda x: f"{x:.4f}")
    df["Cohen's d"] = df["Cohen's d"].map(lambda x: f"{x:.2f}")
    df["Wilcoxon p"] = df["Wilcoxon p"].map(lambda x: f"{x:.4f}")

    md_path = OUT_DIR / "table_4_statistical_tests_summary.md"
    tex_path = OUT_DIR / "table_4_statistical_tests_summary.tex"

    md_text = f"**Table 4.** {CAPTION}\n\n" + df.to_markdown(index=False)
    md_path.write_text(md_text, encoding="utf-8")

    latex = df.to_latex(index=False, escape=False, caption=CAPTION, label="tab:statistical_tests")
    tex_path.write_text(latex, encoding="utf-8")

    print(md_text)
    print(f"\nSaved Markdown: {md_path}")
    print(f"Saved LaTeX   : {tex_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Table 5 - Literature Benchmark Comparison
# Output:
#   outputs/cp4_paper_prep/tables/table_5_literature_benchmark_comparison.tex
#   outputs/cp4_paper_prep/tables/table_5_literature_benchmark_comparison.md

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAPTION = (
    "Comparison with prior literature on TEM virus classification. Multi-seed validation "
    "reported for our work; single-value results reported in prior literature."
)

def main():
    """Generate Table 5 in Markdown and LaTeX formats."""
    rows = [
        {
            "Method": "Matuszewski and Sintorn [1]",
            "Protocol": "Original 60/20/20 random crop split",
            "Accuracy": "0.9310",
            "F1": "0.9210",
            "Multi-Seed Validation": "No",
            "Reported Year": "2021",
        },
        {
            "Method": "Sikder et al. [7]",
            "Protocol": "Original split",
            "Accuracy": "0.9744",
            "F1": "0.9744",
            "Multi-Seed Validation": "No",
            "Reported Year": "2024",
        },
        {
            "Method": "Our ensemble, Protocol A",
            "Protocol": "Official split, ensemble",
            "Accuracy": "0.9709 ± 0.0052; best seed 0.9768",
            "F1": "0.9640 ± 0.0074",
            "Multi-Seed Validation": "Yes, 5 seeds",
            "Reported Year": "This study",
        },
    ]

    df = pd.DataFrame(rows)

    md_path = OUT_DIR / "table_5_literature_benchmark_comparison.md"
    tex_path = OUT_DIR / "table_5_literature_benchmark_comparison.tex"

    md_text = f"**Table 5.** {CAPTION}\n\n" + df.to_markdown(index=False)
    md_path.write_text(md_text, encoding="utf-8")

    latex = df.to_latex(index=False, escape=False, caption=CAPTION, label="tab:literature_benchmark")
    tex_path.write_text(latex, encoding="utf-8")

    print(md_text)
    print(f"\nSaved Markdown: {md_path}")
    print(f"Saved LaTeX   : {tex_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Table 6 - Ensemble Strategy Distribution
# Output:
#   outputs/cp4_paper_prep/tables/table_6_ensemble_strategy_distribution.tex
#   outputs/cp4_paper_prep/tables/table_6_ensemble_strategy_distribution.md

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAPTION = (
    "Distribution of ensemble strategies selected per protocol-seed combination "
    "based on validation macro F1."
)

def main():
    """Generate Table 6 in Markdown and LaTeX formats."""
    rows = [
        {
            "Protocol": "A",
            "n_seeds": 5,
            "Strategies Used": "3× geometric_mean, 1× simple_average, 1× max_confidence",
        },
        {
            "Protocol": "B-G14",
            "n_seeds": 5,
            "Strategies Used": "3× simple_average, 2× max_confidence",
        },
        {
            "Protocol": "A-clean-strict",
            "n_seeds": 3,
            "Strategies Used": "2× max_confidence, 1× geometric_mean",
        },
        {
            "Protocol": "C-G09",
            "n_seeds": 3,
            "Strategies Used": "2× geometric_mean, 1× max_confidence",
        },
    ]

    df = pd.DataFrame(rows)

    md_path = OUT_DIR / "table_6_ensemble_strategy_distribution.md"
    tex_path = OUT_DIR / "table_6_ensemble_strategy_distribution.tex"

    md_text = f"**Table 6.** {CAPTION}\n\n" + df.to_markdown(index=False)
    md_path.write_text(md_text, encoding="utf-8")

    latex = df.to_latex(index=False, escape=False, caption=CAPTION, label="tab:ensemble_strategy")
    tex_path.write_text(latex, encoding="utf-8")

    print(md_text)
    print(f"\nSaved Markdown: {md_path}")
    print(f"Saved LaTeX   : {tex_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Figure 1 - Method Comparison Bar Chart
# Output:
#   outputs/cp4_paper_prep/figures/fig_1_method_comparison_bar_chart.png
#   outputs/cp4_paper_prep/figures/fig_1_method_comparison_bar_chart.pdf

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAPTION = (
    "Test accuracy comparison across protocols and methods. Error bars show standard "
    "deviation across seeds (n=5 for A and B-G14, n=3 for A-clean-strict and C-G09). "
    "Dashed lines show prior literature benchmarks."
)

def main():
    """Generate grouped bar chart for test accuracy across protocols and methods."""
    protocols = ["A", "B-G14", "A-clean-strict", "C-G09"]
    methods = ["Baseline DN201", "DN201+TTA+mixup", "EffNetV2-S+TTA+mixup", "Ensemble"]

    means = np.array([
        [0.9585, 0.9697, 0.9651, 0.9709],
        [0.9589, 0.9636, 0.9605, 0.9663],
        [0.9612, 0.9728, 0.9647, 0.9716],
        [0.9463, 0.9596, 0.9601, 0.9633],
    ])

    sds = np.array([
        [0.0091, 0.0053, 0.0087, 0.0052],
        [0.0039, 0.0055, 0.0056, 0.0042],
        [0.0077, 0.0034, 0.0071, 0.0005],
        [0.0108, 0.0016, 0.0021, 0.0015],
    ])

    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "legend.fontsize": 8,
    })

    cmap = plt.get_cmap("viridis")
    colors = [cmap(x) for x in np.linspace(0.15, 0.85, len(methods))]

    x = np.arange(len(protocols))
    width = 0.18

    fig, ax = plt.subplots(figsize=(9, 5), dpi=300)

    for i, method in enumerate(methods):
        offset = (i - 1.5) * width
        ax.bar(
            x + offset,
            means[:, i],
            width,
            yerr=sds[:, i],
            capsize=3,
            label=method,
            color=colors[i],
            edgecolor="black",
            linewidth=0.4,
        )

    ax.axhline(0.9310, linestyle="--", linewidth=1.2, color="gray", label="Matuszewski 2021 (0.931)")
    ax.axhline(0.9744, linestyle=":", linewidth=1.4, color="black", label="Sikder 2024 (0.9744)")

    ax.set_xticks(x)
    ax.set_xticklabels(protocols)
    ax.set_ylabel("Test accuracy")
    ax.set_xlabel("Evaluation protocol")
    ax.set_ylim(0.90, 0.99)
    ax.set_title("")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="lower right", frameon=True)

    fig.tight_layout()

    png_path = OUT_DIR / "fig_1_method_comparison_bar_chart.png"
    pdf_path = OUT_DIR / "fig_1_method_comparison_bar_chart.pdf"

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    print(CAPTION)
    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Figure 2 - Per-Class F1 Heatmap
# Output:
#   outputs/cp4_paper_prep/figures/fig_2_per_class_f1_heatmap.png
#   outputs/cp4_paper_prep/figures/fig_2_per_class_f1_heatmap.pdf

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
SOURCE_CSV = ROOT / 'outputs/cp4_paper_prep/per_class_f1_aggregate.csv'
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'Adenovirus', 'Astrovirus', 'CCHF', 'Cowpox', 'Ebola', 'Influenza', 'Lassa',
    'Marburg', 'Nipah virus', 'Norovirus', 'Orf', 'Papilloma', 'Rift Valley', 'Rotavirus'
]

PROTOCOLS = ["A", "B_G14", "A_clean_strict", "C_G09"]
PROTOCOL_DISPLAY = ["A", "B-G14", "A-clean-strict", "C-G09"]
METHOD = "DenseNet201_TTA_Mixup"

CAPTION = (
    "Per-class macro F1 across evaluation protocols. Values show mean F1 across seeds "
    "for DenseNet201+TTA+mixup. Color scale: viridis (0.6 to 1.0)."
)

def main():
    """Generate per-class F1 heatmap from per_class_f1_aggregate.csv."""
    if not SOURCE_CSV.exists():
        raise FileNotFoundError(f"Missing source CSV: {SOURCE_CSV}")

    df = pd.read_csv(SOURCE_CSV)
    required = {'protocol', 'method', 'class_name', 'mean_f1'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns {missing}. Found columns: {list(df.columns)}")

    matrix = []
    for cls in CLASS_NAMES:
        row = []
        for protocol in PROTOCOLS:
            sub = df[
                (df['protocol'] == protocol) &
                (df['method'] == METHOD) &
                (df['class_name'] == cls)
            ]
            if sub.empty:
                row.append(np.nan)
            else:
                row.append(float(sub['mean_f1'].iloc[0]))
        matrix.append(row)

    arr = np.array(matrix, dtype=float)

    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 9,
        "axes.labelsize": 10,
        "axes.titlesize": 12,
    })

    fig, ax = plt.subplots(figsize=(7, 7), dpi=300)
    im = ax.imshow(arr, cmap="viridis", vmin=0.6, vmax=1.0, aspect="auto")

    ax.set_xticks(np.arange(len(PROTOCOL_DISPLAY)))
    ax.set_xticklabels(PROTOCOL_DISPLAY, rotation=20, ha="right")
    ax.set_yticks(np.arange(len(CLASS_NAMES)))
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Evaluation protocol")
    ax.set_ylabel("Class")
    ax.set_title("")

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            value = arr[i, j]
            label = "NA" if np.isnan(value) else f"{value:.3f}"
            ax.text(j, i, label, ha="center", va="center", fontsize=7, color="white" if value < 0.82 else "black")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Mean F1")

    fig.tight_layout()

    png_path = OUT_DIR / "fig_2_per_class_f1_heatmap.png"
    pdf_path = OUT_DIR / "fig_2_per_class_f1_heatmap.pdf"

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    print(CAPTION)
    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Figure 3 - Learning Curves
# Output:
#   outputs/cp4_paper_prep/figures/fig_3_learning_curves.png
#   outputs/cp4_paper_prep/figures/fig_3_learning_curves.pdf

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
SOURCE_CSV = ROOT / 'outputs/cp3_6_ensemble/DenseNet201_A_seed42_ttamixup_training_history.csv'
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAPTION_TEMPLATE = (
    "Training and validation macro F1 over epochs for DenseNet201+TTA+mixup on "
    "Protocol A seed 42. Best validation macro F1 achieved at epoch {epoch}, "
    "used for final test evaluation."
)

def find_column(df, candidates):
    """Find a likely column name using case-insensitive exact matching."""
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def main():
    """Generate training and validation macro F1 learning curves."""
    if not SOURCE_CSV.exists():
        raise FileNotFoundError(f"Missing training history CSV: {SOURCE_CSV}")

    df = pd.read_csv(SOURCE_CSV)

    epoch_col = find_column(df, ["epoch", "Epoch"])
    train_col = find_column(df, ["train_macro_f1", "train_f1", "macro_f1_train", "train_macroF1"])
    val_col = find_column(df, ["val_macro_f1", "validation_macro_f1", "val_f1", "macro_f1_val", "valid_macro_f1"])

    if epoch_col is None:
        df = df.copy()
        df["epoch"] = range(1, len(df) + 1)
        epoch_col = "epoch"

    if train_col is None or val_col is None:
        raise ValueError(
            f"Could not identify train/validation macro F1 columns. "
            f"Columns found: {list(df.columns)}"
        )

    df[train_col] = pd.to_numeric(df[train_col], errors="coerce")
    df[val_col] = pd.to_numeric(df[val_col], errors="coerce")
    df[epoch_col] = pd.to_numeric(df[epoch_col], errors="coerce")

    best_idx = df[val_col].idxmax()
    best_epoch = int(df.loc[best_idx, epoch_col])
    best_val = float(df.loc[best_idx, val_col])

    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
    })

    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)

    ax.plot(df[epoch_col], df[train_col], marker="o", linewidth=1.5, markersize=3, label="Training macro F1")
    ax.plot(df[epoch_col], df[val_col], marker="s", linewidth=1.5, markersize=3, label="Validation macro F1")
    ax.axvline(best_epoch, linestyle="--", linewidth=1.2, color="gray", label=f"Best epoch: {best_epoch}")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Macro F1")
    ax.set_title("")
    ax.grid(alpha=0.25)
    ax.legend(frameon=True)

    fig.tight_layout()

    png_path = OUT_DIR / "fig_3_learning_curves.png"
    pdf_path = OUT_DIR / "fig_3_learning_curves.pdf"

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    caption = CAPTION_TEMPLATE.format(epoch=best_epoch)
    print(caption)
    print(f"Best validation macro F1: {best_val:.4f}")
    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")

if __name__ == "__main__":
    main()

In [ ]:
# Figure 4 - Confusion Matrix
# Output:
#   outputs/cp4_paper_prep/figures/fig_4_protocol_a_seed42_ensemble_confusion_matrix.png
#   outputs/cp4_paper_prep/figures/fig_4_protocol_a_seed42_ensemble_confusion_matrix.pdf

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/Riset_TEM_Virus/')
SOURCE_CSV = ROOT / 'outputs/cp4_paper_prep/Ensemble2_A_best_seed_confusion_matrix_normalized.csv'
OUT_DIR = ROOT / 'outputs/cp4_paper_prep/figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAPTION = (
    "Confusion matrix for two-model ensemble on Protocol A seed 42 "
    "(best-seed performance, accuracy 0.9768). Values are row-normalized "
    "(per-class recall). Diagonal mean: 0.971."
)

def main():
    """Generate row-normalized confusion matrix heatmap."""
    if not SOURCE_CSV.exists():
        raise FileNotFoundError(f"Missing normalized confusion matrix CSV: {SOURCE_CSV}")

    df = pd.read_csv(SOURCE_CSV, index_col=0)
    arr = df.to_numpy(dtype=float)
    labels = list(df.index)

    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 8,
        "axes.labelsize": 10,
        "axes.titlesize": 12,
    })

    fig, ax = plt.subplots(figsize=(9, 8), dpi=300)
    im = ax.imshow(arr, cmap="viridis", vmin=0.0, vmax=1.0, aspect="equal")

    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title("")

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            value = arr[i, j]
            if value >= 0.01:
                ax.text(
                    j, i, f"{value*100:.0f}%",
                    ha="center", va="center",
                    fontsize=6,
                    color="white" if value < 0.60 else "black"
                )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Row-normalized recall")

    fig.tight_layout()

    png_path = OUT_DIR / "fig_4_protocol_a_seed42_ensemble_confusion_matrix.png"
    pdf_path = OUT_DIR / "fig_4_protocol_a_seed42_ensemble_confusion_matrix.pdf"

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    print(CAPTION)
    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")

if __name__ == "__main__":
    main()